# 0 Header

**Author:** Siyu Chen  
**Date:** 2025-11-04  
**Project:** Customer Churn Prediction — End-to-End Data Engineering & Machine Learning Notebook  
**Version:** v1.0  

---

### 🎯 Goal
Design and implement a **modular, reproducible data pipeline**  
covering data ingestion (ETL/ELT), validation, transformation, feature engineering,  
modeling, evaluation, and explainability — following the **12-Step Framework**.

### ⚙️ Problem
Raw business data often contains missing, inconsistent, and semantically incorrect records,  
leading to unreliable models and poor business insights.

### 💡 Solution
Develop a **validated, version-controlled pipeline** that:  
- performs schema and domain validation,  
- applies semantic and leakage checks,  
- builds modular ETL and feature-engineering stages,  
- evaluates models with standard metrics and slicing diagnostics,  
- and logs artifacts for full reproducibility.

### 📈 Impact
Ensures **data integrity, model reliability, and transparent results**,  
making the project ready for deployment and BI integration in later versions.

### 🧮 Dataset
Telco Customer Churn Dataset — 7,043 rows × 21 columns (Kaggle public dataset).

### 🧰 Tech Stack
Python · Pandas · NumPy · Scikit-learn · XGBoost · SHAP · Matplotlib  

### ✅ Validation
- Schema and data type validation  
- Domain and semantic consistency checks  
- Row-level integrity assertions  
- Config and artifact snapshots per run


# 1 Setup & Environment
🎯 **Goal:** Set up libraries, configurations, and reproducibility parameters for this project. Ensure consistent results, clean outputs, and a structured workspace for the following analysis.

In [ ]:
# --- Bootstrap (ONLY for notebook / local execution) ---
# NOTE:
# This block exists ONLY to make `src.*` imports work in notebooks.
# It MUST be removed once we migrate to a proper runner / package setup.
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError(f"Could not find project root from cwd={Path.cwd()}")

root_str = str(PROJECT_ROOT)
if root_str not in sys.path:
    sys.path.insert(0, root_str)

# --- Setup ---
import logging
import os
import random

from datetime import datetime, timezone
from typing import Any, Dict, Optional, List

import pandas as pd
import numpy as np
import yaml

from src.utils.path_utils import resolve_dir
from src.utils.artifact_utils import resolve_artifact_path, write_json
from src.utils.time_utils import utc_now_iso

TAG1 = "SETUP"


def _utc_run_id() -> str:
    """Create a stable run_id in UTC (safe for filenames)."""
    return datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")[:19]


def _init_random_seeds(seed: int) -> None:
    """Initialize RNG seeds for reproducibility (Python, NumPy, and hash)."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


def _init_logger(name: str = "churn_pipeline") -> logging.Logger:
    """Initialize and return a project logger."""
    lg = logging.getLogger(name)
    lg.setLevel(logging.INFO)

    # Avoid duplicate handlers in notebooks
    if not lg.handlers:
        handler = logging.StreamHandler()
        formatter = logging.Formatter(
            fmt="%(asctime)s | %(levelname)s | %(message)s",
            datefmt="%H:%M:%S",
        )
        handler.setFormatter(formatter)
        lg.addHandler(handler)

    lg.propagate = False
    return lg


def load_yaml_config(config_path: Path) -> Dict[str, Any]:
    """Load YAML config from disk (fail fast if missing/invalid)."""
    if not isinstance(config_path, Path):
        raise TypeError(f"[{TAG1}] config_path must be Path, got: {type(config_path)}")
    if not config_path.exists():
        raise FileNotFoundError(f"[{TAG1}] YAML config not found: {config_path}")
    with config_path.open("r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    if cfg is None:
        return {}
    if not isinstance(cfg, dict):
        raise TypeError(f"[{TAG1}] YAML content must be a dict, got: {type(cfg)}")
    return cfg


def _merge_list_of_dicts_by_id(base_list: List[Any], override_list: List[Any]) -> List[Any]:
    """
    Merge list[dict] by 'id'. Override items update base items with same id.
    Items without 'id' fall back to full replacement behavior for safety.
    """
    if not all(isinstance(x, dict) for x in base_list + override_list):
        return list(override_list)

    if not all(("id" in x) for x in base_list + override_list):
        return list(override_list)

    base_map = {str(x["id"]): dict(x) for x in base_list}
    for item in override_list:
        sid = str(item["id"])
        if sid in base_map and isinstance(base_map[sid], dict) and isinstance(item, dict):
            base_map[sid] = _deep_merge_dict(base_map[sid], item)
        else:
            base_map[sid] = dict(item)

    # Keep base order, append new ids at end
    base_order = [str(x["id"]) for x in base_list]
    extra_ids = [k for k in base_map.keys() if k not in base_order]
    ordered_ids = base_order + extra_ids
    return [base_map[i] for i in ordered_ids]


def _deep_merge_dict(base: Dict[str, Any], override: Dict[str, Any]) -> Dict[str, Any]:
    """
    Deep merge override into base (returns a new dict).

    Policy:
      - dict + dict => recursive merge
      - list + list:
          * if it's list[dict] with 'id' => merge by id (for orchestrator steps)
          * else => override wins (replace)
      - otherwise => override wins
    """
    if not isinstance(base, dict) or not isinstance(override, dict):
        raise TypeError(f"[{TAG1}] _deep_merge_dict expects dicts")

    out: Dict[str, Any] = dict(base)
    for k, v in override.items():
        if k in out and isinstance(out[k], dict) and isinstance(v, dict):
            out[k] = _deep_merge_dict(out[k], v)
        elif k in out and isinstance(out[k], list) and isinstance(v, list):
            # Special handling for orchestrator steps overrides
            if k == "steps":
                out[k] = _merge_list_of_dicts_by_id(out[k], v)
            else:
                out[k] = list(v)
        else:
            out[k] = v
    return out


def ensure_project_dirs(ctx: Dict[str, Any], *, lg: logging.Logger) -> None:
    """
    Ensure core project directories exist.

    Policy:
      - Required: data.raw, data.interim, data.processed, outputs.models, outputs.artifacts
      - Optional: outputs.figures
      - Fail fast if any required path key is missing in YAML (strict 50k mode)
    """
    required_keys = [
        "data.raw",
        "data.interim",
        "data.processed",
        "outputs.models",
        "outputs.artifacts",
    ]

    optional_keys = [
        "outputs.figures",
    ]

    for k in required_keys:
        p = resolve_dir(ctx, k, tag=f"{TAG1}.PATHS", required=True)
        p.mkdir(parents=True, exist_ok=True)
        lg.info(f"[{TAG1}] 📁 Ensured dir: {k} -> {p}")

    for k in optional_keys:
        p = resolve_dir(ctx, k, tag=f"{TAG1}.PATHS", required=False, fallback=None)
        if p is not None:
            p.mkdir(parents=True, exist_ok=True)
            lg.info(f"[{TAG1}] 📁 Ensured optional dir: {k} -> {p}")


def build_ctx(
    *,
    config_paths: List[Path],
    run_id: Optional[str] = None,
    logger: Optional[logging.Logger] = None,
) -> Dict[str, Any]:
    """
    Build the pipeline context dict (ctx) from multiple YAMLs.

    Merge order matters (industrial standard):
      base -> layer configs -> env override
    Later files override earlier ones.

    Parameters
    ----------
    config_paths : List[Path]
        YAML files in merge order. Example:
          [
            Path("config/base.yaml"),
            Path("config/data_quality.yaml"),
            Path("config/data_integrity.yaml"),
            Path("config/data_split.yaml"),
            Path("config/modeling.yaml"),
            Path("config/overrides/dev.yaml"),
          ]
    """
    if not isinstance(config_paths, list) or len(config_paths) == 0:
        raise TypeError(f"[{TAG1}] config_paths must be a non-empty List[Path]")

    for p in config_paths:
        if not isinstance(p, Path):
            raise TypeError(f"[{TAG1}] config_paths items must be Path, got: {type(p)}")
        if not p.exists():
            raise FileNotFoundError(f"[{TAG1}] YAML not found: {p}")

    lg = logger or _init_logger()
    lg.info(f"[{TAG1}] 🔧 Bootstrapping pipeline context & environment...")

    # --- Load + merge YAMLs ---
    merged_cfg: Dict[str, Any] = {}
    loaded_files: List[str] = []

    for p in config_paths:
        cfg_i = load_yaml_config(p)
        merged_cfg = _deep_merge_dict(merged_cfg, cfg_i)
        loaded_files.append(str(p))

    cfg = merged_cfg

    # --- Seed (SSOT from merged YAML, fallback allowed but explicit) ---
    project_cfg = cfg.get("project", {})
    if not isinstance(project_cfg, dict):
        raise TypeError(f"[{TAG1}] 'project' must be a dict in YAML")

    seed = project_cfg.get("seed", 42)
    if not isinstance(seed, int):
        raise TypeError(f"[{TAG1}] project.seed must be an int")

    _init_random_seeds(seed)

    # --- Run ID ---
    rid = run_id or _utc_run_id()

    # --- ctx skeleton ---
    paths_cfg = cfg.get("paths")
    if not isinstance(paths_cfg, dict):
        raise TypeError(f"[{TAG1}] 'paths' must be a dict in YAML")

    ctx: Dict[str, Any] = {
        "cfg": cfg,            # merged YAML SSOT
        "paths": paths_cfg,    # for path_utils.resolve_dir()
        "run_id": rid,
        "logger": lg,
        "meta": {
            "seed": seed,
            "created_at_utc": utc_now_iso(),
            "python": os.sys.version.split()[0],
            "cwd": str(Path.cwd()),
            "pandas": pd.__version__,
            "numpy": np.__version__,
            "loaded_config_files": loaded_files,
        },
        "config_paths": loaded_files,  # for audit/reproducibility
    }

    # --- Normalize root ---
    root_raw = ctx["paths"].get("root")
    if root_raw is None or not str(root_raw).strip():
        raise KeyError(f"[{TAG1}] paths.root is required in YAML")
    root = Path(str(root_raw)).expanduser()
    ctx["paths"]["root"] = str(root.resolve())

    lg.info(f"[{TAG1}] 📍 Project root resolved to: {ctx['paths']['root']}")
    lg.info(f"[{TAG1}] 🔧 Loaded YAML files (merge order): {loaded_files}")
    lg.info(f"[{TAG1}] 🏷️ run_id: {rid}")
    lg.info(f"[{TAG1}] 🎲 seed: {seed}")

    # --- Ensure directories (strict) ---
    ensure_project_dirs(ctx, lg=lg)

    # --- Config snapshot artifact (merged SSOT) ---
    snapshot_payload: Dict[str, Any] = {
        "run_id": rid,
        "timestamp_utc": utc_now_iso(),
        "config_files": loaded_files,
        "cfg": cfg,  # store FULL merged cfg (SSOT)
        "env": {
            "python": os.sys.version.split()[0],
            "pandas": pd.__version__,
            "numpy": np.__version__,
        },
    }

    snapshot_path = resolve_artifact_path(
        ctx=ctx,
        stage="setup",
        name="config_snapshot",
        run_id=rid,
        tag=TAG1,
    )
    write_json(snapshot_path, snapshot_payload, tag=TAG1, indent=2)
    lg.info(f"[{TAG1}] 📝 Config snapshot saved: {snapshot_path}")

    return ctx


# --- Example usage (notebook) ---
ctx = build_ctx(
    config_paths=[
        PROJECT_ROOT / "config" / "base.yaml",
        PROJECT_ROOT / "config" / "data_quality.yaml",
        PROJECT_ROOT / "config" / "data_integrity.yaml",
        PROJECT_ROOT / "config" / "data_split.yaml",
        PROJECT_ROOT / "config" / "governance.yaml",
        PROJECT_ROOT / "config" / "overrides" / "dev.yaml",
    ]
)
ctx["meta"]

# 2 Data Quality
🎯 **Goal:** Establish a **production-grade, trustworthy, and modeling-ready dataset** by reliably ingesting data and enforcing data contracts, validating semantic and statistical quality, assessing experiment readiness, performing domain-specific diagnostics, and applying standardized cleaning and normalization procedures.

In [2]:
# --- 2.1 Data Ingestion ---
#
# What:
#   Load raw data from configured source, log metadata, and persist an ingestion artifact.
#
# How:
#   - Resolve ingestion mode from YAML SSOT (pipeline.data_quality.ingestion).
#   - CSV override: `source` (Path) is allowed.
#   - DB mode is opt-in via `pipeline.data_quality.ingestion.db.enabled`.
#   - DB connection secrets MUST come from environment variables (names configured in YAML).
#   - YAML only stores non-secret metadata (driver/schema/table/query/env var names).
#   - `db_url` parameter is NOT accepted in DB mode (policy enforcement).
#   - Optional DB override: `table_name` (DEV-only convenience). Prefer YAML `db.query` when provided.
#
# Why:
#   Provide a single, traceable entry point for the pipeline and support reproducibility.

# --- 0) Imports + TAG ---
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.utils.artifact_utils import resolve_artifact_path, write_json
from src.utils.path_utils import resolve_dir
from src.utils.cfg_utils import get_cfg_dict, get_cfg_str, get_cfg_bool

TAG21 = "INGESTION"


# --- 1) Public API ---
def load_data(
    *,
    ctx: Dict[str, Any],
    run_id: Optional[str] = None,
    source: Optional[Path] = None,
    db_url: Optional[str] = None,
    table_name: Optional[str] = None,
) -> pd.DataFrame:
    """
    Load data from CSV or database, log metadata, and save an ingestion report.

    YAML SSOT
    ---------
    pipeline.data_quality.ingestion:
        source: "csv"                # allowed: "csv" | "db"
        csv:
            file_glob: "*.csv"           # used when source="csv"
            encoding: "utf-8"            # used when source="csv"
        db:
            enabled: false             # DB ingestion is opt-in and guarded by this flag

    Behavior / Policy
    -----------------
    - CSV mode (source="csv"):
        * Reads exactly one file matched by `file_glob` under `data.raw`
        * Optional override: `source` (Path) to load a specific file

    - DB mode (source="db"):
        * Requires `db.enabled: true`
        * Connection secrets come from environment variables (names configured in YAML under `db.env`)
        * YAML stores only non-secret metadata (driver/schema/table/query + env var names)
        * `db_url` is NOT accepted in DB mode (policy enforcement)
        * Optional DEV-only override: `table_name` (only when `db.query` is empty)
        * (v1) DB ingestion is currently stubbed (NotImplementedError) unless implemented

    Returns
    -------
    pd.DataFrame
        Loaded dataframe ready for downstream validation and processing.
    report
        module returns both df + report to keep orchestrator contract stable.
    """
    # --- 2) Runtime validation ---
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG21, run_id=run_id)

    # --- 3) Input validation ---
    if source is not None and not isinstance(source, Path):
        raise TypeError(f"[{TAG21}] source must be Path or None")

    # --- 4) Resolve Config (YAML SSOT) ---

    # --- Module（step cfg） ---
    get_cfg_dict(ctx, "pipeline.data_quality.ingestion", tag=TAG21, required=True)
    
    source_mode = get_cfg_str(
        ctx,
        "pipeline.data_quality.ingestion.source",
        tag=TAG21,
        default="csv",
        required=False,
        lower=True,
        allowed={"csv", "db"},
    )

    file_glob = get_cfg_str(
        ctx,
        "pipeline.data_quality.ingestion.csv.file_glob",
        tag=TAG21,
        default="*.csv",
        required=False,
    )

    encoding = get_cfg_str(
        ctx,
        "pipeline.data_quality.ingestion.csv.encoding",
        tag=TAG21,
        default="utf-8",
        required=False,
    )

    db_enabled = get_cfg_bool(
        ctx,
        "pipeline.data_quality.ingestion.db.enabled",
        tag=TAG21,
        default=False,
        required=False,
        strict_type=True,
    )

    use_db = (source_mode == "db")

    db_driver = None
    db_schema = None
    db_table_yaml = None
    db_query = None
    env_host = env_port = env_db = env_user = env_pwd = env_ssl = env_to = None

    if use_db:
        db_driver = get_cfg_str(
            ctx, "pipeline.data_quality.ingestion.db.driver", tag=TAG21,
            default="postgresql", required=False, lower=True
        )
        db_schema = get_cfg_str(
            ctx, "pipeline.data_quality.ingestion.db.schema", tag=TAG21,
            default="public", required=False
        )
        db_table_yaml = get_cfg_str(
            ctx, "pipeline.data_quality.ingestion.db.table", tag=TAG21,
            default="", required=False
        )
        db_query = get_cfg_str(
            ctx, "pipeline.data_quality.ingestion.db.query", tag=TAG21,
            default="", required=False
        )

        env_host = get_cfg_str(ctx, "pipeline.data_quality.ingestion.db.env.host", tag=TAG21, required=False, default="")
        env_port = get_cfg_str(ctx, "pipeline.data_quality.ingestion.db.env.port", tag=TAG21, required=False, default="")
        env_db   = get_cfg_str(ctx, "pipeline.data_quality.ingestion.db.env.database", tag=TAG21, required=False, default="")
        env_user = get_cfg_str(ctx, "pipeline.data_quality.ingestion.db.env.user", tag=TAG21, required=False, default="")
        env_pwd  = get_cfg_str(ctx, "pipeline.data_quality.ingestion.db.env.password", tag=TAG21, required=False, default="")
        env_ssl  = get_cfg_str(ctx, "pipeline.data_quality.ingestion.db.env.sslmode", tag=TAG21, required=False, default="")
        env_to   = get_cfg_str(ctx, "pipeline.data_quality.ingestion.db.env.connect_timeout", tag=TAG21, required=False, default="")
    
    # --- Policy: DB is opt-in via YAML gate; secrets come from env (names in YAML) ---
    if use_db and (not db_enabled):
        raise ValueError(
            f"[{TAG21}] YAML source='db' but db ingestion is disabled "
            f"(pipeline.data_quality.ingestion.db.enabled=false)"
        )

    if (not use_db) and (db_url is not None or table_name is not None):
        raise ValueError(f"[{TAG21}] db_url/table_name provided but YAML source='{source_mode}', expected 'db'")

    if use_db:
        # v2 policy: db_url should NOT be passed (env-only)
        if db_url is not None and str(db_url).strip():
            raise ValueError(f"[{TAG21}] db_url is not allowed (policy: env-only DB secrets)")
        if source is not None:
            raise ValueError(f"[{TAG21}] source path override is not allowed when YAML source='db'")

    # --- 5) Core workflow ---
    lg.info(f"[{TAG21}] ▶ Ingestion start | mode={source_mode} | run_id={rid}")

    # Build ModuleReport skeleton (infra, not business logic)
    report = build_module_report(
        ctx=ctx,
        stage="data_quality",
        step="2.1",
        name="ingestion",
        tag=TAG21,
        cfg_key="pipeline.data_quality.ingestion",
        enabled=True,
        df=None,
        run_id_override=rid,
    )

    report["inputs"] = {
        "source_override": str(source) if isinstance(source, Path) else None,
        "db_url_rejected": bool(db_url is not None and str(db_url).strip()),
        "table_name_override": table_name,
    }

    report["used_config"] = {
        "source_mode": source_mode,
        "file_glob": file_glob if not use_db else None,
        "encoding": encoding if not use_db else None,
        "db_enabled": bool(db_enabled),
        "db_driver": db_driver if use_db else None,
        "db_schema": db_schema if use_db else None,
        "db_table_yaml": db_table_yaml if use_db else None,
        "db_query_provided": bool(db_query) if use_db else None,
    }

    report["refs"]["self"] = make_artifact_ref(
        ctx=ctx,
        stage="data_quality",
        name="ingestion",
        run_id=str(rid),
    )

    start_time = time.time()

    df: Optional[pd.DataFrame] = None
    source_type: Optional[str] = None
    source_path: Optional[str] = None
    table_used: Optional[str] = None

    # --- Actual ingestion (business logic) ---
    if not use_db:
        raw_dir = resolve_dir(ctx, "data.raw", tag=TAG21, required=True)

        if source is None:
            matches = sorted(raw_dir.glob(file_glob))
            if len(matches) == 0:
                raise FileNotFoundError(f"[{TAG21}] No files matched '{file_glob}' under {raw_dir}")
            if len(matches) > 1:
                raise ValueError(f"[{TAG21}] Multiple files matched '{file_glob}' under {raw_dir}")
            source = matches[0]

        lg.info(f"[{TAG21}] 📄 CSV | dir={raw_dir} | glob={file_glob} | encoding={encoding} | selected={source}")

        if not source.exists():
            raise FileNotFoundError(f"[{TAG21}] File not found: {source}")

        df = pd.read_csv(source, encoding=encoding)
        source_type = "csv"
        source_path = str(source)

    else:
        raise NotImplementedError(
            f"[{TAG21}] DB ingestion path is reserved (env-only policy). "
            f"Enable db.enabled and implement env-based connection when needed."
    )

    load_time = round(time.time() - start_time, 3)

    # Safety: df must be resolved here
    if df is None or not isinstance(df, pd.DataFrame):
        raise RuntimeError(f"[{TAG21}] ingestion failed to produce a DataFrame")

    report["checks"]["ingestion"] = {
        "source_type": source_type,
        "source_path": source_path,
        "table_name": table_used,
        "n_rows": int(df.shape[0]),
        "n_cols": int(df.shape[1]),
        "columns": [str(c) for c in df.columns.tolist()],
        "load_time_sec": load_time,
    }

    if df.shape[0] > 0 and df.shape[1] > 0:
        pv = df.iloc[:3, :5]
        report["checks"]["preview"] = {
            "columns": pv.columns.tolist(),
            "rows": pv.astype(object).where(pd.notna(pv), None).values.tolist(),
        }

    # --- 6) Status + Gate-friendly summary ---
    issues: List[str] = []
    if df.shape[0] == 0:       
        issues.append("empty_dataframe")

    overall = "fail" if issues else "pass"

    report = finalize_module_report(
        ctx=ctx,
        report=report,
        overall_status=overall,
        issues=issues,
        metrics={"load_time_sec": load_time},
        notes=[f"source_type={source_type}"],
        df=df,
    )

    if overall == "pass":
        lg.info(f"[{TAG21}] ✅ Data loaded: {df.shape[0]} rows × {df.shape[1]} cols")
    else:
        lg.error(f"[{TAG21}] ❌ Ingestion failed | mode={source_mode} | issues={issues}")

    # --- 7) Persist Artifact---
    out_path = resolve_artifact_path(
        ctx=ctx,
        stage="data_quality",
        name="ingestion",
        run_id=str(rid),
        tag=TAG21,
    )

    write_json(out_path, report, tag=TAG21, indent=2)

    lg.info(f"[{TAG21}] 🧾 Ingestion artifact saved: {out_path}")

    return {"df": df, "report": report}

In [3]:
# --- 2.2 Data Contract Validation ---
#
# What:
#   Validate the raw dataframe against an expected data contract
#   (required columns, basic dtypes, primary key, label presence).
#
# How:
#   - Read schema + policy from YAML SSOT (dataset.schema + pipeline.data_quality.contract)
#   - Run column / dtype / primary-key / label checks (dtype enforcement is policy-driven)
#   - Emit a ModuleReport v1 artifact for orchestrator-friendly consumption
#
# Why:
#   Fail fast when upstream schema changes and keep downstream steps stable.

# --- 0) Import + TAG ---
from typing import Any, Dict, List, Optional

import pandas as pd
from pandas.api.types import (
    is_numeric_dtype,
    is_string_dtype,
    is_categorical_dtype,
    is_bool_dtype,
)

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint
from src.utils.artifact_utils import resolve_artifact_path, write_json
from src.utils.cfg_utils import get_cfg_dict, get_cfg_str, get_cfg_str_list, get_cfg_bool
from src.utils.dataset_utils import get_label_cfg, get_schema_cols, get_primary_keys
from src.utils.serialization_utils import safe_float

TAG22 = "CONTRACT"


# --- Internal helper ---
def _is_acceptable_categorical_dtype(s: pd.Series) -> bool:
    """
    A more production-friendly categorical dtype check.

    Accept: string/object/category/bool (common in raw tables).
    """
    return (
        is_string_dtype(s)
        or s.dtype == "object"
        or is_categorical_dtype(s)
        or is_bool_dtype(s)
    )


def _is_pk_missing_value(s: pd.Series) -> pd.Series:
    """Return True for missing PK values (NaN + empty/blank strings)."""
    miss = s.isna()
    if is_string_dtype(s) or str(s.dtype) == "object":
        try:
            miss = miss | (s.astype(str).str.strip() == "")
        except Exception:
            pass
    return miss


# --- 1) Public API ---
def validate_data_contract(
    *,
    df: pd.DataFrame,
    ctx: Dict[str, Any],
    run_id: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Validate dataframe against a data contract.

    YAML SSOT
    ---------
    dataset:
      keys.primary: ["customerID"]  # supports composite keys too
      label.col: "Churn"
      schema:
        required_columns: [...]
        dtypes.numeric: [...]
        dtypes.categorical: [...]
        allow_extra_columns: true/false

    pipeline.data_quality.contract:
      use_dataset_schema: true/false
      enforce_required_columns: true/false
      enforce_dtypes: true/false    # if false, dtype issues are recorded as warnings only

    Returns
    -------
    dict
        ModuleReport v1 (Contract) (also persisted as artifact).
    """
    # ---- 2) Runtime validation ----
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG22, run_id=run_id)

    # ---- 3) Input validation ----
    if df is None or not isinstance(df, pd.DataFrame):
        raise TypeError(f"[{TAG22}] df must be a pandas DataFrame")

    # ---- 4) Resolve config (YAML SSOT) ----

    # --- Global（SSOT） ---
    primary_keys = get_primary_keys(ctx, TAG22)
    label_col, _, _ = get_label_cfg(ctx, TAG22)

    # --- Module（step cfg） ---
    get_cfg_dict(ctx, "pipeline.data_quality.contract", tag=TAG22, required=True)

    dataset_name = get_cfg_str(
        ctx,
        "dataset.name",
        tag=TAG22,
        default="",
        required=False,
    )

    required_columns = get_cfg_str_list(
        ctx,
        "dataset.schema.required_columns",
        tag=TAG22,
        required=True,
        allow_empty=False,
        min_len=1,
        dedupe=True,
        strict_items=True,
    )

    numeric_columns, categorical_columns = get_schema_cols(ctx, TAG22)

    allow_extra_columns = get_cfg_bool(
        ctx,
        "dataset.schema.allow_extra_columns",
        tag=TAG22,
        default=True,
        required=False,
        strict_type=True,
    )

    use_dataset_schema = get_cfg_bool(
        ctx,
        "pipeline.data_quality.contract.use_dataset_schema",
        tag=TAG22,
        default=True,
        required=True,
        strict_type=True,
    )

    enforce_required_columns = get_cfg_bool(
        ctx,
        "pipeline.data_quality.contract.enforce_required_columns",
        tag=TAG22,
        default=True,
        required=True,
        strict_type=True,
    )

    enforce_dtypes = get_cfg_bool(
        ctx,
        "pipeline.data_quality.contract.enforce_dtypes",
        tag=TAG22,
        default=False,
        required=True,
        strict_type=True,
    )

    if not primary_keys:
        raise ValueError(f"[{TAG22}] dataset.keys.primary must be a non-empty list")

    if label_col not in required_columns:
        raise ValueError(f"[{TAG22}] label_col={label_col!r} must be included in dataset.schema.required_columns")

    missing_pk = sorted([k for k in primary_keys if k not in required_columns])
    if missing_pk:
        raise ValueError(f"[{TAG22}] primary_keys not in required_columns: {missing_pk}")

    num_set = set(numeric_columns)
    cat_set = set(categorical_columns)

    overlap = sorted(list(num_set.intersection(cat_set)))
    if overlap:
        raise ValueError(f"[{TAG22}] numeric_columns and categorical_columns must not overlap: {overlap}")

    unknown_num = sorted([c for c in numeric_columns if c not in required_columns])
    if unknown_num:
        raise ValueError(f"[{TAG22}] numeric_columns must be subset of required_columns. invalid={unknown_num}")

    unknown_cat = sorted([c for c in categorical_columns if c not in required_columns])
    if unknown_cat:
        raise ValueError(f"[{TAG22}] categorical_columns must be subset of required_columns. invalid={unknown_cat}")

    if not use_dataset_schema:
        raise ValueError(f"[{TAG22}] use_dataset_schema=false is not supported yet (SSOT is dataset.schema).")

    # --- 5) Core workflow ---
    lg.info(f"[{TAG22}] 🔍 Starting data contract validation...")

    # Build ModuleReport skeleton (infra)
    report = build_module_report(
        ctx=ctx,
        stage="data_quality",
        step="2.2",
        name="contract",
        tag=TAG22,
        cfg_key="pipeline.data_quality.contract",
        enabled=True,
        df=df,
        run_id_override=rid,
    )

    # Safe-to-log inputs/config
    report["inputs"] = {
        "df_provided": True,
    }

    report["thresholds_used"] = {
        "enforce_required_columns": bool(enforce_required_columns),
        "enforce_dtypes": bool(enforce_dtypes),
        "allow_extra_columns": bool(allow_extra_columns),
    }

    report["used_config"] = {
        "dataset_name": dataset_name,
        "primary_key_cols": list(primary_keys),
        "label_col": label_col,
        "use_dataset_schema": bool(use_dataset_schema),
        # Keep small + stable; you can include full lists since telco is small (21 cols).
        "required_columns": list(required_columns),
        "numeric_columns": list(numeric_columns),
        "categorical_columns": list(categorical_columns),
    }

    report["refs"]["self"] = make_artifact_ref(
        ctx=ctx,
        stage="data_quality",
        name="contract",
        run_id=str(rid),
    )

    # --- Required columns check ---
    existing_cols = set([str(c) for c in df.columns])
    required_cols = set([str(c) for c in required_columns])

    missing_cols = sorted(list(required_cols - existing_cols))
    extra_cols = sorted(list(existing_cols - required_cols))

    # --- Dtype checks ---
    dtype_issues: List[Dict[str, Any]] = []
    for col in numeric_columns:
        if col in df.columns and not is_numeric_dtype(df[col]):
            dtype_issues.append({"column": col, "expected": "numeric", "actual": str(df[col].dtype)})

    for col in categorical_columns:
        if col in df.columns and not _is_acceptable_categorical_dtype(df[col]):
            dtype_issues.append({"column": col, "expected": "categorical/string-like", "actual": str(df[col].dtype)})

    # --- Primary key uniqueness check (supports composite keys) ---
    pk_info: Dict[str, Any] = {
        "columns": list(primary_keys),
        "exists": all(c in df.columns for c in primary_keys),
        "is_unique": None,
        "num_duplicates": None,
        "sample_duplicates": [],
        "pk_missing_count": None,
        "pk_missing_pct": None,
        "status": "pending",
        "error_type": None,
    }

    if pk_info["exists"]:
        n_rows = int(df.shape[0])

        # 1) Missing PK values
        if len(primary_keys) == 1:
            pk = primary_keys[0]
            miss_mask = _is_pk_missing_value(df[pk])
        else:
            miss_mask = pd.Series(False, index=df.index)
            for c in primary_keys:
                miss_mask = miss_mask | _is_pk_missing_value(df[c])

        pk_missing_count = int(miss_mask.sum())
        pk_info["pk_missing_count"] = pk_missing_count
        pk_info["pk_missing_pct"] = float(round(pk_missing_count / max(n_rows, 1), 6))

        # 2) Duplicates ONLY on non-missing subset
        if len(primary_keys) == 1:
            pk = primary_keys[0]
            s_valid = df.loc[~miss_mask, pk]

            n_valid = int(s_valid.shape[0])
            n_unique = int(s_valid.nunique(dropna=True))

            pk_info["is_unique"] = bool(n_unique == n_valid)
            pk_info["num_duplicates"] = int(n_valid - n_unique)

            if not pk_info["is_unique"]:
                pk_info["sample_duplicates"] = [str(x) for x in s_valid.loc[s_valid.duplicated()].head(5).tolist()]
                pk_info["status"] = "fail"
                pk_info["error_type"] = "duplicate_keys"
            else:
                pk_info["status"] = "pass"
        else:
            key_df = df.loc[~miss_mask, list(primary_keys)]
            dup_mask = key_df.duplicated()
            n_dups = int(dup_mask.sum())

            pk_info["is_unique"] = (n_dups == 0)
            pk_info["num_duplicates"] = n_dups

            if n_dups > 0:
                pk_info["sample_duplicates"] = (
                    key_df[dup_mask].head(5).astype(str).agg("|".join, axis=1).tolist()
                )
                pk_info["status"] = "fail"
                pk_info["error_type"] = "duplicate_composite_keys"
            else:
                pk_info["status"] = "pass"
    else:
        pk_info["status"] = "fail"
        pk_info["error_type"] = "missing_primary_key_cols"

    # --- Label column check ---
    label_info: Dict[str, Any] = {
        "column": label_col,
        "exists": label_col in df.columns,
        "unique_values": None,
    }
    if label_info["exists"]:
        label_info["unique_values"] = sorted(df[label_col].dropna().astype(str).str.strip().unique().tolist())[:50]

    # Pack into checks payload
    report["checks"]["contract"] = {
        "missing_columns": missing_cols,
        "extra_columns": extra_cols,
        "dtype_issues": dtype_issues,
        "primary_key": pk_info,
        "label_check": label_info,
    }

    # --- Event hints ---
    ensure_event_hints(report, version=1)

    # 1) Missing required columns (typically non-remediable)
    if missing_cols:
        add_event_hint(
            report,
            code="schema_missing_required_columns",
            severity_signal="hard",
            evidence_path="checks.contract.missing_columns",
            context={"missing_columns": list(missing_cols)},
            remediation={
                "action": "upstream_schema_fix_or_correct_dataset",
                "safe": False,
                "rationale": "Missing required columns cannot be safely reconstructed by deterministic cleaning.",
                "post_check": "re-run 2.2 contract after upstream fix",
            },
        )

    # 2) Extra columns present
    # - If allow_extra_columns=True, treat as INFO (presence signal only)
    # - If allow_extra_columns=False, treat as FIXABLE (drop is deterministic)
    if extra_cols:
        sev_sig = "info" if bool(allow_extra_columns) else "fixable"
        add_event_hint(
            report,
            code="schema_extra_columns_present",
            severity_signal=sev_sig,
            evidence_path="checks.contract.extra_columns",
            context={
                "extra_columns": list(extra_cols),
                "allow_extra_columns": bool(allow_extra_columns),
            },
            remediation={
                "action": "drop_extra_columns" if (not allow_extra_columns) else "review_extra_columns_and_decide_policy",
                "safe": True if (not allow_extra_columns) else True,
                "rationale": (
                    "Dropping unexpected columns is deterministic and auditable. "
                    "When allow_extra_columns=true, keep as observability signal and decide downstream policy."
                ),
                "post_check": "re-run 2.2 to confirm schema alignment",
            },
        )

    # 3) PK columns missing (non-remediable)
    if not bool(pk_info.get("exists", False)):
        add_event_hint(
            report,
            code="pk_missing_primary_key_cols",
            severity_signal="hard",
            evidence_path="checks.contract.primary_key",
            context={"pk_columns": list(primary_keys)},
            remediation={
                "action": "upstream_schema_fix_or_correct_dataset",
                "safe": False,
                "rationale": "Primary keys are required for identity; cannot be safely inferred in cleaning.",
                "post_check": "re-run 2.2 after upstream fix",
            },
        )
    else:
        # 3.1) PK missing values (often fixable by safe row-drop, policy decides thresholds)
        pk_missing_count = int(pk_info.get("pk_missing_count") or 0)
        pk_missing_pct = float(pk_info.get("pk_missing_pct") or 0.0)

        if pk_missing_count > 0:
            add_event_hint(
                report,
                code="pk_missing_values",
                severity_signal="fixable",
                evidence_path="checks.contract.primary_key.pk_missing_count",
                context={
                    "pk_columns": list(primary_keys),
                    "pk_missing_count": pk_missing_count,
                    "pk_missing_pct": float(round(pk_missing_pct, 6)),
                },
                remediation={
                    "action": "drop_rows_with_missing_pk",
                    "safe": True,
                    "rationale": "Dropping rows with missing identity can be deterministic and auditable; does not modify labels.",
                    "post_check": "re-run 2.2 + 2.5 readiness to confirm thresholds still satisfied",
                },
            )

        # 3.2) PK duplicates (hard unless an auditable dedupe policy exists)
        if pk_info.get("is_unique") is False:
            add_event_hint(
                report,
                code="pk_duplicate_values",
                severity_signal="hard",
                evidence_path="checks.contract.primary_key.num_duplicates",
                context={
                    "pk_columns": list(primary_keys),
                    "num_duplicates": int(pk_info.get("num_duplicates") or 0),
                    "sample_duplicates": list(pk_info.get("sample_duplicates") or []),
                },
                remediation={
                    "action": "dedupe_with_auditable_policy",
                    "safe": False,
                    "rationale": "Deduplication may change semantics unless a deterministic, auditable rule is defined (e.g., keep latest by timestamp).",
                    "post_check": "re-run 2.2 (pk unique) + 3.x integrity checks (no leakage / alignment)",
                },
            )

    # 4) Label column missing (non-remediable)
    if not bool(label_info.get("exists", False)):
        add_event_hint(
            report,
            code="label_missing_column",
            severity_signal="hard",
            evidence_path="checks.contract.label_check.exists",
            context={"label_col": str(label_col)},
            remediation={
                "action": "upstream_schema_fix_or_correct_dataset",
                "safe": False,
                "rationale": "Label is required for supervised learning; cannot be created by cleaning.",
                "post_check": "re-run 2.2 after label is present",
            },
        )
    else:
        # 4.1) Label whitespace normalization hint (usually safe + deterministic)
        try:
            raw_vals = df[label_col].dropna().astype(str)
            if int((raw_vals != raw_vals.str.strip()).sum()) > 0:
                add_event_hint(
                    report,
                    code="label_whitespace_detected",
                    severity_signal="fixable",
                    evidence_path="checks.contract.label_check.unique_values",
                    context={"label_col": str(label_col)},
                    remediation={
                        "action": "normalize_string_columns",
                        "safe": True,
                        "rationale": "String trimming is deterministic and preserves label semantics while improving matching.",
                        "post_check": "re-run 2.7 label balance + 2.5 readiness to confirm stability",
                    },
                )
        except Exception:
            pass

    # 5) Dtype issues (often fixable via schema-driven casting; 2.8 decides whether it becomes FIXABLE or HARD)
    if dtype_issues:
        add_event_hint(
            report,
            code="dtype_issues_detected",
            severity_signal="fixable",
            evidence_path="checks.contract.dtype_issues",
            context={
                "n_dtype_issues": int(len(dtype_issues)),
                "enforce_dtypes": bool(enforce_dtypes),
                "examples": list(dtype_issues)[:5],
            },
            remediation={
                "action": "cast_columns_per_schema",
                "safe": True,
                "rationale": "Schema-driven casting can be deterministic (e.g., numeric coerce + audited missing handling).",
                "post_check": "re-run 2.2 dtype checks; ensure no label semantic change",
            },
        )

    # --- 6) Status + gate-friendly summary ---
    issues: List[str] = []
    warnings: List[str] = []

    if enforce_required_columns and missing_cols:
        issues.append("missing_required_columns")
    
    # Extra columns: policy depends on allow_extra_columns.
    if (not allow_extra_columns) and extra_cols:
        issues.append("unexpected_extra_columns")
    elif allow_extra_columns and extra_cols:
        warnings.append("extra_columns_present")
    
    if not pk_info["exists"]:
        issues.append("missing_primary_key_cols")
    else:
        if pk_info.get("is_unique") is False:
            issues.append("primary_key_duplicate_values")

        # PK missing values: warning by default (policy handled by 2.8)
        if (pk_info.get("pk_missing_count") or 0) > 0:
            warnings.append("primary_key_missing_values")
    
    if not label_info["exists"]:
        issues.append("missing_label_column")
    
    # Dtype mismatch: policy-driven
    if enforce_dtypes and dtype_issues:
        issues.append("dtype_mismatch")
    elif (not enforce_dtypes) and dtype_issues:
        warnings.append("dtype_issues_detected")

    overall = "fail" if issues else ("warn" if warnings else "pass")

    report = finalize_module_report(
        ctx=ctx,
        report=report,
        overall_status=overall,
        issues=issues,
        warnings=warnings,
        metrics={
            "n_missing_columns": int(len(missing_cols)),
            "n_extra_columns": int(len(extra_cols)),
            "n_pk_duplicates": int(pk_info.get("num_duplicates") or 0),
            "pk_missing_count": int(pk_info.get("pk_missing_count") or 0),
            "pk_missing_pct": float(safe_float(pk_info.get("pk_missing_pct"), ndigits=6) or 0.0),
            "n_dtype_issues": int(len(dtype_issues)),
        },
        notes=[f"dataset={dataset_name}" if dataset_name else "dataset=unknown"],
        df=df,
    )

    if overall == "pass":
        lg.info(f"[{TAG22}] ✅ Contract pass")
    elif overall == "warn":
        lg.warning(f"[{TAG22}] ⚠ Contract warn | warnings={warnings}")
    else:
        lg.error(f"[{TAG22}] ❌ Contract fail | issues={issues}")

    # --- 7) Persist Artifact ---
    out_path = resolve_artifact_path(
        ctx=ctx,
        stage="data_quality",
        name="contract",
        run_id=str(rid),
        tag=TAG22,
    )
    write_json(out_path, report, tag=TAG22, indent=2)
    lg.info(f"[{TAG22}] 🧾 Contract artifact saved: {out_path}")

    return report

In [4]:
# --- 2.3 Data Health Check ---
#
# What:
#   Run lightweight health checks on the raw dataframe (missingness, cardinality).
#
# How:
#   - Read thresholds from YAML SSOT (pipeline.data_quality.health)
#   - Compute per-column missing ratio / cardinality ratio / top-k values for categoricals
#   - Emit a ModuleReport v1 artifact (orchestrator-friendly)
#
# Why:
#   Catch obvious data quality issues early and provide stable signals for downstream gates.

# --- 0) Import + TAG ---
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from pandas.api.types import is_numeric_dtype, is_string_dtype

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint
from src.utils.artifact_utils import resolve_artifact_path, write_json
from src.utils.serialization_utils import safe_float
from src.utils.cfg_utils import get_cfg_dict, get_cfg_int, get_cfg_float, get_cfg_bool
from src.utils.dataset_utils import get_schema_cols, get_primary_keys
from src.utils.path_utils import resolve_figures_dir
from src.utils.plot_utils import plot_bar_pairs, plot_bar_counter

TAG23 = "HEALTH"


# --- Internal helper ---
def _top_n_pairs(pairs: List[Tuple[str, float]], *, n: int) -> List[Tuple[str, float]]:
    """Return top-N by value desc (stable, small helper)."""
    return sorted(pairs, key=lambda x: x[1], reverse=True)[:n]


def sample_problem_cols_by_marker(
    problem_columns: Dict[str, List[str]],
    *,
    marker: str,
    limit: int = 10,
) -> List[str]:
    """Return up to `limit` column names whose marker list contains `marker`."""
    cols: List[str] = []
    for c, marks in problem_columns.items():
        if isinstance(marks, list) and (marker in marks):
            cols.append(str(c))
    return cols[: int(limit)]


def summarize_marker_counts(problem_columns: Dict[str, List[str]]) -> Dict[str, int]:
    """Count how many columns contain each marker."""
    out: Dict[str, int] = {}
    for _, marks in problem_columns.items():
        if not isinstance(marks, list):
            continue
        for m in marks:
            if not isinstance(m, str):
                continue
            out[m] = out.get(m, 0) + 1
    return out


# --- 1) Public API ---
def compute_data_health(
    *,
    df: pd.DataFrame,
    ctx: Dict[str, Any],
    run_id: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Run lightweight data health checks (missingness, cardinality, basic stats).

    YAML SSOT
    ---------
    pipeline.data_quality.health:
      high_missing_threshold: 0.20
      high_cardinality_ratio: 0.50
      top_k: 3
      plots:
        enabled: true
        top_n : 10

    dataset.schema.dtypes.numeric:
      - ...

    Returns
    -------
    dict
        ModuleReport v1 (Health) (also persisted as artifact). 
    """
    # ---- 2) Runtime validation ---
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG23, run_id=run_id)

    # --- 3) Input validation ---
    if df is None or not isinstance(df, pd.DataFrame):
        raise TypeError(f"[{TAG23}] df must be a pandas DataFrame")

    # --- 4) Resolve config (YAML SSOT) ---

    # --- Global（SSOT） ---
    primary_keys = get_primary_keys(ctx, TAG23)

    numeric_cols, categorical_cols = get_schema_cols(ctx, TAG23)
    numeric_expect = sorted(set([c for c in numeric_cols if c]))
    categorical_expect = sorted(set([c for c in categorical_cols if c]))

    # --- Module（step cfg） ---
    get_cfg_dict(ctx, "pipeline.data_quality.health", tag=TAG23, required=True)

    high_missing_threshold = get_cfg_float(
        ctx,
        "pipeline.data_quality.health.high_missing_threshold",
        tag=TAG23,
        default=0.20,
        required=False,   # <-- allow missing, use default
        min_value=0.0,
        max_value=1.0,
    )

    high_cardinality_ratio = get_cfg_float(
        ctx,
        "pipeline.data_quality.health.high_cardinality_ratio",
        tag=TAG23,
        default=0.50,
        required=False,
        min_value=0.0,
        max_value=1.0,
    )

    top_k = get_cfg_int(
        ctx,
        "pipeline.data_quality.health.top_k",
        tag=TAG23,
        default=3,
        required=False,
        min_value=1,
    )

    plots_enabled = get_cfg_bool(
        ctx,
        "pipeline.data_quality.health.plots.enabled",
        tag=TAG23,
        default=False,
        required=False,
        strict_type=True,
    )
   
    top_n_plot = get_cfg_int(
        ctx,
         "pipeline.data_quality.health.plots.top_n",
        tag=TAG23,
        default=10,
        required=False,
        min_value=3,
        max_value=50,
    )

    # --- 5) Core workflow ---
    lg.info(f"[{TAG23}] 🩺 Starting data health overview...")

    # Build ModuleReport skeleton (infra)
    report = build_module_report(
        ctx=ctx,
        stage="data_quality",
        step="2.3",
        name="health",
        tag=TAG23,
        cfg_key="pipeline.data_quality.health",
        enabled=True,
        df=df,
        run_id_override=rid,
    )

    # Safe-to-log inputs/config
    report["inputs"] = {"df_provided": True}

    report["thresholds_used"] = {
        "high_missing_threshold": float(high_missing_threshold),
        "high_cardinality_ratio": float(high_cardinality_ratio),
        "top_k": int(top_k),
        "top_n_plot": int(top_n_plot),
    }

    report["used_config"] = {
        "numeric_expect": list(numeric_expect),
        "categorical_expect": list(categorical_expect),
        "primary_key_cols": list(primary_keys),
        "plots_enabled": bool(plots_enabled),
        
    }

    report["refs"]["self"] = make_artifact_ref(
        ctx=ctx,
        stage="data_quality",
        name="health",
        run_id=str(rid),
    )

    # --- Actual health computation (business logic) ---
    n_rows = int(df.shape[0])
    n_cols = int(df.shape[1])

    columns_payload: Dict[str, Any] = {}
    problem_columns: Dict[str, List[str]] = {}
    issue_type_counter: Dict[str, int] = {}

    for col in df.columns:
        col_name = str(col)
        s = df[col]

        missing_pct = float(s.isna().mean()) if n_rows > 0 else 0.0
        unique_values = int(s.nunique(dropna=True)) if n_rows > 0 else 0
        card_ratio = float(unique_values / max(n_rows, 1))

        col_info: Dict[str, Any] = {
            "dtype": str(s.dtype),
            "missing_pct": float(round(missing_pct, 6)),
            "unique_values": int(unique_values),
            "cardinality_ratio": float(round(card_ratio, 6)),
        }

        is_num = is_numeric_dtype(s)

        if is_num:
            col_info.update(
                {
                    "min": safe_float(s.min(skipna=True), ndigits=6),
                    "max": safe_float(s.max(skipna=True), ndigits=6),
                    "mean": safe_float(s.mean(skipna=True), ndigits=6),
                    "std": safe_float(s.std(skipna=True), ndigits=6),
                }
            )

            if n_rows > 0:
                zero_mask = (s == 0)
                neg_mask = (s < 0)

                if hasattr(zero_mask, "fillna"):
                    zero_count = int(zero_mask.fillna(False).sum())
                    negative_count = int(neg_mask.fillna(False).sum())
                else:
                    zero_count = int(np.nansum(zero_mask.to_numpy()))
                    negative_count = int(np.nansum(neg_mask.to_numpy()))
            else:
                zero_count = 0
                negative_count = 0

            col_info["zero_count"] = int(zero_count)
            col_info["negative_count"] = int(negative_count)
        else:
            top_values = s.value_counts(dropna=False).head(top_k).to_dict()
            col_info["top_values"] = {str(k): int(v) for k, v in top_values.items()}

        # Signals only (no gating policy here)
        comments: List[str] = []

        if missing_pct > high_missing_threshold:
            comments.append(f"high_missing_ratio")

        if unique_values == 1:
            comments.append("constant_column")

        if (col_name in numeric_expect) and (not is_numeric_dtype(s)) and (is_string_dtype(s) or str(s.dtype) == "object"):
            comments.append("numeric_stored_as_string")

        if (col_name in categorical_expect) and (col_name not in primary_keys) and (card_ratio > high_cardinality_ratio):
            comments.append("high_cardinality_categorical")

        if (col_name in primary_keys) and (not comments):
            comments = ["ok_pk_column"]

        if not comments:
            comments = ["ok"]

        col_info["comment"] = comments
        columns_payload[col_name] = col_info

        # Treat only non-OK markers as "problems"
        non_problem_markers = {"ok", "ok_pk_column"}
        is_problem = any(c not in non_problem_markers for c in comments)

        if is_problem:
            problem_columns[col_name] = comments
            for c in comments:
                if c in non_problem_markers:
                    continue
                issue_type_counter[c] = issue_type_counter.get(c, 0) + 1

    # --- Top offenders (production-friendly short list) ---
    offenders: Dict[str, Any] = {"top_missing": [], "top_cardinality": []}

    if n_rows > 0 and columns_payload:
        missing_rank = [
            (c, float(info.get("missing_pct", 0.0)))
            for c, info in columns_payload.items()
            if isinstance(info, dict) and "missing_pct" in info
        ]
        top_missing = [(c, v) for (c, v) in _top_n_pairs(missing_rank, n=3) if v > high_missing_threshold]
        offenders["top_missing"] = [{"column": c, "missing_pct": float(round(v, 6))} for c, v in top_missing]

        # Top cardinality (ONLY for columns actually flagged as high-cardinality categoricals)
        high_card_cols: List[str] = []
        for c, info in columns_payload.items():
            if not isinstance(info, dict):
                continue
            comments = info.get("comment", [])
            if isinstance(comments, list) and ("high_cardinality_categorical" in comments):
                high_card_cols.append(c)

        card_rank = [
            (c, float(columns_payload[c].get("cardinality_ratio", 0.0)))
            for c in high_card_cols
            if isinstance(columns_payload.get(c), dict)
        ]
        top_card = _top_n_pairs(card_rank, n=3)  # already filtered by your rule; no need to threshold again
        offenders["top_cardinality"] = [{"column": c, "cardinality_ratio": float(round(v, 6))} for c, v in top_card]
    
    report["checks"]["health"] = {
        "n_rows": n_rows,
        "n_cols": n_cols,
        "n_problem_columns": int(len(problem_columns)),
        "problem_columns": list(problem_columns.keys()),
        "issue_types": issue_type_counter,
        "top_offenders": offenders,
        "columns": columns_payload,
    }

    # --- Compatibility aliases for orchestrator (2.8) ---
    # Keep both list and dict forms to avoid downstream schema coupling.
    report["checks"]["health"]["problem_columns_detail"] = dict(problem_columns)  # col -> markers
    report["checks"]["health"]["problem_columns_map"] = dict(problem_columns)     # alias
    report["checks"]["health"]["marker_counts"] = summarize_marker_counts(problem_columns)

    # Optional: common name used by some extractors
    report["checks"]["health"]["problem_cols"] = list(problem_columns.keys())

    # --- Plots ---
    plots_meta = {"requested": bool(plots_enabled), "saved": False, "dir": None, "files": []}

    if plots_enabled:
        try:
            fig_base = resolve_figures_dir(ctx, tag=TAG23)
            if fig_base is None:
                plots_meta["reason"] = "figures_dir_unresolved"
            else:
                module_name = "health"
                fig_dir = fig_base / "data_quality" / module_name
                fig_dir.mkdir(parents=True, exist_ok=True)
                plots_meta["dir"] = str(fig_dir)

                saved_refs: List[Dict[str, Any]] = []
                saved_paths: List[str] = []

                # 1) Top missingness
                missing_pairs = [
                    (c, float(info.get("missing_pct", 0.0)))
                    for c, info in columns_payload.items()
                    if isinstance(info, dict)
                ]
                p1 = fig_dir / f"data_quality_{module_name}_missing_top{top_n_plot}_{rid}.png"
                plot_bar_pairs(
                    missing_pairs,
                    title=f"Top-{top_n_plot} Missingness (run_id={rid})",
                    ylabel="missing_pct",
                    out_path=p1,
                    top_n=int(top_n_plot),
                    sort_desc=True,
                )
                saved_refs.append({"name": "missing_top", "path": str(p1)})
                saved_paths.append(str(p1))

                # 2) Top cardinality ratio
                # Cardinality ratio is most meaningful for categoricals; avoid numeric noise.
                card_pairs = [
                    (c, float(columns_payload[c].get("cardinality_ratio", 0.0)))
                    for c in categorical_expect
                    if isinstance(columns_payload.get(c), dict)
                ]
                # Fallback: if schema categoricals missing, keep old behavior to avoid empty plots.
                if not card_pairs:
                    card_pairs = [
                        (c, float(info.get("cardinality_ratio", 0.0)))
                        for c, info in columns_payload.items()
                        if isinstance(info, dict)
                    ]
                p2 = fig_dir / f"data_quality_{module_name}_cardinality_top{top_n_plot}_{rid}.png"
                plot_bar_pairs(
                    card_pairs,
                    title=f"Top-{top_n_plot} Cardinality Ratio (run_id={rid})",
                    ylabel="cardinality_ratio",
                    out_path=p2,
                    top_n=int(top_n_plot),
                    sort_desc=True,
                )
                saved_refs.append({"name": "cardinality_top", "path": str(p2)})
                saved_paths.append(str(p2))

                # 3) Issue types counter
                p3 = fig_dir / f"data_quality_{module_name}_issue_types_top20_{rid}.png"
                plot_bar_counter(
                    issue_type_counter,
                    title=f"Issue Types Count (run_id={rid})",
                    ylabel="count",
                    out_path=p3,
                    top_n=20,
                )
                saved_refs.append({"name": "issue_types", "path": str(p3)})
                saved_paths.append(str(p3))

                plots_meta["saved"] = True
                plots_meta["files"] = saved_paths

                # Attach figure refs
                report.setdefault("refs", {})
                report["refs"].setdefault("figures", [])
                report["refs"]["figures"].extend(saved_refs)

        except Exception as e:
            plots_meta["saved"] = False
            plots_meta["error"] = str(e)

    report.setdefault("checks", {})
    report["checks"]["plots"] = plots_meta

    # --- Event hints (semantic + remediation hints for 2.8 policy mapping) ---
    # Notes:
    #   - Keep this module policy-free; only provide auditable hints for 2.8.
    #   - severity_signal must be one of: "hard" | "fixable" | "risk" | "info"
    ensure_event_hints(report, version=1)

    n_problem = int(len(problem_columns))

    # Optional: marker-based counts derived from problem_columns (more semantically aligned than issue_type_counter)
    marker_counts = summarize_marker_counts(problem_columns)

    # 1) Any health issues present (umbrella signal)
    if n_problem > 0:
        add_event_hint(
            report,
            code="health_problem_columns_present",
            severity_signal="risk",
            evidence_path="checks.health.n_problem_columns",
            context={
                "n_problem_columns": n_problem,
                "marker_counts": dict(marker_counts),
                # keep small / stable:
                "examples": list(problem_columns.keys())[:10],
            },
            remediation={
                "action": "review_column_health_and_adjust_cleaning_or_feature_policy",
                "safe": True,
                "rationale": (
                    "Health signals are non-blocking diagnostics; remediation is typically deterministic "
                    "(drop/transform/cast) and auditable when configured."
                ),
                "post_check": "re-run 2.3 after cleaning (2.9) to confirm metrics improved",
            },
        )

    # 2) High missingness (risk by default; 2.8 may escalate if extreme)
    n_high_missing = int(sum("high_missing_ratio" in v for v in problem_columns.values()))
    if n_high_missing > 0:
        add_event_hint(
            report,
            code="health_high_missing_ratio",
            severity_signal="risk",
            evidence_path="checks.health.top_offenders.top_missing",
            context={
                "threshold": float(high_missing_threshold),
                "n_high_missing_cols": n_high_missing,
                "top_missing": offenders.get("top_missing", []),
                "sample_cols": sample_problem_cols_by_marker(
                    problem_columns, marker="high_missing_ratio", limit=10
                ),
            },
            remediation={
                "action": "configure_missing_handling_policy",
                "safe": True,
                "rationale": (
                    "Missingness handling can be deterministic (drop/constant/median/imputation) if configured; "
                    "choose strategies that do not alter label semantics or introduce leakage."
                ),
                "post_check": "re-run 2.3 + downstream readiness (2.5) to ensure dataset remains valid",
            },
        )

    # 3) Constant columns (risk; usually safe to drop)
    n_constant = int(sum("constant_column" in v for v in problem_columns.values()))
    if n_constant > 0:
        add_event_hint(
            report,
            code="health_constant_columns",
            severity_signal="risk",
            evidence_path="checks.health.columns",
            context={
                "n_constant_cols": n_constant,
                "sample_cols": sample_problem_cols_by_marker(
                    problem_columns, marker="constant_column", limit=10
                ),
            },
            remediation={
                "action": "drop_constant_columns",
                "safe": True,
                "rationale": (
                    "Constant columns carry no signal and are safe to remove deterministically "
                    "(auditable, no label change)."
                ),
                "post_check": "re-run 2.3 to confirm removed columns no longer appear as issues",
            },
        )

    # 4) Numeric stored as string (typically FIXABLE via schema-driven cast)
    n_num_as_str = int(sum("numeric_stored_as_string" in v for v in problem_columns.values()))
    if n_num_as_str > 0:
        add_event_hint(
            report,
            code="health_numeric_stored_as_string",
            severity_signal="fixable",
            evidence_path="checks.health.columns",
            context={
                "n_numeric_stored_as_string": n_num_as_str,
                "sample_cols": sample_problem_cols_by_marker(
                    problem_columns, marker="numeric_stored_as_string", limit=10
                ),
            },
            remediation={
                "action": "cast_numeric_columns_per_schema",
                "safe": True,
                "rationale": (
                    "Schema-driven casting (e.g., to_numeric(errors='coerce') + audited missing strategy) "
                    "is deterministic and does not change label semantics."
                ),
                "post_check": "re-run 2.2 dtype checks (if enabled) + re-run 2.3 to confirm dtype/missingness improved",
            },
        )

    # 5) High-cardinality categoricals (risk; mitigation is feature policy, not auto-clean)
    n_high_card = int(sum("high_cardinality_categorical" in v for v in problem_columns.values()))
    if n_high_card > 0:
        add_event_hint(
            report,
            code="health_high_cardinality_categorical",
            severity_signal="risk",
            evidence_path="checks.health.top_offenders.top_cardinality",
            context={
                "threshold": float(high_cardinality_ratio),
                "n_high_cardinality_cols": n_high_card,
                "top_cardinality": offenders.get("top_cardinality", []),
                "sample_cols": sample_problem_cols_by_marker(
                    problem_columns, marker="high_cardinality_categorical", limit=10
                ),
                "primary_key_cols": list(primary_keys),
            },
            remediation={
                "action": "apply_cardinality_mitigation_policy",
                "safe": True,
                "rationale": (
                    "High-cardinality categoricals often require encoding strategy (hashing/target encoding) "
                    "or bucketing; avoid leaking label info and keep transformation auditable."
                ),
                "post_check": "re-run 2.3 after applying bucketing/encoding policy (if any) and validate downstream model stability",
            },
        )

    # 6) Plotting failure (non-data; INFO by default)
    if plots_meta.get("requested") and (not plots_meta.get("saved")):
        add_event_hint(
            report,
            code="health_plots_failed",
            severity_signal="info",
            evidence_path="checks.plots",
            context={"plots": dict(plots_meta)},
            remediation={
                "action": "fix_figures_dir_or_plot_runtime",
                "safe": True,
                "rationale": "Plot failures do not change data validity; treat as observability issue.",
                "post_check": "re-run 2.3 with plots enabled to confirm artifacts are produced",
            },
        )

    # --- 6) Status + Gate-friendly summary ---
    issues: List[str] = []
    warnings: List[str] = []
    
    if n_rows == 0:
        issues.append("empty_dataframe")

    if len(problem_columns) > 0:
        warnings.append("column_health_issues")

    if plots_meta.get("requested") and (not plots_meta.get("saved")):
        warnings.append("plots_failed")

    overall = "fail" if issues else ("warn" if warnings else "pass")


    offender_notes: List[str] = []
    if offenders.get("top_missing"):
        offender_notes.append(
            "top_missing=" + ", ".join([f"{x['column']}:{x['missing_pct']:.3f}" for x in offenders["top_missing"]])
        )
    if offenders.get("top_cardinality"):
        offender_notes.append(
            "top_cardinality=" + ", ".join([f"{x['column']}:{x['cardinality_ratio']:.3f}" for x in offenders["top_cardinality"]])
        )

    report = finalize_module_report(
        ctx=ctx,
        report=report,
        overall_status=overall,
        issues=issues,
        warnings=warnings,
        metrics={
            "n_problem_columns": int(len(problem_columns)),
            "n_issue_types": int(len(issue_type_counter)),
            "n_high_missing_cols": int(sum("high_missing_ratio" in v for v in problem_columns.values())),
            "n_high_cardinality_cols": int(sum("high_cardinality_categorical" in v for v in problem_columns.values())),
            "n_numeric_stored_as_string": int(sum("numeric_stored_as_string" in v for v in problem_columns.values())),
        },
        notes=offender_notes,
        df=df,
    )

    if overall == "pass":
        lg.info(f"[{TAG23}] ✅ Health checks completed.")
    elif overall == "warn":
        lg.warning(
            f"[{TAG23}] ⚠ Health check completed with warnings. | "
            f"problem_columns={len(problem_columns)} | issue_types={issue_type_counter} "
            f"{' | '.join(offender_notes) if offender_notes else ''}"
        )
    else:
        lg.error(f"[{TAG23}] ❌ Health checks failed. | issues={issues}")


    # --- 7) Persist Artifact ---
    out_path = resolve_artifact_path(
        ctx=ctx,
        stage="data_quality",
        name="health",
        run_id=str(rid),
        tag=TAG23,
    )
    write_json(out_path, report, tag=TAG23, indent=2)
    lg.info(f"[{TAG23}] 🧾 Health artifact saved: {out_path}")

    return report

In [5]:
# --- 2.4 Semantic Checks ---
#
# What:
#   Apply business logic and domain rules to validate key fields.
#
# How:
#   - Read config from YAML SSOT (pipeline.data_quality.semantic + dataset metadata)
#   - Build a ruleset (telco_v1) (pure)
#   - Run a small rule engine (pure computation)
#   - Wrap engine output into ModuleReport v1 (aligned with 2.1–2.3)
#
# Why:
#   Catch logically impossible or suspicious records early, before cleaning/modeling.

# --- 0) Imports + TAG ---
from dataclasses import dataclass
from typing import Any, Dict, List, Callable, Optional

import pandas as pd
from pandas.api.types import is_bool_dtype

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint
from src.utils.artifact_utils import resolve_artifact_path, write_json
from src.utils.cfg_utils import get_cfg_dict, get_cfg_int, get_cfg_str
from src.utils.dataset_utils import get_label_cfg
from src.utils.pandas_utils import safe_num

TAG24 = "SEMANTIC"


# --- Internal helpers ---
def _as_bool_mask(
    mask: Any,
    *,
    index: pd.Index,
    tag: str,
    rule_name: str,
) -> pd.Series:
    """Coerce rule output into a boolean pd.Series aligned with df index (STRICT)."""
    if not isinstance(mask, pd.Series):
        raise TypeError(f"[{tag}] rule '{rule_name}' must return pd.Series[bool], got {type(mask)}")

    if not mask.index.equals(index):
        raise ValueError(f"[{tag}] rule '{rule_name}' returned misaligned mask index")

    # STRICT: only allow boolean dtype (including pandas BooleanDtype).
    if not is_bool_dtype(mask):
        raise TypeError(f"[{tag}] rule '{rule_name}' must return a boolean Series, got dtype={mask.dtype}")

    # Normalize NA in boolean mask deterministically.
    if mask.isna().any():
        mask = mask.fillna(False)

    return mask.astype(bool)


def _resolve_required_cols_for_ruleset(
    *,
    ruleset: str,
    label_col: str
) -> List[str]:
    """Return minimal required columns for a given ruleset (keep runner robust)."""
    if ruleset == "telco_v1":
        return ["tenure", "MonthlyCharges", "TotalCharges", label_col]
    return [label_col]


# --- Rule + Engine ---
@dataclass(frozen=True)
class SemanticRule:
    """
    A single semantic check rule.

    rule_func returns a boolean mask where True indicates violation.
    """
    name: str
    rule_func: Callable[[pd.DataFrame], pd.Series]
    description: str


    def run(
        self, 
        df: pd.DataFrame,
    ) -> pd.Series:
        """Execute the rule and return a boolean mask of violations."""
        return self.rule_func(df)
        

class SemanticCheckEngine:
    """
    Execute multiple semantic rules and aggregate results.

    Design:
      - Pure computation: input df -> structured dict output
      - No ctx, no file writes, no logging
      - Stable output schema for runner to wrap into ModuleReport
    """

    def __init__(
        self,
        rules: dict[str, SemanticRule],
    ) -> None:
        if not isinstance(rules, dict) or not rules:
            raise TypeError(f"[{TAG24}] rules must be a non-empty dict[str, SemanticRule]")
        self.rules = rules


    def run(
        self,
        df: pd.DataFrame,
        *,
        max_examples: int,
    ) -> Dict[str, Any]:
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"[{TAG24}] engine expects a pandas DataFrame")
        if not isinstance(max_examples, int) or max_examples <= 0:
            raise ValueError(f"[{TAG24}] engine max_examples must be a positive int, got {max_examples!r}")

        checks: Dict[str, Any] = {}
        failed_rules: List[str] = []
        total_violations = 0

        for name, rule in self.rules.items():
            raw_mask = rule.run(df)
            mask = _as_bool_mask(raw_mask, index=df.index, tag=TAG24, rule_name=name)

            n_bad = int(mask.sum())
            total_violations += n_bad

            status = "fail" if n_bad > 0 else "pass"
            if n_bad > 0:
                failed_rules.append(name)

            example_indices = df.index[mask].tolist()[:max_examples]

            checks[name] = {
                "status": status,
                "n_violations": n_bad,
                "example_indices": example_indices,
                "rule": rule.description,
            }

        return {
            "summary": {
                "total_rows": int(df.shape[0]),
                "n_rules": int(len(self.rules)),
                "n_failed_rules": int(len(failed_rules)),
                "failed_rules": failed_rules,
                "total_violations": int(total_violations),
            },
            "checks": checks,
        }

# --- Ruleset Builder ---
def build_telco_semantic_rules(
    *,
    label_col: str
) -> Dict[str, SemanticRule]:
    """Construct semantic rules specific to the Telco Churn dataset."""

    def tenure_range_violation(df: pd.DataFrame) -> pd.Series:
        tenure = safe_num(df["tenure"])
        # Violation includes NaN because tenure is expected to be parseable.
        return tenure.isna() | (tenure < 0) | (tenure > 72)


    def charges_missing_or_unparseable_violation(df: pd.DataFrame) -> pd.Series:
        m = safe_num(df["MonthlyCharges"])
        t = safe_num(df["TotalCharges"])
        # Violation: missing or unparseable numeric values (expected to be cleaned later)
        return m.isna() | t.isna()


    def charges_negative_violation(df: pd.DataFrame) -> pd.Series:
        m = safe_num(df["MonthlyCharges"])
        t = safe_num(df["TotalCharges"])
        # Violation: truly negative charges (more severe)
        return (m.notna() & (m < 0)) | (t.notna() & (t < 0))


    def tenure_total_consistency_violation(df: pd.DataFrame) -> pd.Series:
        tenure = safe_num(df["tenure"])
        total = safe_num(df["TotalCharges"])
        return (
            ((tenure == 0) & total.notna() & (total > 5.0))
            | ((tenure > 0) & (total.isna() | (total <= 0.0)))
        )


    def label_values_violation(df: pd.DataFrame) -> pd.Series:
        # Telco dataset is known to be Yes/No for label; keep it explicit for now.
        s = df[label_col].astype(str).str.strip()
        return ~s.isin({"Yes", "No"})

    return {
        "tenure_range": SemanticRule(
            name="tenure_range",
            rule_func=tenure_range_violation,
            description="0 <= tenure <= 72 (parseable numeric required)",
        ),
        "charges_missing_or_unparseable": SemanticRule(
            name="charges_missing_or_unparseable",
            rule_func=charges_missing_or_unparseable_violation,
            description="MonthlyCharges/TotalCharges must be parseable numeric (missing/unparseable flagged; expected to be handled in cleaning)",
        ),
        "charges_negative": SemanticRule(
            name="charges_negative",
            rule_func=charges_negative_violation,
            description="MonthlyCharges >= 0 and TotalCharges >= 0 (negative values are invalid)",
        ),
        "tenure_total_consistency": SemanticRule(
            name="tenure_total_consistency",
            rule_func=tenure_total_consistency_violation,
            description="If tenure == 0 then TotalCharges ≈ 0; if tenure > 0 then TotalCharges should be > 0",
        ),
        "label_values": SemanticRule(
            name="label_values",
            rule_func=label_values_violation,
            description=f'{label_col} ∈ {{"Yes", "No"}}',
        ),
    }


# --- 1) Public API ---
def run_semantic_checks(
    *,
    df: pd.DataFrame,
    ctx: Dict[str, Any],
    run_id: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Run semantic checks in a ctx-driven, artifact-persisted manner.

    YAML SSOT
    ---------
    pipeline.data_quality.semantic:
      max_examples: 5
      ruleset: "telco_v1"

    Returns
    -------
    dict
        ModuleReport v1 (Semantic) (also persisted as artifact).
    """
    # --- 2) Runtime validation ---
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG24, run_id=run_id)

    # --- 3) Input validation ---
    if df is None or not isinstance(df, pd.DataFrame):
        raise TypeError(f"[{TAG24}] df must be a pandas DataFrame")
    
    # --- 4) Resolve config (YAML SSOT) ---

    # --- Global（SSOT） ---
    label_col, _, _ = get_label_cfg(ctx, TAG24)

    # --- Module（step cfg） ---
    get_cfg_dict(ctx, "pipeline.data_quality.semantic", tag=TAG24, required=True)

    max_examples = get_cfg_int(
        ctx,
        "pipeline.data_quality.semantic.max_examples",
        tag=TAG24,
        default=5,
        required=False,
        min_value=1,
    )

    ruleset = get_cfg_str(
        ctx,
        "pipeline.data_quality.semantic.ruleset",
        tag=TAG24,
        default="telco_v1",
        required=False,
    )

    required_cols = _resolve_required_cols_for_ruleset(ruleset=ruleset, label_col=label_col)
    missing_cols = sorted([c for c in required_cols if c not in df.columns])
    
    # Normalize ruleset early (keep it strict but policy-free)
    ruleset_norm = str(ruleset).strip().lower()

    # --- 5) Core logic (pure computation) ---
    lg.info(f"[{TAG24}] 🧠 Starting semantic checks... (ruleset={ruleset}, max_examples={max_examples})")

    # Build ModuleReport skeleton (infra)
    report = build_module_report(
        ctx=ctx,
        stage="data_quality",
        step="2.4",
        name="semantic",
        tag=TAG24,
        cfg_key="pipeline.data_quality.semantic",
        enabled=True,
        df=df,
        run_id_override=rid,
    )

    report["inputs"] = {"df_provided": True}
    report["thresholds_used"] = {"max_examples": int(max_examples)}
    report["used_config"] = {
        "ruleset": ruleset,
        "label_col": label_col,
        "required_cols": list(required_cols),
    }

    report["refs"]["self"] = make_artifact_ref(
        ctx=ctx,
        stage="data_quality",
        name="semantic",
        run_id=str(rid),
    )

    # --- Preconditions (stable, policy-free) ---
    report["checks"]["semantic_preconditions"] = {
        "ruleset": ruleset_norm,
        "supported_rulesets": ["telco_v1"],
        "required_cols": list(required_cols),
        "missing_required_columns": list(missing_cols),
    }

    # --- Engine output (stable schema) ---
    engine_out: Dict[str, Any]

    unsupported_ruleset = (ruleset_norm != "telco_v1")

    if unsupported_ruleset or missing_cols:
        # Do NOT raise; emit auditable artifact + hard hint, let orchestrator decide.
        reasons: List[str] = []
        if unsupported_ruleset:
            reasons.append("unsupported_ruleset")
        if missing_cols:
            reasons.append("missing_required_columns")

        lg.error(
            f"[{TAG24}] ❌ semantic checks not runnable | reasons={reasons} "
            f"| ruleset={ruleset_norm} | missing_cols={missing_cols}"
        )

        engine_out = {
            "summary": {
                "total_rows": int(df.shape[0]),
                "n_rules": 0,
                "n_failed_rules": 0,
                "failed_rules": [],
                "total_violations": 0,
            },
            "checks": {},
        }
    else:
        rules = build_telco_semantic_rules(label_col=label_col)
        engine = SemanticCheckEngine(rules)
        engine_out = engine.run(df, max_examples=max_examples)

    report["checks"]["semantic"] = engine_out

    # --- Event hints (semantic + remediation hints for 2.8 policy mapping) ---
    # Notes:
    #   - Keep this module policy-free; only provide auditable hints for 2.8.
    #   - severity_signal must be one of: "hard" | "fixable" | "risk" | "info"
    ensure_event_hints(report, version=1)

    if unsupported_ruleset:
        add_event_hint(
            report,
            code="semantic_unsupported_ruleset",
            severity_signal="hard",
            evidence_path="used_config.ruleset",
            context={"ruleset": ruleset_norm, "supported_rulesets": ["telco_v1"]},
            remediation={
                "action": "set_pipeline.data_quality.semantic.ruleset_to_supported_value",
                "safe": False,
                "rationale": "Semantic runner is ruleset-specific; unsupported ruleset cannot be evaluated safely.",
                "post_check": "re-run 2.4 after SSOT fix",
            },
        )

    if missing_cols:
        add_event_hint(
            report,
            code="semantic_missing_required_columns",
            severity_signal="hard",
            evidence_path="checks.semantic_preconditions.missing_required_columns",
            context={"ruleset": ruleset_norm, "missing_columns": list(missing_cols)},
            remediation={
                "action": "upstream_schema_fix_or_adjust_ruleset_requirements",
                "safe": False,
                "rationale": "Semantic checks require specific columns; they cannot be safely reconstructed by cleaning.",
                "post_check": "re-run 2.4 after upstream fix or ruleset adjustment",
            },
        )

    # Rule-level hints (only when runnable)
    if (not unsupported_ruleset) and (not missing_cols):
        checks = engine_out.get("checks", {}) or {}
        for rule_name, rule_out in checks.items():
            if not isinstance(rule_out, dict):
                continue

            n_viol = int(rule_out.get("n_violations", 0) or 0)
            if n_viol <= 0:
                continue

            # Provide semantic meaning WITHOUT making gate decisions here.
            if rule_name == "charges_negative":
                sev_sig = "hard"
                action = "fix_upstream_or_remove_invalid_rows_with_audited_policy"
                safe = False
                rationale = "Negative charges are logically invalid; automatic correction is not safe without a domain-approved policy."
                post_check = "re-run 2.4; if rows removed, re-run 2.5 readiness and 2.7 distribution"
            elif rule_name == "charges_missing_or_unparseable":
                sev_sig = "fixable"
                action = "cast_numeric_and_apply_missing_policy"
                safe = True
                rationale = "Schema-driven casting + deterministic missing handling can be safe and auditable."
                post_check = "re-run 2.2 dtype checks (if enabled) + re-run 2.4"
            elif rule_name == "label_values":
                sev_sig = "fixable"
                action = "normalize_string_columns_and_validate_label_domain"
                safe = True
                rationale = "String normalization (trim/case) is deterministic and preserves label semantics."
                post_check = "re-run 2.4 + 2.5 readiness to confirm label balance"
            elif rule_name == "tenure_range":
                sev_sig = "risk"
                action = "review_outliers_and_define_bounds_policy"
                safe = True
                rationale = "Outliers may be legitimate; typically handled via feature policy rather than auto-clean."
                post_check = "review 2.4 examples; decide clipping/removal policy, then re-run"
            elif rule_name == "tenure_total_consistency":
                sev_sig = "risk"
                action = "review_business_consistency_and_define_deterministic_hook"
                safe = True
                rationale = "Consistency violations are strong signals; remediation depends on a deterministic domain rule/hook."
                post_check = "define deterministic rule/hook, then re-run 2.4"
            else:
                sev_sig = "risk"
                action = "review_semantic_rule_and_define_policy"
                safe = True
                rationale = "Unhandled rule; treat as risk until a remediation policy is defined."
                post_check = "review rule output and define remediation"

            add_event_hint(
                report,
                code=f"semantic_{rule_name}_violations",
                severity_signal=sev_sig,
                evidence_path=f"checks.semantic.checks.{rule_name}",
                context={
                    "rule": rule_name,
                    "n_violations": n_viol,
                    "examples": list(rule_out.get("example_indices") or [])[:5],
                },
                remediation={
                    "action": action,
                    "safe": bool(safe),
                    "rationale": rationale,
                    "post_check": post_check,
                },
            )

    # --- 6) Status + gate-friendly summary ---
    issues: List[str] = []
    warnings: List[str] = []

    if unsupported_ruleset:
        issues.append("semantic_unsupported_ruleset")

    if missing_cols:
        issues.append("semantic_missing_required_columns")

    if issues:
        overall = "fail"
        metrics = {
            "n_missing_required_cols": int(len(missing_cols)),
            "unsupported_ruleset": bool(unsupported_ruleset),
        }
    else:
        total_viol = int(engine_out.get("summary", {}).get("total_violations", 0) or 0)
        if total_viol > 0:
            warnings.append("semantic_violations_detected")
            overall = "warn"
        else:
            overall = "pass"

        metrics = {
            "n_rules": int(engine_out.get("summary", {}).get("n_rules", 0) or 0),
            "n_failed_rules": int(engine_out.get("summary", {}).get("n_failed_rules", 0) or 0),
            "total_violations": int(total_viol),
        }

    report = finalize_module_report(
        ctx=ctx,
        report=report,
        overall_status=overall,
        issues=issues,
        warnings=warnings,
        metrics=metrics,
        notes=[f"ruleset={ruleset_norm}"],
        df=df,
    )

    if overall == "pass":
        lg.info(f"[{TAG24}] ✅ Semantic checks completed.")
    elif overall == "warn":
        lg.warning(f"[{TAG24}] ⚠ Semantic checks completed with warnings. | warnings={warnings}")
    else:
        lg.error(f"[{TAG24}] ❌ Semantic checks failed. | issues={issues}")

    # --- 7) Persist artifact ---
    out_path = resolve_artifact_path(
        ctx=ctx,
        stage="data_quality",
        name="semantic",
        run_id=str(rid),
        tag=TAG24,
    )
    write_json(out_path, report, tag=TAG24, indent=2)
    lg.info(f"[{TAG24}] 🧾 Semantic artifact saved: {out_path}")

    return report

In [6]:
# --- 2.5 Experimental Readiness Check ---
#
# What:
#   Assess whether the dataset is viable for the planned task (classification).
#
# How:
#   - Validate minimal requirements (min rows, label missingness, class balance).
#   - Read thresholds from YAML SSOT via ctx.
#   - Persist a strict JSON artifact for reproducibility.
#
# Why:
#   Avoid expensive modeling on structurally unfit data.

# --- 0) Import + TAG ---
from typing import Any, Dict, List, Optional

import pandas as pd

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint
from src.utils.artifact_utils import resolve_artifact_path, write_json
from src.utils.cfg_utils import get_cfg_dict, get_cfg_int, get_cfg_float
from src.utils.dataset_utils import get_label_cfg
from src.utils.serialization_utils import safe_float

TAG25 = "EXP_READY"

_SUPPORTED_TASK_TYPES: tuple[str, ...] = ("classification",)


# --- 1) Public API ---
def check_experimental_readiness(
    *,
    df: pd.DataFrame,
    ctx: Dict[str, Any],
    run_id: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Assess whether the dataset is viable for the planned task (classification),
    using thresholds from YAML SSOT and persisting a reproducible artifact.

    YAML SSOT
    ---------
    pipeline.data_quality.readiness:
      min_rows: 1000
      max_label_missing_pct: 0.30
      min_positive_frac: 0.05
      max_positive_frac: 0.95

    dataset.label:
      col: "Churn"
      positive: "Yes"
      task_type: "classification"

    Returns
    -------
    dict
        ModuleReport v1 (Readiness) (also persisted as artifact).
    """
    # --- 2) Runtime validation ---
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG25, run_id=run_id)

    # --- 3) Input validation ---
    if df is None or not isinstance(df, pd.DataFrame):
        raise TypeError(f"[{TAG25}] df must be a pandas DataFrame")

    # --- 4) Resolve config (YAML SSOT) ---

    # --- Global（SSOT） ---
    label_col, pos_label, task_type = get_label_cfg(ctx, TAG25)
    
    tt = str(task_type).strip().lower()
    task_type_ok = (tt in _SUPPORTED_TASK_TYPES)

    # --- Module（step cfg） ---
    get_cfg_dict(ctx, "pipeline.data_quality.readiness", tag=TAG25, required=True)

    min_rows = get_cfg_int(
        ctx,
        "pipeline.data_quality.readiness.min_rows",
        tag=TAG25,
        default=0,
        required=True,
        min_value=0,
    )

    max_missing = get_cfg_float(
        ctx,
        "pipeline.data_quality.readiness.max_label_missing_pct",
        tag=TAG25,
        default=1.0,
        required=True,
        min_value=0.0,
        max_value=1.0,
    )

    min_pos = get_cfg_float(
        ctx,
        "pipeline.data_quality.readiness.min_positive_frac",
        tag=TAG25,
        default=0.0,
        required=True,
        min_value=0.0,
        max_value=1.0,
    )

    max_pos = get_cfg_float(
        ctx,
        "pipeline.data_quality.readiness.max_positive_frac",
        tag=TAG25,
        default=1.0,
        required=True,
        min_value=0.0,
        max_value=1.0,
    )

    pos_range_ok = (min_pos <= max_pos)

    n_rows = int(df.shape[0])

    # --- 5) Core workflow ---
    lg.info(f"[{TAG25}] 🚦 Starting experimental readiness check...")

    # Build ModuleReport skeleton (infra)
    report = build_module_report(
        ctx=ctx,
        stage="data_quality",
        step="2.5",
        name="readiness",
        tag=TAG25,
        cfg_key="pipeline.data_quality.readiness",
        enabled=True,
        df=df,
        run_id_override=rid,
    )

    report["inputs"] = {
        "df_provided": True,
    }

    report["thresholds_used"] = {
        "min_rows": int(min_rows),
        "max_label_missing_pct": float(max_missing),
        "min_positive_frac": float(min_pos),
        "max_positive_frac": float(max_pos),
    }

    report["used_config"] = {
        "task_type": task_type,
        "label_col": label_col,
        "positive_label": pos_label,
    }

    report["refs"]["self"] = make_artifact_ref(
        ctx=ctx,
        stage="data_quality",
        name="readiness",
        run_id=str(rid),
    )

    # --- Checks (facts only) ---
    report["checks"]["task_type"] = {
        "status": "pass" if task_type_ok else "fail",
        "task_type": task_type,
        "supported": list(_SUPPORTED_TASK_TYPES),
    }

    report["checks"]["positive_frac_range"] = {
        "status": "pass" if pos_range_ok else "fail",
        "min_positive_frac": float(min_pos),
        "max_positive_frac": float(max_pos),
    }

    size_ok = (n_rows >= min_rows)
    report["checks"]["sample_size"] = {
        "status": "pass" if size_ok else "fail",
        "min_rows_required": int(min_rows),
        "actual_rows": int(n_rows),
    }

    # Label checks: only meaningful when task_type supported AND range config valid
    if (not task_type_ok) or (not pos_range_ok):
        report["checks"]["label"] = {
            "status": "skipped",
            "reason": "unsupported_task_type" if (not task_type_ok) else "invalid_positive_frac_range",
            "label_col": label_col,
            "positive_label": pos_label,
        }
    else:
        if label_col not in df.columns:
            report["checks"]["label"] = {
                "status": "fail",
                "reason": "label_column_not_found",
                "label_col": label_col,
                "positive_label": pos_label,
            }
        else:
            s = df[label_col]

            # Treat blank/whitespace labels as missing (common real-world dirty data).
            try:
                blank_mask = (~s.isna()) & (s.astype(str).str.strip() == "")
            except Exception:
                blank_mask = pd.Series(False, index=s.index)

            missing_mask = s.isna() | blank_mask
            missing_pct = float(safe_float(missing_mask.mean(), ndigits=6) or 0.0)

            s_non_missing = s.loc[~missing_mask].astype(str).str.strip()
            vc = s_non_missing.value_counts()
            total_non_missing = int(vc.sum())

            if total_non_missing == 0:
                report["checks"]["label"] = {
                    "status": "fail",
                    "reason": "all_labels_missing",
                    "missing_pct": float(round(missing_pct, 6)),
                    "total_non_missing": 0,
                }
            else:
                class_counts = {str(k): int(v) for k, v in vc.to_dict().items()}
                n_classes = int(len(class_counts))

                pos_key = str(pos_label).strip()
                pos_count = int(vc.get(pos_key, 0))
                pos_frac = float(safe_float(pos_count / max(total_non_missing, 1), ndigits=6) or 0.0)

                reasons: List[str] = []
                if missing_pct > max_missing:
                    reasons.append("label_missing_too_high")
                if pos_count == 0:
                    reasons.append("positive_label_not_found")
                if pos_count > 0 and (pos_frac < min_pos or pos_frac > max_pos):
                    reasons.append("label_imbalance")

                label_ok = (len(reasons) == 0)

                report["checks"]["label"] = {
                    "status": "pass" if label_ok else "fail",
                    "label_col": label_col,
                    "positive_label": pos_label,
                    "missing_pct": float(round(missing_pct, 6)),
                    "total_non_missing": int(total_non_missing),
                    "n_classes": int(n_classes),
                    "class_counts": class_counts,
                    "positive_count": int(pos_count),
                    "positive_frac": float(round(pos_frac, 6)),
                    "reasons": reasons or ["ok"],
                }

    # --- Event hints (policy-free; for 2.8 mapping) ---
    ensure_event_hints(report, version=1)

    if not task_type_ok:
        add_event_hint(
            report,
            code="readiness_unsupported_task_type",
            severity_signal="hard",
            evidence_path="checks.task_type",
            context={"task_type": task_type, "supported": list(_SUPPORTED_TASK_TYPES)},
            remediation={
                "action": "update_dataset_label_task_type_or_use_task_specific_readiness",
                "safe": False,
                "rationale": "Readiness logic is task-specific; unsupported task_type cannot be evaluated safely here.",
                "post_check": "re-run 2.5 after SSOT fix or selecting correct readiness module",
            },
        )

    if not pos_range_ok:
        add_event_hint(
            report,
            code="readiness_invalid_positive_frac_range",
            severity_signal="hard",
            evidence_path="checks.positive_frac_range",
            context={"min_positive_frac": float(min_pos), "max_positive_frac": float(max_pos)},
            remediation={
                "action": "fix_readiness_thresholds_in_yaml",
                "safe": False,
                "rationale": "Invalid thresholds make readiness gating undefined; must be corrected in SSOT.",
                "post_check": "re-run 2.5 after YAML thresholds are corrected",
            },
        )

    if report["checks"]["sample_size"]["status"] == "fail":
        add_event_hint(
            report,
            code="readiness_insufficient_sample_size",
            severity_signal="hard",
            evidence_path="checks.sample_size",
            context={"min_rows_required": int(min_rows), "actual_rows": int(n_rows)},
            remediation={
                "action": "collect_more_data_or_reduce_min_rows_threshold",
                "safe": False,
                "rationale": "Insufficient sample size makes training/evaluation unreliable; cannot be fixed deterministically by cleaning.",
                "post_check": "re-run 2.5 after data/threshold adjustment",
            },
        )

    label_check = report["checks"].get("label", {}) or {}
    if label_check.get("status") == "fail":
        reason = str(label_check.get("reason", "")).strip() or "label_check_failed"
        sev = "hard" if reason in {"label_column_not_found", "all_labels_missing", "positive_label_not_found"} else "fixable"

        # Some "fixable" label issues are NOT safely auto-remediable (e.g., imbalance / too-missing).
        safe_auto = (sev != "hard")
        if reason in {"label_imbalance", "label_missing_too_high"}:
            safe_auto = False

        add_event_hint(
            report,
            code=f"readiness_label_{reason}",
            severity_signal=sev,
            evidence_path="checks.label",
            context={"label_col": label_col, "positive_label": pos_label, "detail": dict(label_check)},
            remediation={
                "action": (
                    "collect_more_data_or_adjust_label_policy"
                    if reason in {"label_missing_too_high"}
                    else "adjust_sampling_or_split_strategy"
                    if reason in {"label_imbalance"}
                    else "normalize_label_values_or_fix_schema_upstream"
                ),
                "safe": bool(safe_auto),
                "rationale": (
                    "Readiness depends on label availability and distribution. "
                    "Imbalance/missingness usually require data or strategy changes and are not safely auto-fixable."
                ),
                "post_check": "re-run 2.5 after remediation; confirm label balance meets thresholds",
            },
        )

    # --- 6) Status + gate-friendly summary ---
    issues: List[str] = []
    warnings: List[str] = []

    if report["checks"]["task_type"]["status"] == "fail":
        issues.append("unsupported_task_type")

    if report["checks"]["positive_frac_range"]["status"] == "fail":
        issues.append("invalid_positive_frac_range")

    if report["checks"]["sample_size"]["status"] == "fail":
        issues.append("insufficient_sample_size")

    if label_check.get("status") == "fail":
        r = str(label_check.get("reason", "")).strip()
        if r:
            issues.append(r)
        else:
            rs = label_check.get("reasons", [])
            if isinstance(rs, list) and rs:
                issues.extend([str(x).strip() for x in rs if str(x).strip() and str(x).strip() != "ok"])
            else:
                issues.append("label_check_failed")
    elif label_check.get("status") == "pass":
        if int(label_check.get("n_classes", 2)) != 2:
            warnings.append("non_binary_label_detected")

    overall = "fail" if issues else ("warn" if warnings else "pass")

    label_check_ok = (label_check.get("status") == "pass")
    _missing_pct = label_check.get("missing_pct", None)
    _n_classes = label_check.get("n_classes", None)
    _positive_frac = label_check.get("positive_frac", None)

    report = finalize_module_report(
        ctx=ctx,
        report=report,
        overall_status=overall,
        issues=sorted(set(issues)),
        warnings=sorted(set(warnings)),
        metrics={
            "n_rows": int(n_rows),
            "label_missing_pct": float(_missing_pct) if (label_check_ok and _missing_pct is not None) else -1.0,
            "n_classes": int(_n_classes) if (label_check_ok and _n_classes is not None) else -1,
            "positive_frac": float(_positive_frac) if (label_check_ok and _positive_frac is not None) else -1.0,
        },
        notes=[
            f"task_type={task_type}",
            f"label_col={label_col}",
            f"positive_label={pos_label}",
            f"task_type_supported={task_type_ok}",
            f"pos_frac_range_ok={pos_range_ok}",
            "metrics_sentinel=-1 => not_available (only when label_check.status!=pass or value is None)",
        ],
        df=df,
    )

    if overall == "pass":
        lg.info(f"[{TAG25}] ✅ Experimental readiness checks pass.")
    elif overall == "warn":
        lg.warning(f"[{TAG25}] ⚠ Experimental readiness checks warn. | warnings={sorted(set(warnings))}")
    else:
        lg.error(f"[{TAG25}] ❌ Experimental readiness checks failed. | issues={sorted(set(issues))}")

    # --- 7) Persist Artifact ---
    out_path = resolve_artifact_path(
        ctx=ctx,
        stage="data_quality",
        name="readiness",
        run_id=str(rid),
        tag=TAG25,
    )
    write_json(out_path, report, tag=TAG25, indent=2)
    lg.info(f"[{TAG25}] 🧾 Readiness artifact saved: {out_path}")
    
    return report

In [7]:
# --- 2.6 Classification Domain Diagnostics (Core + Addon) ---
# --- 2.6.1 Classification Core ---
#
# What:
#   Task-agnostic, classification-level domain diagnostics for binary classification datasets.
#
# How:
#   - Read label + schema from YAML SSOT via ctx
#   - Run numeric / categorical sanity checks
#   - Slice positive rate by segment columns
#   - Optional correlation-based leakage pre-scan (soft signal)
#   - Persist a ModuleReport v1 artifact (SSOT naming)
#
# Why:
#   Reusable "business-shaped" diagnostics before modeling.

# --- 0) Imports + TAG ---
from typing import Any, Dict, List, Optional

import pandas as pd
import numpy as np

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint
from src.pipeline.data_quality.domain_utils import rate_by_segment, sample_str_list
from src.utils.artifact_utils import resolve_artifact_path, write_json
from src.utils.cfg_utils import get_cfg_dict, get_cfg_int, get_cfg_float, get_cfg_str_list
from src.utils.dataset_utils import get_label_cfg, get_schema_cols
from src.utils.pandas_utils import safe_num, normalize_str_series
from src.utils.serialization_utils import safe_float

TAG26_1 = "DOMAIN_CORE"

_SUPPORTED_TASK_TYPES: tuple[str, ...] = ("classification",)

# --- Internal helpers ---
def _keyify(v: Any) -> str:
    """Convert values to stable string keys for reports (NA -> '<MISSING>')."""
    return "<MISSING>" if pd.isna(v) else str(v)


def _describe_numeric_domain(
    df: pd.DataFrame,
    cols: List[str],
) -> Dict[str, Any]:
    """
    Numeric sanity checks from a domain view:
    - missingness
    - min / max / median / quantiles
    - rough outlier rate via IQR rule (on non-missing values)
    """
    summary: Dict[str, Any] = {}
    n_rows = int(df.shape[0])

    for c in cols:
        if c not in df.columns:
            summary[c] = {"exists": False, "reason": "column_not_found"}
            continue

        x = safe_num(df[c])
        n_non_missing = int(x.notna().sum())
        miss_pct = float(x.isna().mean()) if n_rows > 0 else 0.0

        if n_non_missing == 0:
            summary[c] = {
                "exists": True,
                "missing_pct": float(round(miss_pct, 6)),
                "min": None,
                "max": None,
                "median": None,
                "q25": None,
                "q75": None,
                "q99": None,
                "iqr_outlier_rate": None,
                "note": "all_missing_or_non_numeric"
            }
            continue

        q25 = float(x.quantile(0.25))
        q50 = float(x.quantile(0.50))
        q75 = float(x.quantile(0.75))
        q99 = float(x.quantile(0.99))

        iqr = q75 - q25
        if (not np.isfinite(iqr)) or iqr == 0.0:
            outlier_rate = 0.0
            note = "iqr_zero_or_invalid"
        else:
            lower = q25 - 1.5 * iqr
            upper = q75 + 1.5 * iqr
            mask = x.notna() & ((x < lower) | (x > upper))
            outlier_rate = float(mask.sum() / max(n_non_missing, 1))
            note = "ok"

        summary[c] = {
            "exists": True,
            "missing_pct": float(round(miss_pct, 6)),
            "min": safe_float(x.min(skipna=True), ndigits=6),
            "max": safe_float(x.max(skipna=True), ndigits=6),
            "median": safe_float(q50, ndigits=6),
            "q25": safe_float(q25, ndigits=6),
            "q75": safe_float(q75, ndigits=6),
            "q99": safe_float(q99, ndigits=6),
            "iqr_outlier_rate": float(round(outlier_rate, 6)),
            "n_non_missing": int(n_non_missing),
            "note": note,
        }

    return summary


def _categorical_domain_sanity(
    df: pd.DataFrame,
    cols: List[str],
    *,
    rare_segment_min_share: float,
    top_k: int
) -> Dict[str, Any]:
    """
    Categorical sanity checks:
    - top-k value counts
    - rare categories (share < rare_segment_min_share)
    """
    summary: Dict[str, Any] = {}
    n_rows = int(df.shape[0])
    k = max(int(top_k), 1)

    for c in cols:
        if c not in df.columns:
            summary[c] = {"exists": False, "reason": "column_not_found"}
            continue

        s = normalize_str_series(df[c])
        missing_n = int(s.isna().sum())

        vc_valid = s.dropna().value_counts()
        n_unique = int(vc_valid.shape[0])

        total_valid = int(s.notna().sum())
        if total_valid > 0:
            share_valid = (vc_valid / float(total_valid))
            rare_vals = [_keyify(x) for x in share_valid[share_valid < rare_segment_min_share].index.tolist()]
        else:
            rare_vals = []

        top = vc_valid.head(k).to_dict()

        summary[c] = {
            "exists": True,
            "n_unique": int(n_unique),
            "missing_count": int(missing_n),
            "missing_share": float(round(missing_n / n_rows, 6)) if n_rows > 0 else 0.0,
            "top_values": {_keyify(k2): int(v2) for k2, v2 in top.items()},
            "rare_values_lt_min_share": rare_vals,
            "rare_min_share": float(rare_segment_min_share),
        }

    return summary


def _scan_potential_leakage_generic(
    df: pd.DataFrame,
    *,
    label_col: str,
    positive_label: str,
    candidates: List[str],
    corr_threshold: float,
    min_non_missing: int,
) -> Dict[str, Any]:
    """
    Correlation-based leakage pre-scan (soft signal).
    Flags numeric columns with abs(corr) >= corr_threshold.

    Important:
      - Do NOT treat missing labels as negative.
      - Compute correlation on rows with non-missing labels only.
    """
    if not candidates:
        return {"note": "no_candidates_provided"}

    if label_col not in df.columns:
        return {"error": f"label_col '{label_col}' not found"}

    s_label = df[label_col].astype(object)
    valid = s_label.notna()
    if int(valid.sum()) == 0:
        return {"note": "all_labels_missing; correlation_scan_skipped"}

    y = (s_label[valid].astype(str).str.strip() == positive_label).astype(float).to_numpy()
    if float(np.nanstd(y)) == 0.0:
        return {"note": "label_is_constant; correlation_scan_skipped"}

    out: Dict[str, Any] = {}
    for c in candidates:
        if c not in df.columns:
            out[c] = {"exists": False, "flagged": False, "reason": "column_not_found"}
            continue

        x = safe_num(df[c])[valid]
        n_ok = int(x.notna().sum())
        if n_ok < int(min_non_missing):
            out[c] = {
                "exists": True,
                "flagged": False,
                "reason": "insufficient_non_missing_values",
                "n_non_missing": int(n_ok),
                "min_required": int(min_non_missing),
            }
            continue

        # Fill missing with median (computed on valid rows only)
        med = float(x.median(skipna=True))
        x_filled = x.fillna(med).to_numpy()

        if float(np.nanstd(x_filled)) == 0.0:
            out[c] = {"exists": True, "flagged": False, "reason": "feature_is_constant"}
            continue

        corr = float(np.corrcoef(x_filled, y)[0, 1])
        corr_safe = corr if np.isfinite(corr) else None
        flagged = bool(corr_safe is not None and abs(corr_safe) >= float(corr_threshold))

        out[c] = {
            "exists": True,
            "flagged": bool(flagged),
            "corr_with_label": corr_safe,
            "threshold": float(corr_threshold),
            "n_rows_used": int(len(y)),
        }

    return out


# --- 1) Public API ---
def run_domain_diagnostics_classification_core(
    *,
    df: pd.DataFrame,
    ctx: Dict[str, Any],
    run_id: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Run classification core diagnostics in a ctx-driven, artifact-persisted manner.

    YAML SSOT
    ---------
    dataset.label:
      col: "Churn"
      positive: "Yes"

    dataset.schema.dtypes:
      numeric: [...]
      categorical: [...]

    pipeline.data_quality.domain_diagnostics.core:
      segment_cols: [...]                 # optional; defaults to categorical cols
      rare_segment_min_share: 0.01
      categorical_top_k: 10
      leakage:
        candidates: []
        correlation_threshold: 0.90
        min_non_missing: 10

    Returns
    -------
    dict
        ModuleReport v1 (Domain Diagnostics Core) (also persisted as artifact).
    """
    # --- 2) Runtime validation ---
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG26_1, run_id=run_id)

    # --- 3) Input validation ---
    if df is None or not isinstance(df, pd.DataFrame):
        raise TypeError(f"[{TAG26_1}] df must be a pandas DataFrame")

    # --- 4) Resolve config (YAML SSOT) ---

    # --- Global（SSOT） ---
    label_col, pos_label, task_type = get_label_cfg(ctx, TAG26_1)

    tt = str(task_type).strip().lower()
    task_type_ok = (tt in _SUPPORTED_TASK_TYPES)

    numeric_cols, categorical_cols = get_schema_cols(ctx, TAG26_1)

    # --- Module（step cfg） ---
    get_cfg_dict(ctx, "pipeline.data_quality.domain_diagnostics.core", tag=TAG26_1, required=True)

    segment_cols = get_cfg_str_list(
        ctx,
        "pipeline.data_quality.domain_diagnostics.core.segment_cols",
        tag=TAG26_1,
        default=list(categorical_cols),
        required=False,
        dedupe=True,
        allow_empty=False,
        min_len=1,
        strict_items=True,
    )

    rare_segment_min_share = get_cfg_float(
        ctx,
        "pipeline.data_quality.domain_diagnostics.core.rare_segment_min_share",
        tag=TAG26_1,
        default=0.01,
        required=False,
        min_value=0.0,
        max_value=1.0,
        strict_type=True,
    )

    cat_top_k = get_cfg_int(
        ctx,
        "pipeline.data_quality.domain_diagnostics.core.categorical_top_k",
        tag=TAG26_1,
        default=10,
        required=False,
        min_value=1,
        strict_type=True,
    )

    leakage_candidates = get_cfg_str_list(
        ctx,
        "pipeline.data_quality.domain_diagnostics.core.leakage.candidates",
        tag=TAG26_1,
        default=[],
        required=False,
        dedupe=True,
        allow_empty=True,
        min_len=0,
        strict_items=True,
    )

    corr_threshold = get_cfg_float(
        ctx,
        "pipeline.data_quality.domain_diagnostics.core.leakage.correlation_threshold",
        tag=TAG26_1,
        default=0.90,
        required=False,
        min_value=0.0,
        max_value=1.0,
        strict_type=True,
    )

    min_non_missing = get_cfg_int(
        ctx,
        "pipeline.data_quality.domain_diagnostics.core.leakage.min_non_missing",
        tag=TAG26_1,
        default=10,
        required=False,
        min_value=0,
        max_value=None,
        strict_type=True,
    )

    pos_key = str(pos_label).strip()
    n_rows = int(df.shape[0])

    missing_seg = sorted([c for c in segment_cols if c not in df.columns])
    label_in_segments = (label_col in segment_cols)

    missing_leak = sorted([c for c in leakage_candidates if c not in df.columns])
    min_non_missing_ok = (int(min_non_missing) <= int(n_rows))

    # --- 5) Core workflow ---
    lg.info(
        f"[{TAG26_1}] 🧠 Starting classification core diagnostics... "
        f"(segments={len(segment_cols)})"
    )

    # --- Build ModuleReport skeleton (infra) ---
    report = build_module_report(
        ctx=ctx,
        stage="data_quality",
        step="2.6.1",
        name="domain_core",
        tag=TAG26_1,
        cfg_key="pipeline.data_quality.domain_diagnostics.core",
        enabled=True,
        df=df,
        run_id_override=rid,
    )

    report["inputs"] = {"df_provided": True}

    report["thresholds_used"] = {
        "rare_segment_min_share": float(rare_segment_min_share),
        "categorical_top_k": int(cat_top_k),
        "correlation_threshold": float(corr_threshold),
        "min_non_missing": int(min_non_missing),
        "supported_task_types": list(_SUPPORTED_TASK_TYPES),
    }

    report["used_config"] = {
        "label_col": label_col,
        "positive_label": pos_label,
        "task_type_raw": task_type,
        "task_type_norm": tt,
        "segment_cols": list(segment_cols),
        "leakage_candidates": list(leakage_candidates),
        "numeric_cols": list(numeric_cols),
        "categorical_cols": list(categorical_cols),
    }

    report["refs"]["self"] = make_artifact_ref(
        ctx=ctx,
        stage="data_quality",
        name="domain_core",
        run_id=str(rid),
    )

    # --- Preconditions (facts only) ---
    report["checks"]["preconditions"] = {
        "task_type": {
            "status": "pass" if task_type_ok else "fail",
            "task_type": task_type,
            "expected": list(_SUPPORTED_TASK_TYPES),
        },
        "segment_cols": {
            "status": "pass" if (not missing_seg and (not label_in_segments)) else "fail",
            "n_segment_cols": int(len(segment_cols)),
            "missing_segment_cols": missing_seg,
            "label_in_segments": bool(label_in_segments),
        },
        "leakage_candidates": {
            "status": "pass" if (not missing_leak and min_non_missing_ok) else "warn",
            "n_candidates": int(len(leakage_candidates)),
            "missing_candidates": missing_leak,
            "min_non_missing": int(min_non_missing),
            "n_rows": int(n_rows),
            "min_non_missing_ok": bool(min_non_missing_ok),
        },
    }

    is_empty_df = (n_rows == 0)
    label_missing = (label_col not in df.columns)

    # --- Schema presence (soft signals) ---
    missing_numeric = sorted([c for c in numeric_cols if c not in df.columns])
    missing_categorical = sorted([c for c in categorical_cols if c not in df.columns])

    report["checks"]["schema_presence"] = {
        "missing_numeric_cols": missing_numeric,
        "missing_categorical_cols": missing_categorical,
    }

    # --- Numeric / categorical sanity (still useful even when some preconditions fail) ---
    report["checks"]["numeric_domain_sanity"] = _describe_numeric_domain(df, list(numeric_cols))
    report["checks"]["categorical_domain_sanity"] = _categorical_domain_sanity(
        df,
        list(categorical_cols),
        rare_segment_min_share=float(rare_segment_min_share),
        top_k=int(cat_top_k),
    )

    # --- Positive-rate distribution by segment (soft signal) ---
    rate_by_seg: Dict[str, Any] = {}
    rate_by_seg_failed = False
    many_small_segments = False

    seg_ok = (report["checks"]["preconditions"]["segment_cols"]["status"] == "pass")

    if label_missing or is_empty_df or (not seg_ok) or (not task_type_ok):
        rate_by_seg = {
            "error": "label_missing_or_empty_df_or_invalid_segments_or_task_type",
            "label_col": label_col,
            "n_rows": n_rows,
            "task_type_ok": bool(task_type_ok),
            "segments_ok": bool(seg_ok),
            "missing_segment_cols": missing_seg,
            "label_in_segments": bool(label_in_segments),
        }
        rate_by_seg_failed = True
    else:
        rate_by_seg = rate_by_segment(
            df,
            label_col=label_col,
            positive_label=pos_key,
            segment_cols=list(segment_cols),
            small_segment_min_share=float(rare_segment_min_share),
        )

        if isinstance(rate_by_seg, dict) and ("error" in rate_by_seg):
            rate_by_seg_failed = True
        else:
            small_count = 0
            total_levels = 0
            for _, info in (rate_by_seg or {}).items():
                if not isinstance(info, dict) or not info.get("exists"):
                    continue
                segs = info.get("segments", {})
                if isinstance(segs, dict):
                    for _, s in segs.items():
                        total_levels += 1
                        if isinstance(s, dict) and s.get("small_segment") is True:
                            small_count += 1
            if total_levels > 0 and (small_count / total_levels) > 0.30:
                many_small_segments = True

    report["checks"]["positive_rate_by_segment"] = rate_by_seg

    # --- Optional leakage scan (soft signal) ---
    flagged_feats: List[str] = []
    leak_scan: Dict[str, Any] = {}

    leak_cfg_ok = (len(missing_leak) == 0) and bool(min_non_missing_ok)

    if label_missing or is_empty_df or (not task_type_ok):
        leak_scan = {"note": "label_missing_or_empty_df_or_task_type_mismatch"}
    else:
        if isinstance(leakage_candidates, list) and leakage_candidates:
            if not leak_cfg_ok:
                leak_scan = {
                    "note": "invalid_leakage_config; scan_skipped",
                    "missing_candidates": missing_leak,
                    "min_non_missing": int(min_non_missing),
                    "n_rows": int(n_rows),
                }
            else:
                leak_scan = _scan_potential_leakage_generic(
                    df,
                    label_col=label_col,
                    positive_label=pos_key,
                    candidates=list(leakage_candidates),
                    corr_threshold=float(corr_threshold),
                    min_non_missing=int(min_non_missing),
                )
                if isinstance(leak_scan, dict):
                    flagged_feats = [
                        k for k, v in leak_scan.items()
                        if isinstance(v, dict) and v.get("flagged") is True
                    ]
                    if flagged_feats:
                        leak_scan["flagged_features"] = flagged_feats
        else:
            leak_scan = {"note": "no_candidates_configured"}

    report["checks"]["generic_leakage_scan"] = leak_scan

    # --- Event hints (policy-free; for 2.8 mapping) ---
    ensure_event_hints(report, version=1)

    # 1) Task type mismatch (hard)
    if not task_type_ok:
        add_event_hint(
            report,
            code="domain_core_task_type_mismatch",
            severity_signal="hard",
            evidence_path="checks.preconditions.task_type",
            context={
                "task_type_raw": str(task_type),
                "task_type_norm": str(tt),
                "supported": list(_SUPPORTED_TASK_TYPES),
            },
            remediation={
                "action": "fix_dataset_label_task_type_in_yaml_or_use_correct_domain_module",
                "safe": False,
                "rationale": "Domain core 2.6.1 is designed for binary classification; task mismatch makes outputs unreliable.",
                "post_check": "re-run 2.6.1 after SSOT fix",
            },
        )

    # 2) Segment cols invalid (hard)
    if missing_seg:
        add_event_hint(
            report,
            code="domain_core_missing_segment_cols",
            severity_signal="hard",
            evidence_path="checks.preconditions.segment_cols.missing_segment_cols",
            context={
                "n_missing": int(len(missing_seg)),
                "missing_segment_cols": sample_str_list(missing_seg, limit=10),
                "n_segment_cols": int(len(segment_cols)),
            },
            remediation={
                "action": "fix_segment_cols_config_or_update_schema",
                "safe": False,
                "rationale": "Segment diagnostics require configured columns; missing columns indicate SSOT/schema drift.",
                "post_check": "re-run 2.6.1 after segment_cols are valid",
            },
        )

    if label_in_segments:
        add_event_hint(
            report,
            code="domain_core_label_in_segment_cols",
            severity_signal="hard",
            evidence_path="checks.preconditions.segment_cols.label_in_segments",
            context={
                "label_col": str(label_col),
                "n_segment_cols": int(len(segment_cols)),
            },
            remediation={
                "action": "remove_label_col_from_segment_cols",
                "safe": True,
                "rationale": "Segmenting by label is tautological and breaks diagnostics interpretation.",
                "post_check": "re-run 2.6.1 after config change",
            },
        )

    # 3) Empty df / missing label (hard)
    if is_empty_df:
        add_event_hint(
            report,
            code="domain_core_empty_dataframe",
            severity_signal="hard",
            evidence_path="summary.total_rows",
            context={"n_rows": int(n_rows)},
            remediation={
                "action": "fix_ingestion_or_upstream_filters",
                "safe": False,
                "rationale": "Domain diagnostics cannot run on an empty dataframe.",
                "post_check": "re-run 2.1 ingestion and then 2.6.1",
            },
        )

    if label_missing:
        add_event_hint(
            report,
            code="domain_core_label_col_missing",
            severity_signal="hard",
            evidence_path="used_config.label_col",
            context={"label_col": str(label_col)},
            remediation={
                "action": "fix_schema_or_dataset_source_to_include_label",
                "safe": False,
                "rationale": "Label column is required for positive-rate diagnostics and leakage pre-scan.",
                "post_check": "re-run 2.2 contract + 2.6.1 after upstream fix",
            },
        )

    # 4) Schema drift (risk)
    if missing_numeric:
        add_event_hint(
            report,
            code="domain_core_missing_numeric_schema_cols_in_df",
            severity_signal="risk",
            evidence_path="checks.schema_presence.missing_numeric_cols",
            context={
                "n_missing": int(len(missing_numeric)),
                "examples": sample_str_list(missing_numeric, limit=10),
            },
            remediation={
                "action": "align_schema_or_update_dataset_schema_numeric_list",
                "safe": True,
                "rationale": "Schema drift reduces diagnostics coverage; alignment is deterministic via SSOT update.",
                "post_check": "re-run 2.2 contract and 2.6.1 to confirm schema presence",
            },
        )

    if missing_categorical:
        add_event_hint(
            report,
            code="domain_core_missing_categorical_schema_cols_in_df",
            severity_signal="risk",
            evidence_path="checks.schema_presence.missing_categorical_cols",
            context={
                "n_missing": int(len(missing_categorical)),
                "examples": sample_str_list(missing_categorical, limit=10),
            },
            remediation={
                "action": "align_schema_or_update_dataset_schema_categorical_list",
                "safe": True,
                "rationale": "Schema drift reduces diagnostics coverage; alignment is deterministic via SSOT update.",
                "post_check": "re-run 2.2 contract and 2.6.1 to confirm schema presence",
            },
        )

    # 5) Rate-by-segment issues (risk)
    if rate_by_seg_failed:
        add_event_hint(
            report,
            code="domain_core_positive_rate_by_segment_failed_or_skipped",
            severity_signal="risk",
            evidence_path="checks.positive_rate_by_segment",
            context={
                "task_type_ok": bool(task_type_ok),
                "segments_ok": bool(seg_ok),
                "label_missing": bool(label_missing),
                "empty_df": bool(is_empty_df),
                "n_segment_cols": int(len(segment_cols)),
                "n_missing_segment_cols": int(len(missing_seg)),
                "label_in_segments": bool(label_in_segments),
            },
            remediation={
                "action": "fix_preconditions_and_rerun_segment_diagnostics",
                "safe": True,
                "rationale": "Segment diagnostics are soft signals but valuable; fix inputs/config to restore coverage.",
                "post_check": "re-run 2.6.1 after resolving label/segments/task_type issues",
            },
        )

    if many_small_segments:
        add_event_hint(
            report,
            code="domain_core_many_small_segments_detected",
            severity_signal="risk",
            evidence_path="checks.positive_rate_by_segment",
            context={
                "rare_segment_min_share": float(rare_segment_min_share),
                "heuristic": "small_segments_ratio_gt_0.30",
            },
            remediation={
                "action": "bucket_rare_categories_or_adjust_rare_segment_threshold",
                "safe": True,
                "rationale": "Too many small segments makes segment rates unstable; bucketing is deterministic when configured.",
                "post_check": "re-run 2.6.1 after cleaning/bucketing to confirm segment stability",
            },
        )

    # 6) Leakage config issues (fixable) + leakage flagged (risk)
    if missing_leak:
        add_event_hint(
            report,
            code="domain_core_leakage_candidates_missing_in_df",
            severity_signal="fixable",
            evidence_path="checks.preconditions.leakage_candidates.missing_candidates",
            context={
                "n_missing": int(len(missing_leak)),
                "missing_candidates": sample_str_list(missing_leak, limit=10),
                "n_configured_candidates": int(len(leakage_candidates)),
            },
            remediation={
                "action": "remove_missing_candidates_or_fix_schema",
                "safe": True,
                "rationale": "Leakage scan is optional; removing invalid candidates is deterministic and restores stability.",
                "post_check": "re-run 2.6.1 and confirm leakage scan executes or is cleanly skipped",
            },
        )

    if (not min_non_missing_ok) and isinstance(leakage_candidates, list) and leakage_candidates:
        add_event_hint(
            report,
            code="domain_core_leakage_min_non_missing_exceeds_n_rows",
            severity_signal="fixable",
            evidence_path="checks.preconditions.leakage_candidates",
            context={
                "min_non_missing": int(min_non_missing),
                "n_rows": int(n_rows),
            },
            remediation={
                "action": "lower_min_non_missing_threshold",
                "safe": True,
                "rationale": "Threshold misconfiguration only affects leakage scan; adjusting it is deterministic.",
                "post_check": "re-run 2.6.1 and confirm leakage scan runs",
            },
        )

    if flagged_feats:
        add_event_hint(
            report,
            code="domain_core_potential_leakage_features_flagged",
            severity_signal="risk",
            evidence_path="checks.generic_leakage_scan.flagged_features",
            context={
                "n_flagged": int(len(flagged_feats)),
                "flagged_features": sample_str_list(flagged_feats, limit=10),
                "corr_threshold": float(corr_threshold),
            },
            remediation={
                "action": "investigate_feature_lineage_and_remove_or_transform_leaky_features",
                "safe": True,
                "rationale": "High correlation is a soft pre-scan; treat as investigation trigger to avoid leakage.",
                "post_check": "re-run 3.x leakage_scan and confirm no leakage remains",
            },
        )


    # --- 6) Status + gate-friendly summary ---
    issues: List[str] = []
    warnings: List[str] = []

    # Hard-ish module viability
    if is_empty_df:
        issues.append("empty_dataframe")
    if label_missing:
        issues.append("label_col_missing")
    if not task_type_ok:
        issues.append("task_type_mismatch")
    if missing_seg or label_in_segments:
        issues.append("invalid_segment_cols_config")

    # Soft signals (schema drift / diagnostics health)
    if missing_numeric:
        warnings.append("numeric_schema_cols_missing_in_df")
    if missing_categorical:
        warnings.append("categorical_schema_cols_missing_in_df")
    if rate_by_seg_failed:
        warnings.append("rate_by_segment_failed_or_skipped")
    if many_small_segments:
        warnings.append("many_small_segments_detected")
    if flagged_feats:
        warnings.append("potential_leakage_features_flagged")
    if missing_leak:
        warnings.append("leakage_candidates_missing_in_df")
    if (not min_non_missing_ok) and isinstance(leakage_candidates, list) and leakage_candidates:
        warnings.append("leakage_min_non_missing_exceeds_n_rows")

    _metrics = {
        "n_missing_numeric_schema_cols": int(len(missing_numeric)),
        "n_missing_categorical_schema_cols": int(len(missing_categorical)),
        "n_segment_cols": int(len(segment_cols)),
        "n_leakage_candidates": int(len(leakage_candidates)) if isinstance(leakage_candidates, list) else 0,
        "n_flagged_leakage_features": int(len(flagged_feats)),
        "n_missing_leakage_candidates": int(len(missing_leak)),
    }

    _notes = [f"task_type={task_type}", f"label_col={label_col}", f"positive_label={pos_key}"]

    overall = "fail" if issues else ("warn" if warnings else "pass")

    report = finalize_module_report(
        ctx=ctx,
        report=report,
        overall_status=overall,
        issues=sorted(set(issues)),
        warnings=sorted(set(warnings)),
        metrics=_metrics,
        notes=_notes,
        df=df,
    )

    if overall == "pass":
        lg.info(f"[{TAG26_1}] ✅ Domain core diagnostics completed.")
    elif overall == "warn":
        lg.warning(f"[{TAG26_1}] ⚠ Domain core diagnostics completed with warnings: {sorted(set(warnings))}")
    else:
        lg.error(f"[{TAG26_1}] ❌ Domain core diagnostics failed. | issues={sorted(set(issues))}")

    # --- 7) Persist Artifact ---
    out_path = resolve_artifact_path(
        ctx=ctx,
        stage="data_quality",
        name="domain_core",
        run_id=str(rid),
        tag=TAG26_1,
    )
    write_json(out_path, report, tag=TAG26_1, indent=2)
    lg.info(f"[{TAG26_1}] 🧾 Core diagnostics artifact saved: {out_path}")

    return report

In [8]:
# --- 2.6 Classification Domain Diagnostics (Core + Addon) ---
# --- 2.6.2 Telco Churn Addon ---
#
# What:
#   Telco churn–specific extension on top of 2.6.1 classification core.
#
# How:
#   - Build lifecycle / pricing buckets (tenure + monthly charge quantiles)
#   - Slice churn rate by Telco segments + buckets
#   - Domain-driven leakage hints (TotalCharges ≈ MonthlyCharges * tenure)
#   - Anomaly patterns + optional diagnostic plots
#   - Persist JSON + plots (ctx/YAML SSOT)
#
# Why:
#   Capture domain-specific churn patterns and lifecycle structure
#   that drive feature design and model assumptions.

# --- 0) Import + TAG ---
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.data_quality.domain_utils import rate_by_segment, sample_str_list
from src.pipeline.event_hints import ensure_event_hints, add_event_hint
from src.utils.artifact_utils import resolve_artifact_path, write_json, read_json_if_exists
from src.utils.cfg_utils import get_cfg_dict, get_cfg_float, get_cfg_int, get_cfg_str_list, get_cfg_float_list, get_cfg_bool, get_cfg_value
from src.utils.dataset_utils import get_label_cfg
from src.utils.pandas_utils import safe_num, normalize_str_series
from src.utils.serialization_utils import safe_float
from src.utils.path_utils import resolve_figures_dir
from src.utils.plot_utils import plot_histogram, plot_bar_mean

TAG26_2 = "DOMAIN_TELCO_CHURN"
_SUPPORTED_TASK_TYPES: tuple[str, ...] = ("classification",)


# --- Internal helpers ---
def _build_business_segments_telco(
    df: pd.DataFrame,
    *,
    tenure_bins: List[float],
    tenure_bin_labels: List[str],
    monthly_charge_quantiles: List[float],
) -> pd.DataFrame:
    """
    Create Telco churn business buckets (no cleaning):
      - tenure_bucket (fixed bins)
      - monthly_charge_bucket (quantile bins)
    """
    out = df.copy()

    # Tenure bucket
    if "tenure" in out.columns:
        ten = safe_num(out["tenure"])
        out["tenure_bucket"] = pd.cut(
            ten,
            bins=list(tenure_bins),
            labels=list(tenure_bin_labels),
            include_lowest=True,
        )
    else:
        out["tenure_bucket"] = pd.NA

    # MonthlyCharges quantile bucket
    if "MonthlyCharges" in out.columns:
        mc = safe_num(out["MonthlyCharges"])
        qs = mc.quantile(list(monthly_charge_quantiles)).tolist()
        qs = sorted(list(set([float(x) for x in qs if pd.notna(x)])))

        # pd.cut requires strictly increasing bin edges
        if len(qs) >= 3:
            labels = [f"Q{i+1}" for i in range(len(qs) - 1)]
            out["monthly_charge_bucket"] = pd.cut(
                mc,
                bins=qs,
                labels=labels,
                include_lowest=True,
            )
        else:
            out["monthly_charge_bucket"] = pd.NA
    else:
        out["monthly_charge_bucket"] = pd.NA

    return out


def _pre_scan_leakage_telco(
    df: pd.DataFrame,
    *,
    candidates: List[str],
    label_col: str,
    positive_label: str,
    ratio_band: List[float],
    corr_threshold: float,
    min_non_missing: int,
    action_map: Dict[str, Any],
) -> Dict[str, Any]:
    """
    Telco leakage scan (soft signals):
      - TotalCharges ≈ MonthlyCharges × tenure (median ratio within band)
      - optional abs(corr) with label >= threshold (numeric corr)

    Policy:
      - Do NOT treat missing labels as negative. Exclude missing labels from corr calculation.
    """
    out: Dict[str, Any] = {"features": {}}

    if not candidates:
        return {"note": "no_candidates_provided"}

    ratio_low, ratio_high = float(ratio_band[0]), float(ratio_band[1])

    # Label for corr (exclude missing labels)
    y = None
    valid = None
    pos_key = str(positive_label).strip()

    if label_col in df.columns:
        y_all = normalize_str_series(df[label_col])
        valid = y_all.notna()
        if int(valid.sum()) <= 0:
            return {"error": "all_labels_missing"}
        if int(valid.sum()) > 0:
            y = (y_all[valid] == pos_key).astype(float).to_numpy()
            if np.nanstd(y) == 0.0:
                y = None
        else:
            y = None

    for c in candidates:
        if c not in df.columns:
            out["features"][c] = {
                "exists": False,
                "flagged": False,
                "risk_type": None,
                "derived_feature_flagged": False,
                "recommended_actions": [],
                "reasons": ["column_not_found"],
                "details": {},
            }
            continue

        reasons: List[str] = []
        details: Dict[str, Any] = {}

        # Domain-specific structural hint
        if c == "TotalCharges" and {"MonthlyCharges", "tenure"}.issubset(df.columns):
            mc = safe_num(df["MonthlyCharges"])
            ten = safe_num(df["tenure"])
            tc = safe_num(df[c])

            approx = mc * ten
            ratio_series = tc / approx.replace(0, np.nan)
            ratio_med = safe_float(ratio_series.median(skipna=True))
            details["median_totalcharges_over_monthly_x_tenure"] = ratio_med

            if ratio_med is not None and np.isfinite(ratio_med) and (ratio_low <= ratio_med <= ratio_high):
                reasons.append("approx_linear_function_of_monthlycharges_x_tenure")

        # Optional correlation with label (soft)
        x = safe_num(df[c])
        if valid is not None:
            x = x[valid]

        n_ok = int(x.notna().sum())
        details["n_non_missing"] = n_ok

        if y is not None and n_ok >= int(min_non_missing):
            x_filled = x.fillna(x.median()).to_numpy()
            if np.nanstd(x_filled) == 0.0:
                details["corr_with_label"] = None
                details["corr_note"] = "feature_is_constant"
            else:
                corr = float(np.corrcoef(x_filled, y)[0, 1])
                corr_safe = corr if np.isfinite(corr) else None
                details["corr_with_label"] = corr_safe

                if corr_safe is not None and abs(corr_safe) >= float(corr_threshold):
                    reasons.append(f"high_abs_correlation_with_label({corr_safe:.3f})")
        elif y is not None and n_ok < int(min_non_missing):
            details["corr_note"] = "insufficient_non_missing_values_for_corr"
        else:
            details["corr_note"] = "label_missing_or_constant; correlation_skipped"

        flagged = bool(reasons)

        rec_actions = action_map.get(c)
        if flagged:
            if not isinstance(rec_actions, list) or not rec_actions:
                rec_actions = ["check_multicollinearity", "drop_or_regularize"]
            else:
                rec_actions = [str(x) for x in rec_actions]
        else:
            rec_actions = []

        out["features"][c] = {
            "exists": True,
            "flagged": bool(flagged),
            "risk_type": "derived_feature_risk" if flagged else None,  
            "derived_feature_flagged": bool(flagged),                  
            "recommended_actions": rec_actions,                        
            "reasons": reasons or ["none"],
            "details": details,
        }

    return out


def _domain_anomaly_checks_telco(df: pd.DataFrame) -> Dict[str, Any]:
    """Telco churn anomaly patterns driven by business logic (soft diagnostics)."""
    out: Dict[str, Any] = {}

    if {"tenure", "TotalCharges"}.issubset(df.columns):
        ten = safe_num(df["tenure"])
        tc = safe_num(df["TotalCharges"])
        mask = (ten <= 1) & (tc > 5)
        out["tenure_short_but_high_totalcharges"] = {
            "n": int(mask.sum()),
            "pct": float(round(float(mask.mean()) if int(df.shape[0]) > 0 else 0.0, 6)),
            "note": "Very new customers should not accumulate high TotalCharges.",
        }

    if {"MonthlyCharges", "InternetService"}.issubset(df.columns):
        mc = safe_num(df["MonthlyCharges"])
        isvc = df["InternetService"].astype(str)
        mask = (mc == 0) & (isvc != "No")
        out["monthly_zero_but_has_internet"] = {
            "n": int(mask.sum()),
            "pct": float(round(float(mask.mean()) if int(df.shape[0]) > 0 else 0.0, 6)),
            "note": "Investigate subsidies, billing issues or data errors.",
        }

    return out


def _telco_domain_hypotheses() -> List[Dict[str, str]]:
    """Hypotheses to revisit later in 2.7/2.8 and modeling."""
    return [
        {"id": "H1", "hypothesis": "Month-to-month contracts have higher churn than long-term contracts.", "supporting_fields": "Contract"},
        {"id": "H2", "hypothesis": "High MonthlyCharges segments churn more due to price sensitivity.", "supporting_fields": "MonthlyCharges, monthly_charge_bucket"},
        {"id": "H3", "hypothesis": "Early-life customers (low tenure) churn more.", "supporting_fields": "tenure, tenure_bucket"},
        {"id": "H4", "hypothesis": "Lack of TechSupport and OnlineSecurity increases churn.", "supporting_fields": "TechSupport, OnlineSecurity"},
        {"id": "H5", "hypothesis": "Electronic check payment method is associated with higher churn; paperless billing users also show higher churn and may overlap with month-to-month contracts.", "supporting_fields": "PaymentMethod, PaperlessBilling, Contract"}
    ]


def run_domain_diagnostics_telco_addon(
    *,
    df: pd.DataFrame,
    ctx: Dict[str, Any],
    run_id: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Run Telco churn addon diagnostics (2.6.2) in a ctx-driven, SSOT manner.

    YAML SSOT
    ---------
    pipeline.data_quality.domain_diagnostics.telco_churn:
      segment_cols: [...]
      tenure_bins: [0, 6, 12, 24, 48, 1000]
      tenure_bin_labels: ["0-6m", "6-12m", "12-24m", "24-48m", "48m+"]
      monthly_charge_quantiles: [0.0, 0.33, 0.66, 1.0]
      rare_segment_min_share: 0.01
      leakage:
        candidates: ["TotalCharges"]
        ratio_band: [0.8, 1.2]
        correlation_threshold: 0.70
        actions: {...}
      plots:
        enabled: true
        bins: 30
    
    Returns
    -------
        ModuleReport v1 (Telco-Churn diagnostics) (also persisted as artifact).
    """
    # --- 2) Runtime validation ---
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG26_2, run_id=run_id)

    # --- 3) Input validation ---
    if df is None or not isinstance(df, pd.DataFrame):
        raise TypeError(f"[{TAG26_2}] df must be a pandas DataFrame")

    # --- 4) Resolve SSOT config + dependency preflight (YAML + upstream artifact lookup) ---

    # --- Global（SSOT） ---
    label_col, pos_label, task_type = get_label_cfg(ctx, TAG26_2)
    tt = str(task_type).strip().lower()
    task_type_ok = (tt in _SUPPORTED_TASK_TYPES)
    pos_key = str(pos_label).strip()

    # --- Module（step cfg） ---
    get_cfg_dict(ctx, "pipeline.data_quality.domain_diagnostics.telco_churn", tag=TAG26_2, required=True)

    # --- min_non_missing: telco override -> core fallback -> hard default ---
    raw_telco = get_cfg_value(
        ctx,
        "pipeline.data_quality.domain_diagnostics.telco_churn.leakage.min_non_missing",
        tag=TAG26_2,
        default=None,
        required=False,
    )

    if raw_telco is None:
        # Fallback to core config
        min_non_missing = get_cfg_int(
            ctx,
            "pipeline.data_quality.domain_diagnostics.core.leakage.min_non_missing",
            tag=TAG26_2,
            default=10,
            required=False,
            min_value=0,
            strict_type=True,
        )
    else:
        # Validate telco override strictly (None already handled)
        if isinstance(raw_telco, bool) or not isinstance(raw_telco, int):
            raise TypeError(
                f"[{TAG26_2}] telco_churn.leakage.min_non_missing must be int when provided, got {type(raw_telco).__name__}"
            )
        if raw_telco < 0:
            raise ValueError(f"[{TAG26_2}] telco_churn.leakage.min_non_missing must be >= 0, got {raw_telco}")
        min_non_missing = int(raw_telco)

    if min_non_missing is None:
        min_non_missing = 10

    segment_cols = get_cfg_str_list(
        ctx,
        "pipeline.data_quality.domain_diagnostics.telco_churn.segment_cols",
        tag=TAG26_2,
        default=[],
        required=False,
        dedupe=True,
        allow_empty=True,
        min_len=0,
        strict_items=True,
    )

    tenure_bins = get_cfg_float_list(
        ctx,
        "pipeline.data_quality.domain_diagnostics.telco_churn.tenure_bins",
        tag=TAG26_2,
        default=[0, 6, 12, 24, 48, 1000],
        required=False,
        allow_empty=False,
        min_len=2,
        max_len=None,
        strict_items=True,
        finite=True,
        dedupe=False,
        sort_unique=False,
    )

    tenure_labels = get_cfg_str_list(
        ctx,
        "pipeline.data_quality.domain_diagnostics.telco_churn.tenure_bin_labels",
        tag=TAG26_2,
        default=["0-6m", "6-12m", "12-24m", "24-48m", "48m+"],
        required=False,
        dedupe=False,
        allow_empty=False,
        min_len=1,
        strict_items=True,
    )

    monthly_q = get_cfg_float_list(
        ctx,
        "pipeline.data_quality.domain_diagnostics.telco_churn.monthly_charge_quantiles",
        tag=TAG26_2,
        default=[0.0, 0.33, 0.66, 1.0],
        required=False,
        allow_empty=False,
        min_len=2,
        max_len=None,
        strict_items=True,
        finite=True,
        dedupe=False,
        sort_unique=False,
    )

    rare_segment_min_share = get_cfg_float(
        ctx,
        "pipeline.data_quality.domain_diagnostics.telco_churn.rare_segment_min_share",
        tag=TAG26_2,
        default=0.01,
        required=False,
        min_value=0.0,
        max_value=1.0,
        strict_type=True,
    )

    leakage_candidates = get_cfg_str_list(
        ctx,
        "pipeline.data_quality.domain_diagnostics.telco_churn.leakage.candidates",
        tag=TAG26_2,
        default=["TotalCharges"],
        required=False,
        dedupe=True,
        allow_empty=True,
        min_len=0,
        strict_items=True,
    )

    ratio_band = get_cfg_float_list(
        ctx,
        "pipeline.data_quality.domain_diagnostics.telco_churn.leakage.ratio_band",
        tag=TAG26_2,
        default=[0.8, 1.2],
        required=False,
        allow_empty=False,
        min_len=2,
        max_len=2,
        strict_items=True,
        finite=True,
        dedupe=False,
        sort_unique=False,
    )

    corr_thr = get_cfg_float(
        ctx,
        "pipeline.data_quality.domain_diagnostics.telco_churn.leakage.correlation_threshold",
        tag=TAG26_2,
        default=0.70,
        required=False,
        min_value=0.0,
        max_value=1.0,
        strict_type=True,
    )

    action_map = get_cfg_dict(
        ctx,
        "pipeline.data_quality.domain_diagnostics.telco_churn.leakage.actions",
        tag=TAG26_2,
        required=False,
        default={},
    )

    action_map_ok = isinstance(action_map, dict)

    plots_enabled = get_cfg_bool(
        ctx,
        "pipeline.data_quality.domain_diagnostics.telco_churn.plots.enabled",
        tag=TAG26_2,
        default=False,
        required=False,
        strict_type=True,
    )
   
    bins_plot = get_cfg_int(
        ctx,
        "pipeline.data_quality.domain_diagnostics.telco_churn.plots.bins",
        tag=TAG26_2,
        default=30,
        required=False,
        min_value=1,
        strict_type=True,
    )

    # --- Data facts ---
    df_cols = set(df.columns)
    n_rows = int(df.shape[0])
    is_empty_df = (n_rows == 0)
    label_missing = (label_col not in df.columns)

    required_for_addon = {"tenure", "MonthlyCharges"}
    missing_req = sorted([c for c in required_for_addon if c not in df_cols])
    req_ok = (len(missing_req) == 0)

    missing_seg = sorted([c for c in segment_cols if c not in df_cols]) if segment_cols else []
    label_in_segments = bool(label_col in segment_cols) if segment_cols else False
    seg_cfg_ok = (len(missing_seg) == 0) and (not label_in_segments)

    tenure_bins_ok = (
        isinstance(tenure_bins, list)
        and len(tenure_bins) >= 2
        and all(float(tenure_bins[i]) > float(tenure_bins[i - 1]) for i in range(1, len(tenure_bins)))
    )
    tenure_labels_ok = (isinstance(tenure_labels, list) and len(tenure_labels) == (len(tenure_bins) - 1))

    monthly_q_ok = (
        isinstance(monthly_q, list)
        and len(monthly_q) >= 2
        and all(float(monthly_q[i]) > float(monthly_q[i - 1]) for i in range(1, len(monthly_q)))
    )
    eps = 1e-12
    monthly_q_cover_ok = False
    if isinstance(monthly_q, list) and len(monthly_q) >= 2:
        monthly_q_cover_ok = (abs(float(monthly_q[0]) - 0.0) <= eps) and (abs(float(monthly_q[-1]) - 1.0) <= eps)

    missing_leak = sorted([c for c in leakage_candidates if c not in df_cols]) if leakage_candidates else []
    leak_cfg_ok = (len(missing_leak) == 0)

    ratio_band_ok = (
        isinstance(ratio_band, list)
        and len(ratio_band) == 2
        and float(ratio_band[0]) > 0.0
        and float(ratio_band[1]) > float(ratio_band[0])
    )

    # --- Dependency preflight (facts only) ---
    core_ref = make_artifact_ref(ctx=ctx, stage="data_quality", name="domain_core", run_id=str(rid))
    core_path = resolve_artifact_path(ctx=ctx, stage="data_quality", name="domain_core", run_id=str(rid), tag=TAG26_2)
    core_report = read_json_if_exists(core_path, lg, tag=TAG26_2)

    dep_ok = True
    dep_issue: Optional[str] = None
    core_status: Optional[str] = None

    task_type_drift = False
    core_task_type_raw: Optional[str] = None

    if core_report is None:
        dep_ok = False
        dep_issue = "dependency_domain_core_missing"
    else:
        core_status = (core_report.get("summary") or {}).get("overall_status") or core_report.get("status")
        if core_status not in {"pass", "warn"}:
            dep_ok = False
            dep_issue = f"dependency_domain_core_not_ready:{core_status}"

        # optional: detect SSOT drift vs core artifact config
        core_used_cfg = core_report.get("used_config") if isinstance(core_report, dict) else None
        if isinstance(core_used_cfg, dict):
            core_task_type_raw = core_used_cfg.get("task_type_raw")
            core_tt = str(core_used_cfg.get("task_type_norm") or "").strip().lower()
            if core_tt and (core_tt != tt):
                task_type_drift = True

    # --- 5) Core logic (pure computation) ---
    lg.info(f"[{TAG26_2}] 📊 Starting Telco churn addon diagnostics...")

    # --- Build ModuleReport skeleton (infra) ---
    report = build_module_report(
        ctx=ctx,
        stage="data_quality",
        step="2.6.2",
        name="domain_addon_telco",
        tag=TAG26_2,
        cfg_key="pipeline.data_quality.domain_diagnostics.telco_churn",
        enabled=True,
        df=df,
        run_id_override=rid,
    )

    report["inputs"] = {"df_provided": True}

    report["thresholds_used"] = {
        "rare_segment_min_share": float(rare_segment_min_share),
        "correlation_threshold": float(corr_thr),
        "min_non_missing": int(min_non_missing),
        "ratio_band": [float(ratio_band[0]), float(ratio_band[1])],
        "bins_plot": int(bins_plot),
        "supported_task_types": list(_SUPPORTED_TASK_TYPES),
    }

    report["used_config"] = {
        "label_col": label_col,
        "positive_label": pos_key,
        "task_type_raw": task_type,
        "task_type_norm": tt,
        "segment_cols": list(segment_cols),
        "tenure_bins": list(tenure_bins),
        "tenure_bin_labels": list(tenure_labels),
        "monthly_charge_quantiles": list(monthly_q),
        "leakage_candidates": list(leakage_candidates),
        "ratio_band": [float(ratio_band[0]), float(ratio_band[1])] if isinstance(ratio_band, list) and len(ratio_band) == 2 else None,
        "leakage_actions": action_map if isinstance(action_map, dict) else {},
        "plots_enabled": bool(plots_enabled),
    }

    report["refs"]["self"] = make_artifact_ref(ctx=ctx, stage="data_quality", name="domain_addon_telco", run_id=str(rid))
    report["refs"].setdefault("dependencies", {})
    report["refs"]["dependencies"]["domain_core"] = core_ref

    # --- Preconditions (facts only; like 2.6.1) ---
    report["checks"]["dependency_domain_core"] = {
        "status": "pass" if dep_ok else "fail",
        "ok": bool(dep_ok),
        "issue": dep_issue,
        "core_status": core_status,
        "core_ref": core_ref,
        "task_type_drift": bool(task_type_drift),
        "core_task_type_raw": core_task_type_raw,
        "ssot_task_type_norm": tt,
    }

    report["checks"]["preconditions"] = {
        "task_type": {
            "status": "pass" if task_type_ok else "fail",
            "task_type": task_type,
            "expected": sorted(list(_SUPPORTED_TASK_TYPES)),
        },
        "required_columns": {
            "status": "pass" if req_ok else "fail",
            "required": sorted(list(required_for_addon)),
            "missing_required_columns": missing_req,
        },
        "segment_cols": {
            "status": "pass" if (seg_cfg_ok or (not segment_cols)) else "fail",
            "n_segment_cols": int(len(segment_cols)),
            "missing_segment_cols": missing_seg,
            "label_in_segments": bool(label_in_segments),
            "allow_empty": True,
        },
        "bucket_config": {
            "status": "pass" if (tenure_bins_ok and tenure_labels_ok and monthly_q_ok and monthly_q_cover_ok) else "fail",
            "tenure_bins_ok": bool(tenure_bins_ok),
            "tenure_labels_ok": bool(tenure_labels_ok),
            "monthly_q_ok": bool(monthly_q_ok),
            "monthly_q_cover_ok": bool(monthly_q_cover_ok),
        },
        "leakage_config": {
            "status": "pass" if (leak_cfg_ok and ratio_band_ok and action_map_ok) else "warn",
            "n_candidates": int(len(leakage_candidates)) if isinstance(leakage_candidates, list) else 0,
            "missing_candidates": missing_leak,
            "ratio_band_ok": bool(ratio_band_ok),
            "action_map_ok": bool(action_map_ok),
        },
    }

    # --- Core logic ---
    warnings: List[str] = []

    # Business buckets
    df_seg = df
    if is_empty_df:
        report["checks"]["business_segmentation"] = {"note": "empty_dataframe"}
    else:
        try:
            df_seg = _build_business_segments_telco(
                df,
                tenure_bins=[float(x) for x in tenure_bins],
                tenure_bin_labels=[str(x) for x in tenure_labels],
                monthly_charge_quantiles=[float(x) for x in monthly_q],
            )
            report["checks"]["business_segmentation"] = {"ok": True, "added_cols": ["tenure_bucket", "monthly_charge_bucket"]}
        except Exception as e:
            report["checks"]["business_segmentation"] = {"ok": False, "error": str(e)}
            warnings.append("business_segmentation_failed")
            df_seg = df

    # Positive rate by Telco segments + buckets
    telco_segments = list(segment_cols) + ["tenure_bucket", "monthly_charge_bucket"]

    rate_by_seg_failed = False
    many_small_segments = False

    seg_ok = (report["checks"]["preconditions"]["segment_cols"]["status"] == "pass")
    bucket_ok = (report["checks"]["preconditions"]["bucket_config"]["status"] == "pass")

    if label_missing or is_empty_df or (not task_type_ok) or (not seg_ok) or (not bucket_ok):
        rate_by_seg = {
            "error": "label_missing_or_empty_df_or_invalid_preconditions",
            "label_col": label_col,
            "n_rows": int(n_rows),
            "task_type_ok": bool(task_type_ok),
            "segments_ok": bool(seg_ok),
            "bucket_config_ok": bool(bucket_ok),
            "missing_segment_cols": missing_seg,
            "label_in_segments": bool(label_in_segments),
        }
        rate_by_seg_failed = True
    else:
        rate_by_seg = rate_by_segment(
            df_seg,
            label_col=label_col,
            positive_label=pos_key,
            segment_cols=telco_segments,
            small_segment_min_share=float(rare_segment_min_share),
        )
        if isinstance(rate_by_seg, dict) and ("error" in rate_by_seg):
            rate_by_seg_failed = True
        else:
            small_count = 0
            total_levels = 0
            for _, info in (rate_by_seg or {}).items():
                if not isinstance(info, dict) or not info.get("exists"):
                    continue
                segs = info.get("segments", {})
                if isinstance(segs, dict):
                    for _, s in segs.items():
                        total_levels += 1
                        if isinstance(s, dict) and s.get("small_segment") is True:
                            small_count += 1
            if total_levels > 0 and (small_count / total_levels) > 0.30:
                many_small_segments = True

    report["checks"]["positive_rate_by_segment"] = rate_by_seg

    # Telco leakage pre-scan (soft)
    flagged_feats: List[str] = []
    if label_missing or is_empty_df or (not task_type_ok):
        report["checks"]["telco_leakage_pre_scan"] = {"note": "label_missing_or_empty_df_or_task_type_mismatch"}
    else:
        if isinstance(leakage_candidates, list) and leakage_candidates:
            if not leak_cfg_ok:
                report["checks"]["telco_leakage_pre_scan"] = {
                    "note": "invalid_leakage_config; scan_skipped",
                    "missing_candidates": missing_leak,
                }
            else:
                leak = _pre_scan_leakage_telco(
                    df_seg,
                    candidates=[str(x) for x in leakage_candidates],
                    label_col=label_col,
                    positive_label=pos_key,
                    ratio_band=[float(ratio_band[0]), float(ratio_band[1])] if isinstance(ratio_band, list) and len(ratio_band) == 2 else [0.8, 1.2],
                    corr_threshold=float(corr_thr),
                    min_non_missing=int(min_non_missing),
                    action_map=action_map if isinstance(action_map, dict) else {},
                )
                report["checks"]["telco_leakage_pre_scan"] = leak

                if isinstance(leak, dict) and isinstance(leak.get("features"), dict):
                    flagged_feats = [
                        k for k, v in leak["features"].items()
                        if isinstance(v, dict) and v.get("flagged") is True
                    ]
                    if flagged_feats:
                        report["checks"]["telco_leakage_pre_scan"]["flagged_features"] = flagged_feats
        else:
            report["checks"]["telco_leakage_pre_scan"] = {"note": "no_candidates_configured"}

    # Domain anomalies + hypotheses (soft)
    try:
        report["checks"]["telco_domain_anomalies"] = _domain_anomaly_checks_telco(df_seg)
    except Exception as e:
        report["checks"]["telco_domain_anomalies"] = {"error": str(e)}
        warnings.append("telco_domain_anomalies_failed")

    report["checks"]["telco_domain_hypotheses"] = _telco_domain_hypotheses()

    # Plots
    plots_meta = {"requested": bool(plots_enabled), "saved": False, "dir": None, "files": []}
    if plots_enabled:
        try:
            fig_base = resolve_figures_dir(ctx, tag=TAG26_2)
            if fig_base is None:
                plots_meta["reason"] = "figures_dir_unresolved"
            else:
                module_name = "domain_telco"
                fig_dir = fig_base / "data_quality" / module_name
                fig_dir.mkdir(parents=True, exist_ok=True)
                plots_meta["dir"] = str(fig_dir)

                saved_refs: List[Dict[str, Any]] = []
                saved_paths: List[str] = []

                if "tenure" in df_seg.columns:
                    p = fig_dir / f"data_quality_{module_name}_tenure_hist_{rid}.png"
                    plot_histogram(
                        df_seg["tenure"],
                        title="Tenure Distribution",
                        xlabel="tenure (months)",
                        out_path=p,
                        bins=int(bins_plot),
                    )
                    saved_refs.append({"name": "tenure_hist", "path": str(p)})
                    saved_paths.append(str(p))

                if "MonthlyCharges" in df_seg.columns:
                    p = fig_dir / f"data_quality_{module_name}_monthlycharges_hist_{rid}.png"
                    plot_histogram(
                        df_seg["MonthlyCharges"],
                        title="Monthly Charges Distribution",
                        xlabel="MonthlyCharges",
                        out_path=p,
                        bins=int(bins_plot),
                    )
                    saved_refs.append({"name": "monthlycharges_hist", "path": str(p)})
                    saved_paths.append(str(p))

                if (not label_missing) and (label_col in df_seg.columns):
                    plot_df = df_seg.copy()

                    mask_valid = plot_df[label_col].notna()
                    plot_df = plot_df.loc[mask_valid].copy()
                    plot_df["_pos_flag"] = (plot_df[label_col].astype(str).str.strip() == pos_key).astype(int)

                    if "Contract" in plot_df.columns:
                        p = fig_dir / f"data_quality_{module_name}_contract_churn_{rid}.png"
                        plot_bar_mean(
                            plot_df,
                            group_col="Contract",
                            value_col="_pos_flag",
                            title="Churn Rate by Contract Type",
                            out_path=p,
                        )
                        saved_refs.append({"name": "contract_churn", "path": str(p)})
                        saved_paths.append(str(p))

                    if "tenure_bucket" in plot_df.columns:
                        p = fig_dir / f"data_quality_{module_name}_tenure_bucket_churn_{rid}.png"
                        plot_bar_mean(
                            plot_df,
                            group_col="tenure_bucket",
                            value_col="_pos_flag",
                            title="Churn Rate by Tenure Bucket",
                            out_path=p,
                        )
                        saved_refs.append({"name": "tenure_bucket_churn", "path": str(p)})
                        saved_paths.append(str(p))

                    if "monthly_charge_bucket" in plot_df.columns:
                        p = fig_dir / f"data_quality_{module_name}_monthly_charge_bucket_churn_{rid}.png"
                        plot_bar_mean(
                            plot_df,
                            group_col="monthly_charge_bucket",
                            value_col="_pos_flag",
                            title="Churn Rate by MonthlyCharge Bucket",
                            out_path=p,
                        )
                        saved_refs.append({"name": "monthly_charge_bucket_churn", "path": str(p)})
                        saved_paths.append(str(p))

                plots_meta["saved"] = True
                plots_meta["files"] = saved_paths

                report.setdefault("refs", {})
                report["refs"].setdefault("figures", [])
                report["refs"]["figures"].extend(saved_refs)

        except Exception as e:
            plots_meta["saved"] = False
            plots_meta["error"] = str(e)

    report["checks"]["plots"] = plots_meta

    # Event hints (policy-free; for 2.8 mapping)
    ensure_event_hints(report, version=1)

    # Dependency -> hard
    if not dep_ok:
        add_event_hint(
            report,
            code="domain_telco_dependency_domain_core_not_ready",
            severity_signal="hard",
            evidence_path="checks.dependency_domain_core",
            context={"dep_issue": dep_issue, "core_status": core_status},
            remediation={
                "action": "run_2_6_1_domain_core_and_fix_upstream_issues",
                "safe": True,
                "rationale": "2.6.2 depends on 2.6.1 artifact for consistent assumptions and auditability.",
                "post_check": "core_report.summary.overall_status in {pass,warn}",
            },
        )

    if task_type_drift:
        add_event_hint(
            report,
            code="domain_telco_dependency_task_type_drift",
            severity_signal="hard",
            evidence_path="checks.dependency_domain_core.task_type_drift",
            context={"core_task_type": core_task_type_raw, "ssot_task_type": tt},
            remediation={
                "action": "re_run_2_6_1_after_ssot_change_or_start_new_run_id",
                "safe": True,
                "rationale": "Avoid mixing artifacts produced under different task_type assumptions.",
                "post_check": "core_report.used_config.task_type equals SSOT task_type",
            },
        )

    # Local hard preconditions
    if not task_type_ok:
        add_event_hint(
            report,
            code="domain_telco_task_type_mismatch",
            severity_signal="hard",
            evidence_path="checks.preconditions.task_type",
            context={"task_type_raw": str(task_type), "task_type_norm": tt, "supported": sorted(list(_SUPPORTED_TASK_TYPES))},
            remediation={
                "action": "set_dataset_label_task_type_to_classification_in_yaml",
                "safe": True,
                "rationale": "Telco addon is churn classification-specific.",
                "post_check": "re-run 2.6.2 after SSOT fix",
            },
        )

    if is_empty_df:
        add_event_hint(
            report,
            code="domain_telco_empty_dataframe",
            severity_signal="hard",
            evidence_path="summary.total_rows",
            context={"n_rows": int(n_rows)},
            remediation={
                "action": "fix_ingestion_or_upstream_filters",
                "safe": False,
                "rationale": "Addon diagnostics cannot run on an empty dataframe.",
                "post_check": "df.shape[0] > 0",
            },
        )

    if label_missing:
        add_event_hint(
            report,
            code="domain_telco_label_col_missing",
            severity_signal="hard",
            evidence_path="used_config.label_col",
            context={"label_col": str(label_col)},
            remediation={
                "action": "ensure_label_column_exists_and_matches_ssot",
                "safe": False,
                "rationale": "Churn slicing requires label.",
                "post_check": "df contains label_col",
            },
        )

    if not req_ok:
        add_event_hint(
            report,
            code="domain_telco_missing_required_columns_for_buckets",
            severity_signal="hard",
            evidence_path="checks.preconditions.required_columns.missing_required_columns",
            context={"missing_required_columns": sample_str_list(missing_req, limit=10), "required": sorted(list(required_for_addon))},
            remediation={
                "action": "ensure_tenure_and_monthlycharges_exist",
                "safe": False,
                "rationale": "Buckets depend on tenure and monthly charges.",
                "post_check": "df contains {'tenure','MonthlyCharges'}",
            },
        )

    bucket_cfg_ok = bool(tenure_bins_ok and tenure_labels_ok and monthly_q_ok and monthly_q_cover_ok)
    if not bucket_cfg_ok:
        add_event_hint(
            report,
            code="domain_telco_invalid_bucket_config",
            severity_signal="hard",
            evidence_path="checks.preconditions.bucket_config",
            context={"tenure_bins": tenure_bins, "tenure_labels": tenure_labels, "monthly_q": monthly_q},
            remediation={
                "action": "fix_bucket_config_bins_labels_and_quantiles",
                "safe": True,
                "rationale": "Buckets must be well-defined to produce stable segment rates.",
                "post_check": "bucket_config checks all true",
            },
        )

    if segment_cols and (not seg_cfg_ok):
        add_event_hint(
            report,
            code="domain_telco_invalid_segment_cols_config",
            severity_signal="hard",
            evidence_path="checks.preconditions.segment_cols",
            context={
                "missing_segment_cols": sample_str_list(missing_seg, limit=10),
                "label_in_segments": bool(label_in_segments),
                "label_col": str(label_col),
            },
            remediation={
                "action": "fix_telco_segment_cols_remove_label_and_ensure_columns_exist",
                "safe": True,
                "rationale": "Configured segments must exist to run telco slicing.",
                "post_check": "segment cols are valid or empty",
            },
        )

    # Leak config optional -> fixable
    if missing_leak:
        add_event_hint(
            report,
            code="domain_telco_leakage_candidates_missing_in_df",
            severity_signal="fixable",
            evidence_path="checks.preconditions.leakage_config.missing_candidates",
            context={"missing_candidates": sample_str_list(missing_leak, limit=10), "n_missing": int(len(missing_leak))},
            remediation={
                "action": "remove_missing_candidates_or_fix_schema",
                "safe": True,
                "rationale": "Leakage scan is optional; invalid candidates reduce coverage.",
                "post_check": "re-run 2.6.2 and confirm leakage scan executes or is cleanly skipped",
            },
        )

    if not ratio_band_ok:
        add_event_hint(
            report,
            code="domain_telco_invalid_ratio_band_config",
            severity_signal="fixable",
            evidence_path="checks.preconditions.leakage_config.ratio_band_ok",
            context={"ratio_band": ratio_band},
            remediation={
                "action": "set_ratio_band_as_two_positive_increasing_numbers",
                "safe": True,
                "rationale": "Ratio band misconfiguration only affects a soft leakage hint.",
                "post_check": "ratio_band_ok == True",
            },
        )

    if not action_map_ok:
        add_event_hint(
            report,
            code="domain_telco_invalid_leakage_action_map",
            severity_signal="fixable",
            evidence_path="checks.preconditions.leakage_config.action_map_ok",
            context={"action_map_type": str(type(action_map))},
            remediation={
                "action": "set_leakage_actions_to_a_dict_mapping_feature_to_list_of_actions",
                "safe": True,
                "rationale": "Action map is optional; invalid type reduces interpretability but not core diagnostics.",
                "post_check": "action_map_ok == True",
            },
        )

    if flagged_feats:
        add_event_hint(
            report,
            code="domain_telco_potential_leakage_features_flagged",
            severity_signal="risk",
            evidence_path="checks.telco_leakage_pre_scan.flagged_features",
            context={"n_flagged": int(len(flagged_feats)), "flagged_features": sample_str_list(flagged_feats, limit=10), "corr_threshold": float(corr_thr)},
            remediation={
                "action": "investigate_feature_lineage_and_remove_or_transform_leaky_features",
                "safe": True,
                "rationale": "This is a soft pre-scan; treat as investigation trigger to avoid leakage.",
                "post_check": "re-run 3.x leakage_scan and confirm no leakage remains",
            },
        )

    if plots_meta.get("requested") and (not plots_meta.get("saved")):
        add_event_hint(
            report,
            code="domain_telco_plots_failed",
            severity_signal="info",
            evidence_path="checks.plots",
            context={"plots_meta": plots_meta},
            remediation={
                "action": "check_figures_path_and_plot_utils",
                "safe": True,
                "rationale": "Plots are optional; failures do not invalidate diagnostics.",
                "post_check": "plots.saved == True when enabled",
            },
        )

    if many_small_segments:
        add_event_hint(
            report,
            code="domain_telco_many_small_segments_detected",
            severity_signal="risk",
            evidence_path="checks.positive_rate_by_segment",
            context={"rare_segment_min_share": float(rare_segment_min_share), "heuristic": "small_segments_ratio_gt_0.30"},
            remediation={
                "action": "bucket_rare_categories_or_adjust_rare_segment_threshold",
                "safe": True,
                "rationale": "Too many small segments makes rates unstable; bucketing is deterministic when configured.",
                "post_check": "re-run 2.6.2 after cleaning/bucketing to confirm segment stability",
            },
        )

    if rate_by_seg_failed:
        add_event_hint(
            report,
            code="domain_telco_positive_rate_by_segment_failed_or_skipped",
            severity_signal="risk",
            evidence_path="checks.positive_rate_by_segment",
            context={
                "task_type_ok": bool(task_type_ok),
                "label_missing": bool(label_missing),
                "empty_df": bool(is_empty_df),
                "segments_ok": bool(seg_ok),
                "bucket_config_ok": bool(bucket_ok),
            },
            remediation={
                "action": "fix_preconditions_and_rerun_segment_diagnostics",
                "safe": True,
                "rationale": "Segment diagnostics are soft but valuable; fix inputs/config to restore coverage.",
                "post_check": "re-run 2.6.2 after resolving preconditions",
            },
        )

    # --- 6) Status + gate-friendly summary ---
    issues: List[str] = []
    status_warnings: List[str] = list(warnings)

    # Hard viability
    if not dep_ok:
        issues.append("dependency_domain_core_not_ready")
    if task_type_drift:
        issues.append("dependency_task_type_drift")
    if not task_type_ok:
        issues.append("task_type_mismatch")
    if is_empty_df:
        issues.append("empty_dataframe")
    if label_missing:
        issues.append("label_col_missing")
    if not req_ok:
        issues.append("missing_required_columns_for_buckets")
    if not bucket_cfg_ok:
        issues.append("invalid_bucket_config")
    if segment_cols and (not seg_cfg_ok):
        issues.append("invalid_segment_cols_config")

    # Soft signals
    if rate_by_seg_failed:
        status_warnings.append("rate_by_segment_failed_or_skipped")
    if many_small_segments:
        status_warnings.append("many_small_segments_detected")
    if flagged_feats:
        status_warnings.append("potential_leakage_features_flagged")
    if missing_leak:
        status_warnings.append("leakage_candidates_missing_in_df")
    if plots_meta.get("requested") and (not plots_meta.get("saved")):
        status_warnings.append("plots_failed")

    _metrics = {
        "n_rows": int(n_rows),
        "n_segment_cols": int(len(segment_cols)),
        "n_telco_segments_total": int(len(telco_segments)),
        "n_leakage_candidates": int(len(leakage_candidates)) if isinstance(leakage_candidates, list) else 0,
        "n_flagged_leakage_features": int(len(flagged_feats)),
        "dependency_ok": int(dep_ok),
        "task_type_drift": int(task_type_drift),
    }

    _notes = [
        f"task_type={task_type}",
        f"label_col={label_col}",
        f"positive_label={pos_key}",
        "depends_on=2.6.1(domain_core)",
    ]

    _issues = sorted(set(issues))
    _warnings = sorted(set(status_warnings))

    overall = "fail" if issues else ("warn" if status_warnings else "pass")

    report = finalize_module_report(
        ctx=ctx,
        report=report,
        overall_status=overall,
        issues=_issues,
        warnings=_warnings,
        metrics=_metrics,
        notes=_notes,
        df=df,
    )

    if overall == "pass":
        lg.info(f"[{TAG26_2}] ✅ Telco addon diagnostics completed.")
    elif overall == "warn":
        lg.warning(f"[{TAG26_2}] ⚠ Telco addon diagnostics completed with warnings: {sorted(set(status_warnings))}")
    else:
        lg.error(f"[{TAG26_2}] ❌ Telco addon diagnostics failed. | issues={sorted(set(issues))}")

    # --- 7) Persist Artifact ---
    out_path = resolve_artifact_path(
        ctx=ctx,
        stage="data_quality",
        name="domain_addon_telco",
        run_id=str(rid),
        tag=TAG26_2,
    )
    write_json(out_path, report, tag=TAG26_2, indent=2)
    lg.info(f"[{TAG26_2}] 🧾 Telco addon artifact saved: {out_path}")

    return report

In [9]:
# --- 2.7 Distribution & Balance Check (Classification) ---
#
# What:
#   Generic distribution diagnostics for classification (label balance + separation signals).
#
# How:
#   - Resolve YAML SSOT via ctx:
#       dataset.label (col, positive, task_type)
#       dataset.schema.dtypes (numeric, categorical)
#       pipeline.data_quality.distribution (thresholds)
#   - Compute:
#       • label_balance: counts / shares / majority_share / imbalance_ratio
#       • numeric_by_label: mean/std/median + Cohen's d
#       • categorical_by_label: positive_rate per level + small_segment flags
#   - Persist ModuleReport v1 artifact (SSOT naming).
#
# Why:
#   Surface early imbalance + segment risks + separating features in a reproducible, gate-friendly report.

# --- 0) Imports + TAG ---
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint
from src.utils.artifact_utils import resolve_artifact_path, write_json
from src.utils.cfg_utils import get_cfg_float, get_cfg_dict, get_cfg_int, get_cfg_bool
from src.utils.dataset_utils import get_label_cfg, get_schema_cols
from src.utils.serialization_utils import safe_float
from src.utils.pandas_utils import safe_num, normalize_str_series
from src.utils.path_utils import resolve_figures_dir
from src.utils.plot_utils import plot_bar_pairs

TAG27 = "DISTRIBUTION"
_SUPPORTED_TASK_TYPES: tuple[str, ...] = ("classification",)


# --- Internal Helpers ---
def _keyify(v: Any) -> str:
    """Convert values to stable string keys for reports (NA -> '<MISSING>')."""
    return "<MISSING>" if pd.isna(v) else str(v)


def _compute_label_balance(
    df: pd.DataFrame,
    *,
    label_col: str,
    positive_label: str,
    max_majority_share: float,
) -> Dict[str, Any]:
    """Summarize label distribution and imbalance indicators (exclude missing labels from balance math)."""
    if label_col not in df.columns:
        return {"exists": False, "reason": "label_column_not_found"}

    pos_key = str(positive_label).strip()
    y_all = normalize_str_series(df[label_col])
    missing_count = int(y_all.isna().sum())
    vc_valid = y_all.dropna().value_counts()
    total_valid = int(vc_valid.sum())
    total_all = int(total_valid + missing_count)

    if total_all <= 0:
        return {
            "exists": True,
            "total_all": 0,
            "total_valid": 0,
            "missing_count": 0,
            "value_counts": {},
            "value_pct": {},
            "positive_label": pos_key,
            "positive_share": 0.0,
            "majority_share": 0.0,
            "imbalance_ratio": None,
            "imbalance_flag": "empty",
        }

    vc_dict = { _keyify(k): int(v) for k, v in vc_valid.to_dict().items() }
    if missing_count > 0:
        vc_dict["<MISSING>"] = int(missing_count)

    dist_pct = { k: float(round(v / total_all, 6)) for k, v in vc_dict.items() }

    pos_key = str(positive_label).strip()
    pos_count = int((y_all.dropna() == pos_key).sum())

    if total_valid <= 0:
        return {
            "exists": True,
            "total_all": total_all,
            "total_valid": 0,
            "missing_count": missing_count,
            "missing_share": float(round(float(missing_count / total_all), 6)) if total_all > 0 else 0.0,
            "value_counts": {str(k): int(v) for k, v in vc_dict.items()},
            "value_pct": {str(k): float(v) for k, v in dist_pct.items()},
            "positive_label": pos_key,
            "positive_share": 0.0,
            "majority_share": 0.0,
            "imbalance_ratio": None,
            "imbalance_flag": "all_missing",
        }

    pos_share = float(pos_count / total_valid)
    non_pos_share = float(1.0 - pos_share)
    majority_share = float(max(pos_share, non_pos_share))

    imbalance_flag = "severe_imbalance" if majority_share > float(max_majority_share) else "ok"

    imbalance_ratio = None
    if pos_share > 0 and non_pos_share > 0:
        imbalance_ratio = float(max(pos_share, non_pos_share) / min(pos_share, non_pos_share))

    return {
        "exists": True,
        "total_all": total_all,
        "total_valid": total_valid,
        "missing_count": missing_count,
        "missing_share": float(round(float(missing_count / total_all), 6)) if total_all > 0 else 0.0,
        "value_counts": {str(k): int(v) for k, v in vc_dict.items()},
        "value_pct": {str(k): float(v) for k, v in dist_pct.items()},
        "positive_label": pos_key,
        "positive_share": float(round(pos_share, 6)),
        "majority_share": float(round(majority_share, 6)),
        "imbalance_ratio": float(round(imbalance_ratio, 6)) if imbalance_ratio is not None else None,
        "imbalance_flag": imbalance_flag,
    }


def _cohen_d(
    x: np.ndarray,
    y: np.ndarray,
) -> float:
    """
    Compute Cohen's d effect size between two numeric samples.

    d ≈ (mean_x - mean_y) / pooled_std

    If pooled_std is 0/NaN or sample is too small, returns 0.0.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    if x.size < 2 or y.size < 2:
        return 0.0

    mean_x = float(x.mean())
    mean_y = float(y.mean())
    var_x = float(x.var(ddof=1))
    var_y = float(y.var(ddof=1))

    denom = (x.size + y.size - 2)
    if denom <= 0:
        return 0.0

    pooled = ((x.size - 1) * var_x + (y.size - 1) * var_y) / denom
    if not np.isfinite(pooled) or pooled <= 0:
        return 0.0

    d = (mean_x - mean_y) / float(np.sqrt(pooled))
    return float(d) if np.isfinite(d) else 0.0


def _numeric_distribution_by_label(
    df: pd.DataFrame,
    *,
    label_col: str,
    positive_label: str,
    numeric_cols: List[str],
) -> Dict[str, Any]:
    """Summarize numeric distribution by label and compute Cohen's d (exclude missing labels)."""
    if label_col not in df.columns:
        return {"error": f"label_col '{label_col}' not found"}

    pos_key = str(positive_label).strip()
    y_all = normalize_str_series(df[label_col])
    valid = y_all.notna()
    if int(valid.sum()) <= 0:
        return {"error": "all_labels_missing"}

    y = y_all[valid]
    mask_pos = (y == pos_key)
    mask_neg = (y != pos_key)

    out: Dict[str, Any] = {}

    for col in numeric_cols:
        if col not in df.columns:
            out[col] = {"exists": False, "reason": "column_not_found"}
            continue

        s = safe_num(df[col])[valid]
        x_pos = s[mask_pos].to_numpy()
        x_neg = s[mask_neg].to_numpy()

        d = _cohen_d(x_pos, x_neg)

        pos_n = int(np.isfinite(x_pos).sum())
        neg_n = int(np.isfinite(x_neg).sum())

        summary: Dict[str, Any] = {
            "exists": True,
            "by_label": {
                "positive": {
                    "n": pos_n,
                    "mean": safe_float(np.nanmean(x_pos)) if x_pos.size > 0 else None,
                    "std": safe_float(np.nanstd(x_pos, ddof=1)) if pos_n > 1 else None,
                    "median": safe_float(np.nanmedian(x_pos)) if x_pos.size > 0 else None,
                },
                "non_positive": {
                    "n": neg_n,
                    "mean": safe_float(np.nanmean(x_neg)) if x_neg.size > 0 else None,
                    "std": safe_float(np.nanstd(x_neg, ddof=1)) if neg_n > 1 else None,
                    "median": safe_float(np.nanmedian(x_neg)) if x_neg.size > 0 else None,
                },
            },
            "cohen_d": float(round(float(d), 6)),
            "effect_size_comment": None,
        }

        ad = abs(d) if np.isfinite(d) else 0.0
        if ad < 0.2:
            summary["effect_size_comment"] = "negligible"
        elif ad < 0.5:
            summary["effect_size_comment"] = "small"
        elif ad < 0.8:
            summary["effect_size_comment"] = "medium"
        else:
            summary["effect_size_comment"] = "large"

        out[col] = summary

    return out


def _categorical_distribution_by_label(
    df: pd.DataFrame,
    *,
    label_col: str,
    positive_label: str,
    categorical_cols: List[str],
    min_share: float,
    observed: bool = False,
) -> Tuple[Dict[str, Any], int]:
    """
    Compute positive-class rate per category level and flag tiny segments (exclude missing labels).
    Returns (result_dict, n_small_segments_total).
    """
    if label_col not in df.columns:
        return ({"error": f"label_col '{label_col}' not found"}, 0)

    pos_key = str(positive_label).strip()
    y_all = normalize_str_series(df[label_col])
    valid = y_all.notna()
    if int(valid.sum()) <= 0:
        return ({"error": "all_labels_missing"}, 0)

    y = y_all[valid]
    pos_mask = (y == pos_key)
    n_total = int(valid.sum())

    out: Dict[str, Any] = {}
    n_small_total = 0

    for col in categorical_cols:
        if col not in df.columns:
            out[col] = {"exists": False, "reason": "column_not_found"}
            continue

        g = normalize_str_series(df[col])[valid]
        tab = (
            pd.DataFrame({"group": g, "is_pos": pos_mask})
            .groupby("group", dropna=False, observed=observed)
            .agg(n=("is_pos", "size"), pos_rate=("is_pos", "mean"))
            .sort_values("n", ascending=False)
        )

        seg_dict: Dict[str, Any] = {}
        for k, v in tab.to_dict(orient="index").items():
            n = int(v["n"])
            rate = float(v["pos_rate"])
            share = float(n / n_total) if n_total > 0 else 0.0
            small = bool(share < float(min_share))
            if small:
                n_small_total += 1

            seg_dict[_keyify(k)] = {
                "n": n,
                "share": float(round(share, 6)),
                "positive_rate": float(round(rate, 6)),
                "small_segment": small,
            }

        out[col] = {"exists": True, "n_levels": int(tab.shape[0]), "segments": seg_dict}

    return (out, int(n_small_total))


# --- 1) Public API ---
def run_distribution_balance_checks(
    *,
    df: pd.DataFrame,
    ctx: Dict[str, Any],
    run_id: Optional[str] = None,
    validation_report: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """
    Run distribution & balance checks in a ctx-driven, artifact-persisted manner.

    YAML SSOT
    ---------
    dataset.label:
      col: "Churn"
      positive: "Yes"

    dataset.schema.dtypes:
      numeric: [...]
      categorical: [...]

    pipeline.data_quality.distribution:
      max_majority_share: 0.80
      min_segment_share: 0.01
      plots:
        enabled: true
        top_n: 10
    
    Returns
    -------
        ModuleReport v1 (Distribution) (also persisted as artifact).
    """
    # --- 2) Runtime validation ---
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG27, run_id=run_id)

    # --- 3) Input validation ---
    if df is None or not isinstance(df, pd.DataFrame):
        raise TypeError(f"[{TAG27}] df must be a pandas DataFrame")

    # --- 4) Resolve config (YAML SSOT) ---

    # --- Global（SSOT） ---
    label_col, pos_label, task_type = get_label_cfg(ctx, TAG27)
    tt = str(task_type).strip().lower()
    task_type_ok = (tt in _SUPPORTED_TASK_TYPES)

    numeric_cols, categorical_cols = get_schema_cols(ctx, TAG27)
    
    # --- Module（step cfg） ---
    get_cfg_dict(ctx, "pipeline.data_quality.distribution", tag=TAG27, required=True)
   
    max_majority_share = get_cfg_float(
        ctx,
        "pipeline.data_quality.distribution.max_majority_share",
        tag=TAG27,
        default=0.80,
        required=False,
        min_value=0.0,
        max_value=1.0,
        strict_type=True,
    )
    min_segment_share = get_cfg_float(
        ctx,
        "pipeline.data_quality.distribution.min_segment_share",
        tag=TAG27,
        default=0.01,
        required=False,
        min_value=0.0,
        max_value=1.0,
        strict_type=True,
    )

    plots_enabled = get_cfg_bool(
        ctx,
        "pipeline.data_quality.distribution.plots.enabled",
        tag=TAG27,
        default=False,
        required=False,
        strict_type=True, 
    )

    top_n_plot = get_cfg_int(
        ctx,
        "pipeline.data_quality.distribution.plots.top_n",
        tag=TAG27,
        default=10,
        required=False,
        min_value=3,
        max_value=50,
        strict_type=True,
    )

    # --- Data facts ---
    label_key = str(label_col).strip()
    pos_key = str(pos_label).strip()
    n_rows = int(df.shape[0])
    is_empty_df = (n_rows == 0)
    label_missing = (label_key not in df.columns)

    # --- Schema presence (for drift hints) ---
    missing_numeric = sorted([c for c in numeric_cols if c not in df.columns])
    missing_categorical = sorted([c for c in categorical_cols if c not in df.columns])

    # --- 5) Core workflow ---
    lg.info(f"[{TAG27}] 📊 Starting distribution & balance checks...")

    # --- Build ModuleReport skeleton (infra) ---
    report = build_module_report(
        ctx=ctx,
        stage="data_quality",
        step="2.7",
        name="distribution",
        tag=TAG27,
        cfg_key="pipeline.data_quality.distribution",
        enabled=True,
        df=df,
        run_id_override=rid,
    )

    report["inputs"] = {
        "df_provided": True,
        "validation_report_provided": bool(validation_report),
    }

    report["thresholds_used"] = {
        "max_majority_share": float(max_majority_share),
        "min_segment_share": float(min_segment_share),
        "top_n_plot": int(top_n_plot),
        "supported_task_types": list(_SUPPORTED_TASK_TYPES)
    }

    report["used_config"] = {
        "label_col": str(label_col).strip(),
        "positive_label": str(pos_label).strip(),
        "numeric_cols": list(numeric_cols),
        "categorical_cols": list(categorical_cols),
        "policy": "exclude_missing_labels_from_balance_and_separation",
        "plots_enabled": bool(plots_enabled),
    }

    report["refs"]["self"] = make_artifact_ref(ctx=ctx, stage="data_quality", name="distribution", run_id=str(rid))

    report["checks"]["preconditions"] = {
        "task_type": {"status": "pass" if task_type_ok else "fail", "task_type": task_type, "expected": list(_SUPPORTED_TASK_TYPES)}
    }
    report["checks"]["schema_presence"] = {
        "missing_numeric_cols": missing_numeric,
        "missing_categorical_cols": missing_categorical,
    }

    # --- Initialize locals to avoid UnboundLocalError ---
    label_balance: Dict[str, Any] = {"skipped": True, "reason": "not_computed_yet"}
    numeric_by_label: Dict[str, Any] = {"skipped": True, "reason": "not_computed_yet"}
    categorical_by_label: Dict[str, Any] = {"skipped": True, "reason": "not_computed_yet"}
    n_small_segments: int = 0

    # --- Core logic ---
    if not task_type_ok:
        label_balance = {"skipped": True, "reason": "task_type_mismatch"}
        numeric_by_label = {"skipped": True, "reason": "task_type_mismatch"}
        categorical_by_label = {"skipped": True, "reason": "task_type_mismatch"}
        n_small_segments = 0
    else:
        label_balance = _compute_label_balance(
            df, label_col=label_key, positive_label=pos_key, max_majority_share=float(max_majority_share)
        )

        if is_empty_df or label_missing:
            numeric_by_label = {"skipped": True, "reason": "empty_df_or_missing_label"}
            categorical_by_label = {"skipped": True, "reason": "empty_df_or_missing_label"}
            n_small_segments = 0
        else:
            numeric_by_label = _numeric_distribution_by_label(
                df, label_col=label_key, positive_label=pos_key, numeric_cols=list(numeric_cols)
            )
            categorical_by_label, n_small_segments = _categorical_distribution_by_label(
                df, label_col=label_key, positive_label=pos_key,
                categorical_cols=list(categorical_cols), min_share=float(min_segment_share)
            )

    report["checks"]["label_balance"] = label_balance
    report["checks"]["numeric_by_label"] = numeric_by_label
    report["checks"]["categorical_by_label"] = categorical_by_label
    report["checks"]["categorical_small_segments"] = {"n_small_segments": int(n_small_segments)}

    # --- Aggregate numeric effects safely ---
    sum_abs_d = 0.0
    n_abs_d = 0
    n_large_effect = 0
    if isinstance(numeric_by_label, dict) and (not numeric_by_label.get("error")) and (not numeric_by_label.get("skipped")):
        for _, info in numeric_by_label.items():
            if not isinstance(info, dict) or info.get("exists") is not True:
                continue
            d = info.get("cohen_d")
            if isinstance(d, (int, float)) and np.isfinite(float(d)):
                ad = abs(float(d))
                sum_abs_d += ad
                n_abs_d += 1
                if ad >= 0.8:
                    n_large_effect += 1

    report["checks"]["numeric_effect_summary"] = {
        "n_effect_samples": int(n_abs_d),
        "sum_abs_cohen_d": safe_float(float(sum_abs_d)),
        "n_large_effect": int(n_large_effect),
    }

    # --- Plots: only run when requested + classification ---
    plots_meta: Dict[str, Any] = {"requested": bool(plots_enabled), "saved": False, "dir": None, "files": []}
    if (not plots_enabled) or (not task_type_ok):
        if plots_enabled and (not task_type_ok):
            plots_meta.update({"skipped": True, "reason": "task_type_mismatch"})
        report["checks"]["plots"] = plots_meta
    else:
        try:
            fig_base = resolve_figures_dir(ctx, tag=TAG27)
            if fig_base is None:
                plots_meta["reason"] = "figures_dir_unresolved"
            else:
                module_name = "distribution"
                fig_dir = fig_base / "data_quality" / module_name
                fig_dir.mkdir(parents=True, exist_ok=True)
                plots_meta["dir"] = str(fig_dir)

                saved_refs: List[Dict[str, Any]] = []
                saved_paths: List[str] = []

                # --- Pull computed results from checks (NO extra nesting) ---
                dist_checks = report.get("checks", {}) if isinstance(report.get("checks"), dict) else {}

                # 1) label_counts
                lb = dist_checks.get("label_balance", {}) if isinstance(dist_checks.get("label_balance"), dict) else {}
                vc = lb.get("value_counts", {}) if isinstance(lb.get("value_counts"), dict) else {}

                if vc:
                    label_pairs = [(str(k), float(v)) for k, v in vc.items()]
                    p1 = fig_dir / f"data_quality_{module_name}_label_counts_{rid}.png"
                    plot_bar_pairs(
                        label_pairs,
                        title=f"Label Counts (run_id={rid})",
                        ylabel="count",
                        out_path=p1,
                        top_n=max(len(label_pairs), 1),
                        sort_desc=True,
                    )
                    saved_refs.append({"name": "label_counts", "path": str(p1)})
                    saved_paths.append(str(p1))

                # 2) numeric_effects_topN (|cohen_d|)
                nb = dist_checks.get("numeric_by_label", {}) if isinstance(dist_checks.get("numeric_by_label"), dict) else {}
                numeric_pairs: List[Tuple[str, float]] = []

                # nb is {col: {exists: True, cohen_d: ...}, ...}
                for col, info in nb.items():
                    if not isinstance(info, dict) or info.get("exists") is not True:
                        continue
                    d = info.get("cohen_d", None)
                    try:
                        numeric_pairs.append((str(col), float(abs(float(d)))))
                    except Exception:
                        continue

                if numeric_pairs:
                    p2 = fig_dir / f"data_quality_{module_name}_numeric_effects_top{top_n_plot}_{rid}.png"
                    plot_bar_pairs(
                        numeric_pairs,
                        title=f"Top-{top_n_plot} Numeric Effects |Cohen's d| (run_id={rid})",
                        ylabel="abs_cohen_d",
                        out_path=p2,
                        top_n=int(top_n_plot),
                        sort_desc=True,
                    )
                    saved_refs.append({"name": "numeric_effects_top", "path": str(p2)})
                    saved_paths.append(str(p2))

                # 3) categorical_spread_topN (positive_rate spread)
                cb = dist_checks.get("categorical_by_label", {}) if isinstance(dist_checks.get("categorical_by_label"), dict) else {}
                cat_pairs: List[Tuple[str, float]] = []

                # cb is {col: {exists: True, segments: {...}}, ...}
                for col, info in cb.items():
                    if not isinstance(info, dict) or info.get("exists") is not True:
                        continue
                    segs = info.get("segments", {})
                    if not isinstance(segs, dict) or not segs:
                        continue

                    rates: List[float] = []
                    for _, seg_info in segs.items():
                        if not isinstance(seg_info, dict):
                            continue
                        r = seg_info.get("positive_rate", None)
                        try:
                            rates.append(float(r))
                        except Exception:
                            continue

                    if rates:
                        cat_pairs.append((str(col), float(max(rates) - min(rates))))

                if cat_pairs:
                    p3 = fig_dir / f"data_quality_{module_name}_categorical_spread_top{top_n_plot}_{rid}.png"
                    plot_bar_pairs(
                        cat_pairs,
                        title=f"Top-{top_n_plot} Categorical Spread (run_id={rid})",
                        ylabel="positive_rate_spread",
                        out_path=p3,
                        top_n=int(top_n_plot),
                        sort_desc=True,
                    )
                    saved_refs.append({"name": "categorical_spread_top", "path": str(p3)})
                    saved_paths.append(str(p3))

                if saved_paths:
                    plots_meta["saved"] = True
                    plots_meta["files"] = saved_paths
                else:
                    plots_meta["saved"] = False
                    plots_meta["reason"] = "no_plot_data"
                    plots_meta["files"] = []

                report.setdefault("refs", {})
                report["refs"].setdefault("figures", [])
                report["refs"]["figures"].extend(saved_refs)

        except Exception as e:
            plots_meta["saved"] = False
            plots_meta["error"] = str(e)

    report.setdefault("checks", {})
    report["checks"]["plots"] = plots_meta

    # --- Event hints (policy-free; for 2.8 mapping) ---
    ensure_event_hints(report, version=1)

    # 1) Task type mismatch (hard)
    if not task_type_ok:
        add_event_hint(
            report,
            code="distribution_task_type_mismatch",
            severity_signal="hard",
            evidence_path="checks.preconditions.task_type",
            context={"task_type_raw": str(task_type), "task_type_norm": str(tt)},
            remediation={
                "action": "set_dataset_label_task_type_to_classification_in_yaml_or_use_correct_module",
                "safe": True,
                "rationale": "2.7 distribution checks assume binary classification; task mismatch makes results unreliable.",
                "post_check": "checks.preconditions.task_type.status == 'pass'",
            },
        )

    # 2) Empty df / missing label (hard)
    if is_empty_df:
        add_event_hint(
            report,
            code="distribution_empty_dataframe",
            severity_signal="hard",
            evidence_path="summary.total_rows",
            context={"n_rows": int(df.shape[0])},
            remediation={
                "action": "fix_ingestion_or_upstream_filters",
                "safe": False,
                "rationale": "Distribution checks cannot run on an empty dataframe.",
                "post_check": "df.shape[0] > 0",
            },
        )

    if label_missing:
        add_event_hint(
            report,
            code="distribution_label_col_missing",
            severity_signal="hard",
            evidence_path="used_config.label_col",
            context={"label_col": str(label_key)},
            remediation={
                "action": "ensure_label_column_exists_and_matches_ssot",
                "safe": False,
                "rationale": "Label column is required for balance and separation diagnostics.",
                "post_check": "df contains label_col",
            },
        )

    # 3) Schema drift (risk)
    if missing_numeric:
        add_event_hint(
            report,
            code="distribution_missing_numeric_schema_cols_in_df",
            severity_signal="risk",
            evidence_path="checks.schema_presence.missing_numeric_cols",
            context={
                "n_missing": int(len(missing_numeric)),
                "examples": [str(x) for x in missing_numeric[:10]],
            },
            remediation={
                "action": "align_schema_or_update_dataset_schema_numeric_list",
                "safe": True,
                "rationale": "Schema drift reduces numeric coverage; SSOT alignment restores deterministic behavior.",
                "post_check": "re-run 2.2 contract and 2.7 to confirm numeric presence",
            },
        )

    if missing_categorical:
        add_event_hint(
            report,
            code="distribution_missing_categorical_schema_cols_in_df",
            severity_signal="risk",
            evidence_path="checks.schema_presence.missing_categorical_cols",
            context={
                "n_missing": int(len(missing_categorical)),
                "examples": [str(x) for x in missing_categorical[:10]],
            },
            remediation={
                "action": "align_schema_or_update_dataset_schema_categorical_list",
                "safe": True,
                "rationale": "Schema drift reduces categorical coverage; SSOT alignment restores deterministic behavior.",
                "post_check": "re-run 2.2 contract and 2.7 to confirm categorical presence",
            },
        )

    # 4) Label balance warnings (risk)
    if isinstance(label_balance, dict) and label_balance.get("exists") is True:
        flag = label_balance.get("imbalance_flag")
        if flag in ("empty", "all_missing"):
            add_event_hint(
                report,
                code="distribution_label_balance_unusable",
                severity_signal="risk",
                evidence_path="checks.label_balance.imbalance_flag",
                context={"imbalance_flag": str(flag)},
                remediation={
                    "action": "fix_label_generation_or_upstream_cleaning",
                    "safe": False,
                    "rationale": "Label balance math is not meaningful when labels are empty/all missing.",
                    "post_check": "label_balance.imbalance_flag == 'ok'",
                },
            )

        if flag == "severe_imbalance":
            add_event_hint(
                report,
                code="distribution_severe_label_imbalance",
                severity_signal="risk",
                evidence_path="checks.label_balance",
                context={
                    "majority_share": label_balance.get("majority_share"),
                    "positive_share": label_balance.get("positive_share"),
                    "max_majority_share": float(max_majority_share),
                },
                remediation={
                    "action": "use_stratified_split_and_consider_class_weight_or_resampling",
                    "safe": True,
                    "rationale": "Severe imbalance may bias training/metrics; mitigation should be explicit and validated.",
                    "post_check": "re-run 4.x split strategy + 7.x evaluation with imbalance-aware metrics",
                },
            )

    # 5) Separation calc errors (risk)
    if isinstance(numeric_by_label, dict) and numeric_by_label.get("error"):
        add_event_hint(
            report,
            code="distribution_numeric_by_label_error",
            severity_signal="risk",
            evidence_path="checks.numeric_by_label",
            context={"error": str(numeric_by_label.get('error'))},
            remediation={
                "action": "fix_label_missingness_or_numeric_parsing_and_rerun",
                "safe": True,
                "rationale": "Numeric separation diagnostics failed; restore valid labels and numeric values to recover signal.",
                "post_check": "checks.numeric_by_label has no 'error'",
            },
        )

    if isinstance(categorical_by_label, dict) and categorical_by_label.get("error"):
        add_event_hint(
            report,
            code="distribution_categorical_by_label_error",
            severity_signal="risk",
            evidence_path="checks.categorical_by_label",
            context={"error": str(categorical_by_label.get('error'))},
            remediation={
                "action": "fix_label_missingness_or_categorical_normalization_and_rerun",
                "safe": True,
                "rationale": "Categorical separation diagnostics failed; restore valid labels to recover segment signals.",
                "post_check": "checks.categorical_by_label has no 'error'",
            },
        )

    # 6) Small segments (risk)
    if int(n_small_segments) > 0:
        add_event_hint(
            report,
            code="distribution_small_segments_present",
            severity_signal="risk",
            evidence_path="checks.categorical_small_segments",
            context={"n_small_segments": int(n_small_segments), "min_segment_share": float(min_segment_share)},
            remediation={
                "action": "bucket_rare_categories_or_adjust_min_segment_share",
                "safe": True,
                "rationale": "Tiny segments make estimated rates unstable; bucketing is deterministic when configured.",
                "post_check": "re-run 2.9 cleaning (rare bucketing) then 2.7 to confirm fewer small segments",
            },
        )

    # 7) Plots failed (fixable)
    if plots_meta.get("requested") and (not plots_meta.get("saved")):
        add_event_hint(
            report,
            code="distribution_plots_failed",
            severity_signal="fixable",
            evidence_path="checks.plots",
            context={"plots_meta": plots_meta},
            remediation={
                "action": "ensure_figures_dir_and_plot_deps_available_or_disable_plots",
                "safe": True,
                "rationale": "Plots are best-effort diagnostics; failure doesn't invalidate computed checks.",
                "post_check": "checks.plots.saved == True or checks.plots.requested == False",
            },
        )

    # --- 6) Status + gate-friendly summary ---
    issues: List[str] = []
    warnings: List[str] = []

    if isinstance(label_balance, dict) and label_balance.get("exists"):
        if label_balance.get("imbalance_flag") in ("empty", "all_missing"):
            warnings.append(f"label_balance_{label_balance.get('imbalance_flag')}")

    if isinstance(numeric_by_label, dict) and numeric_by_label.get("error"):
        warnings.append(f"numeric_by_label_error:{numeric_by_label.get('error')}")

    if isinstance(categorical_by_label, dict) and categorical_by_label.get("error"):
        warnings.append(f"categorical_by_label_error:{categorical_by_label.get('error')}")

    if is_empty_df:
        issues.append("empty_dataframe")
    if label_missing:
        issues.append("label_col_missing")
    if not task_type_ok:
        issues.append("task_type_mismatch")

    if missing_numeric:
        warnings.append("numeric_cols_missing_in_df")
    if missing_categorical:
        warnings.append("categorical_cols_missing_in_df")

    if isinstance(label_balance, dict) and label_balance.get("exists") and label_balance.get("imbalance_flag") == "severe_imbalance":
        warnings.append("severe_label_imbalance")

    if int(n_small_segments) > 0:
        warnings.append("small_segments_present")

    if plots_meta.get("requested") and (not plots_meta.get("saved")):
        warnings.append("plots_failed")

    warnings = sorted(set(warnings))

    lb = report.get("checks", {}).get("label_balance", {})
    lb_pos_share = lb.get("positive_share") if isinstance(lb, dict) else None
    lb_majority = lb.get("majority_share") if isinstance(lb, dict) else None
    lb_missing_share = lb.get("missing_share") if isinstance(lb, dict) else None

    avg_abs_cohen_d = safe_float(float(sum_abs_d / n_abs_d)) if n_abs_d > 0 else None

    _metrics = {
        "n_numeric_cols_cfg": int(len(numeric_cols)),
        "n_categorical_cols_cfg": int(len(categorical_cols)),
        "n_numeric_missing": int(len(missing_numeric)),
        "n_categorical_missing": int(len(missing_categorical)),
        "positive_share_valid": lb_pos_share if isinstance(lb_pos_share, (int, float)) else None,
        "majority_share_valid": lb_majority if isinstance(lb_majority, (int, float)) else None,
        "missing_label_share_all": lb_missing_share if isinstance(lb_missing_share, (int, float)) else None,
        "n_small_segments": int(n_small_segments),
        "avg_abs_cohen_d": avg_abs_cohen_d,
        "n_large_effect": int(n_large_effect),
    }

    _notes = [
        f"label_col={label_key}",
        f"positive_label={pos_key}",
        "missing_labels_excluded=true",
        "guidance: use label_balance (imbalance), cohen_d (numeric separation), categorical positive_rate (segment risk).",
    ]

    overall = "fail" if issues else ("warn" if warnings else "pass")

    report = finalize_module_report(
        ctx=ctx,
        report=report,
        overall_status=overall,
        issues=issues,
        warnings=warnings,
        metrics=_metrics,
        notes=_notes,
        df=df,
    )

    if overall == "pass":
        lg.info(f"[{TAG27}] ✅ Distribution & balance checks completed.")
    elif overall == "warn":
        lg.warning(f"[{TAG27}] ⚠ Distribution & balance checks completed with warnings: {warnings}")
    else:
        lg.error(f"[{TAG27}] ❌ Distribution & balance checks failed. | issues={issues}")

    # --- 7) Persist artifact ---
    out_path = resolve_artifact_path(
        ctx=ctx,
        stage="data_quality",
        name="distribution",
        run_id=str(rid),
        tag=TAG27,
    )
    write_json(out_path, report, tag=TAG27, indent=2)
    lg.info(f"[{TAG27}] 🧾 Distribution artifact saved: {out_path}")

    return report

In [10]:
# --- 2.8 Pre-Analysis Summary / Report ---
#
# What:
#   Aggregate 2.2–2.7 artifacts into a compact, run-level pre-modeling summary.
#
# How:
#   - Load artifacts via SSOT naming (resolve_artifact_path), fallback to legacy filenames
#   - Extract contract/readiness status, major issues, and top signals (numeric/categorical)
#   - Optionally attach a domain addon summary (e.g., Telco churn addon)
#   - Persist one JSON ModuleReport v1 under artifacts (SSOT naming)
#
# Why:
#   Repeatable, auditable pre-modeling checklist that reduces cognitive load.

from __future__ import annotations

from typing import Any, Dict, List, Optional, Tuple

import pandas as pd  # used only for optional df validation

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint
from src.pipeline.summary_utils import (
    get_overall_status,
    get_issues,
    get_warnings,
    get_metrics,
    get_total_rows,
    get_total_cols,
    make_upstream_block,
    dedupe_sorted,
    norm_status,
    has_non_empty_list,
    as_dict,
    extract_event_hints,
    dedupe_event_hints,
    collect_upstream_event_hints,
    load_artifact_json_with_runid_fallback,
)
from src.utils.artifact_utils import resolve_artifact_path, write_json
from src.utils.cfg_utils import get_cfg_int

TAG28 = "PRE_SUMMARY"


# --- Internal helpers ---
def _extract_semantic_aggregate(semantic_report: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    """Extract gate-friendly semantic signals from ModuleReport v1 summary.metrics."""
    if not isinstance(semantic_report, dict):
        return {"n_rules": None, "n_failed_rules": None, "total_violations": None, "failed_rules_preview": []}

    m = get_metrics(semantic_report)  # expects ModuleReport v1 summary.metrics
    n_rules = m.get("n_rules", None)
    n_failed_rules = m.get("n_failed_rules", None)
    total_violations = m.get("total_violations", None)

    # Optional drill-down preview (NOT used for gating)
    failed_rules_preview: List[Any] = []
    checks = semantic_report.get("checks", {}) if isinstance(semantic_report.get("checks"), dict) else {}
    sem = checks.get("semantic", {}) if isinstance(checks.get("semantic"), dict) else {}
    sem_sum = sem.get("summary", {}) if isinstance(sem.get("summary"), dict) else {}
    fr = sem_sum.get("failed_rules")
    if isinstance(fr, list):
        failed_rules_preview = list(fr)[:5]

    return {
        "n_rules": n_rules,
        "n_failed_rules": n_failed_rules,
        "total_violations": total_violations,
        "failed_rules_preview": failed_rules_preview,
    }


def extract_semantic_failed_checks(report: Optional[Dict[str, Any]]) -> List[Any]:
    if not isinstance(report, dict):
        return []

    # 1) ModuleReport v1 summary
    s = report.get("summary") if isinstance(report.get("summary"), dict) else {}
    v = s.get("failed_checks")
    if isinstance(v, list):
        return v

    # 2) common legacy
    v = report.get("failed_checks")
    if isinstance(v, list):
        return v

    # 3) ModuleReport v1 checks block (common pattern)
    checks = report.get("checks") if isinstance(report.get("checks"), dict) else {}
    sem = checks.get("semantic") if isinstance(checks.get("semantic"), dict) else {}
    for k in ("failed_checks", "failed_rules", "failed"):
        v = sem.get(k)
        if isinstance(v, list):
            return v

    return []


def _get_dist_aggregate(dist_report: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    """Extract aggregated numeric separation metrics from 2.7 distribution report."""
    if not dist_report or not isinstance(dist_report, dict):
        return {}

    metrics = get_metrics(dist_report)

    checks = dist_report.get("checks", {}) if isinstance(dist_report.get("checks", {}), dict) else {}
    numeric_effect_summary = checks.get("numeric_effect_summary", {})
    if not isinstance(numeric_effect_summary, dict):
        numeric_effect_summary = {}

    label_balance = checks.get("label_balance", {})
    if not isinstance(label_balance, dict):
        label_balance = {}

    return {
        "avg_abs_cohen_d": metrics.get("avg_abs_cohen_d", None),
        "n_large_effect": metrics.get("n_large_effect", None),
        "n_effect_samples": numeric_effect_summary.get("n_effect_samples", None),
        "sum_abs_cohen_d": numeric_effect_summary.get("sum_abs_cohen_d", None),
        "majority_share_valid": metrics.get("majority_share_valid", None),
        "positive_share_valid": metrics.get("positive_share_valid", None),
        "imbalance_flag": label_balance.get("imbalance_flag", None),
    }

def aggregate_upstream_report_signals(
    *,
    contract_report: Optional[Dict[str, Any]],
    health_report: Optional[Dict[str, Any]],
    semantic_report: Optional[Dict[str, Any]],
    readiness_report: Optional[Dict[str, Any]],
    domain_core_report: Optional[Dict[str, Any]] = None,
    domain_addon_telco_report: Optional[Dict[str, Any]] = None,
    distribution_report: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """
    Advisory-only summary (no decision):
    - blockers: ONLY hard-gate blockers (2.2 contract + 2.5 readiness).
    - critical_warnings: soft-gate fails / severe risks (e.g., leakage hints, severe imbalance).
    - warnings: soft-gate warns / missing soft artifacts / minor issues.
    """
    blockers: List[str] = []
    critical_warnings: List[str] = []
    warnings: List[str] = []
    suggested_actions: List[str] = []

    # --- 2.2 Contract (HARD) ---
    c_status = norm_status(get_overall_status(contract_report))
    c_issues = get_issues(contract_report)

    if contract_report is None:
        blockers.append("missing_contract_artifact")
        suggested_actions.append("run_contract_validation")
    else:
        if c_status == "fail" or c_status == "unknown":
            blockers.append("contract_not_pass")
            suggested_actions.append("rerun_or_fix_contract")
        elif c_status == "warn":
            warnings.append("contract_warn")
            suggested_actions.append("review_contract_warnings")

        if has_non_empty_list(c_issues):
            blockers.append("contract_has_issues")
            suggested_actions.append("inspect_contract_issues")


    # --- 2.5 Readiness (HARD) ---
    r_status = norm_status(get_overall_status(readiness_report))
    r_issues = get_issues(readiness_report)

    if readiness_report is None:
        blockers.append("missing_readiness_artifact")
        suggested_actions.append("run_experimental_readiness")
    else:
        if r_status == "fail" or r_status == "unknown":
            blockers.append("readiness_not_pass")
            suggested_actions.append("rerun_or_fix_readiness")
        elif r_status == "warn":
            warnings.append("readiness_warn")
            suggested_actions.append("review_readiness_warnings")

        if has_non_empty_list(r_issues):
            blockers.append("readiness_has_issues")
            suggested_actions.append("inspect_readiness_issues")

    # --- 2.3 Health (SOFT) ---
    h_status = norm_status(get_overall_status(health_report))
    h_issues = get_issues(health_report)

    if health_report is None:
        warnings.append("missing_health_artifact")
        suggested_actions.append("run_health_checks")
    else:
        if h_status == "fail":
            critical_warnings.append("health_failed")
            suggested_actions.append("review_health_fail_reasons")
        elif h_status == "warn":
            warnings.append("health_warn")
            suggested_actions.append("review_problem_columns_missingness_cardinality")

        if has_non_empty_list(h_issues):
            warnings.append("health_has_issues")
            suggested_actions.append("inspect_health_issues")

    # --- 2.4 Semantic (SOFT but can be severe) ---
    s_status = norm_status(get_overall_status(semantic_report))
    s_failed = extract_semantic_failed_checks(semantic_report)

    if semantic_report is None:
        warnings.append("missing_semantic_artifact")
        suggested_actions.append("run_semantic_checks")
    else:
        if s_status == "fail":
            critical_warnings.append("semantic_failed")
            suggested_actions.append("fix_semantic_rules_or_mappings")
        elif s_status == "warn":
            warnings.append("semantic_warn")
            suggested_actions.append("review_semantic_warnings")

        if has_non_empty_list(s_failed):
            critical_warnings.append("semantic_failed_checks_present")
            suggested_actions.append("inspect_failed_semantic_checks")

    # --- 2.6.1 Domain core (SOFT) ---
    dc_status = norm_status(get_overall_status(domain_core_report))
    dc_issues = get_issues(domain_core_report)

    if domain_core_report is None:
        warnings.append("missing_domain_core_artifact")
        suggested_actions.append("run_domain_core_diagnostics")
    else:
        if dc_status == "fail":
            critical_warnings.append("domain_core_fail")
            suggested_actions.append("use_domain_core_findings_to_focus_data_integrity")
        elif dc_status == "warn":
            warnings.append("domain_core_warn")
            suggested_actions.append("use_domain_core_findings_to_focus_data_integrity")

        if has_non_empty_list(dc_issues):
            warnings.append("domain_core_has_issues")
            suggested_actions.append("inspect_domain_core_issues")

    # --- 2.6.2 Domain addon (SOFT + leakage hints) ---
    da_status = norm_status(get_overall_status(domain_addon_telco_report))
    da_warnings = get_warnings(domain_addon_telco_report)
    da_hints = extract_event_hints(domain_addon_telco_report)

    if domain_addon_telco_report is not None:
        if da_status == "fail":
            critical_warnings.append("domain_addon_telco_fail")
            suggested_actions.append("use_domain_addon_telco_findings_to_focus_data_integrity")
        elif da_status == "warn":
            warnings.append("domain_addon_telco_warn")
            suggested_actions.append("use_domain_addon_telco_findings_to_focus_data_integrity")

        # escalate leakage hint from warnings or event_hints
        if isinstance(da_warnings, list) and "potential_leakage_features_flagged" in da_warnings:
            critical_warnings.append("domain_addon_telco_leakage_hints_present")
            suggested_actions.append("prioritize_leakage_checks_in_data_integrity")

        for h in da_hints:
            code = str(h.get("code", "")).strip()
            sev = str(h.get("severity_signal", "")).strip().lower()
            if code and ("leakage" in code or "leaky" in code) and sev in {"risk", "hard", "fixable"}:
                critical_warnings.append("domain_addon_telco_event_hints_leakage")
                suggested_actions.append("inspect_domain_addon_telco_event_hints_and_run_leakage_scan")
                break

    # --- 2.7 Distribution (SOFT but severe imbalance can be critical) ---
    d_status = norm_status(get_overall_status(distribution_report))
    d_issues = get_issues(distribution_report)

    if distribution_report is None:
        warnings.append("missing_distribution_artifact")
        suggested_actions.append("run_distribution_checks")
    else:
        if d_status == "fail":
            critical_warnings.append("distribution_failed")
            suggested_actions.append("inspect_distribution_issues_or_thresholds")
        elif d_status == "warn":
            warnings.append("distribution_warn")
            suggested_actions.append("review_distribution_findings")

        if has_non_empty_list(d_issues):
            warnings.append("distribution_has_issues")
            suggested_actions.append("inspect_distribution_issues")

        rep = as_dict(distribution_report)
        checks = rep.get("checks") if isinstance(rep.get("checks"), dict) else {}
        lb = checks.get("label_balance") if isinstance(checks.get("label_balance"), dict) else {}

        imb = str(lb.get("imbalance_flag", "")).strip().lower()
        if imb == "severe_imbalance":
            critical_warnings.append("severe_label_imbalance")
            suggested_actions.append("plan_imbalance_mitigation_or_sampling")

        ssum = rep.get("summary") if isinstance(rep.get("summary"), dict) else {}
        ws = ssum.get("warnings", [])
        if isinstance(ws, list) and "small_segments_present" in ws:
            warnings.append("small_segments_present")
            suggested_actions.append("review_small_segments_and_split_strategy")

    # --- stable output ---
    return {
        "blockers": sorted(set(blockers)),
        "critical_warnings": sorted(set(critical_warnings)),
        "warnings": sorted(set(warnings)),
        "suggested_actions": sorted(set(suggested_actions)),
        "status_snapshot": {
            "contract": c_status,
            "health": h_status,
            "semantic": s_status,
            "readiness": r_status,
            "domain_core": dc_status,
            "domain_addon_telco": da_status,
            "distribution": d_status,
        },
    }


def _extract_top_numeric_effects(dist_report: Optional[Dict[str, Any]], *, top_k: int = 5) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    if not dist_report:
        return out

    checks = dist_report.get("checks", {}) if isinstance(dist_report.get("checks"), dict) else {}
    numeric = checks.get("numeric_by_label", {})
    if isinstance(numeric, dict) and (numeric.get("skipped") or numeric.get("error")):
        return out
    if not isinstance(numeric, dict):
        return out

    scored: List[Tuple[str, float, Dict[str, Any]]] = []
    for col, info in numeric.items():
        if not isinstance(info, dict):
            continue
        d = info.get("cohen_d", 0.0)
        try:
            d_f = float(d)
        except Exception:
            d_f = 0.0
        scored.append((str(col), abs(d_f), info))

    scored = sorted(scored, key=lambda x: x[1], reverse=True)[: int(top_k)]
    for col, _abs_d, info in scored:
        by_label = info.get("by_label", {}) if isinstance(info.get("by_label"), dict) else {}
        out.append(
            {
                "column": col,
                "cohen_d": float(info.get("cohen_d", 0.0) or 0.0),
                "effect_size_comment": info.get("effect_size_comment", None),
                "positive_summary": by_label.get("positive", {}),
                "non_positive_summary": by_label.get("non_positive", {}),
            }
        )
    return out


def _extract_key_categorical_patterns(
    dist_report: Optional[Dict[str, Any]], *, top_k: int = 3
) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    if not dist_report:
        return out

    checks = dist_report.get("checks", {}) if isinstance(dist_report.get("checks"), dict) else {}
    cat = checks.get("categorical_by_label", {})
    if not isinstance(cat, dict):
        return out
    if cat.get("skipped") or cat.get("error"):
        return out

    spreads: List[Tuple[str, float, Dict[str, Any]]] = []
    for col, info in cat.items():
        if not isinstance(info, dict):
            continue
        segs = info.get("segments", {})
        if not isinstance(segs, dict) or not segs:
            continue

        rates: List[float] = []
        for _, seg_info in segs.items():
            if not isinstance(seg_info, dict):
                continue
            r = seg_info.get("positive_rate", None)
            if r is None:
                continue
            try:
                rates.append(float(r))
            except Exception:
                continue

        if not rates:
            continue

        spread = float(max(rates) - min(rates))
        spreads.append((str(col), spread, info))

    spreads = sorted(spreads, key=lambda x: x[1], reverse=True)[: int(top_k)]
    for col, spread, info in spreads:
        out.append(
            {
                "column": col,
                "positive_rate_spread": float(round(spread, 6)),
                "n_levels": info.get("n_levels", None),
                "segments": info.get("segments", {}),
            }
        )
    return out


def _extract_telco_highlights(telco_report: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    """Extract Telco churn highlights from the 2.6.2 addon report (your current structure)."""
    if not isinstance(telco_report, dict):
        return {"note": "telco_addon_report_not_available"}

    checks = telco_report.get("checks") if isinstance(telco_report.get("checks"), dict) else {}

    churn_struct = checks.get("positive_rate_by_segment", {})
    leakage = checks.get("telco_leakage_pre_scan", {})
    anomalies = checks.get("telco_domain_anomalies", {})
    hypotheses = checks.get("telco_domain_hypotheses", [])
    event_hints = checks.get("event_hints", {})

    key_segments: Dict[str, Any] = {}
    if isinstance(churn_struct, dict):
        for seg_col in ["Contract", "tenure_bucket", "monthly_charge_bucket"]:
            if seg_col in churn_struct:
                key_segments[seg_col] = churn_struct[seg_col]

    return {
        "key_segment_churn_patterns": key_segments,
        "telco_leakage_hints": leakage,
        "telco_anomalies": anomalies,
        "telco_hypotheses": hypotheses,
        "event_hints": event_hints,
    }


# --- 1) Public API ---
def build_pre_analysis_summary(
    *,
    df: Optional[Any] = None,  # optional; 2.8 is artifact-driven
    artifacts: Optional[Dict[str, Any]] = None,  # orchestrator passes this for agg steps
    ctx: Dict[str, Any],
    run_id: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Build a run-level pre-analysis summary (2.8).

    It loads 2.2–2.7 artifacts, extracts key risks & signals, and persists
    one ModuleReport v1 JSON summary under artifacts.
    """
    # --- 2) Runtime validation ---
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG28, run_id=run_id)

    # --- 3) Input validation ---
    if df is not None and not isinstance(df, pd.DataFrame):
        raise TypeError(f"[{TAG28}] df must be a pandas DataFrame when provided (2.8 does not use df)")

    # --- 4) Resolve config (YAML SSOT) ---
    top_k_numeric = get_cfg_int(
        ctx,
        "pipeline.data_quality.summary.top_k_numeric",
        tag=TAG28,
        default=5,
        required=False,
        min_value=1,
        max_value=50,
        strict_type=True,
    )
    top_k_categorical = get_cfg_int(
        ctx,
        "pipeline.data_quality.summary.top_k_categorical",
        tag=TAG28,
        default=3,
        required=False,
        min_value=1,
        max_value=50,
        strict_type=True,
    )

    # --- 5) Core workflow ---
    lg.info(f"[{TAG28}] 📋 Building pre-analysis summary for run_id={rid}...")

    report = build_module_report(
        ctx=ctx,
        stage="data_quality",
        step="2.8",
        name="pre_analysis_summary",
        tag=TAG28,
        cfg_key="pipeline.data_quality.summary",
        enabled=True,
        df=None,
        run_id_override=rid,
    )

    report["inputs"] = {
        "df_provided": df is not None,
        "artifacts_provided": artifacts is not None,
    }
    report["thresholds_used"] = {
        "top_k_numeric": int(top_k_numeric),
        "top_k_categorical": int(top_k_categorical),
    }
    report["used_config"] = {
        "policy": "artifact_driven_summary; ssot_first_then_legacy_fallback",
        "legacy_fallback_dir": "same_directory_as_ssot_artifacts",
    }
    if not isinstance(report.get("refs"), dict):
        report["refs"] = {}
    report["refs"]["self"] = make_artifact_ref(
        ctx=ctx,
        stage="data_quality",
        name="pre_analysis_summary",
        run_id=str(rid),
    )

    ensure_event_hints(report)

    # --- Load artifacts (2.2–2.7) ---
    validate_report, validate_path, validate_meta, validate_rid = load_artifact_json_with_runid_fallback(
        ctx=ctx,
        lg=lg,
        stage="data_quality",
        name="contract",
        run_id=str(rid),
        tag=TAG28,
        legacy_filenames=[f"validate_report_{rid}.json"],
    )

    health_report, health_path, health_meta, health_rid = load_artifact_json_with_runid_fallback(
        ctx=ctx,
        lg=lg,
        stage="data_quality",
        name="health",
        run_id=str(rid),
        tag=TAG28,
        legacy_filenames=[f"data_health_{rid}.json"],
    )

    semantic_report, semantic_path, semantic_meta, semantic_rid = load_artifact_json_with_runid_fallback(
        ctx=ctx,
        lg=lg,
        stage="data_quality",
        name="semantic",
        run_id=str(rid),
        tag=TAG28,
        legacy_filenames=[f"semantic_checks_{rid}.json"],
    )

    exp_ready_report, exp_ready_path, exp_meta, readiness_rid = load_artifact_json_with_runid_fallback(
        ctx=ctx,
        lg=lg,
        stage="data_quality",
        name="readiness",
        run_id=str(rid),
        tag=TAG28,
        legacy_filenames=[f"experimental_readiness_{rid}.json"],
    )

    core_report, core_path, core_meta, core_rid = load_artifact_json_with_runid_fallback(
        ctx=ctx,
        lg=lg,
        stage="data_quality",
        name="domain_core",
        run_id=str(rid),
        tag=TAG28,
        legacy_filenames=[
            f"classification_domain_core_{rid}.json",
        ],
    )

    telco_churn_report, telco_churn_path, telco_churn_meta, telco_churn_rid = load_artifact_json_with_runid_fallback(
        ctx=ctx,
        lg=lg,
        stage="data_quality",
        name="domain_addon_telco",
        run_id=str(rid),
        tag=TAG28,
        legacy_filenames=[
            f"domain_addon_telco_{rid}.json",
        ],
    )
    domain_summary = _extract_telco_highlights(telco_churn_report)

    dist_report, dist_path, dist_meta, dist_rid = load_artifact_json_with_runid_fallback(
        ctx=ctx,
        lg=lg,
        stage="data_quality",
        name="distribution",
        run_id=str(rid),
        tag=TAG28,
        legacy_filenames=[
            f"distribution_balance_{rid}.json",
        ],
    )

    # --- Extract statuses / risks / signals ---
    contract_status = get_overall_status(validate_report)
    contract_issues = get_issues(validate_report)

    exp_status = get_overall_status(exp_ready_report)
    exp_issues = get_issues(exp_ready_report)

    total_rows = None
    total_cols = None
    for rep in (validate_report, health_report, dist_report, core_report, telco_churn_report):
        if total_rows is None:
            total_rows = get_total_rows(rep)
        if total_cols is None:
            total_cols = get_total_cols(rep)
        if total_rows is not None and total_cols is not None:
            break

    problem_columns: List[Any] = []
    issue_types: Dict[str, Any] = {}
    if isinstance(health_report, dict):
        summary_health = health_report.get("summary", {}) or {}
        problem_columns = summary_health.get("problem_columns", []) or []
        issue_types = summary_health.get("issue_types", {}) or {}

    label_balance: Dict[str, Any] = {}
    if isinstance(dist_report, dict):
        checks = dist_report.get("checks", {}) if isinstance(dist_report.get("checks"), dict) else {}
        label_balance = checks.get("label_balance", {}) if isinstance(checks.get("label_balance"), dict) else {}

    top_numeric_effects = _extract_top_numeric_effects(dist_report, top_k=int(top_k_numeric))
    key_categorical_patterns = _extract_key_categorical_patterns(dist_report, top_k=int(top_k_categorical))

    dist_agg = _get_dist_aggregate(dist_report)
    semantic_agg = _extract_semantic_aggregate(semantic_report)

    upstream: Dict[str, Any] = {
        "contract": make_upstream_block(validate_report, name="contract", path=validate_path, meta=validate_meta, key_metrics={"total_rows": get_total_rows(validate_report)}),
        "health": make_upstream_block(health_report, name="health", path=health_path, meta=health_meta, key_metrics={"total_rows": get_total_rows(health_report)}),
        "semantic": make_upstream_block(
            semantic_report,
            name="semantic",
            path=semantic_path,
            meta=semantic_meta,
            key_metrics={
                "n_rules": semantic_agg.get("n_rules"),
                "n_failed_rules": semantic_agg.get("n_failed_rules"),
                "total_violations": semantic_agg.get("total_violations"),
                "failed_rules_preview": semantic_agg.get("failed_rules_preview", []),
            }
        ),
        "readiness": make_upstream_block(exp_ready_report, name="readiness", path=exp_ready_path, meta=exp_meta, key_metrics={"total_rows": get_total_rows(exp_ready_report)}),
        "domain_core": make_upstream_block(core_report, name="domain_core", path=core_path, meta=core_meta, key_metrics={"total_rows": get_total_rows(core_report)}),
        "domain_addon_telco":make_upstream_block(telco_churn_report, name="domain_addon_telco", path=telco_churn_path, meta=telco_churn_meta, key_metrics={"total_rows": get_total_rows(telco_churn_report)}),
        "distribution": make_upstream_block(dist_report, name="distribution", path=dist_path, meta=dist_meta, key_metrics={**dist_agg, "total_rows": get_total_rows(dist_report)}),
    }
    
    # --- refs.dependencies (lineage) ---
    deps: Dict[str, Any] = {}

    if validate_report is not None:
        deps["contract"] = make_artifact_ref(ctx=ctx, stage="data_quality", name="contract", run_id=str(validate_rid))
    if health_report is not None:
        deps["health"] = make_artifact_ref(ctx=ctx, stage="data_quality", name="health", run_id=str(health_rid))
    if semantic_report is not None:
        deps["semantic"] = make_artifact_ref(ctx=ctx, stage="data_quality", name="semantic", run_id=str(semantic_rid))
    if exp_ready_report is not None:
        deps["readiness"] = make_artifact_ref(ctx=ctx, stage="data_quality", name="readiness", run_id=str(readiness_rid))
    if core_report is not None:
        deps["domain_core"] = make_artifact_ref(ctx=ctx, stage="data_quality", name="domain_core", run_id=str(core_rid))
    if telco_churn_report is not None:
        deps["domain_addon_telco"] = make_artifact_ref(
            ctx=ctx,
            stage="data_quality",
            name="domain_addon_telco",
            run_id=str(telco_churn_rid),
        )
    if dist_report is not None:
        deps["distribution"] = make_artifact_ref(ctx=ctx, stage="data_quality", name="distribution", run_id=str(dist_rid))

    if not isinstance(report.get("refs"), dict):
        report["refs"] = {}
    report["refs"]["dependencies"] = deps

    payload: Dict[str, Any] = {
        "run_id": str(rid),
        "high_level": {
            "total_rows": total_rows,
            "total_cols": total_cols,
            "contract_status": contract_status,
            "experimental_readiness_status": exp_status,
        },
        "risks": {
            "contract_issues": contract_issues,
            "experimental_readiness_issues": exp_issues,
            "data_quality_problem_columns": problem_columns,
            "data_quality_issue_types": issue_types,
            "semantic_risks": {
                "n_failed_rules": semantic_agg.get("n_failed_rules"),
                "total_violations": semantic_agg.get("total_violations"),
                "failed_rules_preview": semantic_agg.get("failed_rules_preview", []),
            },
        },
        "signals": {
            "label_balance": label_balance,
            "top_numeric_effects_by_cohen_d": top_numeric_effects,
            "key_categorical_patterns": key_categorical_patterns,
        },
        "refs": {"upstream_blocks": upstream},
    }

    payload["domain_summary"] = domain_summary

    if not isinstance(report.get("checks"), dict):
        report["checks"] = {}
    report["checks"]["pre_analysis_summary"] = payload

    # --- Upstream run_id mismatch detection ---
    pre_warnings: List[str] = []

    upstream_rids: List[str] = []
    for x in [validate_rid, health_rid, semantic_rid, readiness_rid, core_rid, telco_churn_rid, dist_rid]:
        if isinstance(x, str) and x.strip():
            upstream_rids.append(x.strip())

    upstream_rid_set = sorted(set(upstream_rids))
    report["checks"]["upstream_run_ids"] = {"current": str(rid), "set": upstream_rid_set}

    if upstream_rid_set and (len(upstream_rid_set) > 1 or upstream_rid_set[0] != str(rid)):
        pre_warnings.append("upstream_run_id_mismatch")
        add_event_hint(
            report,
            code="upstream_run_id_mismatch",
            severity_signal="risk",
            evidence_path="checks.upstream_run_ids",
            context={"current_run_id": str(rid), "upstream_run_ids": upstream_rid_set},
            remediation={
                "action": "prefer_full_run_for_one_run_id_or_accept_lineage",
                "safe": True,
                "rationale": "Ad-hoc module execution is normal; record dependencies for auditability.",
                "post_check": "verify refs.dependencies exists and points to upstream artifacts used by this summary",
            },
        )

    # --- Collect upstream event hints into 2.8 (single hub) ---
    upstream_for_hints: List[Tuple[str, Optional[Dict[str, Any]]]] = [
        ("data_quality.contract", validate_report),
        ("data_quality.health", health_report),
        ("data_quality.semantic", semantic_report),
        ("data_quality.readiness", exp_ready_report),
        ("data_quality.domain_core", core_report),
        ("data_quality.domain_addon_telco", telco_churn_report),
        ("data_quality.distribution", dist_report),
    ]
    collected_hints = collect_upstream_event_hints(upstream_for_hints, max_hints=200)

    ensure_event_hints(report)
    eh = report["checks"]["event_hints"]
    hints_list = eh.get("hints")
    if isinstance(hints_list, list) and collected_hints:
        hints_list.extend(collected_hints)
    eh["hints"] = dedupe_event_hints(eh.get("hints", []))

    # --- 6) Status + gate-friendly summary ---
    issues: List[str] = []
    warnings: List[str] = []
    notes: List[str] = []

    warnings.extend(pre_warnings)

    # Basic payload sanity
    ps = report.get("checks", {}).get("pre_analysis_summary")
    if not isinstance(ps, dict):
        issues.append("pre_summary_payload_missing")

    # Missing upstream artifacts => warnings (visibility)
    if validate_report is None:
        warnings.append("missing_contract_artifact")
    if exp_ready_report is None:
        warnings.append("missing_readiness_artifact")
    if health_report is None:
        warnings.append("missing_health_artifact")
    if semantic_report is None:
        warnings.append("missing_semantic_artifact")
    if core_report is None:
        warnings.append("missing_domain_core_artifact")
    if telco_churn_report is None:
        warnings.append("missing_domain_addon_telco_artifact")
    if dist_report is None:
        warnings.append("missing_distribution_artifact")


    advisory: Dict[str, Any] = {}
    try:
        advisory = aggregate_upstream_report_signals(
            contract_report=validate_report,
            health_report=health_report,
            semantic_report=semantic_report,
            readiness_report=exp_ready_report,
            domain_core_report=core_report,
            domain_addon_telco_report=telco_churn_report,
            distribution_report=dist_report,
        )
    except Exception as e:
        lg.exception(f"[{TAG28}] advisory aggregation failed: {e}")
        advisory = {
            "blockers": [],
            "critical_warnings": ["advisory_build_failed"],
            "warnings": [],
            "suggested_actions": ["inspect_pre_summary_or_upstream_artifacts"],
            "status_snapshot": {},
            "error": str(e),
        }

    # Persist advisory for auditability
    report["checks"]["advisory"] = advisory

    blockers_list = advisory.get("blockers", [])
    if isinstance(blockers_list, list) and blockers_list:
        warnings.append("hard_blockers_present")
        notes.append(f"hard_blockers={blockers_list}")

    warns_list = advisory.get("warnings", [])
    if isinstance(warns_list, list) and warns_list:
        warnings.extend(warns_list)

    crit_list = advisory.get("critical_warnings", [])
    if not isinstance(crit_list, list):
        crit_list = []

    if crit_list:
        if not isinstance(report.get("summary"), dict):
            report["summary"] = {}
        report["summary"]["critical_warnings"] = crit_list

    notes.append("2.8 is artifact-driven; df is not required.")
    if upstream_rid_set and (len(upstream_rid_set) > 1 or upstream_rid_set[0] != str(rid)):
        notes.append(f"upstream_run_id_set={upstream_rid_set}")

    snapshot = advisory.get("status_snapshot", {})
    if isinstance(snapshot, dict) and snapshot:
        snap_pairs = [f"{k}:{v}" for k, v in snapshot.items() if v is not None]
        if snap_pairs:
            notes.append("status_snapshot=" + ",".join(snap_pairs))

    actions = advisory.get("suggested_actions", [])
    if isinstance(actions, list) and actions:
        notes.append(f"suggested_actions={actions[:8]}")

    if not isinstance(report.get("summary"), dict):
        report["summary"] = {}
    if total_rows is not None:
        report["summary"]["total_rows"] = int(total_rows)
    if total_cols is not None:
        report["summary"]["total_cols"] = int(total_cols)

    issues = dedupe_sorted(issues)
    warnings = dedupe_sorted(warnings)

    overall = "fail" if issues else ("warn" if (warnings or crit_list) else "pass")

    report = finalize_module_report(
        ctx=ctx,
        report=report,
        overall_status=overall,
        issues=issues,
        warnings=warnings,
        metrics={"top_k_numeric": int(top_k_numeric), "top_k_categorical": int(top_k_categorical)},
        notes=notes,
        df=None,
    )

    if overall == "pass":
        lg.info(f"[{TAG28}] ✅ Pre-analysis summary built.")
    elif overall == "warn":
        lg.warning(f"[{TAG28}] ⚠ Pre-analysis summary built. warnings={warnings} critical_warnings={crit_list}")
    else:
        lg.error(f"[{TAG28}] ❌ Pre-analysis summary failed. issues={issues}")

    # --- 7) Persist artifact ---
    out_path = resolve_artifact_path(
        ctx=ctx,
        stage="data_quality",
        name="pre_analysis_summary",
        run_id=str(rid),
        tag=TAG28,
    )
    write_json(out_path, report, tag=TAG28, indent=2)
    lg.info(f"[{TAG28}] 🧾 Pre-analysis summary saved: {out_path}")

    return report

In [11]:
# --- 2.9 Data Cleaning & Standardization ---
#
# Purpose
#   Telco-specific cleaning policy built on top of the generic engine (2.9.1).
#
# What it does
#   - Resolve cleaning policy from YAML SSOT via ctx:
#       pipeline.data_quality.cleaning.*
#       dataset.label.col
#   - Map SSOT config -> engine cleaning_config
#   - Enable Telco domain hooks (e.g., TotalCharges imputation)
#   - Produce SSOT-named artifacts and gate-friendly outputs for the orchestrator
#
# Contract
#   Input:
#     df_raw (pd.DataFrame), ctx (SSOT), run_id optional
#   Output (recommended for 2.x orchestrator):
#     returns (df_clean, module_report_v1)
#     - df_clean: cleaned dataframe for downstream steps
#     - module_report_v1: SSOT-named "data_quality_cleaning_{run_id}.json"
#
# Why this layer exists
#   Project-specific rules must remain isolated and auditable, while the engine
#   stays generic and reusable.

# --- 0) Import + TAG ---
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Tuple

import pandas as pd
from pandas.api.types import is_string_dtype

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint
from src.pipeline.data_quality.cleaning.cleaning_engine import run_generic_cleaning_engine
from src.utils.artifact_utils import resolve_artifact_path, write_json
from src.utils.path_utils import resolve_dir
from src.utils.cfg_utils import get_cfg_dict
from src.utils.dataset_utils import get_label_cfg

TAG29 = "CLEANING"


# --- Internal helpers ---
def telco_total_charges_imputation(df: pd.DataFrame) -> Dict[str, Any]:
    """
    Telco-specific step:
    Impute missing TotalCharges using MonthlyCharges * tenure where possible.

    Notes:
    - Works even if TotalCharges is stored as pandas 'string' dtype.
    - Leaves remaining missing values to the generic numeric-cast step.
    """
    step_name = "telco_total_charges_imputation"
    required_cols = {"TotalCharges", "MonthlyCharges", "tenure"}

    if df is None or not isinstance(df, pd.DataFrame):
        raise TypeError(f"[{TAG29}] telco_total_charges_imputation expects df as pd.DataFrame")

    if not required_cols.issubset(df.columns):
        return {"step": step_name, "note": "required_columns_missing_skip"}

    monthly = pd.to_numeric(df["MonthlyCharges"], errors="coerce")
    tenure = pd.to_numeric(df["tenure"], errors="coerce")
    total_numeric = pd.to_numeric(df["TotalCharges"], errors="coerce")

    missing_mask = total_numeric.isna()
    n_missing_before = int(missing_mask.sum())

    formula_mask = missing_mask & monthly.notna() & tenure.notna()
    n_imputed = int(formula_mask.sum())

    updated = total_numeric.copy()
    updated.loc[formula_mask] = monthly.loc[formula_mask] * tenure.loc[formula_mask]

    if is_string_dtype(df["TotalCharges"].dtype):
        df["TotalCharges"] = updated.astype("string")
    else:
        df["TotalCharges"] = updated

    n_missing_after = int(pd.to_numeric(df["TotalCharges"], errors="coerce").isna().sum())

    return {
        "step": step_name,
        "missing_before": n_missing_before,
        "imputed_by_formula": n_imputed,
        "missing_after": n_missing_after,
        "note": "imputed_totalcharges_via_monthlycharges_times_tenure",
    }


def _build_cleaning_config_from_ctx(ctx: Dict[str, Any], *, tag: str) -> Dict[str, Any]:
    """
    Build engine-compatible cleaning_config from YAML SSOT.

    Expected YAML:
      pipeline.data_quality.cleaning:
        string_columns: null | [..]
        include_mapping_in_report: false
        output_filename_prefix: "telco_cleaned"
        enable_normalize_strings: true
        numeric_cast_columns: {...}
        rare_category: {...}
        value_mappings: {...}
        hooks:
          telco_total_charges_imputation: true
    """
    cfg = get_cfg_dict(ctx, "pipeline.data_quality.cleaning", tag=tag, required=True)

    label_col, _, _ = get_label_cfg(ctx, tag)

    string_columns = cfg.get("string_columns", None)
    if isinstance(string_columns, list):
        string_columns = [c.strip() for c in string_columns if isinstance(c, str) and c.strip()]
    elif string_columns is not None:
        # Any non-list non-null value is treated as invalid and falls back to auto-detect (None).
        string_columns = None

    output_prefix = cfg.get("output_filename_prefix", "telco_cleaned")
    if not isinstance(output_prefix, str) or not output_prefix.strip():
        output_prefix = "telco_cleaned"

    # --- Normalize numeric_cast_columns: allow omitting fill_strategy entirely.
    # If user sets fill_strategy to "none"/null/empty, treat it as "no imputation"
    # and remove the key to avoid any learned-statistic behavior in Step 2.9.
    numeric_cast_columns = cfg.get("numeric_cast_columns", {}) or {}
    if isinstance(numeric_cast_columns, dict):
        normalized_cast: Dict[str, Any] = {}
        for col, spec in numeric_cast_columns.items():
            if not isinstance(col, str) or not col.strip():
                continue
            if not isinstance(spec, dict):
                normalized_cast[col.strip()] = spec
                continue

            spec2 = dict(spec)
            fs = spec2.get("fill_strategy", None)

            if fs is None:
                spec2.pop("fill_strategy", None)
            elif isinstance(fs, str) and fs.strip().lower() in {"none", "skip", "leave_null", ""}:
                spec2.pop("fill_strategy", None)

            normalized_cast[col.strip()] = spec2

        numeric_cast_columns = normalized_cast
    else:
        numeric_cast_columns = {}

    engine_cfg: Dict[str, Any] = {
        "label_col": str(label_col).strip(),
        "string_columns": string_columns,  # None means "auto-detect string-like columns"
        "enable_normalize_strings": bool(cfg.get("enable_normalize_strings", True)),
        "numeric_cast_columns": numeric_cast_columns,
        "rare_category": cfg.get("rare_category", {}) or {},
        "value_mappings": cfg.get("value_mappings", {}) or {},
        "include_mapping_in_report": bool(cfg.get("include_mapping_in_report", False)),
        "output_filename_prefix": output_prefix.strip(),
    }
    return engine_cfg


def _resolve_telco_hooks_from_ctx(ctx: Dict[str, Any], *, tag: str) -> List[Callable[[pd.DataFrame], Dict[str, Any]]]:
    """Resolve which Telco hooks are enabled from YAML SSOT."""
    hooks_cfg = get_cfg_dict(ctx, "pipeline.data_quality.cleaning.hooks", tag=tag, default={})
    steps: List[Callable[[pd.DataFrame], Dict[str, Any]]] = []

    if bool((hooks_cfg or {}).get("telco_total_charges_imputation", False)):
        steps.append(telco_total_charges_imputation)

    return steps


def _extract_engine_signals(engine_report: Dict[str, Any]) -> Dict[str, Any]:
    """
    Extract a small set of gate-friendly signals from the engine report.
    Keep ModuleReport slim and stable; store only what the orchestrator needs.
    """
    steps = engine_report.get("steps", []) or []
    step_names = [s.get("step") for s in steps if isinstance(s, dict)]

    totalcharges_missing_after = None
    for s in steps:
        if not isinstance(s, dict):
            continue
        if s.get("step") == "cast_numeric_columns":
            cols = s.get("columns") or {}
            tc = cols.get("TotalCharges")
            if isinstance(tc, dict):
                totalcharges_missing_after = tc.get("missing_after")
            break

    return {
        "steps_run": step_names,
        "totalcharges_missing_after": totalcharges_missing_after,
    }


# --- 1) Public API ---
def run_cleaning(
    df: pd.DataFrame,
    *,
    ctx: Dict[str, Any],
    artifacts: Optional[Dict[str, Any]] = None,
    run_id: Optional[str] = None,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    Step 2.9 cleaning module (SSOT-driven):
      - reads YAML SSOT
      - runs pure engine + telco hooks
      - persists cleaned parquet (data.interim)
      - persists ModuleReport v1 artifact (outputs.artifacts)

    Returns:
      (df_clean, module_report)
    """
    # --- 2) Runtime validation ---
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG29, run_id=run_id)

    # --- 3) Input validation ---
    if df is None or not isinstance(df, pd.DataFrame):
        raise TypeError(f"[{TAG29}] df must be a pandas DataFrame")

    # --- 4) Resolve config (YAML SSOT) ---
    cleaning_config = _build_cleaning_config_from_ctx(ctx, tag=TAG29)
    hooks = _resolve_telco_hooks_from_ctx(ctx, tag=TAG29)

    interim_dir = resolve_dir(ctx, "data.interim", tag=TAG29, required=True)

    # --- 5) Core workflow ---
    lg.info(f"[{TAG29}] 🧼 Starting cleaning... | run_id={rid}")

    report = build_module_report(
        ctx=ctx,
        stage="data_quality",
        step="2.9",
        name="cleaning",
        tag=TAG29,
        cfg_key="pipeline.data_quality.cleaning",
        enabled=True,
        df=df,
        run_id_override=str(rid),
    )

    report["inputs"] = {
        "df_rows_in": int(df.shape[0]),
        "df_cols_in": int(df.shape[1]),
        "hooks_enabled": [getattr(fn, "__name__", "hook") for fn in hooks],
        "has_pre_analysis_summary": bool(isinstance((artifacts or {}).get("pre_analysis_summary"), dict)),
    }

    rare_cfg = cleaning_config.get("rare_category") or {}
    report["thresholds_used"] = {
        "enable_normalize_strings": bool(cleaning_config.get("enable_normalize_strings", True)),
        "include_mapping_in_report": bool(cleaning_config.get("include_mapping_in_report", False)),
        "rare_category_min_freq": int(rare_cfg.get("min_freq") or 0),
    }

    report["used_config"] = {
        "cfg_key": "pipeline.data_quality.cleaning",
        "label_col": cleaning_config.get("label_col"),
        "output_filename_prefix": cleaning_config.get("output_filename_prefix"),
        "hooks": report["inputs"]["hooks_enabled"],
    }

    report["refs"]["self"] = make_artifact_ref(ctx=ctx, stage="data_quality", name="cleaning", run_id=str(rid))

    ensure_event_hints(report, version=1)

    # --- Run engine (pure) ---
    try:
        df_clean, engine_report = run_generic_cleaning_engine(
            df_raw=df,
            cleaning_config=cleaning_config,
            project_steps=hooks,
        )
    except Exception as e:
        add_event_hint(
            report,
            code="cleaning_engine_crashed",
            severity_signal="hard",
            evidence_path="checks.engine",
            context={"error": f"{type(e).__name__}: {e}"},
            remediation={
                "action": "fix_engine_or_cleaning_config",
                "safe": True,
                "rationale": "Cleaning should be deterministic; crashes imply code/config mismatch.",
                "post_check": "re-run 2.9; confirm parquet saved and status pass/warn.",
            },
        )

        report["checks"]["engine"] = {"error": f"{type(e).__name__}: {e}"}

        report = finalize_module_report(
            ctx=ctx,
            report=report,
            overall_status="fail",
            issues=["cleaning_engine_crashed"],
            warnings=[],
            metrics={},
            notes=[],
            df=df,
        )

        out_path = resolve_artifact_path(
            ctx=ctx,
            stage="data_quality",
            name="cleaning",
            run_id=str(rid),
            tag=TAG29,
        )
        write_json(out_path, report, tag=TAG29, indent=2)
        lg.error(f"[{TAG29}] ❌ Cleaning failed (engine crash). Artifact saved: {out_path}")

        return df, report

    # --- Persist cleaned dataset (parquet) ---
    prefix = str(cleaning_config.get("output_filename_prefix") or "telco_cleaned").strip() or "telco_cleaned"
    cleaned_path = Path(interim_dir) / f"{prefix}_{rid}.parquet"

    parquet_ok = True
    parquet_err: Optional[str] = None
    try:
        df_clean.to_parquet(cleaned_path, index=False)
    except Exception as e:
        parquet_ok = False
        parquet_err = f"{type(e).__name__}: {e}"

    # --- Build checks (keep slim + gate-friendly) ---
    signals = _extract_engine_signals(engine_report)

    report["checks"]["engine_summary"] = {
        "n_steps": int(len(engine_report.get("steps") or [])),
        "signals": signals,
        "shape_before": [int(df.shape[0]), int(df.shape[1])],
        "shape_after": [int(df_clean.shape[0]), int(df_clean.shape[1])],
    }

    report["checks"]["persistence"] = {
        "cleaned_parquet_path": str(cleaned_path),
        "parquet_ok": bool(parquet_ok),
        "parquet_error": parquet_err,
    }

    # --- Event hints (policy-free facts) ---
    if signals.get("totalcharges_missing_after") not in (None, 0):
        add_event_hint(
            report,
            code="totalcharges_missing_after_cleaning",
            severity_signal="risk",
            evidence_path="checks.engine_summary.signals.totalcharges_missing_after",
            context={"missing_after": signals.get("totalcharges_missing_after")},
            remediation={
                "action": "review_totalcharges_imputation_and_casting",
                "safe": True,
                "rationale": "Remaining missing TotalCharges may affect modeling features and integrity checks.",
                "post_check": "re-run 2.2/2.3/2.5 and verify warnings reduced; then run 3.x integrity checks.",
            },
        )

    if not parquet_ok:
        add_event_hint(
            report,
            code="cleaned_parquet_write_failed",
            severity_signal="hard",
            evidence_path="checks.persistence",
            context={"path": str(cleaned_path), "error": parquet_err},
            remediation={
                "action": "install_parquet_engine_or_change_persistence_format",
                "safe": True,
                "rationale": "Downstream reproducibility expects a persisted dataset artifact.",
                "post_check": "confirm parquet write succeeds then re-run pipeline.",
            },
        )

    # --- 6) Status + gate-friendly summary ---
    issues: List[str] = []
    warnings: List[str] = []

    if int(df_clean.shape[0]) == 0:
        issues.append("empty_df_after_cleaning")

    if not parquet_ok:
        issues.append("parquet_write_failed")

    if signals.get("totalcharges_missing_after") not in (None, 0):
        warnings.append("totalcharges_missing_after_cleaning")

    overall = "fail" if issues else ("warn" if warnings else "pass")

    report = finalize_module_report(
        ctx=ctx,
        report=report,
        overall_status=overall,
        issues=issues,
        warnings=warnings,
        metrics={
            "rows_before": int(df.shape[0]),
            "rows_after": int(df_clean.shape[0]),
            "cols_after": int(df_clean.shape[1]),
            "n_steps": int(len(signals.get("steps_run") or [])),
        },
        notes=[f"cleaned_dataset={cleaned_path.name}"],
        df=df_clean,
    )

    if overall == "pass":
        lg.info(f"[{TAG29}] ✅ Cleaning pass | parquet_ok={parquet_ok} | saved={cleaned_path.name}")
    elif overall == "warn":
        lg.warning(f"[{TAG29}] ⚠ Cleaning warn | warnings={warnings} | saved={cleaned_path.name}")
    else:
        lg.error(f"[{TAG29}] ❌ Cleaning fail | issues={issues} | saved={cleaned_path.name}")


    # --- 7) Persist ModuleReport artifact (SSOT naming) ---
    out_path = resolve_artifact_path(
        ctx=ctx,
        stage="data_quality",
        name="cleaning",
        run_id=str(rid),
        tag=TAG29,
    )
    write_json(out_path, report, tag=TAG29, indent=2)
    lg.info(f"[{TAG29}] 🧾 Cleaning artifact saved: {out_path}")

    return df_clean, report

In [12]:
# --- 2.x Data Quality Orchestrator (SSOT-driven) ---
#
# What:
#   Run Step-2 "data_quality" as one controller:
#     - Build the step plan from YAML SSOT (enabled + steps order + gates)
#     - Dispatch each module by contract (source/df/agg + returns_df)
#     - Apply centralized gating (hard/soft + strict) and optionally halt
#     - Persist orchestrator ModuleReport v1 + per-step artifacts (safety net)
#
# How:
#   - ensure_runtime_ctx() -> ctx/logger/run_id
#   - Read SSOT:
#       pipeline.orchestrator.stages.data_quality.{enabled,steps}
#       pipeline.gate_policy.{strict,default}
#   - build_modules_from_ssot(): steps[] -> List[ModuleSpec] (fn resolved from registry)
#   - For each ModuleSpec in SSOT order:
#       invoke_module() -> raw result
#       normalize_module_result() -> {df, report}
#       on missing_impl/exception: build_*_report() + persist_artifact_safety_net()
#       infer status -> should_halt(strict, gate, status)
#   - finalize_module_report() + persist orchestrator artifact
#
# Why:
#   - SSOT defines ordering/gates (no hard-coded orchestration)
#   - Stable contracts let modules evolve independently
#   - Central gating + artifacts make runs reproducible and auditable
#   - Fail-safe reports prevent silent crashes and keep downstream aggregation reliable

# --- 0) Import + TAG ---
from typing import Any, Dict, List, Optional, Set

import pandas as pd

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint
from src.pipeline.orchestrator_utils import get_stage_cfg
from src.pipeline.orchestrator_common import (
    ModuleFn,
    ModuleSpec,
    validate_module_spec,
    infer_module_status,
    should_halt,
    build_missing_impl_report,
    build_exception_report,
    persist_artifact_safety_net,
    invoke_module,
    normalize_module_result,
    has_missing_module_report
)
from src.utils.artifact_utils import resolve_artifact_path, write_json
from src.utils.cfg_utils import get_cfg_str, get_cfg_value

TAG2X = "DQ_ORCH"


# --- Internal helpers ---
def _artifact_key(spec: ModuleSpec) -> str:
    """Return the canonical artifact/report key for a module spec."""
    k = spec.artifact_name or spec.name or spec.step_id
    k = str(k).strip()
    return k if k else str(spec.step_id).strip()


def _ensure_summary_aliases(artifacts: Dict[str, Any]) -> None:
    """Ensure both 'pre_analysis_summary' and 'summary' keys exist when one exists."""
    if not isinstance(artifacts, dict):
        return

    has_pre = "pre_analysis_summary" in artifacts
    has_sum = "summary" in artifacts

    if has_pre and (not has_sum):
        artifacts["summary"] = artifacts["pre_analysis_summary"]
    elif has_sum and (not has_pre):
        artifacts["pre_analysis_summary"] = artifacts["summary"]


def _find_step_spec(modules: List[ModuleSpec], step_id: str) -> Optional[ModuleSpec]:
    sid = str(step_id).strip().lower()
    for m in modules:
        if str(m.step_id).strip().lower() == sid:
            return m
    return None


def build_modules_from_ssot(
    *,
    ctx: Dict[str, Any],
    tag: str,
    registry: Dict[str, ModuleFn],
    include_groups: Optional[Set[str]] = None,
) -> List[ModuleSpec]:
    stage_cfg = get_stage_cfg(ctx, "data_quality", tag=tag)
    groups = stage_cfg.get("groups") or {}
    if not isinstance(groups, dict):
        raise TypeError(f"[{tag}] stage_cfg.groups must be a dict")

    default_gate = get_cfg_str(
        ctx,
        "pipeline.gate_policy.default",
        tag=tag,
        default="soft",
        required=False,
        lower=True,
        allowed={"soft", "hard"},
    )

    group_order = ["ingestion", "diagnostics", "cleaning"]
    modules: List[ModuleSpec] = []

    for group_id in group_order:
        if include_groups is not None and group_id not in include_groups:
            continue

        gcfg = groups.get(group_id)
        if gcfg is None:
            continue
        if not isinstance(gcfg, dict):
            raise TypeError(f"[{tag}] group '{group_id}' must be a dict")

        group_enabled = bool(gcfg.get("enabled", True))
        steps = gcfg.get("steps") or []
        if not isinstance(steps, list):
            raise TypeError(f"[{tag}] group '{group_id}' steps must be a list")

        for i, s in enumerate(steps):
            if not isinstance(s, dict):
                raise TypeError(f"[{tag}] group '{group_id}' steps[{i}] must be dict")

            step_id = str(s.get("id") or "").strip()
            if not step_id:
                raise ValueError(f"[{tag}] group '{group_id}' steps[{i}] missing id")

            cfg_key = str(s.get("cfg_key") or "").strip()
            if not cfg_key:
                raise ValueError(f"[{tag}] group '{group_id}' steps[{i}] missing cfg_key for {step_id}")

            enabled = bool(group_enabled and bool(s.get("enabled", True)))

            gate = s.get("gate", None)
            gate_mode = str(gate).strip().lower() if gate is not None else str(default_gate).strip().lower()
            gate_mode = "hard" if gate_mode == "hard" else "soft"

            sid = step_id.lower()
            kind = "df"
            returns_df = False

            if sid == "ingestion":
                kind = "source"
                returns_df = True
            elif sid == "cleaning":
                kind = "df"
                returns_df = True
            elif sid == "summary":
                kind = "agg"
                returns_df = False

            fn = registry.get(step_id) or registry.get(sid)

            artifact_name = step_id
            if sid == "summary":
                artifact_name = "pre_analysis_summary"

            modules.append(
                ModuleSpec(
                    step_id=step_id,
                    name=step_id,
                    fn=fn,
                    cfg_key=cfg_key,
                    artifact_stage="data_quality",
                    artifact_name=artifact_name,
                    enabled=enabled,
                    kind=kind,
                    gate_default=gate_mode,
                    returns_df=returns_df,
                )
            )

    return modules


def build_dq_registry_from_globals(
    *,
    tag: str = TAG2X,
    fn_name_map: Optional[Dict[str, str]] = None,
    allow_missing: bool = True,
) -> Dict[str, ModuleFn]:
    """
    Notebook registry builder (NO imports).

    Resolves callables from the current notebook globals():
      step_id -> function object

    fn_name_map lets you align SSOT step ids to your actual notebook function names.
    """
    default_map: Dict[str, str] = {
        # SSOT step_id -> notebook function name
        "ingestion": "load_data",
        "contract": "validate_data_contract",
        "health": "compute_data_health",
        "semantic": "run_semantic_checks",
        "readiness": "check_experimental_readiness",
        "domain_core": "run_domain_diagnostics_classification_core",
        "domain_addon_telco": "run_domain_diagnostics_telco_addon",
        "distribution": "run_distribution_balance_checks",
        "summary": "build_pre_analysis_summary",
        "cleaning": "run_cleaning",
    }

    name_map = dict(default_map)
    if fn_name_map:
        name_map.update(fn_name_map)

    reg: Dict[str, ModuleFn] = {}
    g = globals()

    for step_id, fn_name in name_map.items():
        obj = g.get(fn_name)

        if obj is None:
            if not allow_missing:
                raise KeyError(f"[{tag}] registry missing fn '{fn_name}' for step_id='{step_id}'")
            continue

        if not callable(obj):
            raise TypeError(f"[{tag}] '{fn_name}' exists but is not callable (step_id='{step_id}')")

        reg[step_id] = obj  # type: ignore[assignment]

    return reg


def build_dq_registry_from_ssot(
    *,
    ctx: Dict[str, Any],
    tag: str = TAG2X,
    fn_name_map: Optional[Dict[str, str]] = None,
    require_enabled_only: bool = True,
) -> Dict[str, ModuleFn]:
    reg = build_dq_registry_from_globals(tag=tag, fn_name_map=fn_name_map, allow_missing=True)

    if not require_enabled_only:
        return reg

    # Only require functions for enabled steps in SSOT
    modules = build_modules_from_ssot(ctx=ctx, tag=tag, registry=reg, include_groups=None)
    missing: List[str] = []
    for m in modules:
        if m.enabled and (m.fn is None):
            missing.append(f"{m.step_id}")

    if missing:
        raise KeyError(f"[{tag}] registry missing implementations for enabled steps: {missing}")

    return reg


def post_ingestion(
    *,
    ctx: Dict[str, Any],
    registry: Dict[str, ModuleFn],
    df: pd.DataFrame,
    strict: bool,
    rid: str,
    modules: List[ModuleSpec]
) -> Dict[str, Any]:
    """
    Run data_quality diagnostics only (2.2–2.8), assuming df already exists.
    """
    ctx, lg, _ = ensure_runtime_ctx(ctx=ctx, tag=TAG2X, run_id=rid)

    reports: Dict[str, Dict[str, Any]] = {}
    artifacts: Dict[str, Any] = {}
    execution: List[Dict[str, Any]] = []

    halted = False
    halted_at: Optional[str] = None

    for spec in modules:
        if halted:
            execution.append({"step_id": spec.step_id, "status": "skipped", "reason": "halted", "gate": spec.gate_default})
            continue
        if not spec.enabled:
            execution.append({"step_id": spec.step_id, "status": "skipped", "reason": "disabled", "gate": spec.gate_default})
            continue

        gate_mode = spec.gate_default
        lg.info(f"[{TAG2X}] ▶ post_ingestion step_id={spec.step_id} | kind={spec.kind} | gate={gate_mode} | strict={strict}")

        if spec.fn is None:
            rep = build_missing_impl_report(ctx=ctx, rid=rid, tag=TAG2X, stage="data_quality", spec=spec, df=df)
            report_key = _artifact_key(spec)
            reports[report_key] = rep
            persist_artifact_safety_net(ctx=ctx, rid=rid, tag=TAG2X, spec=spec, report=rep)

            st = infer_module_status(rep)
            execution.append({"step_id": spec.step_id, "status": st, "gate": gate_mode, "missing_impl": True})

            artifacts[report_key] = rep

            if should_halt(strict=bool(strict), gate_mode=gate_mode, status=st):
                halted, halted_at = True, spec.step_id
            continue

        try:
            raw = invoke_module(spec=spec, df=df, artifacts=artifacts, ctx=ctx, rid=rid)
            norm = normalize_module_result(spec=spec, res=raw, df_in=df)

            df = norm["df"]
            rep = norm["report"]

            if rep is None:
                rep = build_exception_report(
                    ctx=ctx,
                    rid=rid,
                    tag=TAG2X,
                    stage="data_quality",
                    spec=spec,
                    err=RuntimeError("module_returned_no_report"),
                    df=df,
                )
                persist_artifact_safety_net(ctx=ctx, rid=rid, tag=TAG2X, spec=spec, report=rep)

        except Exception as e:
            lg.exception(f"[{TAG2X}] ❌ post_ingestion step_id={spec.step_id} crashed: {e!r}")
            rep = build_exception_report(ctx=ctx, rid=rid, tag=TAG2X, stage="data_quality", spec=spec, err=e, df=df)
            persist_artifact_safety_net(ctx=ctx, rid=rid, tag=TAG2X, spec=spec, report=rep)

        report_key = _artifact_key(spec)
        reports[report_key] = rep
        artifacts[report_key] = rep

        st = infer_module_status(rep)
        execution.append({"step_id": spec.step_id, "status": st, "gate": gate_mode})

        if should_halt(strict=bool(strict), gate_mode=gate_mode, status=st):
            halted, halted_at = True, spec.step_id

    return {
        "df": df,
        "reports": reports,
        "artifacts": artifacts,
        "execution": execution,
        "halted": halted,
        "halted_at": halted_at,
    }


def enforce_prerequisites(
    *,
    ctx: Dict[str, Any],
    orch: Dict[str, Any],
    tag: str,
    stage: str,
    strict: bool,
    halted: bool,
    halted_at: Optional[str],
    artifacts: Dict[str, Any],
    execution: List[Dict[str, Any]],
    diag_specs: List[ModuleSpec],
    clean_specs: List[ModuleSpec],
) -> Dict[str, Any]:
    """
    Enforce cross-step invariants before running downstream groups (e.g., cleaning).

    Returns:
      {"halted": bool, "halted_at": Optional[str]}
    """
    if halted:
        return {"halted": True, "halted_at": halted_at}

    summary_spec = _find_step_spec(diag_specs, "summary") if diag_specs else None
    cleaning_spec = _find_step_spec(clean_specs, "cleaning") if clean_specs else None

    summary_enabled = bool(summary_spec.enabled) if summary_spec is not None else False
    cleaning_enabled = bool(cleaning_spec.enabled) if cleaning_spec is not None else False

    # Invariant: cleaning requires pre_analysis_summary when summary is enabled
    if summary_enabled and cleaning_enabled:
        _ensure_summary_aliases(artifacts)
        has_pre = "pre_analysis_summary" in artifacts

        orch.setdefault("checks", {})
        orch["checks"].setdefault("prerequisites", {})
        orch["checks"]["prerequisites"]["pre_analysis_summary"] = {
            "required": True,
            "present": bool(has_pre),
            "summary_enabled": True,
            "cleaning_enabled": True,
        }

        if not has_pre:
            # Halt deterministically
            execution.append({
                "step_id": "cleaning",
                "status": "skipped",
                "gate": getattr(cleaning_spec, "gate_default", "hard") if cleaning_spec else "hard",
                "reason": "prerequisite_missing:pre_analysis_summary",
            })

            add_event_hint(
                orch,
                code="dq_cleaning_prereq_missing_pre_analysis_summary",
                severity_signal="hard",
                evidence_path="checks.prerequisites.pre_analysis_summary",
                context={
                    "strict": bool(strict),
                    "artifact_keys_preview": sorted(list(artifacts.keys()))[:30],
                },
                remediation={
                    "action": "fix_summary_module_contract_and_rerun",
                    "safe": True,
                    "rationale": "Cleaning must only run after pre-analysis summary is available; otherwise it becomes non-auditable/unsafe.",
                    "post_check": "Verify artifacts contain 'pre_analysis_summary' before cleaning.",
                },
            )

            return {"halted": True, "halted_at": "pre_analysis_summary_missing"}

    # No prerequisite required (or satisfied)
    return {"halted": False, "halted_at": halted_at}

# --- 1) Public API ---
def run_data_quality_orchestrator(
    *,
    ctx: Dict[str, Any],
    registry: Dict[str, ModuleFn],
    df: Optional[pd.DataFrame] = None,
    run_id: Optional[str] = None,
) -> Dict[str, Any]:
    
    # --- 2) Runtime validation ---
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG2X, run_id=run_id)

    # --- 3) Input validation ---
    if not isinstance(registry, dict):
        raise TypeError(f"[{TAG2X}] registry must be a dict[str, ModuleFn]")
    if (df is not None) and (not isinstance(df, pd.DataFrame)):
        raise TypeError(f"[{TAG2X}] df must be a pandas DataFrame when provided")

    # --- 4) Resolve config (YAML SSOT) ---
    strict = bool(get_cfg_value(ctx, "pipeline.gate_policy.strict", tag=TAG2X, default=True))

    stage_cfg = get_stage_cfg(ctx, "data_quality", tag=TAG2X)
    stage_enabled = bool(stage_cfg.get("enabled", True))

    # --- 5) Core workflows ---
    lg.info(
        f"[{TAG2X}] Run started | stage=data_quality | run_id={rid} | strict={strict} "
        f"| df_input_provided={df is not None}"
    )

    # --- Build ModuleReport skeleton (infra) ---
    orch = build_module_report(
        ctx=ctx,
        stage="data_quality",
        step="2.x",
        name="orchestrator",
        tag=TAG2X,
        cfg_key="pipeline.orchestrator.stages.data_quality",
        enabled=stage_enabled,
        df=df,
        run_id_override=str(rid),
    )
    orch["inputs"] = {"stage_enabled": stage_enabled, "strict": strict, "df_input_provided": df is not None}
    orch["refs"]["self"] = make_artifact_ref(ctx=ctx, stage="data_quality", name="orchestrator", run_id=str(rid))
    ensure_event_hints(orch, version=1)
    orch.setdefault("checks", {})

    # --- Build plan (SSOT -> specs) ---
    ing_specs: List[ModuleSpec] = build_modules_from_ssot(
        ctx=ctx, tag=TAG2X, registry=registry, include_groups={"ingestion"}
    )
    diag_specs: List[ModuleSpec] = build_modules_from_ssot(
        ctx=ctx, tag=TAG2X, registry=registry, include_groups={"diagnostics"}
    )
    clean_specs: List[ModuleSpec] = build_modules_from_ssot(
        ctx=ctx, tag=TAG2X, registry=registry, include_groups={"cleaning"}
    )

    # Optional: validate specs early (fail-fast on malformed SSOT)
    for s in ing_specs + diag_specs + clean_specs:
        validate_module_spec(s, TAG2X)
        
    # Plan snapshot for auditability
    orch["checks"]["plan"] = {
        "ingestion": [m.step_id for m in ing_specs if m.enabled],
        "diagnostics": [m.step_id for m in diag_specs if m.enabled],
        "cleaning": [m.step_id for m in clean_specs if m.enabled],
    }

    lg.info(
        f"[{TAG2X}] Plan | stage_enabled={stage_enabled} "
        f"| ingestion={len(orch['checks']['plan']['ingestion'])} "
        f"| diagnostics={len(orch['checks']['plan']['diagnostics'])} "
        f"| cleaning={len(orch['checks']['plan']['cleaning'])}"
    )

    # --- Early exit: stage disabled ---
    if not stage_enabled:
        orch = finalize_module_report(
            ctx=ctx,
            report=orch,
            overall_status="skipped",
            issues=[],
            warnings=[],
            metrics={"halted": False},
            notes=["stage_disabled=true"],
            df=df,
        )
        out_path = resolve_artifact_path(ctx=ctx, stage="data_quality", name="orchestrator", run_id=str(rid), tag=TAG2X)
        write_json(out_path, orch, tag=TAG2X, indent=2)
        
        lg.info(f"[{TAG2X}] Run finished | overall=skipped | stage_enabled=false | run_id={rid}")
        
        return {
            "df": df,
            "halted": False,
            "halted_at": None,
            "orchestrator_report": orch,
            "reports": {},
            "execution": [],
        }

    # --- Execute plan ---
    reports: Dict[str, Dict[str, Any]] = {}
    execution: List[Dict[str, Any]] = []
    artifacts: Dict[str, Any] = {}
    halted = False
    halted_at: Optional[str] = None
    df_raw: Optional[pd.DataFrame] = None
    df_clean: Optional[pd.DataFrame] = None

    # If caller provides df, we treat it as df_raw baseline for this stage.
    if df is not None:
        df_raw = df

    # --- 2.1 ingestion (only if df is None) ---
    if df is None:
        mods_ing = [m for m in ing_specs if m.step_id.lower() == "ingestion"]
        if not mods_ing:
            raise ValueError(f"[{TAG2X}] ingestion group enabled but no 'ingestion' step found in SSOT")
        spec = mods_ing[0]

        if spec.fn is None:
            rep = build_missing_impl_report(ctx=ctx, rid=str(rid), tag=TAG2X, stage="data_quality", spec=spec, df=None)
            persist_artifact_safety_net(ctx=ctx, rid=str(rid), tag=TAG2X, spec=spec, report=rep)
            
            st = infer_module_status(rep)
            report_key = _artifact_key(spec)

            reports[report_key] = rep
            artifacts[report_key] = rep
            execution.append({"step_id": spec.step_id, "status": st, "gate": spec.gate_default})
            
            halted, halted_at = True, spec.step_id
        
        else:
            try:
                raw = invoke_module(spec=spec, df=None, artifacts=artifacts, ctx=ctx, rid=str(rid))
                norm = normalize_module_result(spec=spec, res=raw, df_in=None)
                
                df = norm["df"]
                rep = norm["report"]

                # Capture df_raw right after ingestion
                if df is not None and isinstance(df, pd.DataFrame):
                    df_raw = df

                # --- Contract fallback: ingestion returned df but no ModuleReport ---
                if rep is None:
                    rep = build_module_report(
                        ctx=ctx,
                        stage="data_quality",
                        step=str(spec.step_id),
                        name=str(spec.name),
                        tag=TAG2X,
                        cfg_key=str(spec.cfg_key),
                        enabled=True,
                        df=df,
                        run_id_override=str(rid),
                    )
                    rep["inputs"] = {"orchestrator": True, "contract_fallback": True}
                    rep["thresholds_used"] = {}
                    rep["used_config"] = {"cfg_key": str(spec.cfg_key)}
                    rep["refs"]["self"] = make_artifact_ref(
                        ctx=ctx,
                        stage=str(spec.artifact_stage),
                        name=str(spec.artifact_name or spec.name),
                        run_id=str(rid),
                    )
                    rep.setdefault("checks", {})
                    rep["checks"]["infra"] = {"warning": "module_returned_df_without_report"}
                    
                    rep = finalize_module_report(
                        ctx=ctx,
                        report=rep,
                        overall_status="warn",
                        issues=[],
                        warnings=["missing_module_report"],
                        metrics={},
                        notes=["module_returned_df_without_report"],
                        df=df,
                    )
                    persist_artifact_safety_net(ctx=ctx, rid=str(rid), tag=TAG2X, spec=spec, report=rep)
            
            except Exception as e:
                rep = build_exception_report(ctx=ctx, rid=str(rid), tag=TAG2X, stage="data_quality", spec=spec, err=e, df=None)
                persist_artifact_safety_net(ctx=ctx, rid=str(rid), tag=TAG2X, spec=spec, report=rep)
                df = None

            report_key = _artifact_key(spec)
            reports[report_key] = rep
            artifacts[report_key] = rep

            st = infer_module_status(rep)
            execution.append({"step_id": spec.step_id, "status": st, "gate": spec.gate_default})
            
            if df is None or should_halt(strict=bool(strict), gate_mode=spec.gate_default, status=st):
                halted, halted_at = True, spec.step_id

    # --- 2.2–2.8 Diagnostics (post-ingestion) ---
    if (not halted) and df is not None:
        post = post_ingestion(
            ctx=ctx,
            registry=registry,
            df=df,
            strict=bool(strict),
            rid=str(rid),
            modules=diag_specs,
        )

        df = post["df"]
        reports.update(post["reports"])
        execution.extend(post["execution"])

        post_artifacts = post.get("artifacts") or {}
        if not isinstance(post_artifacts, dict):
            raise TypeError(f"[{TAG2X}] post_ingestion returned artifacts that is not a dict")

        artifacts.update(post_artifacts)
        _ensure_summary_aliases(artifacts)

        halted = bool(post["halted"])
        halted_at = post["halted_at"] or halted_at

    # --- Prerequisites (before cleaning) ---
    if (not halted) and df is not None:
        pre = enforce_prerequisites(
            ctx=ctx,
            orch=orch,
            tag=TAG2X,
            stage="data_quality",
            strict=bool(strict),
            halted=bool(halted),
            halted_at=halted_at,
            artifacts=artifacts,
            execution=execution,
            diag_specs=diag_specs,
            clean_specs=clean_specs,
        )
        halted = bool(pre["halted"])
        halted_at = pre.get("halted_at") or halted_at

    # --- 2.9 cleaning ---
    if (not halted) and df is not None:
        mods_clean = [m for m in clean_specs if m.step_id.lower() == "cleaning"]
        if mods_clean:
            spec = mods_clean[0]
            if spec.enabled:
                if spec.fn is None:
                    rep = build_missing_impl_report(ctx=ctx, rid=str(rid), tag=TAG2X, stage="data_quality", spec=spec, df=df)
                    persist_artifact_safety_net(ctx=ctx, rid=str(rid), tag=TAG2X, spec=spec, report=rep)
                else:
                    try:
                        raw = invoke_module(spec=spec, df=df, artifacts=artifacts, ctx=ctx, rid=str(rid))
                        norm = normalize_module_result(spec=spec, res=raw, df_in=df)
                        df = norm["df"]
                        rep = norm["report"]

                        # Capture df_clean right after cleaning
                        if df is not None and isinstance(df, pd.DataFrame):
                            df_clean = df

                    except Exception as e:
                        rep = build_exception_report(ctx=ctx, rid=str(rid), tag=TAG2X, stage="data_quality", spec=spec, err=e, df=df)
                        persist_artifact_safety_net(ctx=ctx, rid=str(rid), tag=TAG2X, spec=spec, report=rep)

                report_key = _artifact_key(spec)
                reports[report_key] = rep
                artifacts[report_key] = rep

                st = infer_module_status(rep)
                execution.append({"step_id": spec.step_id, "status": st, "gate": spec.gate_default})

                if should_halt(strict=bool(strict), gate_mode=spec.gate_default, status=st):
                    halted, halted_at = True, spec.step_id

    orch["checks"]["execution"] = execution
    orch["checks"]["reports_present"] = {k: True for k in reports.keys()}
    orch["checks"]["halted"] = {"halted": bool(halted), "halted_at": halted_at or ""}
    orch["checks"]["datasets"] = {
        "df_raw_provided": df_raw is not None,
        "df_clean_provided": df_clean is not None,
        "df_raw_shape": [int(df_raw.shape[0]), int(df_raw.shape[1])] if isinstance(df_raw, pd.DataFrame) else None,
        "df_clean_shape": [int(df_clean.shape[0]), int(df_clean.shape[1])] if isinstance(df_clean, pd.DataFrame) else None,
    }

    if halted:
        add_event_hint(
            orch,
            code="dq_orchestrator_halted",
            severity_signal="hard",
            evidence_path="checks.halted",
            context={"halted": True, "halted_at": halted_at or "", "strict": bool(strict)},
            remediation={
                "action": "fix_failed_step_then_rerun",
                "safe": True,
                "rationale": "Hard-gated step failed; downstream outputs may be invalid.",
                "post_check": "Confirm halted step becomes pass/warn before continuing.",
            },
        )
        lg.error(f"[{TAG2X}] Halted | halted_at={halted_at or ''} | strict={strict}")

    # --- 6) Status + gate-friendly summary ---
    issues: List[str] = []
    warnings: List[str] = []
    
    if halted:
        issues.append("pipeline_halted")
    if halted_at == "pre_analysis_summary_missing":
        issues.append("missing_pre_analysis_summary")

    # Any step returned warn
    if any((x.get("status") == "warn") for x in execution):
        warnings.append("one_or_more_steps_warn")
    # Any missing implementation under soft gate (non-fatal but important)
    if any((x.get("missing_impl") is True and str(x.get("gate", "")).lower() == "soft") for x in execution):
        warnings.append("soft_gated_missing_impl")
    # Ingestion fallback: module returned df but no ModuleReport
    if any(has_missing_module_report(rep) for rep in reports.values() if isinstance(rep, dict)):
        warnings.append("missing_module_report_from_module")
    
    orch["checks"]["warnings"] = {
        "one_or_more_steps_warn": [x.get("step_id") for x in execution if x.get("status") == "warn"],
        "soft_gated_missing_impl": [
            x.get("step_id")
            for x in execution
            if x.get("missing_impl") is True and str(x.get("gate", "")).lower() == "soft"
        ],
        "missing_module_report_from_module": [
            k for k, rep in reports.items() if isinstance(rep, dict) and has_missing_module_report(rep)
        ],
    }

    overall = "fail" if issues else ("warn" if warnings else "pass")

    orch = finalize_module_report(
        ctx=ctx,
        report=orch,
        overall_status=overall,
        issues=issues,
        warnings=warnings,
        metrics={"halted": bool(halted), "n_reports": int(len(reports))},
        notes=["dq_orchestrator_split:2.1+post(2.2-2.8)+2.9"],
        df=df,
    )

    # --- 7) Persist Artifact ---
    out_path = resolve_artifact_path(
        ctx=ctx,
        stage="data_quality",
        name="orchestrator",
        run_id=str(rid),
        tag=TAG2X,
    )
    write_json(out_path, orch, tag=TAG2X, indent=2)

    lg.info(
        f"[{TAG2X}] Run finished | overall={overall} | halted={halted} | halted_at={halted_at or ''} | n_reports={len(reports)}"
    )

    return {
        "df": df,
        "df_raw": df_raw,
        "df_clean": df_clean,
        "reports": reports,
        "execution": execution,
        "halted": halted,
        "halted_at": halted_at,
        "orchestrator_report": orch,
    }


def run_dq_valve(
    *,
    ctx: Dict[str, Any],
    df: Optional[pd.DataFrame],
    run_id: Optional[str] = None,
    registry: Optional[Dict[str, ModuleFn]] = None,
    fn_name_map: Optional[Dict[str, str]] = None,
) -> Dict[str, Any]:
    """
    Notebook one-call entrypoint (NO imports).
    Priority: explicit registry > resolve from globals().
    """
    reg = registry or build_dq_registry_from_globals(tag=TAG2X, fn_name_map=fn_name_map, allow_missing=True)
    return run_data_quality_orchestrator(ctx=ctx, registry=reg, df=df, run_id=run_id)

reg2 = build_dq_registry_from_globals(allow_missing=True)
out2 = run_dq_valve(ctx=ctx, df=None, registry=reg2)

rid = out2["orchestrator_report"]["run_id"]
df_raw = out2.get("df_raw")
df_clean = out2.get("df_clean")

print("[DQ] rid:", rid, "halted:", out2["halted"], "halted_at:", out2["halted_at"])

00:19:49 | INFO | [DQ_ORCH] Run started | stage=data_quality | run_id=20260222-231948-167 | strict=True | df_input_provided=False
00:19:49 | INFO | [DQ_ORCH] Plan | stage_enabled=True | ingestion=1 | diagnostics=8 | cleaning=1
00:19:49 | INFO | [INGESTION] ▶ Ingestion start | mode=csv | run_id=20260222-231948-167
00:19:49 | INFO | [INGESTION] 📄 CSV | dir=D:\DS_project\telco-churn-project\data\raw | glob=*.csv | encoding=utf-8 | selected=D:\DS_project\telco-churn-project\data\raw\Telco-Customer-Churn.csv
00:19:49 | INFO | [INGESTION] ✅ Data loaded: 7043 rows × 21 cols
00:19:49 | INFO | [INGESTION] 🧾 Ingestion artifact saved: D:\DS_project\telco-churn-project\artifacts\data_quality_ingestion_20260222-231948-167.json
00:19:49 | INFO | [DQ_ORCH] ▶ post_ingestion step_id=contract | kind=df | gate=hard | strict=True
00:19:49 | INFO | [CONTRACT] 🔍 Starting data contract validation...
00:19:49 | WARNING | [CONTRACT] ⚠ Contract warn | warnings=['dtype_issues_detected']
00:19:49 | INFO | [CONTRA

[DQ] rid: 20260222-231948-167 halted: False halted_at: None


# 3 Data Integrity & Anti-Leakage
🎯 **Goal:** Establish a leak-safe, temporally consistent modeling dataset by enforcing row identity, temporal ordering, label–feature time alignment, automated leakage risk scanning, and a final integrity gate — ensuring that all downstream splits, features, and models only use information available at prediction time rather than leaked future knowledge.

In [13]:
# --- 3.1 Row Identity & Duplicate Integrity ---
#
# What:
#   Validate row identity: primary key uniqueness and (optional) label consistency per entity.
#
# How:
#   - Read SSOT config: dataset.keys.primary, dataset.label.col, pipeline.data_integrity.row_identity
#   - Build a stable row_id (supports composite keys) and detect duplicates / label conflicts
#   - Emit ModuleReport v1 + artifact for gate-friendly orchestration
#
# Why:
#   Ensures the unit of analysis is stable before splits and leakage checks.

from typing import Any, Dict, List, Optional

import pandas as pd

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint
from src.pipeline.data_integrity.row_identity import build_row_id_series
from src.utils.artifact_utils import resolve_artifact_path, write_json
from src.utils.cfg_utils import get_cfg_dict, get_cfg_int, get_cfg_bool
from src.utils.dataset_utils import get_label_cfg, get_primary_keys

TAG31 = "ROW_IDENTITY"


def run_row_identity_integrity(
    *,
    df: pd.DataFrame,
    ctx: Dict[str, Any],
    run_id: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Run row identity & duplicate integrity checks in a ctx-driven, artifact-persisted manner.

    YAML SSOT
    ---------
    dataset:
      keys:
        primary: ["customerID"]   # supports composite keys
      label:
        col: "Churn"

    pipeline.data_integrity.row_identity:
      max_duplicate_examples: 20
      label_conflict_check: true

    Returns
    -------
    ModuleReport v1 (Row Identity & Duplicate Integrity) (also persisted as artifact).
    """
    # --- 2) Runtime validation ---
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG31, run_id=run_id)

    # --- 3) Input validation ---
    if df is None or not isinstance(df, pd.DataFrame):
        raise TypeError(f"[{TAG31}] df must be a pandas DataFrame")

    # --- 4) Resolve config (YAML SSOT) ---
    label_col, _, _ = get_label_cfg(ctx, TAG31)
    id_columns = get_primary_keys(ctx, TAG31)

    row_cfg = get_cfg_dict(ctx, "pipeline.data_integrity.row_identity", tag=TAG31, required=False, default={}) or {}

    max_examples = get_cfg_int(
        ctx,
        "pipeline.data_integrity.row_identity.max_duplicate_examples",
        tag=TAG31,
        default=int(row_cfg.get("max_duplicate_examples", 20) or 20),
        required=False,
        strict_type=False,
        min_value=1,
        max_value=200,
    )

    label_conflict_check = get_cfg_bool(
        ctx,
        "pipeline.data_integrity.row_identity.label_conflict_check",
        tag=TAG31,
        default=bool(row_cfg.get("label_conflict_check", True)),
        required=False,
        strict_type=False,  # allow "true"/"false"/1/0 if needed
    )

    # --- 5) Core workflow ---
    lg.info(f"[{TAG31}] 🧬 Starting row identity checks... | run_id={rid}")

    report = build_module_report(
        ctx=ctx,
        stage="data_integrity",
        step="3.1",
        name="row_identity",
        tag=TAG31,
        cfg_key="pipeline.data_integrity.row_identity",
        enabled=True,
        df=df,
        run_id_override=str(rid),
    )

    report["inputs"] = {
        "df_cleaned": True,
        "id_columns": list(id_columns),
        "label_col": label_col,
    }

    report["thresholds_used"] = {
        "max_duplicate_examples": int(max_examples),
        "label_conflict_check": bool(label_conflict_check),
    }

    report["used_config"] = {
        "label_col": label_col,
        "primary_keys": list(id_columns),
    }

    if not isinstance(report.get("refs"), dict):
        report["refs"] = {}

    report["refs"]["self"] = make_artifact_ref(
        ctx=ctx,
        stage="data_integrity",
        name="row_identity",
        run_id=str(rid),
    )

    ensure_event_hints(report, version=1)

    # --- Facts -> checks ---
    checks: Dict[str, Any] = {"identity": {}, "label_conflicts": {}}

    # A) No PK configured -> warn (not fail), skip identity-based checks
    if not id_columns:
        checks["identity"] = {"status": "skipped", "reason": "primary_keys_not_configured"}
        checks["label_conflicts"] = {"status": "skipped", "reason": "identity_skipped_no_primary_keys"}

        add_event_hint(
            report,
            code="row_identity_primary_keys_not_configured",
            severity_signal="risk",
            evidence_path="inputs.id_columns",
            context={"id_columns": [], "reason": "primary_keys_not_configured"},
            remediation={
                "action": "configure_dataset_primary_keys",
                "safe": True,
                "rationale": "Primary keys are required for row identity, deduplication, and reliable split strategy.",
                "post_check": "set dataset.keys.primary in YAML and re-run 3.1",
            },
        )

        overall = "warn"
        report["checks"].update(checks)

        report = finalize_module_report(
            ctx=ctx,
            report=report,
            overall_status=overall,
            issues=[],
            warnings=["primary_keys_not_configured"],
            metrics={
                "n_rows": int(df.shape[0]),
                "n_cols": int(df.shape[1]),
                "n_id_cols": 0,
                "label_conflict_check": int(bool(label_conflict_check)),
            },
            notes=[],
            df=df,
        )

        out_path = resolve_artifact_path(ctx=ctx, stage="data_integrity", name="row_identity", run_id=str(rid), tag=TAG31)
        write_json(out_path, report, tag=TAG31, indent=2)
        lg.info(f"[{TAG31}] 🧾 Row identity artifact saved (warn): {out_path}")
        return report

    # B) Build stable row_id (use missing_token to avoid NaN duplicate ambiguity)
    try:
        id_series = build_row_id_series(df, id_columns, tag=TAG31, missing_token="__MISSING__")
    except Exception as e:
        checks["identity"] = {"status": "fail", "error": f"{type(e).__name__}: {e}"}
        checks["label_conflicts"] = {"status": "skipped", "reason": "identity_build_failed"}

        add_event_hint(
            report,
            code="row_identity_build_failed",
            severity_signal="hard",
            evidence_path="checks.identity",
            context={"error": f"{type(e).__name__}: {e}", "id_columns": list(id_columns)},
            remediation={
                "action": "fix_primary_key_columns_or_schema",
                "safe": True,
                "rationale": "Identity series construction failed; keys might be missing or invalid.",
                "post_check": "re-run 2.2 contract and confirm PK columns exist",
            },
        )

        report["checks"].update(checks)

        report = finalize_module_report(
            ctx=ctx,
            report=report,
            overall_status="fail",
            issues=["failed_to_build_identity"],
            warnings=[],
            metrics={"n_rows": int(df.shape[0]), "n_cols": int(df.shape[1]), "n_id_cols": int(len(id_columns))},
            notes=[],
            df=df,
        )

        out_path = resolve_artifact_path(ctx=ctx, stage="data_integrity", name="row_identity", run_id=str(rid), tag=TAG31)
        write_json(out_path, report, tag=TAG31, indent=2)
        lg.info(f"[{TAG31}] 🧾 Row identity artifact saved (fail): {out_path}")
        return report

    # Identity duplicate check
    df_tmp = df.assign(_row_id=id_series)

    total_rows = int(df_tmp.shape[0])
    total_unique_ids = int(df_tmp["_row_id"].nunique(dropna=False))
    num_duplicates = int(total_rows - total_unique_ids)
    duplicate_share = float(num_duplicates / total_rows) if total_rows > 0 else 0.0

    dup_mask = df_tmp["_row_id"].duplicated(keep=False)

    duplicate_keys: List[Any] = []
    duplicate_examples: List[Dict[str, Any]] = []

    if num_duplicates > 0:
        duplicate_keys = (
            df_tmp.loc[dup_mask, "_row_id"]
            .drop_duplicates()
            .head(int(max_examples))
            .tolist()
        )

        for key in duplicate_keys:
            rows = df_tmp[df_tmp["_row_id"].eq(key)].head(3)
            duplicate_examples.append(
                {
                    "id_value": str(key),
                    "n_rows": int(rows.shape[0]),
                    "example_rows": rows.drop(columns=["_row_id"], errors="ignore").to_dict(orient="records"),
                }
            )

    checks["identity"] = {
        "status": "pass" if num_duplicates == 0 else "fail",
        "total_rows": int(total_rows),
        "total_unique_ids": int(total_unique_ids),
        "num_duplicates": int(num_duplicates),
        "duplicate_share": float(duplicate_share),
        "sample_duplicate_keys": [str(k) for k in duplicate_keys],
        "sample_duplicate_examples": duplicate_examples,
    }

    if num_duplicates > 0:
        add_event_hint(
            report,
            code="row_identity_duplicate_ids_detected",
            severity_signal="hard",
            evidence_path="checks.identity.num_duplicates",
            context={
                "num_duplicates": int(num_duplicates),
                "duplicate_share": float(duplicate_share),
                "sample_keys": [str(x) for x in duplicate_keys[: min(5, len(duplicate_keys))]],
            },
            remediation={
                "action": "deduplicate_or_fix_upstream_joins",
                "safe": True,
                "rationale": "Duplicate IDs break the assumption of one row per entity snapshot.",
                "post_check": "re-run 3.1 and confirm num_duplicates == 0",
            },
        )

    # Label conflict check (only when enabled and label exists)
    if not bool(label_conflict_check):
        checks["label_conflicts"] = {"status": "skipped", "reason": "label_conflict_check_disabled"}
    elif (not label_col) or (label_col not in df_tmp.columns):
        checks["label_conflicts"] = {"status": "skipped", "reason": "label_col_missing_or_not_found", "label_col": label_col}
        add_event_hint(
            report,
            code="row_identity_label_col_missing_skip",
            severity_signal="risk",
            evidence_path="inputs.label_col",
            context={"label_col": label_col, "available_cols": list(df_tmp.columns)},
            remediation={
                "action": "fix_dataset_label_config_or_schema",
                "safe": True,
                "rationale": "Label conflict checks require a valid label column; skipping may hide per-entity inconsistencies.",
                "post_check": "confirm label_col exists then re-run 3.1",
            },
        )
    else:
        # Normalize label strings to avoid false conflicts caused by trailing/leading spaces
        label_norm = df_tmp[label_col].astype("string").str.strip()
        df_norm = df_tmp.assign(_label_norm=label_norm)

        # Only consider non-null normalized labels for conflicts
        grp = (
            df_norm.dropna(subset=["_label_norm"])
            .groupby("_row_id", dropna=False)["_label_norm"]
            .nunique(dropna=True)
        )
        conflict_mask = grp > 1
        num_conflict_ids = int(conflict_mask.sum())

        conflict_ids_raw = grp[conflict_mask].index.to_series().head(int(max_examples)).tolist()

        examples: List[Dict[str, Any]] = []
        for key in conflict_ids_raw:
            rows = df_norm[df_norm["_row_id"].eq(key)].head(5)
            examples.append(
                {
                    "id_value": str(key),
                    "n_rows": int(rows.shape[0]),
                    "label_values": rows["_label_norm"].dropna().unique().tolist(),
                    "example_rows": rows.drop(columns=["_row_id", "_label_norm"], errors="ignore").to_dict(orient="records"),
                }
            )

        checks["label_conflicts"] = {
            "status": "pass" if num_conflict_ids == 0 else "fail",
            "enabled": True,
            "label_col": label_col,
            "num_ids_with_conflicting_labels": int(num_conflict_ids),
            "sample_conflict_ids": [str(x) for x in conflict_ids_raw],
            "sample_conflict_examples": examples,
        }

        if num_conflict_ids > 0:
            add_event_hint(
                report,
                code="row_identity_label_conflicts_detected",
                severity_signal="hard",
                evidence_path="checks.label_conflicts.num_ids_with_conflicting_labels",
                context={"num_conflict_ids": int(num_conflict_ids), "label_col": label_col},
                remediation={
                    "action": "resolve_label_conflicts_per_id",
                    "safe": True,
                    "rationale": "Conflicting labels per entity invalidate supervised learning assumptions.",
                    "post_check": "re-run 3.1 and confirm conflicts==0",
                },
            )

    report["checks"].update(checks)

    # --- 6) Status + gate-friendly summary ---
    issues: List[str] = []
    warnings: List[str] = []

    if checks["identity"].get("status") == "fail":
        issues.append("duplicate_ids_detected")

    if checks["label_conflicts"].get("status") == "fail":
        issues.append("label_conflicts_detected")

    overall = "fail" if issues else ("warn" if warnings else "pass")

    report = finalize_module_report(
        ctx=ctx,
        report=report,
        overall_status=overall,
        issues=issues,
        warnings=warnings,
        metrics={
            "n_rows": int(df.shape[0]),
            "n_cols": int(df.shape[1]),
            "n_id_cols": int(len(id_columns)),
            "label_conflict_check": int(bool(label_conflict_check)),
        },
        notes=[],
        df=df,
    )

    if overall == "pass":
        lg.info(f"[{TAG31}] ✅ Row identity pass.")
    elif overall == "warn":
        lg.warning(f"[{TAG31}] ⚠ Row identity warn. warnings={warnings}")
    else:
        lg.error(f"[{TAG31}] ❌ Row identity fail. issues={issues}")

    # --- 7) Persist artifact ---
    out_path = resolve_artifact_path(ctx=ctx, stage="data_integrity", name="row_identity", run_id=str(rid), tag=TAG31)
    write_json(out_path, report, tag=TAG31, indent=2)
    lg.info(f"[{TAG31}] 🧾 Row identity artifact saved: {out_path}")

    return report

In [14]:
# --- 3.2 Temporal & Snapshot Integrity ---
#
# What:
#   Validate snapshot timestamp correctness (when configured) and detect
#   duplicate snapshots / out-of-order sequences per entity.
#
# How:
#   - Read SSOT config: pipeline.data_integrity.temporal (+ dataset.keys.primary fallback)
#   - Parse timestamp column, enforce optional bounds, and run sequence checks
#   - Emit ModuleReport v1 + artifact for gate-friendly orchestration
#
# Why:
#   Prevent temporal leakage and ensure downstream split strategy is compatible
#   with the dataset’s time structure.

# --- 0) Import + TAG ---
from typing import Any, Dict, List, Optional

import pandas as pd

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint
from src.pipeline.data_integrity.row_identity import build_row_id_series
from src.utils.artifact_utils import resolve_artifact_path, write_json
from src.utils.cfg_utils import get_cfg_dict, get_cfg_int, get_cfg_bool, get_cfg_str, get_cfg_str_list
from src.utils.dataset_utils import get_primary_keys

TAG32 = "TEMPORAL_INTEGRITY"


# --- Internal helpers ---
def _parse_ts_bound(raw: Any) -> Optional[pd.Timestamp]:
    """Parse ISO-like timestamp bound from config; normalize to tz-naive UTC Timestamp."""
    if raw is None:
        return None

    if isinstance(raw, str):
        s = raw.strip()
        if not s:
            return None
        ts = pd.to_datetime(s, errors="coerce", utc=True)
        if pd.isna(ts):
            return None
        # Convert to tz-naive UTC for safe comparisons
        return ts.tz_localize(None)

    if isinstance(raw, pd.Timestamp):
        ts = raw
        # Normalize tz-aware -> tz-naive UTC
        if ts.tz is not None:
            ts = ts.tz_convert("UTC").tz_localize(None)
        return ts

    return None


def _has_out_of_order_original(ts_series: pd.Series) -> bool:
    """
    Out-of-order definition: in original appearance order, timestamps should be non-decreasing.
    NaT comparisons yield False; missing handling is covered by allow_missing_timestamps policy.
    """
    ts_seq = ts_series.values
    if len(ts_seq) < 2:
        return False
    prev = ts_seq[:-1].astype("datetime64[ns]")
    curr = ts_seq[1:].astype("datetime64[ns]")
    return (curr < prev).any()


# --- 1) Public API ---
def run_temporal_snapshot_integrity(
    *,
    df: pd.DataFrame,
    ctx: Dict[str, Any],
    run_id: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Run temporal & snapshot integrity checks in a ctx-driven, artifact-persisted manner.

    YAML SSOT
    ---------
    pipeline.data_integrity.temporal:
      timestamp_col: null
      entity_id_columns: []
      allow_missing_timestamps: false
      min_allowed_timestamp: null
      max_allowed_timestamp: null
      max_duplicate_examples: 20
      max_out_of_order_examples: 20
      out_of_order_definition: "original_order"

    Returns
    -------
    ModuleReport v1 (Temporal & Snapshot Integrity) (also persisted as artifact).
    """

    # --- 2) Runtime validation ---
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG32, run_id=run_id)

    # --- 3) Input validation ---
    if df is None or not isinstance(df, pd.DataFrame):
        raise TypeError(f"[{TAG32}] df must be a pandas DataFrame")

    # --- 4) Resolve config (YAML SSOT) ---
    temporal_cfg = get_cfg_dict(ctx, "pipeline.data_integrity.temporal", tag=TAG32, default={}) or {}

    timestamp_col = get_cfg_str(
        ctx,
        "pipeline.data_integrity.temporal.timestamp_col",
        tag=TAG32,
        required=False,
        default="",
        strict_type=False,
    )
    timestamp_col = (timestamp_col or "").strip()
    if timestamp_col.lower() in {"none", "null"}:
        timestamp_col = ""
    
    allow_missing_ts = get_cfg_bool(
        ctx,
        "pipeline.data_integrity.temporal.allow_missing_timestamps",
        tag=TAG32,
        default=False,
        required=False,
        strict_type=True,
    )

    max_dup_examples = get_cfg_int(
        ctx,
        "pipeline.data_integrity.temporal.max_duplicate_examples",
        tag=TAG32,
        default=20,
        required=False,
        strict_type=False,
        min_value=1,
        max_value=200,
    )

    max_oo_examples = get_cfg_int(
        ctx,
        "pipeline.data_integrity.temporal.max_out_of_order_examples",
        tag=TAG32,
        default=20,
        required=False,
        strict_type=False,
        min_value=1,
        max_value=200,
    )

    min_ts = _parse_ts_bound(temporal_cfg.get("min_allowed_timestamp"))
    max_ts = _parse_ts_bound(temporal_cfg.get("max_allowed_timestamp"))

    # entity_id_columns semantics:
    # - If key exists in YAML: respect it (even empty list => explicitly disable sequences)
    # - If key missing: fallback to dataset.keys.primary
    if "entity_id_columns" in temporal_cfg:
        entity_cols = get_cfg_str_list(ctx, "pipeline.data_integrity.temporal.entity_id_columns", tag=TAG32, required=False)
        entity_cols_source = "yaml.temporal.entity_id_columns"
    else:
        entity_cols = get_primary_keys(ctx, TAG32)
        entity_cols_source = "dataset.keys.primary_fallback"

    out_of_order_def = get_cfg_str(
        ctx,
        "pipeline.data_integrity.temporal.out_of_order_definition",
        tag=TAG32,
        required=False,
        default="original_order",
        strict_type=False,
    ).strip() or "original_order"

    if out_of_order_def not in {"original_order"}:
        out_of_order_def_raw = out_of_order_def
        out_of_order_def = "original_order"
    else:
        out_of_order_def_raw = None

    # --- 5) Core workflows ---
    lg.info(f"[{TAG32}] ⏱️ Starting temporal & snapshot integrity checks... | run_id={rid}")

    # Build ModuleReport skeleton (infra)
    report = build_module_report(
        ctx=ctx,
        stage="data_integrity",
        step="3.2",
        name="temporal",
        tag=TAG32,
        cfg_key="pipeline.data_integrity.temporal",
        enabled=True,
        df=df,
        run_id_override=str(rid),
    )

    report["inputs"] = {
        "df_cleaned": True,
        "timestamp_col": timestamp_col or None,
        "entity_id_columns": list(entity_cols),
    }

    report["thresholds_used"] = {
        "allow_missing_timestamps": bool(allow_missing_ts),
        "min_allowed_timestamp": min_ts.isoformat() if isinstance(min_ts, pd.Timestamp) else None,
        "max_allowed_timestamp": max_ts.isoformat() if isinstance(max_ts, pd.Timestamp) else None,
        "max_duplicate_examples": int(max_dup_examples),
        "max_out_of_order_examples": int(max_oo_examples),
        "out_of_order_definition": out_of_order_def,
    }

    report["used_config"] = {
        "timestamp_col": timestamp_col or None,
        "entity_id_columns_source": entity_cols_source,
    }

    if not isinstance(report.get("refs"), dict):
        report["refs"] = {}

    report["refs"]["self"] = make_artifact_ref(
        ctx=ctx,
        stage="data_integrity",
        name="temporal",
        run_id=str(rid),
    )

    ensure_event_hints(report, version=1)

    if out_of_order_def_raw is not None:
        add_event_hint(
            report,
            code="temporal_unknown_out_of_order_definition",
            severity_signal="risk",
            evidence_path="thresholds_used.out_of_order_definition",
            context={"provided": out_of_order_def_raw, "fallback": "original_order"},
            remediation={
                "action": "use_supported_out_of_order_definition",
                "safe": True,
                "rationale": "Unknown out_of_order_definition; falling back to original_order.",
                "post_check": "re-run 3.2 and confirm sequences behavior matches expectation",
            },
        )

    # --- Core computation (facts -> checks) ---
    facts: Dict[str, Any] = {"temporal": None, "sequences": None}
    checks: Dict[str, Any] = {"temporal": {}, "sequences": {}}

    # Case A: timestamp not configured or missing -> skip everything (gate-friendly)
    if (not timestamp_col) or (timestamp_col not in df.columns):
        reason = "timestamp_column_not_configured_or_missing"
        checks["temporal"] = {"status": "skipped", "reason": reason, "timestamp_col": timestamp_col or None}
        checks["sequences"] = {"status": "skipped", "reason": reason}

        add_event_hint(
            report,
            code="temporal_timestamp_missing_skip",
            severity_signal="info",
            evidence_path="inputs.timestamp_col",
            context={"timestamp_col": timestamp_col or None, "available_cols": list(df.columns)},
            remediation={
                "action": "configure_timestamp_col_if_applicable",
                "safe": True,
                "rationale": "Temporal checks require a snapshot timestamp column; otherwise they are skipped.",
                "post_check": "re-run 3.2 and confirm temporal/sequences status",
            },
        )

        report["checks"].update(checks)

        report = finalize_module_report(
            ctx=ctx,
            report=report,
            overall_status="skipped",
            issues=[],
            warnings=[],
            metrics={"n_rows": int(df.shape[0]), "n_cols": int(df.shape[1])},
            notes=["temporal_checks_skipped_no_timestamp"],
            df=df,
        )

        out_path = resolve_artifact_path(ctx=ctx, stage="data_integrity", name="temporal", run_id=str(rid), tag=TAG32)
        write_json(out_path, report, tag=TAG32, indent=2)
        lg.info(f"[{TAG32}] 🧾 Temporal integrity artifact saved (skipped): {out_path}")
        return report

    # Parse timestamp column
    ts_parsed = pd.to_datetime(df[timestamp_col], errors="coerce", utc=True)
    ts_parsed = ts_parsed.dt.tz_localize(None)  # tz-naive UTC

    total_rows = int(df.shape[0])
    num_missing_ts = int(ts_parsed.isna().sum())

    min_observed_ts = ts_parsed.min()
    max_observed_ts = ts_parsed.max()

    out_of_range_mask = pd.Series(False, index=df.index)
    if isinstance(min_ts, pd.Timestamp):
        out_of_range_mask |= (ts_parsed < min_ts)
    if isinstance(max_ts, pd.Timestamp):
        out_of_range_mask |= (ts_parsed > max_ts)
    num_out_of_range = int(out_of_range_mask.sum())

    facts["temporal"] = {
        "total_rows": int(total_rows),
        "num_missing_or_invalid_timestamps": int(num_missing_ts),
        "min_observed_timestamp": None if pd.isna(min_observed_ts) else min_observed_ts.isoformat(),
        "max_observed_timestamp": None if pd.isna(max_observed_ts) else max_observed_ts.isoformat(),
        "num_out_of_range_rows": int(num_out_of_range),
    }

    # Temporal status
    temporal_status = "pass"
    temporal_reasons: List[str] = []

    if pd.isna(min_observed_ts) or pd.isna(max_observed_ts):
        temporal_status = "fail"
        temporal_reasons.append("all_timestamps_nat_after_parsing")

    if num_missing_ts > 0:
        if not bool(allow_missing_ts):
            temporal_status = "fail"
            temporal_reasons.append("missing_or_invalid_timestamps_not_allowed")
        else:
            if temporal_status != "fail":
                temporal_status = "warn"
            temporal_reasons.append("missing_or_invalid_timestamps_allowed")

    if num_out_of_range > 0:
        temporal_status = "fail"
        temporal_reasons.append("out_of_range_timestamps_detected")

    checks["temporal"] = {
        "status": temporal_status,
        **(facts["temporal"] or {}),
        "reasons": temporal_reasons,
    }

    # Sequence checks: controlled by entity_id_columns
    if not entity_cols:
        checks["sequences"] = {"status": "skipped", "reason": "entity_id_columns_not_configured"}
        add_event_hint(
            report,
            code="temporal_sequences_skipped_no_entity_id_columns",
            severity_signal="info",
            evidence_path="inputs.entity_id_columns",
            context={"entity_id_columns": [], "source": entity_cols_source},
            remediation={
                "action": "configure_entity_id_columns_if_sequence_checks_needed",
                "safe": True,
                "rationale": "Sequence checks require entity_id_columns; otherwise they are skipped.",
                "post_check": "re-run 3.2 and confirm sequences status",
            },
        )
        facts["sequences"] = None
    else:
        # Build stable entity identity
        row_id = build_row_id_series(df, entity_cols, tag=TAG32, missing_token="__MISSING__")
        df_tmp = df.assign(_row_id=row_id, _ts=ts_parsed)

        # Duplicate snapshots: same entity + same timestamp (parsed)
        grp_counts = df_tmp.groupby(["_row_id", "_ts"], dropna=False).size()
        dup_snapshot_mask = grp_counts > 1
        num_duplicate_snapshots = int(dup_snapshot_mask.sum())

        duplicate_snapshot_examples: List[Dict[str, Any]] = []
        if num_duplicate_snapshots > 0:
            dup_keys = grp_counts[dup_snapshot_mask].reset_index().head(int(max_dup_examples))
            for _, r in dup_keys.iterrows():
                key_row_id = r["_row_id"]
                key_ts = r["_ts"]
                cond = (df_tmp["_row_id"].eq(key_row_id)) & (df_tmp["_ts"].eq(key_ts))
                rows = df_tmp[cond].head(5)
                duplicate_snapshot_examples.append(
                    {
                        "row_id": str(key_row_id),
                        "timestamp": None if pd.isna(key_ts) else pd.Timestamp(key_ts).isoformat(),
                        "n_rows": int(rows.shape[0]),
                        "example_rows": rows.drop(columns=["_row_id", "_ts"], errors="ignore").to_dict(orient="records"),
                    }
                )

        # Out-of-order (definition)
        out_of_order_entities: List[Dict[str, Any]] = []
        num_entities_with_oo = 0

        grouped = df_tmp.groupby("_row_id", dropna=False, sort=False)
        for rid_key, sub in grouped:
            has_oo = _has_out_of_order_original(sub["_ts"])

            if has_oo:
                num_entities_with_oo += 1
                if len(out_of_order_entities) < int(max_oo_examples):
                    preview = sub["_ts"].head(10).tolist()
                    out_of_order_entities.append(
                        {
                            "row_id": str(rid_key),
                            "n_rows": int(sub.shape[0]),
                            "example_timestamps_in_order": [
                                None if pd.isna(x) else pd.Timestamp(x).isoformat() for x in preview
                            ],
                        }
                    )

        facts["sequences"] = {
            "num_duplicate_snapshots": int(num_duplicate_snapshots),
            "duplicate_snapshot_examples": duplicate_snapshot_examples,
            "num_entities_with_out_of_order_timestamps": int(num_entities_with_oo),
            "out_of_order_entity_examples": out_of_order_entities,
        }

        # Sequence status
        sequence_status = "pass"
        seq_reasons: List[str] = []
        if int(num_duplicate_snapshots) > 0:
            sequence_status = "fail"
            seq_reasons.append("duplicate_snapshots_detected")
        if int(num_entities_with_oo) > 0:
            sequence_status = "fail"
            seq_reasons.append("out_of_order_sequences_detected")

        checks["sequences"] = {
            "status": sequence_status,
            **(facts["sequences"] or {}),
            "reasons": seq_reasons,
        }

        # Event hints for sequences
        if int(num_duplicate_snapshots) > 0:
            add_event_hint(
                report,
                code="temporal_duplicate_snapshots_detected",
                severity_signal="hard",
                evidence_path="checks.sequences.num_duplicate_snapshots",
                context={"num_duplicate_snapshots": int(num_duplicate_snapshots), "timestamp_col": timestamp_col},
                remediation={
                    "action": "deduplicate_entity_timestamp_pairs",
                    "safe": True,
                    "rationale": "Multiple rows for the same entity at the same timestamp break snapshot integrity.",
                    "post_check": "re-run 3.2 and confirm duplicate_snapshots==0",
                },
            )

        if int(num_entities_with_oo) > 0:
            add_event_hint(
                report,
                code="temporal_out_of_order_sequences_detected",
                severity_signal="risk",
                evidence_path="checks.sequences.num_entities_with_out_of_order_timestamps",
                context={
                    "num_entities_with_out_of_order_timestamps": int(num_entities_with_oo),
                    "timestamp_col": timestamp_col,
                    "out_of_order_definition": out_of_order_def,
                },
                remediation={
                    "action": "sort_or_reconstruct_sequences_if_order_is_semantic",
                    "safe": True,
                    "rationale": "Out-of-order sequences can indicate mixed snapshots or ingestion ordering issues.",
                    "post_check": "re-run 3.2 and confirm out_of_order==0 (if required)",
                },
            )

    report["checks"].update(checks)

    # Event hints for temporal
    if temporal_status == "fail" and "all_timestamps_nat_after_parsing" in temporal_reasons:
        add_event_hint(
            report,
            code="temporal_all_timestamps_nat",
            severity_signal="hard",
            evidence_path="checks.temporal.min_observed_timestamp",
            context={"timestamp_col": timestamp_col},
            remediation={
                "action": "fix_timestamp_parsing_or_source_schema",
                "safe": True,
                "rationale": "All timestamps became NaT after parsing; temporal structure is unusable.",
                "post_check": "re-run 3.2 and confirm observed range is valid",
            },
        )

    if num_missing_ts > 0 and (not bool(allow_missing_ts)):
        add_event_hint(
            report,
            code="temporal_missing_timestamps_not_allowed",
            severity_signal="hard",
            evidence_path="checks.temporal.num_missing_or_invalid_timestamps",
            context={"num_missing_or_invalid_timestamps": int(num_missing_ts), "timestamp_col": timestamp_col},
            remediation={
                "action": "clean_or_impute_timestamps_or_allow_missing",
                "safe": True,
                "rationale": "Missing/invalid timestamps violate the configured temporal policy.",
                "post_check": "re-run 3.2 and confirm missing count meets policy",
            },
        )

    if num_out_of_range > 0:
        add_event_hint(
            report,
            code="temporal_out_of_range_timestamps",
            severity_signal="hard",
            evidence_path="checks.temporal.num_out_of_range_rows",
            context={
                "num_out_of_range_rows": int(num_out_of_range),
                "min_allowed_timestamp": min_ts.isoformat() if isinstance(min_ts, pd.Timestamp) else None,
                "max_allowed_timestamp": max_ts.isoformat() if isinstance(max_ts, pd.Timestamp) else None,
            },
            remediation={
                "action": "filter_or_fix_out_of_range_rows",
                "safe": True,
                "rationale": "Out-of-range timestamps may indicate data corruption or temporal leakage risk.",
                "post_check": "re-run 3.2 and confirm out_of_range==0",
            },
        )

    # --- 6) Status + gate-friendly summary ---
    issues: List[str] = []
    warnings: List[str] = []

    if checks["temporal"].get("status") == "fail":
        issues.append("temporal_checks_failed")
    elif checks["temporal"].get("status") == "warn":
        warnings.append("temporal_checks_warning")

    seq_status = checks["sequences"].get("status")
    if seq_status == "fail":
        issues.append("sequence_checks_failed")

    overall = "fail" if issues else ("warn" if warnings else "pass")

    report = finalize_module_report(
        ctx=ctx,
        report=report,
        overall_status=overall,
        issues=issues,
        warnings=warnings,
        metrics={
            "n_rows": int(df.shape[0]),
            "n_cols": int(df.shape[1]),
            "n_entity_cols": int(len(entity_cols)),
            "n_missing_ts": int(num_missing_ts),
            "n_out_of_range": int(num_out_of_range),
            "out_of_order_definition": out_of_order_def,
        },
        notes=[],
        df=df,
    )

    if overall == "pass":
        lg.info(f"[{TAG32}] ✅ Temporal integrity pass.")
    elif overall == "warn":
        lg.warning(f"[{TAG32}] ⚠ Temporal integrity warn. warnings={warnings}")
    else:
        lg.error(f"[{TAG32}] ❌ Temporal integrity fail. issues={issues}")

    # --- 8) Persist artifact ---
    out_path = resolve_artifact_path(ctx=ctx, stage="data_integrity", name="temporal", run_id=str(rid), tag=TAG32)
    write_json(out_path, report, tag=TAG32, indent=2)
    lg.info(f"[{TAG32}] 🧾 Temporal integrity artifact saved: {out_path}")

    return report

In [15]:
# --- 3.3 Label–Feature Alignment & Causality Guards ---
#
# What:
#   Validate label quality and its alignment with the feature space, and add
#   basic guards against trivial label leakage (alias / risky feature names).
#
# How:
#   - Read SSOT: dataset.label (col, positive, task_type)
#   - Read module cfg: pipeline.data_integrity.label_alignment
#   - Summarize label health (missingness / cardinality / balance)
#   - Detect high-risk feature names and obvious label aliases on a sample
#   - Emit ModuleReport v1 + artifact (SSOT naming)
#
# Why:
#   Catch degenerate labels early and prevent trivial leakage before split/modeling.

from typing import Any, Dict, List, Optional

import pandas as pd

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint
from src.utils.artifact_utils import resolve_artifact_path, write_json
from src.utils.cfg_utils import (
    get_cfg_dict,
    get_cfg_int,
    get_cfg_float,
    get_cfg_str_list,
    normalize_str_list,
)
from src.utils.dataset_utils import get_label_cfg
from src.utils.serialization_utils import safe_float

TAG33 = "LABEL_ALIGNMENT"


# --- Internal helpers ---
def _infer_task_type_auto(label_series: pd.Series) -> str:
    """Infer task_type: unique non-null <= 20 => classification else regression."""
    non_null = label_series.dropna()
    n_unique = int(non_null.nunique())
    return "classification" if n_unique <= 20 else "regression"


def _normalize_label_strings(series: pd.Series) -> pd.Series:
    """Normalize label values for robust comparisons (strip spaces)."""
    return series.astype("string").str.strip()


def _detect_label_aliases_on_sample(
    df: pd.DataFrame,
    *,
    label_col: str,
    feature_cols: List[str],
    sample_size: int,
    min_non_missing: int,
    max_examples: int,
) -> Dict[str, Any]:
    """
    Detect obvious label aliases by equality on a sample.

    Missing policy:
      - Compare only on rows where BOTH label and feature are non-missing.
      - Require >= min_non_missing comparable rows.
    """
    total_rows = int(df.shape[0])
    if total_rows <= 0 or not feature_cols:
        return {
            "sample_size": 0,
            "min_non_missing": int(min_non_missing),
            "aliases": [],
            "examples": [],
            "skipped_too_few_non_missing": [],
        }

    n = int(min(max(sample_size, 1), total_rows))
    sample_idx = df.index[:n]

    label_s = df.loc[sample_idx, label_col]
    aliases: List[str] = []
    examples: List[Dict[str, Any]] = []
    skipped: List[Dict[str, Any]] = []

    for col in feature_cols:
        feat_s = df.loc[sample_idx, col]
        mask = label_s.notna() & feat_s.notna()
        k = int(mask.sum())
        if k < int(min_non_missing):
            skipped.append({"feature": col, "n_comparable": int(k)})
            continue

        lhs = label_s[mask].astype("string").str.strip().reset_index(drop=True)
        rhs = feat_s[mask].astype("string").str.strip().reset_index(drop=True)

        if lhs.equals(rhs):
            aliases.append(col)
            if len(examples) < int(max_examples):
                preview_n = min(5, int(k))
                examples.append(
                    {
                        "feature": col,
                        "n_comparable": int(k),
                        "preview_pairs": [
                            {
                                "label": None if pd.isna(lhs.iloc[i]) else str(lhs.iloc[i]),
                                "feature_value": None if pd.isna(rhs.iloc[i]) else str(rhs.iloc[i]),
                            }
                            for i in range(preview_n)
                        ],
                    }
                )

    return {
        "sample_size": int(n),
        "min_non_missing": int(min_non_missing),
        "aliases": aliases,
        "examples": examples,
        "skipped_too_few_non_missing": skipped[: int(max_examples)],
    }


# --- 1) Public API ---
def run_label_feature_alignment(
    *,
    df: pd.DataFrame,
    ctx: Dict[str, Any],
    run_id: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Run label–feature alignment checks in a ctx-driven, artifact-persisted manner.

    YAML SSOT
    ---------
    dataset.label:
      col: "Churn"
      positive: "Yes"
      task_type: "classification"

    pipeline.data_integrity.label_alignment:
      min_positive_share: 0.01
      max_positive_share: 0.99
      forbidden_name_substrings: ["label","target","churn",...]
      sample_size_for_equality: 1000
      max_examples: 20
      min_non_missing_for_equality: 10

    Returns
    -------
    ModuleReport v1 (Label–Feature Alignment) (also persisted as artifact).
    """

    # --- 2) Runtime validation ---
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG33, run_id=run_id)

    # --- 3) Input validation ---
    if df is None or not isinstance(df, pd.DataFrame):
        raise TypeError(f"[{TAG33}] df must be a pandas DataFrame")

    # --- 4) Resolve config (YAML SSOT) ---
    label_col, pos_label, task_type = get_label_cfg(ctx, TAG33)

    get_cfg_dict(ctx, "pipeline.data_integrity.label_alignment", tag=TAG33, default={})

    min_pos_share = get_cfg_float(
        ctx,
        "pipeline.data_integrity.label_alignment.min_positive_share",
        tag=TAG33,
        required=False,
        default=0.0,
        strict_type=False,
        min_value=0.0,
        max_value=1.0,
    )

    max_pos_share = get_cfg_float(
        ctx,
        "pipeline.data_integrity.label_alignment.max_positive_share",
        tag=TAG33,
        required=False,
        default=1.0,
        strict_type=False,
        min_value=0.0,
        max_value=1.0,
    )

    orig_min_pos_share = float(min_pos_share)
    orig_max_pos_share = float(max_pos_share)

    swapped_bounds = False
    if orig_min_pos_share > orig_max_pos_share:
        swapped_bounds = True
        min_pos_share, max_pos_share = max_pos_share, min_pos_share

    forbidden_raw = get_cfg_str_list(
        ctx,
        "pipeline.data_integrity.label_alignment.forbidden_name_substrings",
        tag=TAG33,
        required=False,
        default=[],
    )
    # Use utils normalizer, then lowercase+dedup for matching
    forbidden_substrings = [
        s.lower()
        for s in normalize_str_list(
            forbidden_raw,
            tag=TAG33,
            path="pipeline.data_integrity.label_alignment.forbidden_name_substrings",
        )
    ]

    sample_size = get_cfg_int(
        ctx,
        "pipeline.data_integrity.label_alignment.sample_size_for_equality",
        tag=TAG33,
        default=1000,
        required=False,
        strict_type=False,
        min_value=1,
        max_value=200000,
    )

    max_examples = get_cfg_int(
        ctx,
        "pipeline.data_integrity.label_alignment.max_examples",
        tag=TAG33,
        default=20,
        required=False,
        strict_type=False,
        min_value=1,
        max_value=200,
    )

    min_non_missing = get_cfg_int(
        ctx,
        "pipeline.data_integrity.label_alignment.min_non_missing_for_equality",
        tag=TAG33,
        default=50,
        required=False,
        strict_type=False,
        min_value=1,
        max_value=100000,
    )

    # Guard: min_non_missing should not exceed sample size / rows
    total_rows = int(df.shape[0])
    sample_n_effective = int(min(max(int(sample_size), 1), max(total_rows, 0)))

    min_non_missing_raw = int(min_non_missing)
    min_non_missing_effective = int(min(min_non_missing_raw, max(sample_n_effective, 1)))

    # We'll attach a hint later after report exists
    min_non_missing_clamped = (min_non_missing_effective != min_non_missing_raw)

    # --- 5) Core workflows ---
    lg.info(f"[{TAG33}] 🎯 Starting label–feature alignment checks... | run_id={rid}")

    # Build ModuleReport skeleton (infra)
    report = build_module_report(
        ctx=ctx,
        stage="data_integrity",
        step="3.3",
        name="label_alignment",
        tag=TAG33,
        cfg_key="pipeline.data_integrity.label_alignment",
        enabled=True,
        df=df,
        run_id_override=str(rid),
    )

    report["inputs"] = {
        "df_cleaned": True,
        "label_col": label_col,
        "positive_label": pos_label,
        "task_type_ssot": task_type,
    }

    report["thresholds_used"] = {
        "min_positive_share": safe_float(min_pos_share),
        "max_positive_share": safe_float(max_pos_share),
        "forbidden_name_substrings": list(forbidden_substrings),
        "sample_size_for_equality": int(sample_size),
        "min_non_missing_for_equality": int(min_non_missing_effective),  # <-- use effective
        "max_examples": int(max_examples),
    }

    report["used_config"] = {
        "label_col": label_col,
        "positive_label": pos_label,
        "task_type_ssot": task_type,
        "min_non_missing_for_equality_raw": int(min_non_missing_raw),  # <-- keep raw for audit
    }

    if not isinstance(report.get("refs"), dict):
        report["refs"] = {}

    report["refs"]["self"] = make_artifact_ref(
        ctx=ctx,
        stage="data_integrity",
        name="label_alignment",
        run_id=str(rid),
    )

    ensure_event_hints(report, version=1)

    # --- Core computation (facts -> checks) ---
    checks: Dict[str, Any] = {"label_health": {}, "name_risks": {}, "label_aliases": {}}

    # Defensive: label col exists
    if (not label_col) or (label_col not in df.columns):
        checks["label_health"] = {
            "status": "fail",
            "reason": "label_column_missing",
            "label_col": label_col,
        }
        checks["name_risks"] = {"status": "skipped", "reason": "label_unavailable"}
        checks["label_aliases"] = {"status": "skipped", "reason": "label_unavailable"}

        report["checks"].update(checks)

        # --- Event hints (in this early-return path) ---
        if swapped_bounds:
            add_event_hint(
                report,
                code="label_alignment_positive_share_bounds_swapped",
                severity_signal="risk",
                evidence_path="thresholds_used",
                context={
                    "original_min_positive_share": safe_float(orig_min_pos_share),
                    "original_max_positive_share": safe_float(orig_max_pos_share),
                },
                remediation={
                    "action": "fix_positive_share_bounds",
                    "safe": True,
                    "rationale": "min_positive_share was greater than max_positive_share; bounds were swapped to keep checks runnable.",
                    "post_check": "update YAML and re-run 3.3 to confirm thresholds are intended",
                },
            )

        if min_non_missing_clamped:
            add_event_hint(
                report,
                code="label_alignment_min_non_missing_clamped",
                severity_signal="risk",
                evidence_path="thresholds_used.min_non_missing_for_equality",
                context={
                    "configured_min_non_missing_for_equality": int(min_non_missing_raw),
                    "effective_min_non_missing_for_equality": int(min_non_missing_effective),
                    "sample_size_for_equality": int(sample_size),
                    "sample_n_effective": int(sample_n_effective),
                    "total_rows": int(total_rows),
                },
                remediation={
                    "action": "adjust_min_non_missing_or_sample_size",
                    "safe": True,
                    "rationale": "min_non_missing_for_equality exceeded available sample size; it was clamped to keep alias checks meaningful.",
                    "post_check": "re-run 3.3 and confirm label_aliases results are based on sufficient comparable rows",
                },
            )

        add_event_hint(
            report,
            code="label_alignment_label_missing",
            severity_signal="hard",
            evidence_path="inputs.label_col",
            context={"label_col": label_col, "available_cols": list(df.columns)},
            remediation={
                "action": "fix_dataset_label_config_or_schema",
                "safe": True,
                "rationale": "Label column is required for supervised learning and integrity checks.",
                "post_check": "re-run 3.3 and confirm label exists",
            },
        )

        # --- Status + finalize ---
        report = finalize_module_report(
            ctx=ctx,
            report=report,
            overall_status="fail",
            issues=["label_missing"],
            warnings=[],
            metrics={"n_rows": int(df.shape[0]), "n_cols": int(df.shape[1])},
            notes=[],
            df=df,
        )

        out_path = resolve_artifact_path(
            ctx=ctx, stage="data_integrity", name="label_alignment", run_id=str(rid), tag=TAG33
        )
        write_json(out_path, report, tag=TAG33, indent=2)
        lg.info(f"[{TAG33}] 🧾 Label alignment artifact saved: {out_path}")
        return report

    label_s_raw = df[label_col]
    total_rows = int(df.shape[0])
    n_missing = int(label_s_raw.isna().sum())
    n_unique_non_null = int(label_s_raw.nunique(dropna=True))

    top_counts = label_s_raw.value_counts(dropna=False).head(10)
    top_values = {str(k): int(v) for k, v in top_counts.items()}

    task_type_ssot = (str(task_type).strip().lower() if task_type is not None else "")
    task_type_effective = task_type_ssot or _infer_task_type_auto(label_s_raw)

    label_status = "pass"
    label_reasons: List[str] = []

    if n_unique_non_null <= 1:
        label_status = "fail"
        label_reasons.append("degenerate_label_single_value")

    if n_missing > 0:
        label_reasons.append("label_has_missing_values")

    positive_share: Optional[float] = None
    pos_share_status: str = "skipped"
    pos_share_out_of_bounds = False

    if task_type_effective == "classification" and pos_label is not None and str(pos_label).strip():
        pos_share_status = "pass"
        label_norm = _normalize_label_strings(label_s_raw)
        pos_norm = str(pos_label).strip()

        non_missing = label_norm.notna()
        denom = int(non_missing.sum())

        pos_count = int(label_norm[non_missing].eq(pos_norm).sum())
        positive_share = float(pos_count / denom) if denom > 0 else None

        if positive_share is not None and (
            positive_share < float(min_pos_share) or positive_share > float(max_pos_share)
        ):
            pos_share_status = "warn"
            pos_share_out_of_bounds = True
            label_reasons.append("positive_share_out_of_bounds")
            if label_status != "fail":
                label_status = "warn"

    checks["label_health"] = {
        "status": label_status,
        "task_type_effective": task_type_effective,
        "n_missing": int(n_missing),
        "n_unique_non_null": int(n_unique_non_null),
        "top_values": top_values,
        "positive_label": pos_label,
        "positive_share": safe_float(positive_share) if positive_share is not None else None,
        "positive_share_status": pos_share_status,
        "reasons": label_reasons,
    }

    # Feature columns (3.3 by default uses all non-label cols)
    feature_cols = [c for c in df.columns if c != label_col]
    feature_cols_source = "df.columns_minus_label"

    # Name-based risks
    high_risk_by_name: List[str] = []
    if forbidden_substrings:
        for col in feature_cols:
            col_lower = col.lower()
            if any(sub in col_lower for sub in forbidden_substrings):
                high_risk_by_name.append(col)

    name_risk_status = "pass" if not high_risk_by_name else "warn"

    checks["name_risks"] = {
        "status": name_risk_status,
        "feature_cols_source": feature_cols_source,
        "n_feature_cols": int(len(feature_cols)),
        "forbidden_name_substrings": list(forbidden_substrings),
        "high_risk_by_name": high_risk_by_name[: int(max_examples)],
    }

    # Alias checks
    alias_facts = _detect_label_aliases_on_sample(
        df,
        label_col=label_col,
        feature_cols=feature_cols,
        sample_size=int(sample_size),
        min_non_missing=int(min_non_missing_effective),  # <-- only one, use effective
        max_examples=int(max_examples),
    )

    aliases = alias_facts.get("aliases") or []
    alias_status = "pass" if not aliases else "fail"

    checks["label_aliases"] = {
        "status": alias_status,
        "feature_cols_source": feature_cols_source,
        **alias_facts,
    }

    report["checks"].update(checks)

    # --- Event hints ---
    if swapped_bounds:
        add_event_hint(
            report,
            code="label_alignment_positive_share_bounds_swapped",
            severity_signal="risk",
            evidence_path="thresholds_used",
            context={
                "original_min_positive_share": safe_float(orig_min_pos_share),
                "original_max_positive_share": safe_float(orig_max_pos_share),
            },
            remediation={
                "action": "fix_positive_share_bounds",
                "safe": True,
                "rationale": "min_positive_share was greater than max_positive_share; bounds were swapped to keep checks runnable.",
                "post_check": "update YAML and re-run 3.3 to confirm thresholds are intended",
            },
        )

    if min_non_missing_clamped:
        add_event_hint(
            report,
            code="label_alignment_min_non_missing_clamped",
            severity_signal="risk",
            evidence_path="thresholds_used.min_non_missing_for_equality",
            context={
                "configured_min_non_missing_for_equality": int(min_non_missing_raw),
                "effective_min_non_missing_for_equality": int(min_non_missing_effective),
                "sample_size_for_equality": int(sample_size),
                "sample_n_effective": int(sample_n_effective),
                "total_rows": int(total_rows),
            },
            remediation={
                "action": "adjust_min_non_missing_or_sample_size",
                "safe": True,
                "rationale": "min_non_missing_for_equality exceeded available sample size; it was clamped to keep alias checks meaningful.",
                "post_check": "re-run 3.3 and confirm label_aliases results are based on sufficient comparable rows",
            },
        )

    if n_unique_non_null <= 1:
        add_event_hint(
            report,
            code="label_alignment_degenerate_label",
            severity_signal="hard",
            evidence_path="checks.label_health.n_unique_non_null",
            context={"label_col": label_col, "n_unique_non_null": int(n_unique_non_null)},
            remediation={
                "action": "fix_label_definition_or_filter_bad_rows",
                "safe": True,
                "rationale": "A degenerate label cannot support supervised learning.",
                "post_check": "re-run 3.3 and confirm n_unique_non_null > 1",
            },
        )

    if n_missing > 0:
        add_event_hint(
            report,
            code="label_alignment_label_missing_values",
            severity_signal="risk",
            evidence_path="checks.label_health.n_missing",
            context={"label_col": label_col, "n_missing": int(n_missing), "total_rows": int(total_rows)},
            remediation={
                "action": "drop_or_impute_missing_labels",
                "safe": True,
                "rationale": "Missing labels reduce usable training data and may bias results.",
                "post_check": "confirm missing labels are handled before modeling",
            },
        )

    if pos_share_out_of_bounds:
        add_event_hint(
            report,
            code="label_alignment_positive_share_out_of_bounds",
            severity_signal="risk",
            evidence_path="checks.label_health.positive_share",
            context={
                "label_col": label_col,
                "positive_label": pos_label,
                "positive_share": safe_float(positive_share),
                "min_positive_share": safe_float(min_pos_share),
                "max_positive_share": safe_float(max_pos_share),
            },
            remediation={
                "action": "adjust_bounds_or_apply_resampling_strategy",
                "safe": True,
                "rationale": "Extreme imbalance can harm model performance and evaluation reliability.",
                "post_check": "re-run 3.3 and confirm positive_share is acceptable",
            },
        )

    if high_risk_by_name:
        add_event_hint(
            report,
            code="label_alignment_high_risk_feature_names",
            severity_signal="risk",
            evidence_path="checks.name_risks.high_risk_by_name",
            context={
                "count": int(len(high_risk_by_name)),
                "examples": high_risk_by_name[: min(10, len(high_risk_by_name))],
            },
            remediation={
                "action": "review_or_remove_risky_features",
                "safe": True,
                "rationale": "Feature names suggesting the target may indicate leakage or post-outcome signals.",
                "post_check": "confirm risky features are excluded from training",
            },
        )

    if aliases:
        add_event_hint(
            report,
            code="label_alignment_label_alias_detected",
            severity_signal="hard",
            evidence_path="checks.label_aliases.aliases",
            context={
                "aliases": list(aliases[: min(20, len(aliases))]),
                "sample_size": int(alias_facts.get("sample_size") or 0),
                "min_non_missing_for_equality": int(min_non_missing_effective),
            },
            remediation={
                "action": "remove_label_alias_features",
                "safe": True,
                "rationale": "A feature identical to the label causes trivial leakage and invalid evaluation.",
                "post_check": "re-run 3.3 and confirm aliases == []",
            },
        )

    # --- 6) Status + gate-friendly summary ---
    issues: List[str] = []
    warnings: List[str] = []

    if checks["label_health"].get("status") == "fail":
        issues.append("label_health_failed")
    elif checks["label_health"].get("status") == "warn":
        warnings.append("label_health_warning")

    if checks["name_risks"].get("status") == "warn":
        warnings.append("high_risk_feature_names")

    if checks["label_aliases"].get("status") == "fail":
        issues.append("label_alias_detected")

    overall = "fail" if issues else ("warn" if warnings else "pass")

    report = finalize_module_report(
        ctx=ctx,
        report=report,
        overall_status=overall,
        issues=issues,
        warnings=warnings,
        metrics={
            "n_rows": int(df.shape[0]),
            "n_cols": int(df.shape[1]),
            "n_feature_cols": int(len(feature_cols)),
            "n_missing_label": int(n_missing),
            "n_unique_label_non_null": int(n_unique_non_null),
            "positive_share": safe_float(positive_share) if positive_share is not None else None,
            "n_high_risk_by_name": int(len(high_risk_by_name)),
            "n_label_aliases": int(len(aliases)),
            "task_type_effective": task_type_effective,
            "feature_cols_source": feature_cols_source,
        },
        notes=[],
        df=df,
    )

    if overall == "pass":
        lg.info(f"[{TAG33}] ✅ Label alignment pass.")
    elif overall == "warn":
        lg.warning(f"[{TAG33}] ⚠ Label alignment warn. warnings={warnings}")
    else:
        lg.error(f"[{TAG33}] ❌ Label alignment fail. issues={issues}")

    # --- 7) Persist artifact ---
    out_path = resolve_artifact_path(
        ctx=ctx,
        stage="data_integrity",
        name="label_alignment",
        run_id=str(rid),
        tag=TAG33,
    )
    write_json(out_path, report, tag=TAG33, indent=2)
    lg.info(f"[{TAG33}] 🧾 Label alignment artifact saved: {out_path}")

    return report

In [16]:
# --- 3.4 Leakage Risk Scan & Feature Tagging ---
#
# What:
#   Scan the feature space for potential leakage signals and produce reusable
#   feature tags that downstream steps (4.x / 5.x) can consume.
#
# How:
#   - Resolve YAML SSOT via ctx:
#       dataset.label (col, positive, task_type)
#       pipeline.data_integrity.leakage_scan (thresholds + name patterns)
#   - Compute per-feature:
#       dtype / feature_type / missing / unique / unique_ratio
#       name-based tags (high_risk_name, id_like_name)
#       stats-based tags (high_cardinality, constant)
#       label-based signals (corr vs encoded label, deterministic mapping)
#       derived-proxy scan: target ≈ product(components)
#   - Assign risk_level {high, medium, low}
#   - Persist ModuleReport v1 artifact (SSOT naming)
#
# Why:
#   Leakage often comes from subtle correlations and identifier-like columns.
#   This step creates a reusable "feature catalogue" with tags and evidence.

# --- 0) Import + TAG ---
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint
from src.utils.artifact_utils import resolve_artifact_path, write_json
from src.utils.cfg_utils import (
    get_cfg_dict,
    get_cfg_float,
    get_cfg_int,
    get_cfg_str_list,
    normalize_str_list,
    get_cfg_bool,
)
from src.utils.dataset_utils import get_label_cfg
from src.utils.serialization_utils import safe_float, to_native_scalar

TAG34 = "LEAKAGE_SCAN"


# --- Internal helpers ---
def _infer_feature_type(series: pd.Series) -> str:
    """Infer a coarse feature type for tagging purposes."""
    if pd.api.types.is_bool_dtype(series):
        return "bool"
    if pd.api.types.is_numeric_dtype(series):
        return "numeric"
    if pd.api.types.is_datetime64_any_dtype(series):
        return "datetime"
    return "categorical"


def _encode_label_for_correlation(label_s: pd.Series, pos_label: Optional[str]) -> Tuple[Optional[pd.Series], Optional[int], str]:
    """
    Encode label into numeric series for correlation.
    Returns:
      (label_encoded, n_unique_non_null, encoding_type)
    """
    if label_s is None or not isinstance(label_s, pd.Series):
        return None, None, "none"

    # Normalize strings for stable encoding
    s_norm = label_s.astype("string").str.strip()
    n_unique = int(s_norm.nunique(dropna=True))

    # If label is already numeric -> use it
    if pd.api.types.is_numeric_dtype(label_s):
        encoded = pd.to_numeric(label_s, errors="coerce").astype(float)
        return encoded, n_unique, "numeric"

    # Binary classification: encode by pos_label (most stable)
    if pos_label is not None and str(pos_label).strip():
        pos = str(pos_label).strip()
        encoded = s_norm.eq(pos).astype(float)
        encoded = encoded.where(label_s.notna(), np.nan)  # keep missing as NaN
        return encoded, n_unique, "binary_pos_label"

    # Fallback: factorize normalized labels
    codes, _ = pd.factorize(s_norm, sort=True)
    encoded = pd.Series(codes, index=label_s.index, dtype="float")
    encoded = encoded.where(encoded >= 0, np.nan)  # treat missing as NaN
    return encoded, n_unique, "factorize"


def _safe_numeric_series(series: pd.Series) -> pd.Series:
    """Cast series to numeric safely (NaN on errors)."""
    return pd.to_numeric(series, errors="coerce").astype(float)


def _is_deterministic_wrt_label(
    *,
    feature_s: pd.Series,
    label_s: pd.Series,
    feature_name: str,
    label_col: str,
    feature_n_unique: int,
    label_n_unique: int,
) -> Tuple[bool, Optional[Dict[str, Any]]]:
    """
    Check whether each feature value maps to exactly one label value.
    Returns (is_deterministic, example_payload).
    """
    if feature_s is None or label_s is None:
        return False, None
    if not label_col:
        return False, None

    if feature_n_unique <= 0 or label_n_unique <= 0:
        return False, None

    label_s_norm = label_s.astype("string").str.strip()

    df_tmp = pd.concat([feature_s, label_s_norm], axis=1)
    df_tmp.columns = [feature_name, label_col]
    df_tmp = df_tmp.dropna(subset=[feature_name, label_col])

    if df_tmp.shape[0] <= 0:
        return False, None

    vc = df_tmp.groupby(feature_name)[label_col].nunique(dropna=True)
    if vc.empty:
        return False, None

    is_det = int(vc.max()) == 1
    if not is_det:
        return False, None

    example_rows_raw = df_tmp.head(5).to_dict(orient="records")
    example_rows = [{k: to_native_scalar(v) for k, v in row.items()} for row in example_rows_raw]

    payload = {
        "feature": feature_name,
        "n_unique_feature_values": int(feature_n_unique),
        "n_unique_labels": int(label_n_unique),
        "example_rows": example_rows,
    }
    return True, payload


def _scan_derived_proxies(
    *,
    df: pd.DataFrame,
    rules: List[Dict[str, Any]],
    min_non_missing_default: int,
    max_examples: int,
) -> Dict[str, Any]:
    """
    Scan derived-proxy patterns like: target ≈ product(components).

    A rule example:
      {
        "target": "TotalCharges",
        "components": ["MonthlyCharges", "tenure"],
        "op": "product",
        "ratio_band": [0.8, 1.2],
        "min_in_band_share": 0.90,
        "min_non_missing": 200
      }

    Returns:
      {
        "flagged": { target: evidence_dict, ... },
        "examples": [ ... ],
        "skipped": [ ... ]
      }
    """
    flagged: Dict[str, Any] = {}
    examples: List[Dict[str, Any]] = []
    skipped: List[Dict[str, Any]] = []

    if df is None or not isinstance(df, pd.DataFrame):
        return {"flagged": {}, "examples": [], "skipped": [{"reason": "df_invalid"}]}

    for r in rules or []:
        target = str((r or {}).get("target") or "").strip()
        comps = (r or {}).get("components") or []
        op = str((r or {}).get("op") or "product").strip().lower()
        ratio_band = (r or {}).get("ratio_band") or [0.8, 1.2]
        min_in_band_share = float((r or {}).get("min_in_band_share", 0.90))
        min_non_missing = int((r or {}).get("min_non_missing", min_non_missing_default))

        if (not target) or (target not in df.columns):
            skipped.append({"target": target, "reason": "target_missing"})
            continue

        if (not isinstance(comps, list)) or (len(comps) != 2) or (not all(isinstance(x, str) for x in comps)):
            skipped.append({"target": target, "reason": "components_invalid", "components": comps})
            continue

        c1, c2 = comps[0], comps[1]
        if (c1 not in df.columns) or (c2 not in df.columns):
            skipped.append({"target": target, "reason": "components_missing", "components": [c1, c2]})
            continue

        if op != "product":
            skipped.append({"target": target, "reason": "unsupported_op", "op": op})
            continue

        y = _safe_numeric_series(df[target])
        x1 = _safe_numeric_series(df[c1])
        x2 = _safe_numeric_series(df[c2])

        denom = x1 * x2
        denom_finite = denom.replace([np.inf, -np.inf], np.nan).notna()
        y_finite = y.replace([np.inf, -np.inf], np.nan).notna()

        mask = y.notna() & denom.notna() & denom_finite & (denom != 0.0) & y_finite
        n = int(mask.sum())
        if n < int(min_non_missing):
            skipped.append({"target": target, "reason": "too_few_non_missing", "n": int(n), "min_non_missing": int(min_non_missing)})
            continue

        ratio = (y[mask] / denom[mask]).replace([np.inf, -np.inf], np.nan).dropna()
        if ratio.shape[0] <= 0:
            skipped.append({"target": target, "reason": "ratio_empty_after_clean", "n": int(n)})
            continue

        rb0 = float(ratio_band[0]) if isinstance(ratio_band, (list, tuple)) and len(ratio_band) >= 2 else 0.8
        rb1 = float(ratio_band[1]) if isinstance(ratio_band, (list, tuple)) and len(ratio_band) >= 2 else 1.2
        lo, hi = (rb0, rb1) if rb0 <= rb1 else (rb1, rb0)

        in_band = ratio.between(lo, hi, inclusive="both")
        in_band_share = float(in_band.mean()) if ratio.shape[0] > 0 else 0.0
        median_ratio = float(ratio.median())

        flagged_now = (in_band_share >= float(min_in_band_share)) and (lo <= median_ratio <= hi)

        evidence = {
            "target": target,
            "components": [c1, c2],
            "op": "product",
            "n_used": int(ratio.shape[0]),
            "min_non_missing": int(min_non_missing),
            "ratio_band": [safe_float(lo), safe_float(hi)],
            "min_in_band_share": safe_float(min_in_band_share),
            "in_band_share": safe_float(in_band_share),
            "median_ratio": safe_float(median_ratio),
            "flagged": bool(flagged_now),
        }

        if flagged_now:
            flagged[target] = evidence
            if len(examples) < int(max_examples):
                # Small preview: first 5 valid rows
                idx = ratio.index[:5]
                preview_rows_raw = pd.DataFrame(
                    {
                        target: y.loc[idx],
                        c1: x1.loc[idx],
                        c2: x2.loc[idx],
                        "ratio": (y.loc[idx] / (x1.loc[idx] * x2.loc[idx])),
                    }
                ).to_dict(orient="records")
                preview_rows = [{k: to_native_scalar(v) for k, v in row.items()} for row in preview_rows_raw]
                examples.append({"evidence": evidence, "preview_rows": preview_rows})
        else:
            skipped.append({"target": target, "reason": "not_flagged", "evidence": evidence})

    return {"flagged": flagged, "examples": examples, "skipped": skipped[: int(max_examples)]}


# --- 1) Public API ---
def run_leakage_scan_and_feature_tagging(
    *,
    df: pd.DataFrame,
    ctx: Dict[str, Any],
    run_id: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Run leakage-oriented feature scan and produce reusable feature tags.

    YAML SSOT
    ---------
    dataset.label:
      col: "Churn"
      positive: "Yes"
      task_type: "classification"

    pipeline.data_integrity.leakage_scan:
      high_risk_name_substrings: [...]
      id_like_name_substrings: [...]
      high_cardinality_threshold: 0.90
      corr_threshold: 0.98
      max_examples: 30
      
      derived_proxies:
        enabled: true
        min_non_missing: 200
        max_examples: 10
        rules:
          - target: "TotalCharges"
            components: ["MonthlyCharges", "tenure"]
            op: "product"
            ratio_band: [0.8, 1.2]
            min_in_band_share: 0.90

    Returns
    -------
    ModuleReport v1 (Leakage Risk Scan & Feature Tagging) (also persisted as artifact).
    """

    # --- 2) Runtime validation ---
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG34, run_id=run_id)

    # --- 3) Input validation ---
    if df is None or not isinstance(df, pd.DataFrame):
        raise TypeError(f"[{TAG34}] df must be a pandas DataFrame")

    total_rows = int(df.shape[0])
    total_cols = int(df.shape[1])

    # --- 4) Resolve config (YAML SSOT) ---
    label_col, pos_label, task_type = get_label_cfg(ctx, TAG34)

    get_cfg_dict(ctx, "pipeline.data_integrity.leakage_scan", tag=TAG34, default={})

    high_risk_raw = get_cfg_str_list(
        ctx,
        "pipeline.data_integrity.leakage_scan.high_risk_name_substrings",
        tag=TAG34,
        required=False,
        default=[],
    )
    id_like_raw = get_cfg_str_list(
        ctx,
        "pipeline.data_integrity.leakage_scan.id_like_name_substrings",
        tag=TAG34,
        required=False,
        default=[],
    )

    high_risk_substrings = [
        s.lower()
        for s in normalize_str_list(
            high_risk_raw,
            tag=TAG34,
            path="pipeline.data_integrity.leakage_scan.high_risk_name_substrings",
        )
    ]
    
    id_like_substrings = [
        s.lower()
        for s in normalize_str_list(
            id_like_raw,
            tag=TAG34,
            path="pipeline.data_integrity.leakage_scan.id_like_name_substrings",
        )
    ]

    high_cardinality_threshold = get_cfg_float(
        ctx,
        "pipeline.data_integrity.leakage_scan.high_cardinality_threshold",
        tag=TAG34,
        required=False,
        default=0.90,
        strict_type=False,
        min_value=0.0,
        max_value=1.0,
    )

    corr_threshold = get_cfg_float(
        ctx,
        "pipeline.data_integrity.leakage_scan.corr_threshold",
        tag=TAG34,
        required=False,
        default=0.98,
        strict_type=False,
        min_value=0.0,
        max_value=1.0,
    )

    max_examples = get_cfg_int(
        ctx,
        "pipeline.data_integrity.leakage_scan.max_examples",
        tag=TAG34,
        required=False,
        default=30,
        strict_type=False,
        min_value=1,
        max_value=200,
    )

    feature_cols_cfg = get_cfg_str_list(
        ctx,
        "pipeline.data_integrity.leakage_scan.feature_cols",
        tag=TAG34,
        required=False,
        default=[],
    )
    feature_cols_cfg = normalize_str_list(
        feature_cols_cfg,
        tag=TAG34,
        path="pipeline.data_integrity.leakage_scan.feature_cols",
    )


    derived_cfg = get_cfg_dict(
        ctx,
        "pipeline.data_integrity.leakage_scan.derived_proxies",
        tag=TAG34,
        required=False,
        default={},
    ) or {}

    derived_enabled = get_cfg_bool(
        ctx,
        "pipeline.data_integrity.leakage_scan.derived_proxies.enabled",
        tag=TAG34,
        required=False,
        default=bool(derived_cfg.get("enabled", False)),
        strict_type=False,  # allow "true"/"false"/1/0 style if you want
    )

    derived_min_non_missing = get_cfg_int(
        ctx,
        "pipeline.data_integrity.leakage_scan.derived_proxies.min_non_missing",
        tag=TAG34,
        required=False,
        default=int(derived_cfg.get("min_non_missing", 200) or 200),
        strict_type=False,
        min_value=1,
        max_value=10_000_000,
    )

    derived_max_examples = get_cfg_int(
        ctx,
        "pipeline.data_integrity.leakage_scan.derived_proxies.max_examples",
        tag=TAG34,
        required=False,
        default=int(derived_cfg.get("max_examples", 10) or 10),
        strict_type=False,
        min_value=1,
        max_value=200,
    )

    derived_rules = derived_cfg.get("rules") or []
    if not isinstance(derived_rules, list):
        derived_rules = []

    # --- 5) Core workflows ---
    lg.info(f"[{TAG34}] 🔍 Starting leakage risk scan & feature tagging... | run_id={rid}")

    # --- Build ModuleReport skeleton (infra) ---
    report = build_module_report(
        ctx=ctx,
        stage="data_integrity",
        step="3.4",
        name="leakage_scan",
        tag=TAG34,
        cfg_key="pipeline.data_integrity.leakage_scan",
        enabled=True,
        df=df,
        run_id_override=str(rid),
    )

    report["inputs"] = {
        "label_col": label_col,
        "positive_label": pos_label,
        "task_type_ssot": task_type,
        "feature_cols_cfg_provided": bool(feature_cols_cfg),
        "derived_proxy_enabled": bool(derived_enabled),
    }

    report["thresholds_used"] = {
        "high_risk_name_substrings": list(high_risk_substrings),
        "id_like_name_substrings": list(id_like_substrings),
        "high_cardinality_threshold": safe_float(high_cardinality_threshold),
        "corr_threshold": safe_float(corr_threshold),
        "max_examples": int(max_examples),
        "feature_cols_cfg": list(feature_cols_cfg),
        "derived_proxies": {
            "enabled": bool(derived_enabled),
            "min_non_missing": int(derived_min_non_missing),
            "max_examples": int(derived_max_examples),
            "n_rules": int(len(derived_rules)),
        },
    }

    report["used_config"] = {
        "label_col": label_col,
        "positive_label": pos_label,
        "task_type_ssot": task_type,
    }

    if not isinstance(report.get("refs"), dict):
        report["refs"] = {}

    report["refs"]["self"] = make_artifact_ref(
        ctx=ctx,
        stage="data_integrity",
        name="leakage_scan",
        run_id=str(rid),
    )

    ensure_event_hints(report, version=1)

    # --- Core computation (facts -> checks) ---
    checks: Dict[str, Any] = {
        "scan_overview": {},
        "derived_proxy_scan": {"status": "skipped", "flagged": {}, "examples": [], "skipped": []},
        "feature_catalogue": {},
        "examples": {"high_corr_with_label": [], "deterministic_wrt_label": []},
    }

    issues: List[str] = []
    warnings: List[str] = []

    if total_rows <= 0 or total_cols <= 0:
        checks["scan_overview"] = {
            "status": "fail",
            "reason": "empty_dataframe",
            "total_rows": int(total_rows),
            "total_cols": int(total_cols),
        }
        issues.append("empty_dataframe")

        report["checks"].update(checks)

        add_event_hint(
            report,
            code="leakage_scan_empty_dataframe",
            severity_signal="hard",
            evidence_path="summary.total_rows",
            context={"total_rows": int(total_rows), "total_cols": int(total_cols)},
            remediation={
                "action": "fix_upstream_ingestion_or_filters",
                "safe": True,
                "rationale": "Leakage scanning requires a non-empty dataframe.",
                "post_check": "re-run 3.4 and confirm df has rows/cols",
            },
        )

        report = finalize_module_report(
            ctx=ctx,
            report=report,
            overall_status="fail",
            issues=issues,
            warnings=warnings,
            metrics={"n_rows": int(total_rows), "n_cols": int(total_cols)},
            notes=[],
            df=df,
        )

        out_path = resolve_artifact_path(ctx=ctx, stage="data_integrity", name="leakage_scan", run_id=str(rid), tag=TAG34)
        write_json(out_path, report, tag=TAG34, indent=2)
        lg.info(f"[{TAG34}] 🧾 Leakage scan artifact saved: {out_path}")
        return report

    label_available = bool(label_col) and (label_col in df.columns)
    label_series = df[label_col] if label_available else None

    label_encoded, label_n_unique, label_encoding_type = (
        _encode_label_for_correlation(label_series, pos_label) if label_available else (None, None, "none")
    )

    # Resolve feature columns
    if feature_cols_cfg:
        feature_cols = [c for c in feature_cols_cfg if c in df.columns and c != label_col]
        feature_cols_source = "cfg.feature_cols"
    else:
        feature_cols = [c for c in df.columns if c != label_col]
        feature_cols_source = "df.columns_minus_label"

    # Derived proxy scan (optional, before per-feature loop)
    derived_flagged: Dict[str, Any] = {}
    derived_examples: List[Dict[str, Any]] = []
    if derived_enabled and derived_rules:
        derived_out = _scan_derived_proxies(
            df=df,
            rules=derived_rules,
            min_non_missing_default=int(derived_min_non_missing),
            max_examples=int(min(derived_max_examples, max_examples)),
        )
        derived_flagged = (derived_out or {}).get("flagged") or {}
        derived_examples = (derived_out or {}).get("examples") or []
        checks["derived_proxy_scan"] = {
            "status": "pass",
            "flagged": derived_flagged,
            "examples": derived_examples,
            "skipped": (derived_out or {}).get("skipped") or [],
        }
    else:
        checks["derived_proxy_scan"] = {
            "status": "skipped",
            "reason": "disabled_or_no_rules",
            "flagged": {},
            "examples": [],
            "skipped": [],
        }

    feature_reports: Dict[str, Any] = {}
    risk_level_counts: Dict[str, int] = {"high": 0, "medium": 0, "low": 0}
    tag_counts: Dict[str, int] = {}

    high_corr_examples: List[Dict[str, Any]] = []
    deterministic_examples: List[Dict[str, Any]] = []

    n_high_risk = 0
    n_medium_risk = 0
    n_derived_proxy = 0

    for col in feature_cols:
        series = df[col]
        dtype_str = str(series.dtype)
        ftype = _infer_feature_type(series)

        n_missing = int(series.isna().sum())
        n_unique = int(series.nunique(dropna=True))
        unique_ratio = (float(n_unique) / float(total_rows)) if total_rows > 0 else 0.0

        tags: List[str] = []
        notes: List[str] = []
        derived_proxy_evidence: Optional[Dict[str, Any]] = None

        # Constant / near-constant
        if n_unique <= 1:
            tags.append("constant")
            notes.append("Feature has 1 or 0 unique non-null value(s).")

        # Name-based patterns
        col_lower = str(col).lower()

        if any(sub in col_lower for sub in id_like_substrings):
            tags.append("id_like_name")

        if any(sub in col_lower for sub in high_risk_substrings):
            tags.append("high_risk_name")
            notes.append("Feature name matches high-risk leakage patterns.")

        # High-cardinality detection
        if total_rows > 0 and unique_ratio >= float(high_cardinality_threshold):
            tags.append("high_cardinality")
            notes.append(
                f"unique_ratio={unique_ratio:.4f} exceeds high_cardinality_threshold={float(high_cardinality_threshold):.4f}."
            )

        # Datetime tagging
        if ftype == "datetime":
            tags.append("datetime")

        # Derived proxy tagging (optional evidence)
        if col in derived_flagged:
            tags.append("derived_proxy")
            n_derived_proxy += 1
            derived_proxy_evidence = derived_flagged.get(col)
            notes.append("Feature appears to be a derived proxy from other features (see derived_proxy evidence).")

        # Label-based checks (optional)
        corr_with_label: Optional[float] = None

        if label_available and label_encoded is not None and n_unique > 1:
            # Correlation for numeric/bool features
            if ftype in {"numeric", "bool"}:
                s_num = _safe_numeric_series(series)
                aligned = pd.concat([label_encoded, s_num], axis=1, join="inner").dropna()
                if aligned.shape[0] > 1:
                    corr = aligned.iloc[:, 0].corr(aligned.iloc[:, 1])
                    if pd.notna(corr):
                        corr_with_label = float(corr)
                        if abs(corr_with_label) >= float(corr_threshold):
                            tags.append("near_label")
                            notes.append(
                                f"Abs correlation with label is {abs(corr_with_label):.4f}, "
                                f"above corr_threshold={float(corr_threshold):.4f}."
                            )
                            if len(high_corr_examples) < int(max_examples):
                                high_corr_examples.append(
                                    {
                                        "feature": col,
                                        "corr_with_label": float(corr_with_label),
                                        "n_rows_used": int(aligned.shape[0]),
                                    }
                                )

            # Deterministic mapping for low-cardinality categoricals/bool
            if (
                ftype in {"categorical", "bool"}
                and label_series is not None
                and label_n_unique is not None
                and n_unique <= max(int(label_n_unique) * 5, 50)
            ):
                is_det, payload = _is_deterministic_wrt_label(
                    feature_s=series,
                    label_s=label_series,
                    feature_name=col,
                    label_col=str(label_col),
                    feature_n_unique=int(n_unique),
                    label_n_unique=int(label_n_unique),
                )
                if is_det:
                    tags.append("deterministic_wrt_label")
                    notes.append("Each feature value maps to exactly one label value; feature is deterministic w.r.t. label.")
                    if payload is not None and len(deterministic_examples) < int(max_examples):
                        deterministic_examples.append(payload)

        # Risk level assignment
        risk_level = "low"

        # High: strong leakage indicators
        if ("near_label" in tags) or ("deterministic_wrt_label" in tags) or ("high_risk_name" in tags):
            risk_level = "high"
        # Medium: identifier/context/derived proxy
        elif ("id_like_name" in tags) or ("high_cardinality" in tags) or ("datetime" in tags) or ("derived_proxy" in tags):
            risk_level = "medium"

        if risk_level == "high":
            n_high_risk += 1
        elif risk_level == "medium":
            n_medium_risk += 1

        risk_level_counts[risk_level] = int(risk_level_counts.get(risk_level, 0)) + 1

        for t in tags:
            tag_counts[t] = int(tag_counts.get(t, 0)) + 1

        feature_reports[col] = {
            "dtype": dtype_str,
            "feature_type": ftype,
            "n_missing": int(n_missing),
            "n_unique_non_null": int(n_unique),
            "unique_ratio": safe_float(unique_ratio),
            "tags": list(tags),
            "risk_level": risk_level,
            "corr_with_label": safe_float(corr_with_label) if corr_with_label is not None else None,
            "derived_proxy": derived_proxy_evidence,
            "notes": list(notes),
        }

    checks["scan_overview"] = {
        "status": "pass",
        "label_available": bool(label_available),
        "label_encoding_type": str(label_encoding_type),
        "label_n_unique_non_null": int(label_n_unique) if label_n_unique is not None else None,
        "feature_cols_source": feature_cols_source,
        "n_features_scanned": int(len(feature_cols)),
        "risk_level_counts": risk_level_counts,
        "tag_counts": tag_counts,
    }

    checks["feature_catalogue"] = feature_reports
    checks["examples"]["high_corr_with_label"] = high_corr_examples
    checks["examples"]["deterministic_wrt_label"] = deterministic_examples

    report["checks"].update(checks)

    # --- Event hints ---
    if not label_available:
        add_event_hint(
            report,
            code="leakage_scan_label_missing_skip_label_based_checks",
            severity_signal="risk",
            evidence_path="inputs.label_col",
            context={"label_col": label_col, "available_cols": list(df.columns)},
            remediation={
                "action": "fix_dataset_label_config_or_schema",
                "safe": True,
                "rationale": "Label-based leakage signals (corr/deterministic) were skipped because label column is missing.",
                "post_check": "re-run 3.4 with a valid label column to enable label-based checks",
            },
        )

    if derived_flagged:
        add_event_hint(
            report,
            code="leakage_scan_derived_proxy_features_detected",
            severity_signal="risk",
            evidence_path="checks.derived_proxy_scan.flagged",
            context={
                "n_flagged": int(len(list(derived_flagged.keys()))),
                "flagged_features": list(derived_flagged.keys())[: min(20, int(max_examples))],
            },
            remediation={
                "action": "review_derived_proxy_features_for_collinearity_or_proxy_risk",
                "safe": True,
                "rationale": "Derived proxy features may inflate importance, harm interpretability, or hide leakage via post-outcome construction.",
                "post_check": "confirm derived proxy handling (drop/regularize/constrain) is documented and consistent in 4.x/5.x",
            },
        )

    if n_high_risk > 0:
        add_event_hint(
            report,
            code="leakage_scan_high_risk_features_detected",
            severity_signal="risk",
            evidence_path="checks.scan_overview.risk_level_counts",
            context={
                "n_high_risk": int(n_high_risk),
                "examples": [x for x, v in list(feature_reports.items()) if v.get("risk_level") == "high"][
                    : min(10, int(max_examples))
                ],
            },
            remediation={
                "action": "exclude_or_review_high_risk_features",
                "safe": True,
                "rationale": "High-risk features may leak label information and invalidate evaluation.",
                "post_check": "confirm high-risk features are excluded in 4.x/5.x feature selection",
            },
        )

    if n_medium_risk > 0:
        add_event_hint(
            report,
            code="leakage_scan_medium_risk_features_detected",
            severity_signal="info",
            evidence_path="checks.scan_overview.risk_level_counts",
            context={
                "n_medium_risk": int(n_medium_risk),
                "examples": [x for x, v in list(feature_reports.items()) if v.get("risk_level") == "medium"][
                    : min(10, int(max_examples))
                ],
            },
            remediation={
                "action": "treat_medium_risk_features_as_keys_or_context",
                "safe": True,
                "rationale": "Medium-risk features are often identifiers/time/context/derived proxy columns; use with caution.",
                "post_check": "document handling of id-like/datetime/high-cardinality/derived-proxy columns",
            },
        )

    if high_corr_examples:
        add_event_hint(
            report,
            code="leakage_scan_near_label_correlation_detected",
            severity_signal="risk",
            evidence_path="checks.examples.high_corr_with_label",
            context={"count": int(len(high_corr_examples)), "examples": high_corr_examples[: min(10, int(max_examples))]},
            remediation={
                "action": "review_correlated_features_for_leakage",
                "safe": True,
                "rationale": "Near-deterministic correlation to label is a strong leakage indicator.",
                "post_check": "confirm correlated features are removed or justified",
            },
        )

    if deterministic_examples:
        add_event_hint(
            report,
            code="leakage_scan_deterministic_wrt_label_detected",
            severity_signal="hard",
            evidence_path="checks.examples.deterministic_wrt_label",
            context={"count": int(len(deterministic_examples)), "examples": deterministic_examples[: min(5, int(max_examples))]},
            remediation={
                "action": "remove_deterministic_features",
                "safe": True,
                "rationale": "Deterministic features w.r.t. label cause trivial leakage.",
                "post_check": "re-run 3.4 and confirm deterministic_wrt_label is empty",
            },
        )

    # --- 6) Status + gate-friendly summary ---
    if not label_available:
        warnings.append("label_missing_skip_label_based_checks")

    # Deterministic is hard-leakage: fail
    if deterministic_examples:
        issues.append("deterministic_features_detected")

    # Near-label correlation / high-risk names: treat as warn (policy may escalate later in orchestrator)
    if n_high_risk > 0 or high_corr_examples:
        warnings.append("potential_leakage_signals_detected")

    if derived_flagged:
        warnings.append("derived_proxy_features_detected")

    notes = [f"medium_risk_features={int(n_medium_risk)}"]

    overall = "fail" if issues else ("warn" if warnings else "pass")

    report = finalize_module_report(
        ctx=ctx,
        report=report,
        overall_status=overall,
        issues=issues,
        warnings=warnings,
        metrics={
            "n_rows": int(total_rows),
            "n_cols": int(total_cols),
            "n_features_scanned": int(len(feature_cols)),
            "n_high_risk": int(n_high_risk),
            "n_medium_risk": int(n_medium_risk),
            "n_derived_proxy": int(n_derived_proxy),
            "label_available": bool(label_available),
            "label_encoding_type": str(label_encoding_type),
            "corr_threshold": safe_float(corr_threshold),
            "high_cardinality_threshold": safe_float(high_cardinality_threshold),
            "feature_cols_source": feature_cols_source,
        },
        notes=notes,
        df=df,
    )

    if overall == "pass":
        lg.info(f"[{TAG34}] ✅ Leakage scan pass.")
    elif overall == "warn":
        lg.warning(f"[{TAG34}] ⚠ Leakage scan warn. warnings={warnings}")
    else:
        lg.error(f"[{TAG34}] ❌ Leakage scan fail. issues={issues}")

    # --- 7) Persist artifact ---
    out_path = resolve_artifact_path(
        ctx=ctx,
        stage="data_integrity",
        name="leakage_scan",
        run_id=str(rid),
        tag=TAG34,
    )
    write_json(out_path, report, tag=TAG34, indent=2)
    lg.info(f"[{TAG34}] 🧾 Leakage scan artifact saved: {out_path}")

    return report

In [17]:
# --- 3.5 Integrity Summary ---
#
# What:
#   Aggregate 3.1–3.4 artifacts into a compact, run-level integrity summary.
#
# How:
#   - Load artifacts via SSOT naming (resolve_artifact_path), fallback to legacy filenames
#   - Extract per-module effective status + key risk signals (esp. leakage overview)
#   - Collect upstream event_hints and dedupe into one hub report
#   - Persist one JSON ModuleReport v1 under artifacts (SSOT naming)
#
# Why:
#   Repeatable, auditable integrity checklist + single gate hook before Step 4.

# --- 0) Imports + TAG ---
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd  # used only for optional df validation

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint
from src.pipeline.summary_utils import (
    get_overall_status,
    get_total_rows,
    get_total_cols,
    make_upstream_block,
    dedupe_sorted,
    norm_status,
    dedupe_event_hints,
    collect_upstream_event_hints,
    load_artifact_json_with_runid_fallback,
)
from src.utils.artifact_utils import resolve_artifact_path, write_json
from src.utils.cfg_utils import get_cfg_dict, get_cfg_int

TAG35 = "INTEGRITY_SUMMARY"


# --- Internal helpers (no nested helpers) ---
def _safe_bool(v: Any, *, default: bool = False) -> bool:
    """Parse a bool-like value safely."""
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return bool(v)
    if isinstance(v, str):
        s = v.strip().lower()
        if s in {"true", "1", "yes", "y"}:
            return True
        if s in {"false", "0", "no", "n"}:
            return False
    return bool(default)


def _safe_rid(x: Any, *, fallback: str) -> str:
    """Return a clean run_id string for lineage refs."""
    if isinstance(x, str) and x.strip():
        return x.strip()
    return fallback


def _top_k_counts(counts: Dict[str, Any], *, k: int) -> List[Dict[str, Any]]:
    """Return top-k items from a count dict, sorted by count desc."""
    if not isinstance(counts, dict) or k <= 0:
        return []
    scored: List[Tuple[str, int]] = []
    for key, val in counts.items():
        if not isinstance(key, str):
            continue
        try:
            n = int(val or 0)
        except Exception:
            n = 0
        scored.append((key, n))
    scored = sorted(scored, key=lambda x: x[1], reverse=True)[: int(k)]
    return [{"key": k_, "count": int(n_)} for k_, n_ in scored]


def _extract_leakage_overview(
    leakage_report: Optional[Dict[str, Any]],
    *,
    top_k: int,
) -> Dict[str, Any]:
    """
    Extract leakage overview from 3.4 ModuleReport v1.
    Expected:
      checks.scan_overview.risk_level_counts
      checks.scan_overview.tag_counts
    """
    if not isinstance(leakage_report, dict):
        return {
            "risk_level_counts": {},
            "tag_counts": {},
            "n_high_risk": 0,
            "n_medium_risk": 0,
            "top_tags": [],
        }

    checks = leakage_report.get("checks") if isinstance(leakage_report.get("checks"), dict) else {}
    overview = checks.get("scan_overview") if isinstance(checks.get("scan_overview"), dict) else {}

    risk_level_counts = (
        overview.get("risk_level_counts") if isinstance(overview.get("risk_level_counts"), dict) else {}
    )
    tag_counts = overview.get("tag_counts") if isinstance(overview.get("tag_counts"), dict) else {}

    try:
        n_high = int(risk_level_counts.get("high", 0) or 0)
    except Exception:
        n_high = 0
    try:
        n_med = int(risk_level_counts.get("medium", 0) or 0)
    except Exception:
        n_med = 0

    return {
        "risk_level_counts": dict(risk_level_counts),
        "tag_counts": dict(tag_counts),
        "n_high_risk": int(n_high),
        "n_medium_risk": int(n_med),
        "top_tags": _top_k_counts(tag_counts, k=int(top_k)),
    }


def _combine_statuses(statuses: List[str]) -> str:
    """Combine statuses with precedence: fail > warn > pass > skipped/unknown."""
    s = [norm_status(x) for x in statuses]
    if "fail" in s:
        return "fail"
    if "warn" in s:
        return "warn"
    if "pass" in s:
        return "pass"
    return "skipped"


def _get_effective_status(rep: Optional[Dict[str, Any]], *, check_key: Optional[str] = None) -> str:
    """Prefer checks.<check_key>.status when available, otherwise fall back to summary.overall_status."""
    if not isinstance(rep, dict):
        return "unknown"
    if check_key:
        checks = rep.get("checks") if isinstance(rep.get("checks"), dict) else {}
        ck = checks.get(check_key) if isinstance(checks.get(check_key), dict) else {}
        s = ck.get("status")
        s_norm = norm_status(s)
        if s_norm != "unknown":
            return s_norm
    return norm_status(get_overall_status(rep))


def _collect_upstream_run_ids(*rids: Any) -> List[str]:
    out: List[str] = []
    for x in rids:
        if isinstance(x, str) and x.strip():
            out.append(x.strip())
    return sorted(set(out))


# --- 1) Public API ---
def build_integrity_summary(
    *,
    df: Optional[Any] = None,  # optional; 3.5 is artifact-driven
    ctx: Dict[str, Any],
    run_id: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Build a run-level integrity summary (3.5).

    YAML SSOT (optional)
    --------------------
    pipeline.data_integrity.summary:
      max_examples: 20
      warn_on_missing_reports: true
      fail_on_missing_reports: false
      warn_on_high_risk_leakage: true
      max_event_hints: 200
    """
    # --- 2) Runtime validation ---
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG35, run_id=run_id)

    # --- 3) Input validation ---
    if df is not None and not isinstance(df, pd.DataFrame):
        raise TypeError(f"[{TAG35}] df must be a pandas DataFrame when provided (3.5 does not use df)")

    # --- 4) Resolve config (YAML SSOT) ---
    cfg = get_cfg_dict(ctx, "pipeline.data_integrity.summary", tag=TAG35, required=False, default={}) or {}

    max_examples = get_cfg_int(
        ctx,
        "pipeline.data_integrity.summary.max_examples",
        tag=TAG35,
        default=int(cfg.get("max_examples", 20) or 20),
        required=False,
        min_value=1,
        max_value=200,
        strict_type=False,
    )

    max_event_hints = get_cfg_int(
        ctx,
        "pipeline.data_integrity.summary.max_event_hints",
        tag=TAG35,
        default=int(cfg.get("max_event_hints", 200) or 200),
        required=False,
        min_value=0,
        max_value=5000,
        strict_type=False,
    )

    fail_on_missing = _safe_bool(cfg.get("fail_on_missing_reports"), default=False)
    warn_on_missing = _safe_bool(cfg.get("warn_on_missing_reports"), default=True)
    warn_on_high_risk_leakage = _safe_bool(cfg.get("warn_on_high_risk_leakage"), default=True)

    # --- 5) Core workflow ---
    lg.info(f"[{TAG35}] 📊 Building integrity summary for run_id={rid}...")

    report = build_module_report(
        ctx=ctx,
        stage="data_integrity",
        step="3.5",
        name="integrity_summary",
        tag=TAG35,
        cfg_key="pipeline.data_integrity.summary",
        enabled=True,
        df=None,
        run_id_override=str(rid),
    )

    # --- 5.1) Base blocks (inputs / thresholds / used_config / refs.self / event_hints) ---
    report["inputs"] = {"df_provided": df is not None}
    report["thresholds_used"] = {
        "max_examples": int(max_examples),
        "max_event_hints": int(max_event_hints),
        "fail_on_missing_reports": bool(fail_on_missing),
        "warn_on_missing_reports": bool(warn_on_missing),
        "warn_on_high_risk_leakage": bool(warn_on_high_risk_leakage),
    }
    report["used_config"] = {"summary_cfg_keys": sorted(list(cfg.keys()))}

    if not isinstance(report.get("refs"), dict):
        report["refs"] = {}
    report["refs"]["self"] = make_artifact_ref(
        ctx=ctx, stage="data_integrity", name="integrity_summary", run_id=str(rid)
    )

    if not isinstance(report.get("checks"), dict):
        report["checks"] = {}
    ensure_event_hints(report, version=1)

    # --- 5.2) Load artifacts (3.1–3.4) SSOT-first + legacy fallback ---
    row_report, row_path, row_meta, row_rid = load_artifact_json_with_runid_fallback(
        ctx=ctx,
        lg=lg,
        stage="data_integrity",
        name="row_identity",
        run_id=str(rid),
        tag=TAG35,
        legacy_filenames=[f"row_identity_{rid}.json"],
    )

    temporal_report, temporal_path, temporal_meta, temporal_rid = load_artifact_json_with_runid_fallback(
        ctx=ctx,
        lg=lg,
        stage="data_integrity",
        name="temporal",
        run_id=str(rid),
        tag=TAG35,
        legacy_filenames=[f"temporal_integrity_{rid}.json", f"temporal_{rid}.json"],
    )

    alignment_report, alignment_path, alignment_meta, alignment_rid = load_artifact_json_with_runid_fallback(
        ctx=ctx,
        lg=lg,
        stage="data_integrity",
        name="label_alignment",
        run_id=str(rid),
        tag=TAG35,
        legacy_filenames=[f"label_alignment_{rid}.json"],
    )

    leakage_report, leakage_path, leakage_meta, leakage_rid = load_artifact_json_with_runid_fallback(
        ctx=ctx,
        lg=lg,
        stage="data_integrity",
        name="leakage_scan",
        run_id=str(rid),
        tag=TAG35,
        legacy_filenames=[f"leakage_scan_{rid}.json"],
    )

    # --- 5.3) Compute effective statuses + shape ---
    row_status = _get_effective_status(row_report, check_key="identity")
    temporal_status = _get_effective_status(temporal_report, check_key="temporal")
    alignment_status = _get_effective_status(alignment_report, check_key="label_health")
    leakage_status = _get_effective_status(leakage_report, check_key="scan_overview")

    total_rows = None
    total_cols = None
    for rep in (row_report, temporal_report, alignment_report, leakage_report):
        if total_rows is None:
            total_rows = get_total_rows(rep)
        if total_cols is None:
            total_cols = get_total_cols(rep)
        if total_rows is not None and total_cols is not None:
            break

    # --- 5.4) Leakage overview (facts) ---
    leakage_overview = _extract_leakage_overview(leakage_report, top_k=int(max_examples))
    n_high_risk = int(leakage_overview.get("n_high_risk", 0) or 0)
    n_medium_risk = int(leakage_overview.get("n_medium_risk", 0) or 0)

    # --- 5.5) Upstream blocks (refs.upstream) ---
    upstream: Dict[str, Any] = {
        "row_identity": make_upstream_block(
            row_report, name="row_identity", path=row_path, meta=row_meta, key_metrics={"total_rows": get_total_rows(row_report)}
        ),
        "temporal": make_upstream_block(
            temporal_report, name="temporal", path=temporal_path, meta=temporal_meta, key_metrics={"total_rows": get_total_rows(temporal_report)}
        ),
        "label_alignment": make_upstream_block(
            alignment_report, name="label_alignment", path=alignment_path, meta=alignment_meta, key_metrics={"total_rows": get_total_rows(alignment_report)}
        ),
        "leakage_scan": make_upstream_block(
            leakage_report,
            name="leakage_scan",
            path=leakage_path,
            meta=leakage_meta,
            key_metrics={
                "total_rows": get_total_rows(leakage_report),
                "n_high_risk": int(n_high_risk),
                "n_medium_risk": int(n_medium_risk),
            },
        ),
    }

    # --- 5.6) refs.dependencies (lineage) ---
    deps: Dict[str, Any] = {}
    if row_report is not None:
        deps["row_identity"] = make_artifact_ref(
            ctx=ctx, stage="data_integrity", name="row_identity", run_id=_safe_rid(row_rid, fallback=str(rid))
        )
    if temporal_report is not None:
        deps["temporal"] = make_artifact_ref(
            ctx=ctx, stage="data_integrity", name="temporal", run_id=_safe_rid(temporal_rid, fallback=str(rid))
        )
    if alignment_report is not None:
        deps["label_alignment"] = make_artifact_ref(
            ctx=ctx, stage="data_integrity", name="label_alignment", run_id=_safe_rid(alignment_rid, fallback=str(rid))
        )
    if leakage_report is not None:
        deps["leakage_scan"] = make_artifact_ref(
            ctx=ctx, stage="data_integrity", name="leakage_scan", run_id=_safe_rid(leakage_rid, fallback=str(rid))
        )
    report["refs"]["dependencies"] = deps

    # --- 5.7) checks payload (hub-style) ---
    payload: Dict[str, Any] = {
        "run_id": str(rid),
        "high_level": {
            "total_rows": total_rows,
            "total_cols": total_cols,
            "row_identity_status": row_status,
            "temporal_status": temporal_status,
            "label_alignment_status": alignment_status,
            "leakage_scan_status": leakage_status,
        },
        "leakage_overview": leakage_overview,
        "refs": {"upstream": upstream},
    }
    report["checks"]["integrity_summary"] = payload

    # --- 5.8) Policy-free hints (facts -> hints) ---
    # NOTE: evidence_path now points to an existing node (payload already set).
    if n_high_risk > 0:
        add_event_hint(
            report,
            code="leakage_high_risk_features_present",
            severity_signal="risk",
            evidence_path="checks.integrity_summary.leakage_overview",
            context={
                "n_high_risk": int(n_high_risk),
                "risk_level_counts": leakage_overview.get("risk_level_counts", {}),
            },
            remediation={
                "action": "exclude_or_isolate_high_risk_features",
                "safe": True,
                "rationale": "High-risk leakage signals may indicate post-outcome features or future information.",
                "post_check": "drop flagged features and rerun 3.4 + model validation",
            },
        )

    # --- 5.9) Upstream run_id mismatch detection (facts -> hint) ---
    pre_warnings: List[str] = []
    upstream_rid_set = _collect_upstream_run_ids(row_rid, temporal_rid, alignment_rid, leakage_rid)
    report["checks"]["upstream_run_ids"] = {"current": str(rid), "set": upstream_rid_set}

    if upstream_rid_set and (len(upstream_rid_set) > 1 or upstream_rid_set[0] != str(rid)):
        pre_warnings.append("upstream_run_id_mismatch")
        add_event_hint(
            report,
            code="upstream_run_id_mismatch",
            severity_signal="risk",
            evidence_path="checks.upstream_run_ids",
            context={"current_run_id": str(rid), "upstream_run_ids": upstream_rid_set},
            remediation={
                "action": "prefer_full_run_for_one_run_id_or_accept_lineage",
                "safe": True,
                "rationale": "Ad-hoc module execution is normal; record dependencies for auditability.",
                "post_check": "verify refs.dependencies exists and points to upstream artifacts used by this summary",
            },
        )

    # --- 5.10) Collect upstream event hints into 3.5 (hub) ---
    upstream_for_hints: List[Tuple[str, Optional[Dict[str, Any]]]] = [
        ("data_integrity.row_identity", row_report),
        ("data_integrity.temporal", temporal_report),
        ("data_integrity.label_alignment", alignment_report),
        ("data_integrity.leakage_scan", leakage_report),
    ]
    collected_hints = collect_upstream_event_hints(upstream_for_hints, max_hints=int(max_event_hints))

    eh = report["checks"]["event_hints"]
    hints_list = eh.get("hints")
    if isinstance(hints_list, list) and collected_hints:
        hints_list.extend(collected_hints)

    # Dedup + trim
    deduped = dedupe_event_hints(eh.get("hints", []))
    if int(max_event_hints) > 0:
        deduped = deduped[: int(max_event_hints)]
    eh["hints"] = deduped

    # --- 6) Status + gate-friendly summary (warnings/issues only) ---
    issues: List[str] = []
    warnings: List[str] = []
    notes: List[str] = []

    warnings.extend(pre_warnings)

    missing: List[str] = []
    if row_report is None:
        missing.append("missing_row_identity_artifact")
    if temporal_report is None:
        missing.append("missing_temporal_artifact")
    if alignment_report is None:
        missing.append("missing_label_alignment_artifact")
    if leakage_report is None:
        missing.append("missing_leakage_scan_artifact")

    if missing:
        if fail_on_missing:
            issues.extend(missing)
            add_event_hint(
                report,
                code="integrity_summary_missing_subreports",
                severity_signal="hard",
                evidence_path="checks.integrity_summary",
                context={"missing": missing},
                remediation={
                    "action": "run_3x_modules_then_rerun_3_5",
                    "safe": True,
                    "rationale": "3.5 depends on 3.1–3.4 artifacts. Missing artifacts reduce auditability and gating confidence.",
                    "post_check": "confirm 3.1–3.4 artifacts exist with the same run_id",
                },
            )
        elif warn_on_missing:
            warnings.extend(missing)
            add_event_hint(
                report,
                code="integrity_summary_missing_subreports",
                severity_signal="risk",
                evidence_path="checks.integrity_summary",
                context={"missing": missing},
                remediation={
                    "action": "run_3x_modules_then_rerun_3_5",
                    "safe": True,
                    "rationale": "Missing integrity artifacts reduce auditability.",
                    "post_check": "confirm refs.dependencies points to used upstream artifacts",
                },
            )

    # Policy-driven warning (status-only)
    if warn_on_high_risk_leakage and n_high_risk > 0:
        warnings.append("leakage_high_risk_features_present")

    combined = _combine_statuses([row_status, temporal_status, alignment_status, leakage_status])

    issues = dedupe_sorted(issues)
    warnings = dedupe_sorted(warnings)

    if issues or combined == "fail":
        overall_final = "fail"
    elif warnings or combined == "warn":
        overall_final = "warn"
    elif combined == "pass":
        overall_final = "pass"
    else:
        overall_final = "skipped"

    # Notes (non-gating)
    notes.append("3.5 is artifact-driven; df is not required.")
    if upstream_rid_set and (len(upstream_rid_set) > 1 or upstream_rid_set[0] != str(rid)):
        notes.append(f"upstream_run_id_set={upstream_rid_set}")
    if n_high_risk > 0:
        notes.append(f"leakage_n_high_risk={n_high_risk}")

    if not isinstance(report.get("summary"), dict):
        report["summary"] = {}
    if total_rows is not None:
        report["summary"]["total_rows"] = int(total_rows)
    if total_cols is not None:
        report["summary"]["total_cols"] = int(total_cols)

    report = finalize_module_report(
        ctx=ctx,
        report=report,
        overall_status=overall_final,
        issues=issues,
        warnings=warnings,
        metrics={
            "leakage_n_high_risk": int(n_high_risk),
            "leakage_n_medium_risk": int(n_medium_risk),
            "upstream_missing_count": int(len(missing)),
        },
        notes=notes,
        df=None,
    )

    if overall_final == "pass":
        lg.info(f"[{TAG35}] ✅ Integrity summary built.")
    elif overall_final == "warn":
        lg.warning(f"[{TAG35}] ⚠ Integrity summary built. warnings={warnings}")
    else:
        lg.error(f"[{TAG35}] ❌ Integrity summary failed. issues={issues}")

    # --- 7) Persist artifact ---
    out_path = resolve_artifact_path(
        ctx=ctx, stage="data_integrity", name="integrity_summary", run_id=str(rid), tag=TAG35
    )
    write_json(out_path, report, tag=TAG35, indent=2)
    lg.info(f"[{TAG35}] 🧾 Integrity summary saved: {out_path}")

    return report

In [18]:
# --- 3.x Data Integrity & Anti-Leakage Orchestrator ---
#
# What:
#   Run Step-3 "data_integrity" as one controller:
#     - Build the step plan from YAML SSOT (enabled + steps order + gates)
#     - Dispatch each module by contract (df/agg)
#     - Apply centralized gating (hard/soft + strict) and optionally halt
#     - Persist orchestrator ModuleReport v1 + per-step artifacts (safety net)
#
# Notes:
#   - Keep df stable: only steps with returns_df=True may update df
#   - For agg steps, pass df=None to invoke_module() to avoid contract ambiguity

# --- 0) Import + TAG ---
import logging
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint
from src.pipeline.orchestrator_utils import get_stage_cfg
from src.pipeline.orchestrator_common import (
    ModuleFn,
    ModuleSpec,
    validate_module_spec,
    infer_module_status,
    should_halt,
    build_missing_impl_report,
    build_exception_report,
    persist_artifact_safety_net,
    invoke_module,
    normalize_module_result,
    has_missing_module_report,
)
from src.utils.artifact_utils import resolve_artifact_path, write_json, read_json_if_exists
from src.utils.cfg_utils import get_cfg_str, get_cfg_value
from src.utils.path_utils import resolve_dir

TAG3X = "DI_ORCH"


# --- Internal helpers ---
def _artifact_key(spec: ModuleSpec) -> str:
    """Return the canonical artifact/report key for a module spec."""
    k = spec.artifact_name or spec.name or spec.step_id
    k = str(k).strip()
    return k if k else str(spec.step_id).strip()


def _ensure_summary_aliases(artifacts: Dict[str, Any]) -> None:
    """
    Ensure both 'integrity_summary' and 'summary' keys exist when one exists.
    This keeps downstream usage stable even if SSOT step_id is 'summary'.
    """
    if not isinstance(artifacts, dict):
        return

    has_integrity = "integrity_summary" in artifacts
    has_summary = "summary" in artifacts

    if has_integrity and (not has_summary):
        artifacts["summary"] = artifacts["integrity_summary"]
    elif has_summary and (not has_integrity):
        artifacts["integrity_summary"] = artifacts["summary"]


def build_modules_from_ssot(
    *,
    ctx: Dict[str, Any],
    tag: str,
    registry: Dict[str, ModuleFn],
) -> List[ModuleSpec]:
    """
    Build Step-3 module specs from SSOT:
      pipeline.orchestrator.stages.data_integrity.{enabled,steps}
    """
    stage_cfg = get_stage_cfg(ctx, "data_integrity", tag=tag)
    stage_enabled = bool(stage_cfg.get("enabled", True))

    steps = stage_cfg.get("steps") or []
    if not isinstance(steps, list):
        raise TypeError(f"[{tag}] stage_cfg.steps must be a list")

    default_gate = get_cfg_str(
        ctx,
        "pipeline.gate_policy.default",
        tag=tag,
        default="soft",
        required=False,
        lower=True,
        allowed={"soft", "hard"},
    )

    modules: List[ModuleSpec] = []

    for i, s in enumerate(steps):
        if not isinstance(s, dict):
            raise TypeError(f"[{tag}] steps[{i}] must be dict")

        step_id = str(s.get("id") or "").strip()
        if not step_id:
            raise ValueError(f"[{tag}] steps[{i}] missing id")

        cfg_key = str(s.get("cfg_key") or "").strip()
        if not cfg_key:
            raise ValueError(f"[{tag}] steps[{i}] missing cfg_key for {step_id}")

        enabled = bool(stage_enabled and bool(s.get("enabled", True)))

        gate = s.get("gate", None)
        gate_mode = str(gate).strip().lower() if gate is not None else str(default_gate).strip().lower()
        gate_mode = "hard" if gate_mode == "hard" else "soft"

        sid = step_id.strip().lower()

        # Step-3 kinds:
        # - row_identity/temporal/label_alignment/leakage_scan take df
        # - summary is artifact-driven aggregator
        kind = "df"
        returns_df = False
        artifact_name = step_id

        if sid == "summary":
            kind = "agg"
            returns_df = False
            artifact_name = "integrity_summary"  # align with your 3.5 artifact name

        fn = registry.get(step_id) or registry.get(sid)

        modules.append(
            ModuleSpec(
                step_id=step_id,
                name=step_id,
                fn=fn,
                cfg_key=cfg_key,
                artifact_stage="data_integrity",
                artifact_name=artifact_name,
                enabled=enabled,
                kind=kind,
                gate_default=gate_mode,
                returns_df=returns_df,
            )
        )

    return modules


def build_di_registry_from_globals(
    *,
    tag: str = TAG3X,
    fn_name_map: Optional[Dict[str, str]] = None,
    allow_missing: bool = True,
) -> Dict[str, ModuleFn]:
    """
    Notebook registry builder (NO imports).

    Resolves callables from the current notebook globals():
      step_id -> function object
    """
    default_map: Dict[str, str] = {
        "row_identity": "run_row_identity_integrity",
        "temporal": "run_temporal_snapshot_integrity",
        "label_alignment": "run_label_feature_alignment",
        "leakage_scan": "run_leakage_scan_and_feature_tagging",
        "summary": "build_integrity_summary",
    }

    name_map = dict(default_map)
    if fn_name_map:
        name_map.update(fn_name_map)

    reg: Dict[str, ModuleFn] = {}
    g = globals()

    for step_id, fn_name in name_map.items():
        obj = g.get(fn_name)

        if obj is None:
            if not allow_missing:
                raise KeyError(f"[{tag}] registry missing fn '{fn_name}' for step_id='{step_id}'")
            continue

        if not callable(obj):
            raise TypeError(f"[{tag}] '{fn_name}' exists but is not callable (step_id='{step_id}')")

        reg[step_id] = obj  # type: ignore[assignment]

    return reg


def build_di_registry_from_ssot(
    *,
    ctx: Dict[str, Any],
    tag: str = TAG3X,
    fn_name_map: Optional[Dict[str, str]] = None,
    require_enabled_only: bool = True,
) -> Dict[str, ModuleFn]:
    """
    Build registry from globals(), and (optionally) require implementations
    for enabled steps in SSOT.
    """
    reg = build_di_registry_from_globals(tag=tag, fn_name_map=fn_name_map, allow_missing=True)

    if not require_enabled_only:
        return reg

    modules = build_modules_from_ssot(ctx=ctx, tag=tag, registry=reg)
    missing: List[str] = []
    for m in modules:
        if m.enabled and (m.fn is None):
            missing.append(str(m.step_id))

    if missing:
        raise KeyError(f"[{tag}] registry missing implementations for enabled steps: {missing}")

    return reg


def extract_cleaned_parquet_path(*, rep: Dict[str, Any], ctx: Dict[str, Any], tag: str) -> str:
    """Extract the cleaned parquet path from Step-2 cleaning ModuleReport."""
    if not isinstance(rep, dict):
        raise TypeError(f"[{tag}] cleaning report must be a dict")

    # 1) Preferred: checks.persistence.cleaned_parquet_path (your current output)
    checks = rep.get("checks") or {}
    persistence = (checks.get("persistence") or {}) if isinstance(checks, dict) else {}
    p = str(persistence.get("cleaned_parquet_path") or "").strip()
    if p:
        return p

    # 2) Optional future-proof: refs.outputs.cleaned_path
    refs = rep.get("refs") or {}
    outputs = (refs.get("outputs") or {}) if isinstance(refs, dict) else {}
    p2 = str(outputs.get("cleaned_path") or "").strip()
    if p2:
        return p2

    # 3) Last resort: parse summary.notes like "cleaned_dataset=xxx.parquet"
    summary = rep.get("summary") or {}
    notes = summary.get("notes") or []
    if isinstance(notes, list):
        for n in notes:
            s = str(n)
            if s.startswith("cleaned_dataset=") and s.endswith(".parquet"):
                fname = s.split("=", 1)[1].strip()
                interim_dir = resolve_dir(ctx, "data.interim", required=True, tag=tag)
                return str(interim_dir / fname)

    raise KeyError(f"[{tag}] cannot find cleaned parquet path in cleaning report")


def _resolve_logger(*, ctx: Dict[str, Any], lg: Any = None) -> Any:
    """Resolve a usable logger. Fallback to stdlib logging if ctx has none."""
    if lg is not None:
        return lg
    ctx_lg = ctx.get("logger", None) if isinstance(ctx, dict) else None
    if ctx_lg is not None:
        return ctx_lg
    # Fallback logger (safe for notebook usage)
    return logging.getLogger(TAG3X)


def load_step3_input_df(
    *,
    ctx: Dict[str, Any],
    rid: str,
    tag: str,
    lg=None,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Load Step-3 input df from Step-2 cleaning artifact (materialized parquet)."""
    lg = _resolve_logger(ctx=ctx, lg=lg)

    clean_rep_path = resolve_artifact_path(
        ctx=ctx, stage="data_quality", name="cleaning", run_id=str(rid), tag=tag
    )

    rep = read_json_if_exists(clean_rep_path, lg)
    if not rep:
        raise FileNotFoundError(f"[{tag}] missing Step-2 cleaning artifact: {str(clean_rep_path)}")

    parquet_path = extract_cleaned_parquet_path(rep=rep, ctx=ctx, tag=tag)
    df = pd.read_parquet(parquet_path)

    meta = {
        "source": "step2_cleaning_parquet",
        "upstream_stage": "data_quality",
        "upstream_step": "2.9",
        "upstream_run_id": str(rid),
        "cleaning_artifact_path": str(clean_rep_path),
        "cleaned_parquet_path": str(parquet_path),
    }
    return df, meta


# --- 1) Public API ---
def run_data_integrity_orchestrator(
    *,
    ctx: Dict[str, Any],
    registry: Dict[str, ModuleFn],
    df: pd.DataFrame,
    run_id: Optional[str] = None,
    input_meta: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """
    Run Step-3 data_integrity (3.1–3.5) with SSOT-driven plan + centralized gating.

    Returns:
      {
        "df": df,
        "reports": {key: ModuleReport},
        "execution": [{step_id,status,gate,...}],
        "halted": bool,
        "halted_at": Optional[str],
        "orchestrator_report": ModuleReport,
      }
    """

    # --- 2) Runtime validation ---
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG3X, run_id=run_id)

    # --- 3) Input validation ---
    if not isinstance(registry, dict):
        raise TypeError(f"[{TAG3X}] registry must be a dict[str, ModuleFn]")
    if df is None or (not isinstance(df, pd.DataFrame)):
        raise TypeError(f"[{TAG3X}] df must be a pandas DataFrame")

    # --- 4) Resolve config (YAML SSOT) ---
    strict = bool(get_cfg_value(ctx, "pipeline.gate_policy.strict", tag=TAG3X, default=True))

    stage_cfg = get_stage_cfg(ctx, "data_integrity", tag=TAG3X)
    stage_enabled = bool(stage_cfg.get("enabled", True))

    # --- 5) Core workflows ---
    lg.info(
        f"[{TAG3X}] Run started | stage=data_integrity | run_id={rid} | strict={strict} "
        f"| df_rows={int(df.shape[0])} | df_cols={int(df.shape[1])}"
    )

    # --- Build ModuleReport skeleton (infra) ---
    orch = build_module_report(
        ctx=ctx,
        stage="data_integrity",
        step="3.x",
        name="orchestrator",
        tag=TAG3X,
        cfg_key="pipeline.orchestrator.stages.data_integrity",
        enabled=stage_enabled,
        df=df,
        run_id_override=str(rid),
    )
    orch.setdefault("refs", {})

    orch.setdefault("inputs", {})
    orch["inputs"].update(
        {
            "stage_enabled": stage_enabled,
            "strict": bool(strict),
            "df_input_provided": True,
        }
    )

    if isinstance(input_meta, dict) and input_meta:
        orch["inputs"]["input_meta"] = input_meta

    orch["refs"]["self"] = make_artifact_ref(
        ctx=ctx, stage="data_integrity", name="orchestrator", run_id=str(rid)
    )
    ensure_event_hints(orch, version=1)
    orch.setdefault("checks", {})

    # --- Build plan (SSOT -> specs) ---
    modules: List[ModuleSpec] = build_modules_from_ssot(ctx=ctx, tag=TAG3X, registry=registry)

    for s in modules:
        validate_module_spec(s, TAG3X)

    orch["checks"]["plan"] = [m.step_id for m in modules if m.enabled]
    orch["checks"]["artifact_keys"] = {str(m.step_id): _artifact_key(m) for m in modules if m.enabled}

    lg.info(
        f"[{TAG3X}] Plan | stage_enabled={stage_enabled} | steps={len(orch['checks']['plan'])} "
        f"| enabled_steps={orch['checks']['plan']}"
    )

    # --- Early exit: stage disabled ---
    if not stage_enabled:
        orch = finalize_module_report(
            ctx=ctx,
            report=orch,
            overall_status="skipped",
            issues=[],
            warnings=[],
            metrics={"halted": False, "n_reports": 0},
            notes=["stage_disabled=true"],
            df=df,
        )
        out_path = resolve_artifact_path(ctx=ctx, stage="data_integrity", name="orchestrator", run_id=str(rid), tag=TAG3X)
        write_json(out_path, orch, tag=TAG3X, indent=2)

        lg.info(f"[{TAG3X}] Run finished | overall=skipped | stage_enabled=false | run_id={rid}")

        return {
            "df": df,
            "halted": False,
            "halted_at": None,
            "orchestrator_report": orch,
            "reports": {},
            "execution": [],
        }

    # --- Execute plan ---
    reports: Dict[str, Dict[str, Any]] = {}
    execution: List[Dict[str, Any]] = []
    artifacts: Dict[str, Any] = {}
    halted = False
    halted_at: Optional[str] = None

    for spec in modules:
        if halted:
            execution.append({"step_id": spec.step_id, "status": "skipped", "reason": "halted", "gate": spec.gate_default})
            continue

        if not spec.enabled:
            execution.append({"step_id": spec.step_id, "status": "skipped", "reason": "disabled", "gate": spec.gate_default})
            continue

        gate_mode = spec.gate_default
        lg.info(f"[{TAG3X}] ▶ step_id={spec.step_id} | kind={spec.kind} | gate={gate_mode} | strict={strict}")

        # --- Missing implementation ---
        if spec.fn is None:
            rep = build_missing_impl_report(ctx=ctx, rid=str(rid), tag=TAG3X, stage="data_integrity", spec=spec, df=df)
            report_key = _artifact_key(spec)

            reports[report_key] = rep
            artifacts[report_key] = rep
            persist_artifact_safety_net(ctx=ctx, rid=str(rid), tag=TAG3X, spec=spec, report=rep)

            st = infer_module_status(rep)
            execution.append({"step_id": spec.step_id, "status": st, "gate": gate_mode, "missing_impl": True})

            if should_halt(strict=bool(strict), gate_mode=gate_mode, status=st):
                halted, halted_at = True, spec.step_id
            continue

        # --- Invoke module (with safety net) ---
        rep: Dict[str, Any]
        try:
            df_for_call = df if str(spec.kind).lower() == "df" else None

            raw = invoke_module(spec=spec, df=df_for_call, artifacts=artifacts, ctx=ctx, rid=str(rid))
            norm = normalize_module_result(spec=spec, res=raw, df_in=df)

            # Keep df stable: only update when module contract says returns_df=True
            if bool(getattr(spec, "returns_df", False)) and (norm.get("df") is not None):
                df = norm["df"]

            rep = norm["report"]

            # Contract fallback: module returned no ModuleReport
            if rep is None:
                rep = build_exception_report(
                    ctx=ctx,
                    rid=str(rid),
                    tag=TAG3X,
                    stage="data_integrity",
                    spec=spec,
                    err=RuntimeError("module_returned_no_report"),
                    df=df,
                )
                persist_artifact_safety_net(ctx=ctx, rid=str(rid), tag=TAG3X, spec=spec, report=rep)

        except Exception as e:
            lg.exception(f"[{TAG3X}] ❌ step_id={spec.step_id} crashed: {e!r}")
            rep = build_exception_report(ctx=ctx, rid=str(rid), tag=TAG3X, stage="data_integrity", spec=spec, err=e, df=df)
            persist_artifact_safety_net(ctx=ctx, rid=str(rid), tag=TAG3X, spec=spec, report=rep)

        report_key = _artifact_key(spec)
        reports[report_key] = rep
        artifacts[report_key] = rep

        st = infer_module_status(rep)
        execution.append({"step_id": spec.step_id, "status": st, "gate": gate_mode})

        # Summary alias should be stabilized after summary step (or at end)
        if str(spec.step_id).strip().lower() == "summary":
            _ensure_summary_aliases(artifacts)

        if should_halt(strict=bool(strict), gate_mode=gate_mode, status=st):
            halted, halted_at = True, spec.step_id

    # Ensure aliases even if summary disabled/missing
    _ensure_summary_aliases(artifacts)

    orch["checks"]["execution"] = execution
    orch["checks"]["reports_present"] = {k: True for k in reports.keys()}
    orch["checks"]["halted"] = {"halted": bool(halted), "halted_at": halted_at or ""}

    # Add a concise status map for auditability
    orch["checks"]["status_map"] = {str(x.get("step_id")): str(x.get("status")) for x in execution if x.get("step_id")}

    # Soft-gated failures should surface as warnings (even if pipeline continues)
    soft_fail_steps = [
        x.get("step_id")
        for x in execution
        if (str(x.get("gate", "")).lower() == "soft") and (str(x.get("status", "")).lower() == "fail")
    ]
    orch["checks"]["soft_gated_failures"] = soft_fail_steps

    # --- Step-4 readiness (split gate) ---
    blocked = bool(halted and bool(strict))
    ready_for_step4 = (not blocked)

    reasons: List[str] = []
    if halted:
        reasons.append("halted")
    if halted_at:
        reasons.append(f"halted_at:{halted_at}")
    if soft_fail_steps:
        reasons.append("soft_gated_failures_present")

    orch["checks"]["ready_for_step4"] = {
        "ready": bool(ready_for_step4),
        "blocked": bool(blocked),
        "reasons": reasons,
        "strict": bool(strict),
        "halted": bool(halted),
        "halted_at": halted_at or "",
        "soft_gated_fail_steps": list(soft_fail_steps),
        "policy_note": (
            "Step-4 readiness is blocked only by hard-gated failures under strict mode; "
            "soft-gated failures are surfaced as warnings but do not block by default."
        ),
    }


    if halted:
        add_event_hint(
            orch,
            code="di_orchestrator_halted",
            severity_signal="hard",
            evidence_path="checks.halted",
            context={"halted": True, "halted_at": halted_at or "", "strict": bool(strict)},
            remediation={
                "action": "fix_failed_step_then_rerun",
                "safe": True,
                "rationale": "Hard-gated step failed; downstream outputs may be invalid.",
                "post_check": "Confirm halted step becomes pass/warn/skipped (as expected) before continuing.",
            },
        )
        lg.error(f"[{TAG3X}] Halted | halted_at={halted_at or ''} | strict={strict}")

    # --- 6) Status + gate-friendly summary ---
    issues: List[str] = []
    warnings: List[str] = []

    if halted:
        issues.append("pipeline_halted")

    if any((x.get("status") == "warn") for x in execution):
        warnings.append("one_or_more_steps_warn")

    if soft_fail_steps:
        warnings.append("soft_gated_failures_present")

    if any((x.get("missing_impl") is True and str(x.get("gate", "")).lower() == "soft") for x in execution):
        warnings.append("soft_gated_missing_impl")

    if any(has_missing_module_report(rep) for rep in reports.values() if isinstance(rep, dict)):
        warnings.append("missing_module_report_from_module")

    orch["checks"]["warnings"] = {
        "one_or_more_steps_warn": [x.get("step_id") for x in execution if x.get("status") == "warn"],
        "soft_gated_failures_present": list(soft_fail_steps),
        "soft_gated_missing_impl": [
            x.get("step_id")
            for x in execution
            if x.get("missing_impl") is True and str(x.get("gate", "")).lower() == "soft"
        ],
        "missing_module_report_from_module": [
            k for k, rep in reports.items() if isinstance(rep, dict) and has_missing_module_report(rep)
        ],
    }

    overall = "fail" if issues else ("warn" if warnings else "pass")

    orch = finalize_module_report(
        ctx=ctx,
        report=orch,
        overall_status=overall,
        issues=issues,
        warnings=warnings,
        metrics={"halted": bool(halted), "n_reports": int(len(reports))},
        notes=["di_orchestrator:3.1-3.5 (SSOT-driven)"],
        df=df,
    )

    # --- 7) Persist Artifact ---
    out_path = resolve_artifact_path(
        ctx=ctx,
        stage="data_integrity",
        name="orchestrator",
        run_id=str(rid),
        tag=TAG3X,
    )
    write_json(out_path, orch, tag=TAG3X, indent=2)

    lg.info(
        f"[{TAG3X}] Run finished | overall={overall} | halted={halted} | halted_at={halted_at or ''} | n_reports={len(reports)}"
    )

    return {
        "df": df,
        "reports": reports,
        "execution": execution,
        "halted": halted,
        "halted_at": halted_at,
        "orchestrator_report": orch,
        "ready_for_step4": bool(ready_for_step4),
    }


def run_di_valve(
    *,
    ctx: Dict[str, Any],
    run_id: Optional[str] = None,
    df: Optional[pd.DataFrame] = None,
    registry: Optional[Dict[str, ModuleFn]] = None,
    fn_name_map: Optional[Dict[str, str]] = None,
    input_from_step2: bool = True,
) -> Dict[str, Any]:
    """
    Notebook one-call entrypoint (NO imports).
    Priority:
      1) explicit df
      2) auto-load Step-2 (2.9) cleaned parquet by run_id
    """
    rid = str(run_id).strip() if run_id is not None else str(ctx.get("run_id", "")).strip()
    if not rid:
        raise ValueError(f"[{TAG3X}] run_id is missing (provide run_id or ensure ctx['run_id'] exists)")

    reg = registry or build_di_registry_from_ssot(
        ctx=ctx,
        tag=TAG3X,
        fn_name_map=fn_name_map,
        require_enabled_only=True,
    )

    input_meta: Optional[Dict[str, Any]] = None
    if df is None:
        if not input_from_step2:
            raise ValueError(f"[{TAG3X}] df is None and input_from_step2=False; cannot resolve input df")
        df, input_meta = load_step3_input_df(ctx=ctx, rid=rid, tag=TAG3X)

    # NOTE: runtime validation happens once inside run_data_integrity_orchestrator()
    return run_data_integrity_orchestrator(
        ctx=ctx,
        registry=reg,
        df=df,
        run_id=rid,
        input_meta=input_meta,
    )

out3 = run_di_valve(ctx=ctx, run_id=rid)

print("[DI] halted:", out3["halted"], "halted_at:", out3["halted_at"], "ready4:", out3["ready_for_step4"])


00:19:53 | INFO | [DI_ORCH] Run started | stage=data_integrity | run_id=20260222-231948-167 | strict=True | df_rows=7043 | df_cols=21
00:19:53 | INFO | [DI_ORCH] Plan | stage_enabled=True | steps=5 | enabled_steps=['row_identity', 'temporal', 'label_alignment', 'leakage_scan', 'summary']
00:19:53 | INFO | [DI_ORCH] ▶ step_id=row_identity | kind=df | gate=hard | strict=True
00:19:53 | INFO | [ROW_IDENTITY] 🧬 Starting row identity checks... | run_id=20260222-231948-167
00:19:53 | INFO | [ROW_IDENTITY] ✅ Row identity pass.
00:19:53 | INFO | [ROW_IDENTITY] 🧾 Row identity artifact saved: D:\DS_project\telco-churn-project\artifacts\data_integrity_row_identity_20260222-231948-167.json
00:19:53 | INFO | [DI_ORCH] ▶ step_id=temporal | kind=df | gate=hard | strict=True
00:19:53 | INFO | [TEMPORAL_INTEGRITY] ⏱️ Starting temporal & snapshot integrity checks... | run_id=20260222-231948-167
00:19:53 | INFO | [TEMPORAL_INTEGRITY] 🧾 Temporal integrity artifact saved (skipped): D:\DS_project\telco-chur

[DI] halted: False halted_at: None ready4: True


# 4 Split Strategy
🎯 **Goal:** Establish a **reproducible, leak-safe train/val/test split** by resolving the split plan from YAML SSOT, applying Step 3 integrity signals as constraints, building the split assignment, validating split quality, and producing an auditable summary.

## Sub-steps

| Step | Module | Kind | Gate | Output |
|------|--------|------|------|--------|
| 4.1 | Split Plan Resolver | agg | **hard** | `split_strategy_split_plan_{run_id}.json` |
| 4.2 | Leakage-Aware Constraints | agg | soft | `split_strategy_leakage_constraints_{run_id}.json` |
| 4.3 | Split Builder | df | **hard** | `split_assignment_{run_id}.parquet` + JSON |
| 4.4 | Split Diagnostics | df | soft | `split_strategy_split_diagnostics_{run_id}.json` |
| 4.5 | Split Summary | agg | soft | `split_strategy_split_summary_{run_id}.json` |
| 4.x | Orchestrator | — | gate | `split_strategy_orchestrator_{run_id}.json` |

## Design Principles
- **Contract-first**: All modules emit ModuleReport v1
- **YAML SSOT**: All split parameters come from `config/data_split.yaml`
- **Artifact separation**: Row-level data → parquet; metadata → JSON
- **Step 3 linkage**: 4.2 reads Step 3 `leakage_scan` + `temporal` artifacts to enforce constraints
- **Hard gates**: group_overlap > 0 / time_inversion > 0 / missing split_assignment → fail


In [ ]:
# --- 4.1 Split Plan Resolver ---
#
# What:
#   Resolve the split plan from YAML SSOT and validate logical consistency.
#
# How:
#   - Read pipeline.split_strategy.split from YAML via ctx (SSOT)
#   - Validate: group strategy requires group_col; time_based requires timestamp_col
#   - Emit event_hints (hard) for invalid configurations
#   - Persist ModuleReport v1 artifact (SSOT naming)
#
# Why:
#   Fail fast on invalid split configuration before any data is touched.
#   Provides a stable, auditable record of the chosen split plan.

# --- 0) Import + TAG ---
from typing import Any, Dict, List, Optional

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint

from src.utils.artifact_utils import resolve_artifact_path, write_json
from src.utils.cfg_utils import get_cfg_dict, get_cfg_str, get_cfg_float, get_cfg_int, get_cfg_value

TAG41 = "SPLIT_PLAN"

_ALLOWED_SPLIT_TYPES = {"holdout", "cv"}
_ALLOWED_STRATEGIES = {"random", "stratified", "group", "time_based"}


def _norm_optional_str(x: Any) -> Optional[str]:
    """Normalize optional string config values like null/None/'null'/'none'/'' -> None."""
    if x is None:
        return None
    s = str(x).strip()
    if not s:
        return None
    if s.lower() in {"none", "null"}:
        return None
    return s

# --- 1) public API ---
def resolve_split_plan(
    *,
    ctx: Dict[str, Any],
    run_id: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Step 4.1 - Resolve split plan from YAML SSOT and validate configuration.

    SSOT:
      pipeline.split_strategy.split:
        type: "holdout" | "cv"
        strategy: "random" | "stratified" | "group" | "time_based"
        test_size: float
        val_size: float
        random_state: int
        group_col: str|null
        timestamp_col: str|null
    """
    # --- 2) Runtime validation ---
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG41, run_id=run_id)

    # --- 3) Input validation (minimal for 4.1) ---
    if not isinstance(ctx, dict):
        raise TypeError(f"[{TAG41}] ctx must be a dict")

    # --- 4) Resolve config (YAML SSOT) ---
    base = "pipeline.split_strategy.split"

    split_cfg = get_cfg_dict(ctx, base, tag=TAG41, default={})
    if not isinstance(split_cfg, dict):
        raise TypeError(f"[{TAG41}] {base} must be a dict")

    split_type = get_cfg_str(
        ctx,
        f"{base}.type",
        tag=TAG41,
        default="holdout",
        lower=True,
        allowed=_ALLOWED_SPLIT_TYPES,
        strict_type=False,
    )

    strategy = get_cfg_str(
        ctx,
        f"{base}.strategy",
        tag=TAG41,
        default="stratified",
        lower=True,
        allowed=_ALLOWED_STRATEGIES,
        strict_type=False,
    )

    test_size = get_cfg_float(
        ctx,
        f"{base}.test_size",
        tag=TAG41,
        default=0.20,
        min_value=0.0,
        max_value=1.0,
        strict_type=False,
    )

    val_size = get_cfg_float(
        ctx,
        f"{base}.val_size",
        tag=TAG41,
        default=0.20,
        min_value=0.0,
        max_value=1.0,
        strict_type=False,
    )

    # Prefer project.seed if present; otherwise 42
    default_seed = int(get_cfg_value(ctx, "project.seed", tag=TAG41, default=42) or 42)
    random_state = get_cfg_int(
        ctx,
        f"{base}.random_state",
        tag=TAG41,
        default=default_seed,
        min_value=0,
        strict_type=False,
    )

    group_col = _norm_optional_str(split_cfg.get("group_col"))
    timestamp_col = _norm_optional_str(split_cfg.get("timestamp_col"))

    # --- 5) Core workflow ---
    lg.info(f"[{TAG41}] 🔧 Resolving split plan | run_id={rid}")

    report = build_module_report(
        ctx=ctx,
        stage="split_strategy",
        step="4.1",
        name="split_plan",
        tag=TAG41,
        cfg_key=base,
        enabled=True,
        df=None,
        run_id_override=str(rid),
    )

    report.setdefault("refs", {})
    report.setdefault("inputs", {})
    report.setdefault("checks", {})
    ensure_event_hints(report, version=1)

    report["refs"]["self"] = make_artifact_ref(ctx=ctx, stage="split_strategy", name="split_plan", run_id=str(rid))

    report["inputs"] = {
        "split_cfg_key": base,
    }

    report["thresholds_used"] = {
        "allowed_split_types": sorted(list(_ALLOWED_SPLIT_TYPES)),
        "allowed_strategies": sorted(list(_ALLOWED_STRATEGIES)),
        "size_constraints": {"require_train_gt_0": True, "require_each_in_0_1": True},
    }

    report["used_config"] = {
        "cfg_keys": [
            base,
            f"{base}.type",
            f"{base}.strategy",
            f"{base}.test_size",
            f"{base}.val_size",
            f"{base}.random_state",
            f"{base}.group_col",
            f"{base}.timestamp_col",
        ]
    }

    # --- Facts (config validation) ---
    validation_errors: List[str] = []

    # Strategy-specific requirements
    if strategy == "group" and not group_col:
        validation_errors.append("strategy='group' requires group_col")

        add_event_hint(
            report,
            code="split_group_col_missing",
            severity_signal="hard",
            evidence_path="checks.split_plan.group_col",
            context={"strategy": strategy, "group_col": group_col},
            remediation={
                "action": "set_group_col",
                "safe": True,
                "rationale": "Group split requires an entity/group column to prevent leakage across groups.",
                "post_check": "Re-run 4.1 and confirm group_col is resolved.",
            },
        )

    if strategy == "time_based" and not timestamp_col:
        validation_errors.append("strategy='time_based' requires timestamp_col")

        add_event_hint(
            report,
            code="split_timestamp_col_missing",
            severity_signal="hard",
            evidence_path="checks.split_plan.timestamp_col",
            context={"strategy": strategy, "timestamp_col": timestamp_col},
            remediation={
                "action": "set_timestamp_col",
                "safe": True,
                "rationale": "Time-based split requires a timestamp column to avoid time leakage.",
                "post_check": "Re-run 4.1 and confirm timestamp_col is resolved.",
            },
        )

    # Size validations
    # (typed getters already constrain [0,1], but we still enforce strict (0,1) and sum < 1)
    if not (0.0 < float(test_size) < 1.0):
        validation_errors.append("test_size must be in (0,1)")
    if not (0.0 < float(val_size) < 1.0):
        validation_errors.append("val_size must be in (0,1)")

    if float(test_size) + float(val_size) >= 1.0:
        validation_errors.append("test_size + val_size must be < 1.0")

        add_event_hint(
            report,
            code="split_sizes_invalid_sum",
            severity_signal="hard",
            evidence_path="checks.split_plan",
            context={"test_size": float(test_size), "val_size": float(val_size)},
            remediation={
                "action": "adjust_split_sizes",
                "safe": True,
                "rationale": "Holdout/CV split requires train portion > 0.",
                "post_check": "Ensure test_size + val_size < 1.0, then re-run 4.1.",
            },
        )

    pct_train = round(1.0 - float(test_size) - float(val_size), 6)

    split_plan = {
        "split_type": str(split_type),
        "strategy": str(strategy),
        "test_size": float(test_size),
        "val_size": float(val_size),
        "pct_train_approx": float(pct_train),
        "random_state": int(random_state),
        "group_col": group_col,
        "timestamp_col": timestamp_col,
        "validation_errors": list(validation_errors),
        "valid": bool(len(validation_errors) == 0),
    }

    report["checks"]["split_plan"] = split_plan

    lg.info(
        f"[{TAG41}] 📌 Plan | type={split_type} | strategy={strategy} | "
        f"train≈{pct_train} | test={test_size} | val={val_size} | seed={random_state}"
    )

    # --- 6) Status + gate-friendly summary ---
    issues: List[str] = []
    warnings: List[str] = []

    # Treat any validation error as fail for 4.1 (config must be correct before touching data)
    if validation_errors:
        issues.append("invalid_split_config")

    overall = "fail" if issues else ("warn" if warnings else "pass")

    report = finalize_module_report(
        ctx=ctx,
        report=report,
        overall_status=overall,
        issues=issues,
        warnings=warnings,
        metrics={
            "split_type": str(split_type),
            "strategy": str(strategy),
            "test_size": float(test_size),
            "val_size": float(val_size),
            "pct_train_approx": float(pct_train),
            "random_state": int(random_state),
            "n_validation_errors": int(len(validation_errors)),
        },
        notes=[f"split_plan_resolved_at_utc_run_id={rid}"],
        df=None,
    )

    if overall == "pass":
        lg.info(f"[{TAG41}] ✅ Split plan pass")
    elif overall == "warn":
        lg.warning(f"[{TAG41}] ⚠ Split plan warn | warnings={warnings}")
    else:
        lg.error(f"[{TAG41}] ❌ Split plan fail | issues={issues} | errors={validation_errors[:5]}")

    # --- 7) Persist Artifact ---
    out_path = resolve_artifact_path(
        ctx=ctx,
        stage="split_strategy",
        name="split_plan",
        run_id=str(rid),
        tag=TAG41,
    )
    write_json(out_path, report, tag=TAG41, indent=2)
    lg.info(f"[{TAG41}] 🧾 Split plan artifact saved: {out_path}")

    return report

In [ ]:
# --- 4.2 Leakage-Aware Constraints ---
#
# What:
#   Translate Step 3 integrity signals into split constraints and surface
#   mismatches between the YAML plan and data structure as event_hints.
#
# How:
#   - Load Step 3 artifacts: leakage_scan (high-risk features) + temporal (timestamp status)
#   - Load 4.1 split_plan artifact
#   - Compare data signals vs chosen strategy → emit event_hints (risk/info)
#   - Produce effective_constraints: may override strategy if signals are critical
#   - Persist ModuleReport v1 artifact (SSOT naming)
#
# Why:
#   Bridging Step 3 (integrity signals) → Step 4 (split decisions) is the
#   key architectural link that prevents temporal and entity leakage.

# --- 0) Import + TAG ---
from typing import Any, Dict, List, Optional

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint
from src.utils.artifact_utils import resolve_artifact_path, write_json, read_json_if_exists
from src.utils.cfg_utils import get_cfg_bool, get_cfg_str, get_cfg_int
from src.utils.serialization_utils import safe_float

TAG42 = "SPLIT_CONSTRAINTS"


# --- Internal helpers ---
def _load_step3_artifact(
    ctx: Dict[str, Any],
    *,
    name: str,
    run_id: str,
    lg: Any,
) -> Optional[Dict[str, Any]]:
    """Load a data_integrity artifact by run_id."""
    path = resolve_artifact_path(ctx=ctx, stage="data_integrity", name=name, run_id=run_id, tag=TAG42)
    return read_json_if_exists(path, lg=lg, tag=TAG42)


def _is_timestamp_active(temporal_rep: Optional[Dict[str, Any]]) -> bool:
    """Return True if Step 3 temporal check found an active timestamp column."""
    if not isinstance(temporal_rep, dict):
        return False
    checks = temporal_rep.get("checks", {}) if isinstance(temporal_rep.get("checks"), dict) else {}
    temporal = checks.get("temporal", {}) if isinstance(checks.get("temporal"), dict) else {}
    ts_col = temporal.get("timestamp_col")
    enabled = temporal.get("enabled", False)
    return bool(enabled and ts_col)


def _count_high_risk_features(leakage_rep: Optional[Dict[str, Any]]) -> int:
    """Count features tagged as high risk in 3.4 leakage scan."""
    if not isinstance(leakage_rep, dict):
        return 0
    checks = leakage_rep.get("checks", {}) if isinstance(leakage_rep.get("checks"), dict) else {}
    scan = checks.get("scan_overview", {}) if isinstance(checks.get("scan_overview"), dict) else {}
    risk_counts = scan.get("risk_level_counts", {})
    return int(risk_counts.get("high", 0)) if isinstance(risk_counts, dict) else 0


def _has_entity_groups(leakage_rep: Optional[Dict[str, Any]]) -> bool:
    """
    Infer entity/group structure from leakage scan.
    Proxy: if any feature is tagged 'id_like_name' with low null rate.
    """
    if not isinstance(leakage_rep, dict):
        return False
    checks = leakage_rep.get("checks", {}) if isinstance(leakage_rep.get("checks"), dict) else {}
    features = checks.get("features", {}) if isinstance(checks.get("features"), dict) else {}
    for feat, info in features.items():
        if not isinstance(info, dict):
            continue
        tags = info.get("tags", [])
        if isinstance(tags, list) and "id_like_name" in tags:
            return True
    return False


# --- 1) Public API ---
def run_leakage_constraints(
    *,
    ctx: Dict[str, Any],
    run_id: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Translate Step 3 signals into split constraints and detect plan mismatches.

    YAML SSOT
    ---------
    pipeline.split_strategy.constraints:
      enforce_time_split_if_timestamp: true
      enforce_group_split_if_entity: true
      step3_leakage_artifact_name: "leakage_scan"
      step3_temporal_artifact_name: "temporal"

    Returns
    -------
    dict
        ModuleReport v1 (Leakage Constraints), also persisted as artifact.
    """
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG42, run_id=run_id)

    enforce_time = get_cfg_bool(
        ctx, "pipeline.split_strategy.constraints.enforce_time_split_if_timestamp",
        tag=TAG42, default=True, required=False,
    )
    enforce_group = get_cfg_bool(
        ctx, "pipeline.split_strategy.constraints.enforce_group_split_if_entity",
        tag=TAG42, default=True, required=False,
    )
    leakage_artifact = get_cfg_str(
        ctx, "pipeline.split_strategy.constraints.step3_leakage_artifact_name",
        tag=TAG42, default="leakage_scan", required=False,
    )
    temporal_artifact = get_cfg_str(
        ctx, "pipeline.split_strategy.constraints.step3_temporal_artifact_name",
        tag=TAG42, default="temporal", required=False,
    )

    report = build_module_report(
        ctx=ctx, run_id=rid,
        stage="split_strategy", step="4.2", name="leakage_constraints",
        tag=TAG42, cfg_key="pipeline.split_strategy.constraints", enabled=True,
    )
    ensure_event_hints(report)

    lg.info(f"[{TAG42}] Loading Step 3 artifacts | run_id={rid}")

    # --- Load Step 3 artifacts ---
    leakage_rep = _load_step3_artifact(ctx, name=leakage_artifact, run_id=rid, lg=lg)
    temporal_rep = _load_step3_artifact(ctx, name=temporal_artifact, run_id=rid, lg=lg)

    if leakage_rep is None:
        lg.warning(f"[{TAG42}] Step 3 leakage_scan artifact not found — constraints will use YAML plan as-is")
    if temporal_rep is None:
        lg.warning(f"[{TAG42}] Step 3 temporal artifact not found — timestamp signal unavailable")

    # --- Load 4.1 plan artifact ---
    plan_path = resolve_artifact_path(ctx=ctx, stage="split_strategy", name="split_plan", run_id=rid, tag=TAG42)
    plan_rep = read_json_if_exists(plan_path, lg=lg, tag=TAG42)
    plan_checks = plan_rep.get("checks", {}).get("split_plan", {}) if isinstance(plan_rep, dict) else {}
    plan_strategy = str(plan_checks.get("strategy") or "stratified").strip().lower()

    # --- Analyze signals ---
    timestamp_active = _is_timestamp_active(temporal_rep)
    entity_groups_detected = _has_entity_groups(leakage_rep)
    n_high_risk = _count_high_risk_features(leakage_rep)

    issues: List[str] = []
    warnings: List[str] = []
    strategy_override: Optional[str] = None
    override_reason: Optional[str] = None

    # Signal 1: Timestamp active but plan uses random/stratified split
    if timestamp_active and enforce_time and plan_strategy not in {"time_based"}:
        msg = (
            f"Step 3 detected an active timestamp column, but the split strategy is '{plan_strategy}'. "
            f"Consider using 'time_based' split to prevent temporal leakage."
        )
        warnings.append(msg)
        add_event_hint(report, code="TIMESTAMP_DETECTED_BUT_RANDOM_SPLIT",
                       severity_signal="risk",
                       evidence_path="checks.effective_constraints.timestamp_active",
                       context={"detected_strategy": plan_strategy, "signal": "timestamp_active"},
                       remediation="Set pipeline.split_strategy.split.strategy=time_based and timestamp_col in data_split.yaml.")
        lg.warning(f"[{TAG42}] {msg}")

    # Signal 2: Entity/group structure but plan uses random/stratified split
    if entity_groups_detected and enforce_group and plan_strategy not in {"group"}:
        msg = (
            f"Step 3 detected id-like features suggesting entity/group structure, "
            f"but the split strategy is '{plan_strategy}'. "
            f"Consider using 'group' split to prevent entity leakage."
        )
        warnings.append(msg)
        add_event_hint(report, code="ENTITY_DETECTED_BUT_RANDOM_SPLIT",
                       severity_signal="risk",
                       evidence_path="checks.effective_constraints.entity_groups_detected",
                       context={"detected_strategy": plan_strategy, "signal": "entity_groups_detected"},
                       remediation="Set pipeline.split_strategy.split.strategy=group and group_col in data_split.yaml.")
        lg.warning(f"[{TAG42}] {msg}")

    # Signal 3: High-risk features detected — inform downstream (not a split override)
    if n_high_risk > 0:
        add_event_hint(report, code="HIGH_RISK_FEATURES_IN_SPLIT",
                       severity_signal="info",
                       evidence_path="checks.effective_constraints.n_high_risk_features",
                       context={"n_high_risk": n_high_risk},
                       remediation="Inspect leakage_scan artifact. Verify these features are excluded or handled in Step 5.")

    # --- Effective constraints (passed to 4.3) ---
    effective_constraints = {
        "strategy_override": strategy_override,
        "override_reason": override_reason,
        "timestamp_active": bool(timestamp_active),
        "entity_groups_detected": bool(entity_groups_detected),
        "n_high_risk_features": int(n_high_risk),
        "leakage_artifact_loaded": leakage_rep is not None,
        "temporal_artifact_loaded": temporal_rep is not None,
        "plan_strategy": plan_strategy,
    }

    overall = "fail" if issues else ("warn" if warnings else "pass")
    report["checks"]["effective_constraints"] = effective_constraints
    report["checks"]["step3_signals"] = {
        "timestamp_active": bool(timestamp_active),
        "entity_groups_detected": bool(entity_groups_detected),
        "n_high_risk_features": int(n_high_risk),
    }

    metrics = {
        "timestamp_active": int(timestamp_active),
        "entity_groups_detected": int(entity_groups_detected),
        "n_high_risk_features": int(n_high_risk),
        "strategy_override": strategy_override,
        "n_warnings": len(warnings),
    }

    out_path = resolve_artifact_path(ctx=ctx, stage="split_strategy", name="leakage_constraints", run_id=rid, tag=TAG42)
    finalized = finalize_module_report(
        report=report,
        overall_status=overall,
        issues=issues,
        warnings=warnings,
        metrics=metrics,
        notes=["Translates Step 3 signals into split strategy constraints. Override only applied when critical."],
        refs=[make_artifact_ref(ctx=ctx, stage="split_strategy", name="leakage_constraints", run_id=rid)],
        ctx=ctx,
    )
    write_json(out_path, finalized, tag=TAG42)
    lg.info(f"[{TAG42}] Constraints complete | status={overall} | timestamp_active={timestamp_active} | entity_groups={entity_groups_detected}")
    return finalized


In [ ]:
# --- 4.3 Split Builder ---
#
# What:
#   Generate a reproducible split assignment (train / val / test) using the
#   effective plan from 4.1 and constraints from 4.2.
#
# How:
#   - Load 4.1 plan artifact + 4.2 constraints artifact by run_id
#   - Call split_engine.build_holdout_split(df, ...) with merged params
#   - Save split_assignment.parquet to data/interim/ (row-level data)
#   - Persist JSON artifact with metadata + parquet path (SSOT naming)
#
# Why:
#   Reproducible split assignment is the foundation for leak-safe training.
#   Separating row-level data (parquet) from metadata (JSON) keeps artifacts
#   lightweight and audit-friendly.

# --- 0) Import + TAG ---
from pathlib import Path
from typing import Any, Dict, Optional

import pandas as pd

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint
from src.pipeline.data_split.split_engine import build_holdout_split
from src.utils.artifact_utils import resolve_artifact_path, write_json, read_json_if_exists
from src.utils.cfg_utils import get_cfg_dict, get_cfg_float, get_cfg_int, get_cfg_str, get_cfg_bool
from src.utils.dataset_utils import get_label_cfg
from src.utils.path_utils import resolve_dir
from src.utils.serialization_utils import safe_float

TAG43 = "SPLIT_BUILDER"


# --- 1) Public API ---
def run_split_builder(
    *,
    df: pd.DataFrame,
    ctx: Dict[str, Any],
    run_id: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Build a reproducible split assignment and persist it as parquet + JSON artifact.

    YAML SSOT
    ---------
    pipeline.split_strategy.split:
      type: "holdout"
      strategy: "stratified"
      test_size: 0.20
      val_size: 0.20
      random_state: 42
      group_col: null
      timestamp_col: null

    Returns
    -------
    dict
        ModuleReport v1 (Split Builder), also persisted as artifact.
    """
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG43, run_id=run_id)

    if df is None or not isinstance(df, pd.DataFrame):
        raise TypeError(f"[{TAG43}] df must be a pandas DataFrame")

    # --- Load 4.1 plan artifact ---
    plan_path = resolve_artifact_path(ctx=ctx, stage="split_strategy", name="split_plan", run_id=rid, tag=TAG43)
    plan_rep = read_json_if_exists(plan_path, lg=lg, tag=TAG43)
    plan_checks = plan_rep.get("checks", {}).get("split_plan", {}) if isinstance(plan_rep, dict) else {}

    # --- Load 4.2 constraints artifact (for potential overrides) ---
    constraints_path = resolve_artifact_path(ctx=ctx, stage="split_strategy", name="leakage_constraints", run_id=rid, tag=TAG43)
    constraints_rep = read_json_if_exists(constraints_path, lg=lg, tag=TAG43)
    effective = constraints_rep.get("checks", {}).get("effective_constraints", {}) if isinstance(constraints_rep, dict) else {}

    # --- Resolve effective params (constraints can override plan) ---
    label_col, pos_label, _ = get_label_cfg(ctx, TAG43)

    split_type = str(plan_checks.get("split_type") or "holdout").strip().lower()
    strategy = str(effective.get("strategy_override") or plan_checks.get("strategy") or "stratified").strip().lower()
    test_size = float(plan_checks.get("test_size") or 0.20)
    val_size = float(plan_checks.get("val_size") or 0.20)
    random_state = int(plan_checks.get("random_state") or 42)

    override_applied = (effective.get("strategy_override") is not None)
    override_reason = effective.get("override_reason")

    lg.info(
        f"[{TAG43}] Effective params: type={split_type} | strategy={strategy} "
        f"| test={test_size} | val={val_size} | seed={random_state} "
        f"| override={override_applied}"
    )

    report = build_module_report(
        ctx=ctx, run_id=rid,
        stage="split_strategy", step="4.3", name="split_builder",
        tag=TAG43, cfg_key="pipeline.split_strategy.split", enabled=True, df=df,
    )
    ensure_event_hints(report)

    if override_applied and override_reason:
        add_event_hint(report, code="SPLIT_STRATEGY_OVERRIDE",
                       severity_signal="info",
                       evidence_path="checks.split_assignment.strategy_used",
                       context={"override": strategy, "reason": override_reason},
                       remediation="This is expected when Step 3 detected timestamp or entity constraints.")

    issues = []
    warnings = []

    # --- Execute split ---
    if split_type == "holdout":
        try:
            split_df, engine_report = build_holdout_split(
                df,
                label_col=label_col,
                positive_label=pos_label,
                strategy=strategy,
                test_size=test_size,
                val_size=val_size,
                random_state=random_state,
            )
        except Exception as exc:
            issues.append(f"Split engine failed: {exc}")
            add_event_hint(report, code="SPLIT_ENGINE_FAILURE", severity_signal="hard",
                           evidence_path="checks.split_assignment",
                           context={"error": str(exc)},
                           remediation="Check split parameters (test_size, val_size, random_state) in data_split.yaml.")
            out_path = resolve_artifact_path(ctx=ctx, stage="split_strategy", name="split_builder", run_id=rid, tag=TAG43)
            finalized = finalize_module_report(
                report=report, overall_status="fail",
                issues=issues, warnings=warnings, metrics={}, notes=[],
                refs=[make_artifact_ref(ctx=ctx, stage="split_strategy", name="split_builder", run_id=rid)],
                ctx=ctx,
            )
            write_json(out_path, finalized, tag=TAG43)
            return finalized
    else:
        issues.append(f"split_type '{split_type}' not yet implemented (only 'holdout' supported in v1)")
        out_path = resolve_artifact_path(ctx=ctx, stage="split_strategy", name="split_builder", run_id=rid, tag=TAG43)
        finalized = finalize_module_report(
            report=report, overall_status="fail",
            issues=issues, warnings=warnings, metrics={}, notes=[],
            refs=[make_artifact_ref(ctx=ctx, stage="split_strategy", name="split_builder", run_id=rid)],
            ctx=ctx,
        )
        write_json(out_path, finalized, tag=TAG43)
        return finalized

    # --- Save parquet (row-level) ---
    interim_dir = resolve_dir(ctx, "data.interim", tag=TAG43)
    interim_dir.mkdir(parents=True, exist_ok=True)
    parquet_path = interim_dir / f"split_assignment_{rid}.parquet"
    split_df.to_parquet(parquet_path, index=True)
    lg.info(f"[{TAG43}] Saved split_assignment.parquet → {parquet_path}")

    # Warn if fallback was used
    if engine_report.get("strategy_used") in {"random_fallback", "random_fallback_val"}:
        fallback_reason = engine_report.get("fallback_reason", "unknown")
        warnings.append(f"Stratification fallback: {fallback_reason}")
        add_event_hint(report, code="SPLIT_STRATIFY_FALLBACK", severity_signal="risk",
                       evidence_path="checks.split_assignment.engine_report.strategy_used",
                       context={"strategy_used": engine_report.get("strategy_used"), "reason": fallback_reason},
                       remediation="Stratification failed (likely rare class). Check label distribution.")

    # Assemble checks
    report["checks"]["split_assignment"] = {
        "parquet_path": str(parquet_path),
        "engine_report": engine_report,
        "override_applied": bool(override_applied),
        "override_reason": override_reason,
    }

    metrics = {
        "n_train": engine_report.get("n_train"),
        "n_val": engine_report.get("n_val"),
        "n_test": engine_report.get("n_test"),
        "pct_train": safe_float(engine_report.get("pct_train")),
        "pct_val": safe_float(engine_report.get("pct_val")),
        "pct_test": safe_float(engine_report.get("pct_test")),
        "positive_rate_overall": safe_float(engine_report.get("positive_rate_overall")),
        "positive_rate_train": safe_float(engine_report.get("positive_rate_train")),
        "positive_rate_val": safe_float(engine_report.get("positive_rate_val")),
        "positive_rate_test": safe_float(engine_report.get("positive_rate_test")),
    }

    overall = "fail" if issues else ("warn" if warnings else "pass")
    out_path = resolve_artifact_path(ctx=ctx, stage="split_strategy", name="split_builder", run_id=rid, tag=TAG43)
    finalized = finalize_module_report(
        report=report,
        overall_status=overall,
        issues=issues,
        warnings=warnings,
        metrics=metrics,
        notes=[f"Split assignment saved to: {parquet_path.name}"],
        refs=[make_artifact_ref(ctx=ctx, stage="split_strategy", name="split_builder", run_id=rid)],
        ctx=ctx,
    )
    write_json(out_path, finalized, tag=TAG43)
    lg.info(f"[{TAG43}] Split builder complete | status={overall} | {engine_report.get('n_train')}/{engine_report.get('n_val')}/{engine_report.get('n_test')} (train/val/test)")
    return finalized


In [ ]:
# --- 4.4 Split Diagnostics ---
#
# What:
#   Validate split quality: label balance, group overlap, time inversion, KS drift.
#
# How:
#   - Load split_assignment.parquet path from 4.3 artifact
#   - Compute four checks: label_balance / group_overlap / time_inversion / drift_ks
#   - KS test (scipy.stats.ks_2samp) on top numeric features (train vs val)
#   - Emit event_hints for hard violations and soft warnings
#   - Persist ModuleReport v1 artifact (SSOT naming)
#
# Why:
#   Prove the split is trustworthy before modeling begins.

# --- 0) Imports + TAG ---
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy import stats as scipy_stats

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint
from src.utils.artifact_utils import resolve_artifact_path, write_json, read_json_if_exists
from src.utils.cfg_utils import get_cfg_dict, get_cfg_float, get_cfg_int, get_cfg_bool
from src.utils.dataset_utils import get_label_cfg, get_schema_cols
from src.utils.serialization_utils import safe_float

TAG44 = "SPLIT_DIAGNOSTICS"


# --- Internal helpers ---
def _positive_rate_for_split(
    df: pd.DataFrame,
    split_df: pd.DataFrame,
    split_val: str,
    label_col: str,
    positive_label: str,
) -> Dict[str, Any]:
    """Compute positive rate in a named split partition."""
    idx = split_df.index[split_df["split"] == split_val]
    sub = df.loc[df.index.isin(idx), label_col]
    n = int(sub.notna().sum())
    if n == 0:
        return {"n": 0, "positive_rate": None}
    pos_rate = float((sub.astype(str).str.strip() == str(positive_label).strip()).sum() / n)
    return {"n": n, "positive_rate": round(pos_rate, 4)}


def _check_label_balance(
    df: pd.DataFrame,
    split_df: pd.DataFrame,
    *,
    label_col: str,
    positive_label: str,
    max_imbalance_pct: float,
) -> Dict[str, Any]:
    """Check label balance across train/val/test splits."""
    splits = ["train", "val", "test"]
    results: Dict[str, Any] = {}

    all_valid = df.loc[df.index.isin(split_df.index[split_df["split"].isin(splits)]), label_col]
    n_all = int(all_valid.notna().sum())
    overall_rate = (
        float((all_valid.astype(str).str.strip() == str(positive_label).strip()).sum() / n_all)
        if n_all > 0 else 0.0
    )

    warnings = []
    for sp in splits:
        stats = _positive_rate_for_split(df, split_df, sp, label_col, positive_label)
        results[sp] = stats
        rate = stats.get("positive_rate")
        if rate is not None and abs(rate - overall_rate) > max_imbalance_pct:
            warnings.append(f"{sp}: positive_rate={rate:.3f} deviates >{max_imbalance_pct:.0%} from overall {overall_rate:.3f}")

    return {
        "overall_positive_rate": round(overall_rate, 4),
        "by_split": results,
        "max_imbalance_pct": float(max_imbalance_pct),
        "warnings": warnings,
        "ok": len(warnings) == 0,
    }


def _check_group_overlap(
    split_df: pd.DataFrame,
    df: pd.DataFrame,
    *,
    group_col: Optional[str],
) -> Dict[str, Any]:
    """Check that no group appears in more than one split (only for group-based splits)."""
    if not group_col or group_col not in df.columns:
        return {"applicable": False, "reason": "group_col not configured or not in df"}

    splits = ["train", "val", "test"]
    groups_by_split: Dict[str, set] = {}
    for sp in splits:
        idx = split_df.index[split_df["split"] == sp]
        grps = set(df.loc[df.index.isin(idx), group_col].dropna().astype(str).unique())
        groups_by_split[sp] = grps

    overlaps = []
    split_list = list(groups_by_split.keys())
    for i in range(len(split_list)):
        for j in range(i + 1, len(split_list)):
            s1, s2 = split_list[i], split_list[j]
            shared = groups_by_split[s1] & groups_by_split[s2]
            if shared:
                overlaps.append({
                    "splits": [s1, s2],
                    "overlap_count": len(shared),
                    "examples": sorted(list(shared))[:5],
                })

    total_overlap = sum(o["overlap_count"] for o in overlaps)
    return {
        "applicable": True,
        "group_col": str(group_col),
        "overlap_count": int(total_overlap),
        "overlapping_pairs": overlaps,
        "ok": total_overlap == 0,
    }


def _check_time_inversion(
    split_df: pd.DataFrame,
    df: pd.DataFrame,
    *,
    timestamp_col: Optional[str],
) -> Dict[str, Any]:
    """Check that train timestamps < val timestamps < test timestamps."""
    if not timestamp_col or timestamp_col not in df.columns:
        return {"applicable": False, "reason": "timestamp_col not configured or not in df"}

    splits = ["train", "val", "test"]
    ts_by_split: Dict[str, pd.Series] = {}
    for sp in splits:
        idx = split_df.index[split_df["split"] == sp]
        ts = pd.to_datetime(df.loc[df.index.isin(idx), timestamp_col], errors="coerce").dropna()
        ts_by_split[sp] = ts

    inversions = []
    ordered_pairs = [("train", "val"), ("val", "test"), ("train", "test")]
    for earlier, later in ordered_pairs:
        ts_e = ts_by_split.get(earlier, pd.Series(dtype="datetime64[ns]"))
        ts_l = ts_by_split.get(later, pd.Series(dtype="datetime64[ns]"))
        if ts_e.empty or ts_l.empty:
            continue
        max_earlier = ts_e.max()
        min_later = ts_l.min()
        if max_earlier >= min_later:
            inversions.append({
                "pair": [earlier, later],
                "max_earlier": str(max_earlier),
                "min_later": str(min_later),
            })

    return {
        "applicable": True,
        "timestamp_col": str(timestamp_col),
        "inversion_count": len(inversions),
        "inversions": inversions,
        "ok": len(inversions) == 0,
    }


def _check_drift_ks(
    df: pd.DataFrame,
    split_df: pd.DataFrame,
    *,
    numeric_cols: List[str],
    top_k: int,
    ks_warn_threshold: float,
    p_value_threshold: float,
) -> Dict[str, Any]:
    """KS test between train and val for each numeric feature."""
    if not numeric_cols:
        return {"applicable": False, "reason": "no numeric columns configured"}

    idx_train = split_df.index[split_df["split"] == "train"]
    idx_val = split_df.index[split_df["split"] == "val"]

    if len(idx_train) == 0 or len(idx_val) == 0:
        return {"applicable": False, "reason": "train or val split is empty"}

    results = []
    for col in numeric_cols:
        if col not in df.columns:
            continue
        x_train = pd.to_numeric(df.loc[df.index.isin(idx_train), col], errors="coerce").dropna().values
        x_val = pd.to_numeric(df.loc[df.index.isin(idx_val), col], errors="coerce").dropna().values
        if len(x_train) < 2 or len(x_val) < 2:
            continue
        ks_stat, p_val = scipy_stats.ks_2samp(x_train, x_val)
        warned = bool(ks_stat > ks_warn_threshold or p_val < p_value_threshold)
        results.append({
            "feature": col,
            "ks_statistic": float(round(ks_stat, 4)),
            "p_value": float(round(p_val, 4)),
            "warned": warned,
        })

    results.sort(key=lambda x: x["ks_statistic"], reverse=True)
    top_results = results[:top_k]
    n_warned = int(sum(1 for r in results if r["warned"]))
    max_ks = float(results[0]["ks_statistic"]) if results else None
    top_feature = results[0]["feature"] if results else None

    return {
        "applicable": True,
        "method": "ks_2samp",
        "n_features_tested": int(len(results)),
        "n_warned": int(n_warned),
        "max_ks_statistic": max_ks,
        "top_feature": top_feature,
        "ks_warn_threshold": float(ks_warn_threshold),
        "p_value_threshold": float(p_value_threshold),
        "top_k_results": top_results,
    }


# --- 1) Public API ---
def run_split_diagnostics(
    *,
    df: pd.DataFrame,
    ctx: Dict[str, Any],
    run_id: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Validate split quality via four checks.

    YAML SSOT
    ---------
    pipeline.split_strategy.diagnostics:
      max_label_imbalance_pct: 0.10
      max_group_overlap: 0
      max_time_inversion: 0
      drift:
        top_k: 5
        ks_warn_threshold: 0.10
        p_value_threshold: 0.05
      min_train_size: 500
      min_val_size: 100
      min_test_size: 100

    Returns
    -------
    dict
        ModuleReport v1 (Split Diagnostics), also persisted as artifact.
    """
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG44, run_id=run_id)

    if df is None or not isinstance(df, pd.DataFrame):
        raise TypeError(f"[{TAG44}] df must be a pandas DataFrame")

    # --- Config ---
    label_col, pos_label, _ = get_label_cfg(ctx, TAG44)
    numeric_cols, _ = get_schema_cols(ctx, TAG44)

    max_imbalance_pct = get_cfg_float(
        ctx, "pipeline.split_strategy.diagnostics.max_label_imbalance_pct",
        tag=TAG44, default=0.10, required=False, min_value=0.0, max_value=1.0,
    )
    top_k = get_cfg_int(
        ctx, "pipeline.split_strategy.diagnostics.drift.top_k",
        tag=TAG44, default=5, required=False, min_value=1, max_value=50,
    )
    ks_warn_threshold = get_cfg_float(
        ctx, "pipeline.split_strategy.diagnostics.drift.ks_warn_threshold",
        tag=TAG44, default=0.10, required=False, min_value=0.0, max_value=1.0,
    )
    p_value_threshold = get_cfg_float(
        ctx, "pipeline.split_strategy.diagnostics.drift.p_value_threshold",
        tag=TAG44, default=0.05, required=False, min_value=0.0, max_value=1.0,
    )
    min_train_size = get_cfg_int(
        ctx, "pipeline.split_strategy.diagnostics.min_train_size",
        tag=TAG44, default=500, required=False, min_value=1,
    )
    min_val_size = get_cfg_int(
        ctx, "pipeline.split_strategy.diagnostics.min_val_size",
        tag=TAG44, default=100, required=False, min_value=1,
    )
    min_test_size = get_cfg_int(
        ctx, "pipeline.split_strategy.diagnostics.min_test_size",
        tag=TAG44, default=100, required=False, min_value=1,
    )

    # Load split_plan config (for group_col / timestamp_col)
    split_plan_path = resolve_artifact_path(
        ctx=ctx, stage="split_strategy", name="split_plan", run_id=rid, tag=TAG44
    )
    split_plan_rep = read_json_if_exists(split_plan_path, lg=lg, tag=TAG44)
    plan_checks = split_plan_rep.get("checks", {}).get("split_plan", {}) if isinstance(split_plan_rep, dict) else {}
    group_col = plan_checks.get("group_col")
    timestamp_col = plan_checks.get("timestamp_col")

    # Load split_assignment.parquet from 4.3 artifact
    builder_path = resolve_artifact_path(
        ctx=ctx, stage="split_strategy", name="split_builder", run_id=rid, tag=TAG44
    )
    builder_rep = read_json_if_exists(builder_path, lg=lg, tag=TAG44)
    parquet_path_str = None
    if isinstance(builder_rep, dict):
        parquet_path_str = builder_rep.get("checks", {}).get("split_assignment", {}).get("parquet_path")

    if not parquet_path_str:
        raise FileNotFoundError(f"[{TAG44}] Cannot find parquet_path in 4.3 builder artifact. Run 4.3 first.")

    from pathlib import Path
    parquet_path = Path(str(parquet_path_str))
    if not parquet_path.exists():
        raise FileNotFoundError(f"[{TAG44}] split_assignment.parquet not found at: {parquet_path}")

    split_df = pd.read_parquet(parquet_path)
    lg.info(f"[{TAG44}] Loaded split_assignment: {split_df.shape} | splits={split_df['split'].value_counts().to_dict()}")

    report = build_module_report(
        ctx=ctx, run_id=rid,
        stage="split_strategy", step="4.4", name="split_diagnostics",
        tag=TAG44, cfg_key="pipeline.split_strategy.diagnostics", enabled=True, df=df,
    )
    ensure_event_hints(report)

    issues: List[str] = []
    warnings: List[str] = []

    # --- Check 1: Split sizes ---
    n_train = int((split_df["split"] == "train").sum())
    n_val = int((split_df["split"] == "val").sum())
    n_test = int((split_df["split"] == "test").sum())
    size_checks = {"n_train": n_train, "n_val": n_val, "n_test": n_test}
    if n_train < min_train_size:
        warnings.append(f"train size {n_train} < min_train_size {min_train_size}")
        add_event_hint(report, code="SPLIT_TRAIN_TOO_SMALL", severity_signal="risk",
                       evidence_path="checks.diagnostics.size_checks.n_train",
                       context={"n_train": n_train, "min_required": min_train_size},
                       remediation="Increase dataset size or reduce test/val fractions.")
    if n_val < min_val_size:
        warnings.append(f"val size {n_val} < min_val_size {min_val_size}")
    if n_test < min_test_size:
        warnings.append(f"test size {n_test} < min_test_size {min_test_size}")

    # --- Check 2: Label balance ---
    lb_result = _check_label_balance(
        df, split_df,
        label_col=label_col, positive_label=pos_label,
        max_imbalance_pct=max_imbalance_pct,
    )
    for w in lb_result.get("warnings", []):
        warnings.append(f"label_balance: {w}")
        add_event_hint(report, code="SPLIT_LABEL_IMBALANCE", severity_signal="risk",
                       evidence_path="checks.diagnostics.label_balance",
                       context={"warning": w},
                       remediation="Consider stratified split or resampling.")

    # --- Check 3: Group overlap ---
    group_result = _check_group_overlap(split_df, df, group_col=group_col)
    if group_result.get("applicable") and not group_result.get("ok"):
        n_overlap = group_result.get("overlap_count", 0)
        issues.append(f"Group overlap detected: {n_overlap} groups in multiple splits")
        add_event_hint(report, code="SPLIT_GROUP_OVERLAP", severity_signal="hard",
                       evidence_path="checks.diagnostics.group_overlap",
                       context={"overlap_count": n_overlap},
                       remediation="Use GroupKFold or GroupShuffleSplit for group-based data.")

    # --- Check 4: Time inversion ---
    time_result = _check_time_inversion(split_df, df, timestamp_col=timestamp_col)
    if time_result.get("applicable") and not time_result.get("ok"):
        n_inv = time_result.get("inversion_count", 0)
        issues.append(f"Time inversion detected: {n_inv} split pairs out of order")
        add_event_hint(report, code="SPLIT_TIME_INVERSION", severity_signal="hard",
                       evidence_path="checks.diagnostics.time_inversion",
                       context={"inversion_count": n_inv},
                       remediation="Use time-based split (TimeSeriesSplit) to prevent future leakage.")

    # --- Check 5: KS drift ---
    drift_result = _check_drift_ks(
        df, split_df,
        numeric_cols=numeric_cols, top_k=top_k,
        ks_warn_threshold=ks_warn_threshold, p_value_threshold=p_value_threshold,
    )
    if drift_result.get("applicable") and drift_result.get("n_warned", 0) > 0:
        n_warned = drift_result["n_warned"]
        warnings.append(f"KS drift: {n_warned} feature(s) exceeded threshold (max KS={drift_result.get('max_ks_statistic'):.3f})")
        add_event_hint(report, code="SPLIT_DRIFT_DETECTED", severity_signal="risk",
                       evidence_path="checks.diagnostics.drift_ks",
                       context={"n_warned": n_warned, "max_ks": drift_result.get("max_ks_statistic"),
                                "top_feature": drift_result.get("top_feature")},
                       remediation="Distribution shift between train and val. Check data shuffling or consider stratification.")

    # Determine status
    if issues:
        overall = "fail"
    elif warnings:
        overall = "warn"
    else:
        overall = "pass"

    report["checks"]["diagnostics"] = {
        "size_checks": size_checks,
        "label_balance": lb_result,
        "group_overlap": group_result,
        "time_inversion": time_result,
        "drift_ks": drift_result,
    }

    metrics = {
        "n_train": n_train, "n_val": n_val, "n_test": n_test,
        "label_balance_ok": lb_result.get("ok"),
        "group_overlap_count": group_result.get("overlap_count") if group_result.get("applicable") else None,
        "time_inversion_count": time_result.get("inversion_count") if time_result.get("applicable") else None,
        "drift_n_warned": drift_result.get("n_warned") if drift_result.get("applicable") else None,
        "drift_max_ks": safe_float(drift_result.get("max_ks_statistic")),
    }

    out_path = resolve_artifact_path(ctx=ctx, stage="split_strategy", name="split_diagnostics", run_id=rid, tag=TAG44)
    finalized = finalize_module_report(
        report=report,
        overall_status=overall,
        issues=issues,
        warnings=warnings,
        metrics=metrics,
        notes=["KS drift: train vs val. Group overlap and time inversion are hard-fail checks."],
        refs=[make_artifact_ref(ctx=ctx, stage="split_strategy", name="split_diagnostics", run_id=rid)],
        ctx=ctx,
    )
    write_json(out_path, finalized, tag=TAG44)
    lg.info(f"[{TAG44}] Diagnostics complete | status={overall} | artifact={out_path.name}")
    return finalized


In [ ]:
# --- 4.5 Split Summary ---
#
# What:
#   Aggregate 4.1–4.4 artifacts into a compact, run-level split strategy summary.
#
# How:
#   - Load artifacts via SSOT naming, fallback to latest run_id if missing
#   - Extract: chosen split type + reason, constraints enforced,
#     diagnostics highlights (label balance, overlap, drift), refs
#   - Persist one JSON ModuleReport v1 under artifacts (SSOT naming)
#
# Why:
#   One-page tech summary for audit and interview demos — mirrors 2.8 and 3.5 patterns.

# --- 0) Imports + TAG ---
from typing import Any, Dict, List, Optional

import pandas as pd

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint
from src.pipeline.summary_utils import (
    get_overall_status,
    get_issues,
    get_warnings,
    get_metrics,
    get_total_rows,
    norm_status,
    make_upstream_block,
    dedupe_sorted,
    dedupe_event_hints,
    collect_upstream_event_hints,
    load_artifact_json_with_runid_fallback,
)
from src.utils.artifact_utils import resolve_artifact_path, write_json
from src.utils.cfg_utils import get_cfg_int, get_cfg_bool

TAG45 = "SPLIT_SUMMARY"

_UPSTREAM_STEPS = [
    ("split_strategy", "split_plan",          "4.1 Split Plan"),
    ("split_strategy", "leakage_constraints", "4.2 Leakage Constraints"),
    ("split_strategy", "split_builder",       "4.3 Split Builder"),
    ("split_strategy", "split_diagnostics",   "4.4 Split Diagnostics"),
]


# --- Internal helpers ---
def _extract_split_plan_summary(rep: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    """Extract key fields from 4.1 split_plan report."""
    if not isinstance(rep, dict):
        return {"available": False}
    checks = rep.get("checks", {}) if isinstance(rep.get("checks"), dict) else {}
    plan = checks.get("split_plan", {}) if isinstance(checks.get("split_plan"), dict) else {}
    return {
        "available": True,
        "split_type": plan.get("split_type"),
        "strategy": plan.get("strategy"),
        "test_size": plan.get("test_size"),
        "val_size": plan.get("val_size"),
        "random_state": plan.get("random_state"),
        "validation_errors": plan.get("validation_errors", []),
    }


def _extract_constraints_summary(rep: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    """Extract key fields from 4.2 leakage_constraints report."""
    if not isinstance(rep, dict):
        return {"available": False}
    checks = rep.get("checks", {}) if isinstance(rep.get("checks"), dict) else {}
    c = checks.get("effective_constraints", {}) if isinstance(checks.get("effective_constraints"), dict) else {}
    return {
        "available": True,
        "strategy_override": c.get("strategy_override"),
        "override_reason": c.get("override_reason"),
        "timestamp_active": c.get("timestamp_active"),
        "entity_groups_detected": c.get("entity_groups_detected"),
        "n_high_risk_features": c.get("n_high_risk_features"),
    }


def _extract_builder_summary(rep: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    """Extract key fields from 4.3 split_builder report."""
    if not isinstance(rep, dict):
        return {"available": False}
    checks = rep.get("checks", {}) if isinstance(rep.get("checks"), dict) else {}
    sa = checks.get("split_assignment", {}) if isinstance(checks.get("split_assignment"), dict) else {}
    eng = sa.get("engine_report", {}) if isinstance(sa.get("engine_report"), dict) else {}
    return {
        "available": True,
        "n_train": eng.get("n_train"),
        "n_val": eng.get("n_val"),
        "n_test": eng.get("n_test"),
        "pct_train": eng.get("pct_train"),
        "pct_val": eng.get("pct_val"),
        "pct_test": eng.get("pct_test"),
        "strategy_used": eng.get("strategy_used"),
        "fallback_reason": eng.get("fallback_reason"),
        "parquet_path": sa.get("parquet_path"),
    }


def _extract_diagnostics_summary(rep: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    """Extract key fields from 4.4 split_diagnostics report."""
    if not isinstance(rep, dict):
        return {"available": False}
    checks = rep.get("checks", {}) if isinstance(rep.get("checks"), dict) else {}
    diag = checks.get("diagnostics", {}) if isinstance(checks.get("diagnostics"), dict) else {}
    return {
        "available": True,
        "label_balance_ok": diag.get("label_balance", {}).get("ok") if isinstance(diag.get("label_balance"), dict) else None,
        "group_overlap": diag.get("group_overlap", {}).get("overlap_count") if isinstance(diag.get("group_overlap"), dict) else None,
        "time_inversion": diag.get("time_inversion", {}).get("inversion_count") if isinstance(diag.get("time_inversion"), dict) else None,
        "drift_top_feature": diag.get("drift_ks", {}).get("top_feature") if isinstance(diag.get("drift_ks"), dict) else None,
        "drift_max_ks": diag.get("drift_ks", {}).get("max_ks_statistic") if isinstance(diag.get("drift_ks"), dict) else None,
        "n_drift_warned": diag.get("drift_ks", {}).get("n_warned", 0) if isinstance(diag.get("drift_ks"), dict) else None,
    }


# --- 1) Public API ---
def build_split_summary(
    *,
    ctx: Dict[str, Any],
    run_id: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Aggregate 4.1–4.4 artifacts into a compact split strategy summary.

    YAML SSOT
    ---------
    pipeline.split_strategy.summary:
      max_examples: 20
      warn_on_missing_reports: true

    Returns
    -------
    dict
        ModuleReport v1 (Split Summary), also persisted as artifact.
    """
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG45, run_id=run_id)

    warn_on_missing = get_cfg_bool(
        ctx, "pipeline.split_strategy.summary.warn_on_missing_reports",
        tag=TAG45, default=True, required=False,
    )

    report = build_module_report(
        ctx=ctx, run_id=rid,
        stage="split_strategy", step="4.5", name="split_summary",
        tag=TAG45, cfg_key="pipeline.split_strategy.summary", enabled=True,
    )
    ensure_event_hints(report)

    lg.info(f"[{TAG45}] Building split summary | run_id={rid}")

    # Load upstream artifacts
    upstream_reports: Dict[str, Optional[Dict[str, Any]]] = {}
    for stage, name, label in _UPSTREAM_STEPS:
        rep, path, meta = load_artifact_json_with_runid_fallback(
            ctx=ctx, stage=stage, name=name, run_id=rid, tag=TAG45
        )
        upstream_reports[name] = rep
        if rep is None:
            msg = f"Upstream artifact missing: {label} ({stage}_{name}_{rid}.json)"
            if warn_on_missing:
                add_event_hint(report, code="SPLIT_SUMMARY_MISSING_UPSTREAM",
                               severity_signal="risk", evidence_path=f"artifacts.{name}",
                               context={"label": label}, remediation="Re-run the missing step.")
            lg.warning(f"[{TAG45}] {msg}")

    # Extract summaries
    plan_sum = _extract_split_plan_summary(upstream_reports.get("split_plan"))
    constraints_sum = _extract_constraints_summary(upstream_reports.get("leakage_constraints"))
    builder_sum = _extract_builder_summary(upstream_reports.get("split_builder"))
    diag_sum = _extract_diagnostics_summary(upstream_reports.get("split_diagnostics"))

    # Upstream status block
    upstream_block = make_upstream_block(upstream_reports, _UPSTREAM_STEPS)

    # Collect and dedupe event hints
    all_hints = collect_upstream_event_hints(upstream_reports)
    all_hints = dedupe_event_hints(all_hints)

    if all_hints:
        report["checks"]["event_hints"]["hints"].extend(all_hints)

    # Issues / warnings
    issues = []
    warnings = []
    for _, name, label in _UPSTREAM_STEPS:
        rep = upstream_reports.get(name)
        if rep is None:
            issues.append(f"Missing upstream report: {label}")
        else:
            s = norm_status(get_overall_status(rep))
            if s == "fail":
                issues.extend([f"{label}: {i}" for i in get_issues(rep)])
            elif s == "warn":
                warnings.extend([f"{label}: {w}" for w in get_warnings(rep)])

    # Determine overall status
    upstream_statuses = [
        norm_status(get_overall_status(r)) for r in upstream_reports.values() if r is not None
    ]
    if any(s == "fail" for s in upstream_statuses) or issues:
        overall = "fail"
    elif any(s == "warn" for s in upstream_statuses) or warnings:
        overall = "warn"
    else:
        overall = "pass"

    # Assemble checks
    report["checks"]["upstream"] = upstream_block
    report["checks"]["split_plan_summary"] = plan_sum
    report["checks"]["constraints_summary"] = constraints_sum
    report["checks"]["builder_summary"] = builder_sum
    report["checks"]["diagnostics_summary"] = diag_sum

    # Metrics
    metrics = {
        "n_train": builder_sum.get("n_train"),
        "n_val": builder_sum.get("n_val"),
        "n_test": builder_sum.get("n_test"),
        "pct_train": builder_sum.get("pct_train"),
        "pct_val": builder_sum.get("pct_val"),
        "pct_test": builder_sum.get("pct_test"),
        "n_upstream_missing": sum(1 for r in upstream_reports.values() if r is None),
        "n_upstream_failed": sum(
            1 for r in upstream_reports.values()
            if r is not None and norm_status(get_overall_status(r)) == "fail"
        ),
    }

    out_path = resolve_artifact_path(ctx=ctx, stage="split_strategy", name="split_summary", run_id=rid, tag=TAG45)
    finalized = finalize_module_report(
        report=report,
        overall_status=overall,
        issues=dedupe_sorted(issues),
        warnings=dedupe_sorted(warnings),
        metrics=metrics,
        notes=["Aggregated from 4.1 split_plan, 4.2 leakage_constraints, 4.3 split_builder, 4.4 split_diagnostics."],
        refs=[make_artifact_ref(ctx=ctx, stage="split_strategy", name="split_summary", run_id=rid)],
        ctx=ctx,
    )
    write_json(out_path, finalized, tag=TAG45)
    lg.info(f"[{TAG45}] Summary complete | status={overall} | artifact={out_path.name}")
    return finalized


In [ ]:
# --- 4.x Split Strategy Orchestrator (SSOT-driven) ---
#
# What:
#   Run Step-4 "split_strategy" as one controller:
#     - Build the step plan from YAML SSOT (enabled + steps order + gates)
#     - Dispatch each module by contract (agg / df)
#     - Apply centralized gating (hard/soft + strict) and optionally halt
#     - Persist orchestrator ModuleReport v1 + per-step artifacts (safety net)
#
# How:
#   - ensure_runtime_ctx() -> ctx/logger/run_id
#   - Read SSOT: pipeline.orchestrator.stages.split_strategy.{enabled,steps}
#   - build_modules_from_ssot(): steps[] -> List[ModuleSpec] (fn resolved from registry)
#   - For each ModuleSpec in SSOT order:
#       invoke_module() -> raw result
#       normalize_module_result() -> {df, report}
#       on missing_impl/exception: build_*_report() + persist_artifact_safety_net()
#       infer status -> should_halt(strict, gate, status)
#   - finalize_module_report() + persist orchestrator artifact
#
# Why:
#   - SSOT defines ordering/gates (no hard-coded orchestration)
#   - Stable contracts let modules evolve independently
#   - Central gating + artifacts make runs reproducible and auditable

# --- 0) Import + TAG ---
from typing import Any, Dict, List, Optional

import pandas as pd

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint
from src.pipeline.orchestrator_utils import get_stage_cfg
from src.pipeline.orchestrator_common import (
    ModuleFn,
    ModuleSpec,
    validate_module_spec,
    infer_module_status,
    should_halt,
    build_missing_impl_report,
    build_exception_report,
    persist_artifact_safety_net,
    invoke_module,
    normalize_module_result,
    has_missing_module_report,
)
from src.utils.artifact_utils import resolve_artifact_path, write_json
from src.utils.cfg_utils import get_cfg_str, get_cfg_value

TAG4X = "SS_ORCH"


# --- Internal helpers ---
def _artifact_key_4x(spec: ModuleSpec) -> str:
    k = spec.artifact_name or spec.name or spec.step_id
    k = str(k).strip()
    return k if k else str(spec.step_id).strip()


def _build_modules_from_ssot_4x(
    *,
    ctx: Dict[str, Any],
    tag: str,
    registry: Dict[str, ModuleFn],
) -> List[ModuleSpec]:
    stage_cfg = get_stage_cfg(ctx, "split_strategy", tag=tag)
    stage_enabled = bool(stage_cfg.get("enabled", True))

    steps = stage_cfg.get("steps") or []
    if not isinstance(steps, list):
        raise TypeError(f"[{tag}] stage_cfg.steps must be a list")

    default_gate = get_cfg_str(
        ctx,
        "pipeline.gate_policy.default",
        tag=tag,
        default="soft",
        required=False,
    )

    specs: List[ModuleSpec] = []
    for step_cfg in steps:
        if not isinstance(step_cfg, dict):
            continue

        step_id = str(step_cfg.get("id", "")).strip()
        cfg_key = str(step_cfg.get("cfg_key", "")).strip()
        gate = str(step_cfg.get("gate", default_gate)).strip().lower()
        enabled = bool(step_cfg.get("enabled", True))
        fn = registry.get(step_id)

        # Determine kind and returns_df from registry metadata or step config
        kind_map = {
            "split_plan":          ("agg", False),
            "leakage_constraints": ("agg", False),
            "split_builder":       ("df",  False),
            "split_diagnostics":   ("df",  False),
            "split_summary":       ("agg", False),
        }
        kind, returns_df = kind_map.get(step_id, ("agg", False))

        spec = ModuleSpec(
            step_id=step_id,
            name=step_id,
            fn=fn,
            cfg_key=cfg_key,
            artifact_stage="split_strategy",
            artifact_name=step_id,
            kind=kind,
            gate_default=gate,
            enabled=enabled and stage_enabled,
            returns_df=returns_df,
        )
        validate_module_spec(spec, tag=tag)
        specs.append(spec)

    return specs


# --- 1) Public API ---
def run_split_strategy_orchestrator(
    *,
    ctx: Dict[str, Any],
    df: Optional[pd.DataFrame] = None,
    run_id: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Run the Step-4 Split Strategy stage as one SSOT-driven controller.

    YAML SSOT
    ---------
    pipeline.orchestrator.stages.split_strategy:
      enabled: true
      steps:
        - id: "split_plan"         gate: "hard"
        - id: "leakage_constraints" gate: "soft"
        - id: "split_builder"      gate: "hard"
        - id: "split_diagnostics"  gate: "soft"
        - id: "split_summary"      gate: "soft"

    Parameters
    ----------
    ctx  : pipeline context (cfg + logger + run_id)
    df   : cleaned DataFrame from Step 2.9 (required by split_builder/diagnostics)
    run_id : optional override

    Returns
    -------
    dict
        Orchestrator ModuleReport v1 (split_strategy).
    """
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG4X, run_id=run_id)

    # --- Registry ---
    registry: Dict[str, ModuleFn] = {
        "split_plan":          resolve_split_plan,
        "leakage_constraints": run_leakage_constraints,
        "split_builder":       run_split_builder,
        "split_diagnostics":   run_split_diagnostics,
        "split_summary":       build_split_summary,
    }

    specs = _build_modules_from_ssot_4x(ctx=ctx, tag=TAG4X, registry=registry)

    strict = bool(ctx.get("cfg", {}).get("pipeline", {}).get("gate_policy", {}).get("strict", True))

    report = build_module_report(
        ctx=ctx,
        run_id=rid,
        stage="split_strategy",
        step="4.x",
        name="orchestrator",
        tag=TAG4X,
        cfg_key="pipeline.orchestrator.stages.split_strategy",
        enabled=True,
        df=df,
    )
    ensure_event_hints(report)

    artifacts: Dict[str, Any] = {}
    current_df = df
    halted = False
    halt_reason: Optional[str] = None

    for spec in specs:
        if not spec.enabled:
            lg.info(f"[{TAG4X}] Skipping disabled step: {spec.step_id}")
            continue

        lg.info(f"[{TAG4X}] Running step: {spec.step_id} | kind={spec.kind} | gate={spec.gate_default}")

        # agg modules don't need df
        invoke_df = current_df if spec.kind == "df" else None

        if spec.fn is None:
            raw = None
            step_report = build_missing_impl_report(
                ctx=ctx, run_id=rid, spec=spec, tag=TAG4X
            )
            persist_artifact_safety_net(
                ctx=ctx, run_id=rid, spec=spec, report=step_report, tag=TAG4X
            )
        else:
            try:
                raw = invoke_module(spec=spec, df=invoke_df, ctx=ctx, run_id=rid, tag=TAG4X)
                result = normalize_module_result(raw=raw, spec=spec, tag=TAG4X)
                step_report = result.get("report") or {}
                if spec.returns_df and result.get("df") is not None:
                    current_df = result["df"]
            except Exception as exc:
                lg.exception(f"[{TAG4X}] Step '{spec.step_id}' raised: {exc}")
                step_report = build_exception_report(
                    ctx=ctx, run_id=rid, spec=spec, exc=exc, tag=TAG4X
                )
                persist_artifact_safety_net(
                    ctx=ctx, run_id=rid, spec=spec, report=step_report, tag=TAG4X
                )

        ak = _artifact_key_4x(spec)
        artifacts[ak] = step_report

        status = infer_module_status(step_report, tag=TAG4X)
        lg.info(f"[{TAG4X}] Step '{spec.step_id}' → status={status}")

        if should_halt(strict=strict, gate=spec.gate_default, status=status):
            halt_reason = f"Step '{spec.step_id}' status={status} (gate={spec.gate_default}, strict={strict})"
            add_event_hint(
                report,
                code="SPLIT_ORCH_HALT",
                severity_signal="hard",
                evidence_path=f"artifacts.{ak}.summary.overall_status",
                context={"step_id": spec.step_id, "status": status, "gate": spec.gate_default},
                remediation="Inspect the failing step artifact and fix upstream data or config issues.",
            )
            halted = True
            lg.error(f"[{TAG4X}] Halting pipeline. Reason: {halt_reason}")
            break

    # Final status
    all_statuses = [
        infer_module_status(r, tag=TAG4X)
        for r in artifacts.values()
        if isinstance(r, dict)
    ]
    if halted:
        orch_status = "fail"
    elif any(s == "fail" for s in all_statuses):
        orch_status = "warn"
    elif any(s == "warn" for s in all_statuses):
        orch_status = "warn"
    else:
        orch_status = "pass"

    report["checks"]["orchestrator"] = {
        "steps_run": [s.step_id for s in specs if s.enabled],
        "halted": halted,
        "halt_reason": halt_reason,
        "step_statuses": {
            _artifact_key_4x(s): infer_module_status(artifacts.get(_artifact_key_4x(s)), tag=TAG4X)
            for s in specs if s.enabled
        },
    }
    report["checks"]["artifacts"] = artifacts

    out_path = resolve_artifact_path(ctx=ctx, stage="split_strategy", name="orchestrator", run_id=rid, tag=TAG4X)
    finalized = finalize_module_report(
        report=report,
        overall_status=orch_status,
        issues=[halt_reason] if halt_reason else [],
        warnings=[],
        metrics={"n_steps": len(specs), "halted": int(halted)},
        notes=[f"Orchestrated {len(specs)} split_strategy steps."],
        refs=[make_artifact_ref(ctx=ctx, stage="split_strategy", name="orchestrator", run_id=rid)],
        ctx=ctx,
    )
    write_json(out_path, finalized, tag=TAG4X)
    lg.info(f"[{TAG4X}] Orchestrator complete | status={orch_status} | artifact={out_path.name}")
    return finalized


# --- 2) Execute ---
if "ctx" in dir() and "df_clean" in dir():
    report_4x = run_split_strategy_orchestrator(ctx=ctx, df=df_clean, run_id=run_id)
    print(f"[4.x] status={report_4x.get('summary', {}).get('overall_status')} | halted={report_4x.get('checks', {}).get('orchestrator', {}).get('halted')}")


# 11 Governance, Lineage & Reproducibility (Control Plane)
🎯 **Goal:** Establish a single, auditable **control plane** for each pipeline run by capturing immutable run metadata, configuration + environment snapshots, and a complete artifact index (stable refs + hashes) — enabling every downstream consumer to reliably reconstruct **what data was used, what checks were executed, what outputs were produced**, and whether the run is **approved for consumption/release**, supporting debugging, governance, and long-term maintainability.

In [19]:
# --- 11.1 Governance Run Manifest & Reproducibility Snapshot ---
#
# What:
#   Build a run-level manifest that captures:
#   - Core run metadata (run_id, timestamps, paths).
#   - Dataset fingerprint (raw / clean shapes and lightweight signals).
#   - Index preview of JSON artifacts produced by the pipeline.
#   - Reproducibility anchors (e.g., setup/config_snapshot).
#
# How:
#   - Resolve YAML SSOT via ctx: pipeline.governance.manifest
#   - Scan outputs.artifacts for SSOT-named artifacts (bounded).
#   - Persist a ModuleReport v1 artifact under stage=governance, name=manifest.
#
# Why:
#   - A single control-plane entrypoint to understand what happened in a run.
#   - Aligns with "data governance" / audit / reproducibility story for interviews.

# --- 0) Import + TAG ---
from typing import Any, Dict, List, Optional, Tuple
from pathlib import Path
import hashlib
import heapq

import pandas as pd

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint

from src.utils.artifact_utils import resolve_artifact_path, write_json, read_json_if_exists
from src.utils.cfg_utils import (
    get_cfg_bool,
    get_cfg_int,
    get_cfg_str,
    get_cfg_str_list,
)
from src.utils.path_utils import resolve_dir
from src.utils.time_utils import utc_now_iso, utc_from_timestamp_iso

TAG111 = "MANIFEST"


# --- Internal helpers ---
_ALLOWED_SORT_BY = {"name", "mtime"}
_ALLOWED_HASH_ALGO = {"sha256", "md5"}


def _hash_file(path: Path, *, algo: str, block_size: int) -> str:
    """Compute file hash with a configurable algorithm."""
    algo_l = str(algo).strip().lower()
    if algo_l not in _ALLOWED_HASH_ALGO:
        raise ValueError(f"[{TAG111}] unsupported hash_algo={algo!r}. allowed={sorted(_ALLOWED_HASH_ALGO)}")

    h = hashlib.sha256() if algo_l == "sha256" else hashlib.md5()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(int(block_size)), b""):
            h.update(chunk)
    return h.hexdigest()


def _infer_step_hint_from_filename(name: str) -> str:
    """
    Infer a rough step hint from SSOT filenames:
      {stage}_{name}_{run_id}.json
    """
    lowered = str(name).lower()

    mapping = [
        # 1
        ("setup_config_snapshot_", "1_setup_config_snapshot"),
        # 2.x
        ("data_quality_ingestion_", "2.1_ingestion"),
        ("data_quality_contract_", "2.2_contract"),
        ("data_quality_health_", "2.3_health"),
        ("data_quality_semantic_", "2.4_semantic"),
        ("data_quality_readiness_", "2.5_readiness"),
        ("data_quality_domain_core_", "2.6.1_domain_core"),
        ("data_quality_domain_addon_", "2.6.2_domain_addon"),
        ("data_quality_distribution_", "2.7_distribution"),
        ("data_quality_pre_analysis_summary_", "2.8_pre_analysis_summary"),
        ("data_quality_cleaning_", "2.9_cleaning"),
        ("data_quality_orchestrator_", "2.x_dq_orchestrator"),
        # 3.x
        ("data_integrity_row_identity_", "3.1_row_identity"),
        ("data_integrity_temporal_", "3.2_temporal"),
        ("data_integrity_label_alignment_", "3.3_label_alignment"),
        ("data_integrity_leakage_scan_", "3.4_leakage_scan"),
        ("data_integrity_integrity_summary_", "3.5_integrity_summary"),
        ("data_integrity_orchestrator_", "3.x_di_orchestrator"),
        # 11.x (new governance stage)
        ("governance_manifest_", "11.1_manifest"),
        ("governance_lineage_audit_", "11.2_lineage_audit"),
        ("governance_inventory_", "11.3_inventory"),
        ("governance_release_gate_", "11.4_release_gate"),
        # Backward compat (older stage name)
        ("experiment_tracking_manifest_", "11.1_manifest_legacy"),
    ]

    for key, label in mapping:
        if key in lowered:
            return label

    return "unknown"


def _extract_clean_parquet_path_from_dq_cleaning_artifact(
    *,
    ctx: Dict[str, Any],
    run_id: str,
    lg: Any,
    tag: str,
) -> Optional[str]:
    """Read 2.9 cleaning artifact and return cleaned parquet path if parquet_ok."""
    rid = str(run_id).strip()
    if not rid:
        return None

    p = resolve_artifact_path(
        ctx=ctx,
        stage="data_quality",
        name="cleaning",
        run_id=rid,
        tag=tag,
    )

    rep = read_json_if_exists(p, lg=lg, tag=tag)
    if not isinstance(rep, dict):
        return None

    persistence = ((rep.get("checks") or {}).get("persistence") or {})
    if not isinstance(persistence, dict):
        return None

    if not bool(persistence.get("parquet_ok", False)):
        return None

    path = persistence.get("cleaned_parquet_path")
    if not isinstance(path, str) or not path.strip():
        return None

    return str(Path(path))


def _resolve_manifest_cfg(ctx: Dict[str, Any]) -> Dict[str, Any]:
    """
    Resolve canonical config for 11.1 from YAML SSOT:
      pipeline.governance.manifest
    """
    base = "pipeline.governance.manifest"

    # --- scan ---
    include_suffixes: List[str] = get_cfg_str_list(
        ctx,
        f"{base}.scan.include_suffixes",
        tag=TAG111,
        default=[".json"],
        allow_empty=False,
        dedupe=True,
    )

    max_artifacts = get_cfg_int(
        ctx,
        f"{base}.scan.max_artifacts",
        tag=TAG111,
        default=500,
        min_value=1,
    )

    filter_by_run_id = get_cfg_bool(
        ctx,
        f"{base}.scan.filter_by_run_id",
        tag=TAG111,
        default=True,
        strict_type=False,
    )

    recursive = get_cfg_bool(
        ctx,
        f"{base}.scan.recursive",
        tag=TAG111,
        default=False,
        strict_type=False,
    )

    sort_by = get_cfg_str(
        ctx,
        f"{base}.scan.sort_by",
        tag=TAG111,
        default="name",
        lower=True,
        allowed=_ALLOWED_SORT_BY,
        strict_type=True,
    )

    include_hash_preview = get_cfg_bool(
        ctx,
        f"{base}.scan.include_hash_preview",
        tag=TAG111,
        default=True,
        strict_type=False,
    )

    hash_algo = get_cfg_str(
        ctx,
        f"{base}.scan.hash_algo",
        tag=TAG111,
        default="sha256",
        lower=True,
        allowed=_ALLOWED_HASH_ALGO,
        strict_type=True,
    )

    hash_block_size = get_cfg_int(
        ctx,
        f"{base}.scan.hash_block_size",
        tag=TAG111,
        default=65536,
        min_value=1024,
    )

    # --- preview ---
    preview_max = get_cfg_int(
        ctx,
        f"{base}.preview.max_items",
        tag=TAG111,
        default=30,
        min_value=1,
    )

    # --- fingerprint ---
    sample_columns = get_cfg_int(
        ctx,
        f"{base}.fingerprint.sample_columns",
        tag=TAG111,
        default=5,
        min_value=1,
    )

    include_columns_hash = get_cfg_bool(
        ctx,
        f"{base}.fingerprint.include_columns_hash",
        tag=TAG111,
        default=True,
        strict_type=False,
    )

    include_shape = get_cfg_bool(
        ctx,
        f"{base}.fingerprint.include_shape",
        tag=TAG111,
        default=True,
        strict_type=False,
    )

    include_dtypes = get_cfg_bool(
        ctx,
        f"{base}.fingerprint.include_dtypes",
        tag=TAG111,
        default=True,  # align with your YAML
        strict_type=False,
    )

    include_null_rate = get_cfg_bool(
        ctx,
        f"{base}.fingerprint.include_null_rate",
        tag=TAG111,
        default=False,
        strict_type=False,
    )

    # --- refs ---
    include_setup = get_cfg_bool(
        ctx,
        f"{base}.refs.include_setup_config_snapshot",
        tag=TAG111,
        default=True,
        strict_type=False,
    )

    require_setup = get_cfg_bool(
        ctx,
        f"{base}.refs.require_setup_config_snapshot",
        tag=TAG111,
        default=False,
        strict_type=False,
    )

    setup_stage = get_cfg_str(
        ctx,
        f"{base}.refs.setup_config_snapshot.stage",
        tag=TAG111,
        default="setup",
        strict_type=True,
    )

    setup_name = get_cfg_str(
        ctx,
        f"{base}.refs.setup_config_snapshot.name",
        tag=TAG111,
        default="config_snapshot",
        strict_type=True,
    )

    return {
        "cfg_key": base,
        "scan": {
            "include_suffixes": include_suffixes,
            "max_artifacts": max_artifacts,
            "filter_by_run_id": filter_by_run_id,
            "recursive": recursive,
            "sort_by": sort_by,
            "include_hash_preview": bool(include_hash_preview),
            "hash_algo": hash_algo,
            "hash_block_size": hash_block_size,
        },
        "preview_max": preview_max,
        "fingerprint": {
            "sample_columns": sample_columns,
            "include_columns_hash": include_columns_hash,
            "include_shape": include_shape,
            "include_dtypes": include_dtypes,
            "include_null_rate": include_null_rate,
        },
        "refs": {
            "include_setup_config_snapshot": include_setup,
            "require_setup_config_snapshot": require_setup,
            "setup_config_snapshot": {"stage": setup_stage, "name": setup_name},
        },
    }


def _columns_sha256(cols: List[str]) -> str:
    """Hash column names for a stable lightweight fingerprint."""
    joined = "\n".join([str(c) for c in cols])
    return hashlib.sha256(joined.encode("utf-8")).hexdigest()


def _build_dataset_fingerprint(
    *,
    df: pd.DataFrame,
    cfg: Dict[str, Any],
) -> Dict[str, Any]:
    """Build a lightweight dataset fingerprint according to config."""
    fp_cfg = cfg["fingerprint"]
    k = int(fp_cfg["sample_columns"])

    cols = list(df.columns)
    out: Dict[str, Any] = {}

    if bool(fp_cfg["include_shape"]):
        out["n_rows"] = int(df.shape[0])
        out["n_cols"] = int(df.shape[1])

    out["sample_columns"] = cols[:k]

    if bool(fp_cfg["include_columns_hash"]):
        out["columns_sha256"] = _columns_sha256(cols)

    if bool(fp_cfg["include_dtypes"]):
        # Keep it lightweight: only for sample columns
        out["sample_dtypes"] = {c: str(df[c].dtype) for c in cols[:k]}
        out["dtypes_scope"] = "sampled"

    if bool(fp_cfg["include_null_rate"]):
        # Keep it lightweight: only for sample columns
        out["sample_null_rate"] = {c: float(df[c].isna().mean()) for c in cols[:k]}
        out["null_rate_scope"] = "sampled"

    return out


def _artifact_matches_run_id(p: Path, *, run_id: str) -> bool:
    """
    SSOT naming: {stage}_{name}_{run_id}.json
    We consider a match if the last '_' chunk of stem equals run_id.
    """
    stem = p.stem
    parts = stem.split("_")
    if not parts:
        return False
    return parts[-1] == str(run_id)


def _scan_artifacts_preview(
    *,
    artifact_dir: Path,
    cfg: Dict[str, Any],
    run_id: str,
    exclude_names: Optional[List[str]] = None,
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    """Scan artifacts dir and return a bounded preview + scan summary."""
    scan_cfg = cfg["scan"]

    include_suffixes = [str(s).lower() for s in (scan_cfg.get("include_suffixes") or [])]
    suffix_set = set(include_suffixes) if include_suffixes else {".json"}

    exclude_set = set([str(x).strip() for x in (exclude_names or []) if str(x).strip()])

    max_artifacts = int(scan_cfg.get("max_artifacts", 500))
    preview_max = int(cfg.get("preview_max", 30))
    sort_by = str(scan_cfg.get("sort_by", "name"))
    filter_by_run_id = bool(scan_cfg.get("filter_by_run_id", True))
    recursive = bool(scan_cfg.get("recursive", False))
    include_hash_preview = bool(scan_cfg.get("include_hash_preview", False))

    scanned = 0
    filtered_out = 0
    excluded = 0
    stopped_early = False
    stop_reason: Optional[str] = None

    if not artifact_dir.exists():
        return [], {
            "dir_exists": False,
            "recursive": recursive,
            "scanned": 0,
            "indexed": 0,
            "filtered_out": 0,
            "excluded": 0,
            "excluded_names": sorted(list(exclude_set)) if exclude_set else [],
            "include_suffixes": list(suffix_set),
            "max_artifacts": int(max_artifacts),
            "filter_by_run_id": bool(filter_by_run_id),
            "sort_by": sort_by,
            "include_hash_preview": bool(include_hash_preview),
            "hash_algo": str(scan_cfg.get("hash_algo")) if include_hash_preview else None,
            "hash_scope": "preview_only" if include_hash_preview else "disabled",
        }

    # --- iterate files (no big list) ---
    if recursive:
        it = (p for p in artifact_dir.rglob("*") if p.is_file())
    else:
        it = (p for p in artifact_dir.iterdir() if p.is_file())

    # candidates storage (bounded)
    # - for name: keep list up to max_artifacts
    # - for mtime: keep heap top-K by mtime
    candidates: List[Path] = []
    heap: List[Tuple[float, Path]] = []

    for p in it:
        scanned += 1

        if exclude_set and (p.name in exclude_set):
            filtered_out += 1
            excluded += 1
            continue

        if p.suffix.lower() not in suffix_set:
            filtered_out += 1
            continue

        if filter_by_run_id and (not _artifact_matches_run_id(p, run_id=run_id)):
            filtered_out += 1
            continue

        # keep bounded
        if sort_by == "mtime":
            try:
                mtime = float(p.stat().st_mtime)
            except Exception:
                mtime = 0.0

            if len(heap) < max_artifacts:
                heapq.heappush(heap, (mtime, p))
            else:
                # keep top-K newest
                if mtime > heap[0][0]:
                    heapq.heapreplace(heap, (mtime, p))
        else:
            # name mode: keep first max_artifacts matches (bounded)
            if len(candidates) < max_artifacts:
                candidates.append(p)
            else:
                # already have enough indexed items
                # (we still count scanned/filtered_out above, but stop early to be safe)
                stopped_early = True
                stop_reason = "max_artifacts_reached"
                break

    # finalize candidates ordering
    if sort_by == "mtime":
        # newest first
        candidates = [p for _, p in sorted(heap, key=lambda x: x[0], reverse=True)]
    else:
        # name ordering (within bounded set)
        candidates = sorted(candidates, key=lambda x: x.name)

    indexed = int(len(candidates))
    preview_paths = candidates[:preview_max]

    preview: List[Dict[str, Any]] = []
    for p in preview_paths:
        try:
            st = p.stat()
            item: Dict[str, Any] = {
                "name": p.name,
                "path": str(p),
                "size_bytes": int(st.st_size),
                "modified_at_utc": utc_from_timestamp_iso(st.st_mtime),
                "suffix": p.suffix,
                "step_hint": _infer_step_hint_from_filename(p.name),
            }
        except Exception:
            item = {
                "name": p.name,
                "path": str(p),
                "size_bytesL": None,
                "modified_at_utc": None,
                "suffix": p.suffix,
                "step_hint": _infer_step_hint_from_filename(p.name),
            }

        # IMPORTANT: hash only for preview items
        if include_hash_preview:
            item["hash_algo"] = str(scan_cfg.get("hash_algo", "sha256"))
            item["hash"] = _hash_file(
                p,
                algo=str(scan_cfg.get("hash_algo", "sha256")),
                block_size=int(scan_cfg.get("hash_block_size", 65536)),
            )

        preview.append(item)

    summary: Dict[str, Any] = {
        "dir_exists": True,
        "recursive": bool(recursive),
        "scanned": int(scanned),
        "indexed": int(indexed),
        "filtered_out": int(filtered_out),
        "excluded": int(excluded),
        "stopped_early": bool(stopped_early),
        "stop_reason": stop_reason,
        "excluded_names": sorted(list(exclude_set)) if exclude_set else [],
        "include_suffixes": list(suffix_set),
        "max_artifacts": int(max_artifacts),
        "filter_by_run_id": bool(filter_by_run_id),
        "sort_by": sort_by,
        "include_hash_preview": bool(include_hash_preview),
        "hash_algo": str(scan_cfg.get("hash_algo")) if include_hash_preview else None,
        "hash_scope": "preview_only" if include_hash_preview else "disabled",
    }

    return preview, summary



# --- 1) Public API ---
def build_run_manifest(
    *,
    ctx: Dict[str, Any],
    df_raw: Optional[pd.DataFrame] = None,
    df_clean: Optional[pd.DataFrame] = None,
    run_id: Optional[str] = None,
    extra_metadata: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """
    Step 11.1 - Build and persist the run-level manifest (ModuleReport v1).
    """
    # --- 2) Runtime validation ---
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG111, run_id=run_id)

    # --- 3) Input validation ---
    if df_raw is not None and (not isinstance(df_raw, pd.DataFrame)):
        raise TypeError(f"[{TAG111}] df_raw must be a pandas DataFrame when provided")
    if df_clean is not None and (not isinstance(df_clean, pd.DataFrame)):
        raise TypeError(f"[{TAG111}] df_clean must be a pandas DataFrame when provided")
    if extra_metadata is not None and (not isinstance(extra_metadata, dict)):
        raise TypeError(f"[{TAG111}] extra_metadata must be a dict when provided")

    # --- 4) Resolve config (YAML SSOT) ---
    cfg = _resolve_manifest_cfg(ctx)

    artifacts_dir = resolve_dir(ctx, "outputs.artifacts", required=True, tag=TAG111)

    # --- 5) Core workflows ---
    lg.info(f"[{TAG111}] 📦 Building governance manifest | run_id={rid}")

    # --- Build ModuleReport skeleton ---
    report = build_module_report(
        ctx=ctx,
        stage="governance",
        step="11.1",
        name="manifest",
        tag=TAG111,
        cfg_key=str(cfg["cfg_key"]),
        enabled=True,
        df=df_clean if isinstance(df_clean, pd.DataFrame) else None,
        run_id_override=str(rid),
    )

    report.setdefault("refs", {})
    report.setdefault("inputs", {})
    report.setdefault("checks", {})

    ensure_event_hints(report, version=1)

    report["inputs"].update(
        {
            "df_raw_provided": bool(isinstance(df_raw, pd.DataFrame)),
            "df_clean_provided": bool(isinstance(df_clean, pd.DataFrame)),
            "artifacts_dir": str(artifacts_dir),
        }
    )
    if isinstance(extra_metadata, dict) and extra_metadata:
        report["inputs"]["extra_metadata_keys"] = sorted([str(k) for k in extra_metadata.keys()])

    report["thresholds_used"] = {
        "scan": dict(cfg["scan"]),
        "preview_max": int(cfg["preview_max"]),
        "fingerprint": dict(cfg["fingerprint"]),
        "refs": dict(cfg["refs"]),
    }

    report["used_config"] = {
        "cfg_keys": [
            "pipeline.governance.manifest",
            "pipeline.governance.manifest.scan",
            "pipeline.governance.manifest.preview",
            "pipeline.governance.manifest.fingerprint",
            "pipeline.governance.manifest.refs",
        ]
    }

    report["refs"]["self"] = make_artifact_ref(
        ctx=ctx,
        stage="governance",
        name="manifest",
        run_id=str(rid),
    )

    # Track hint conditions (emit hints in a single block before Status)
    setup_path: Optional[str] = None
    raw_missing = df_raw is None
    clean_missing = df_clean is None
    missing_required_setup = False
    missing_optional_setup = False

    # --- 5.1) Reproducibility anchor: setup/config_snapshot ---
    refs_cfg = cfg["refs"]
    if bool(refs_cfg["include_setup_config_snapshot"]):
        ss = refs_cfg["setup_config_snapshot"]
        expected_setup_path = resolve_artifact_path(
            ctx=ctx,
            stage=str(ss["stage"]),
            name=str(ss["name"]),
            run_id=str(rid),
            tag=TAG111,
        )
        setup_path = str(expected_setup_path)
        setup_exists = bool(Path(expected_setup_path).exists())

        report["refs"]["setup_config_snapshot"] = make_artifact_ref(
            ctx=ctx,
            stage=str(ss["stage"]),
            name=str(ss["name"]),
            run_id=str(rid),
        )
        report["checks"]["setup_config_snapshot"] = {
            "expected_path": str(expected_setup_path),
            "exists": bool(setup_exists),
        }

        if bool(refs_cfg["require_setup_config_snapshot"]) and (not bool(setup_exists)):
            missing_required_setup = True
        elif not bool(setup_exists):
            missing_optional_setup = True

    # --- 5.2) Dataset fingerprint (raw / clean) ---
    report["checks"]["dataset_fingerprint"] = {"raw": {"provided": False}, "clean": {"provided": False}}

    if isinstance(df_raw, pd.DataFrame):
        report["checks"]["dataset_fingerprint"]["raw"] = {
            "provided": True,
            **_build_dataset_fingerprint(df=df_raw, cfg=cfg),
        }

    if isinstance(df_clean, pd.DataFrame):
        report["checks"]["dataset_fingerprint"]["clean"] = {
            "provided": True,
            **_build_dataset_fingerprint(df=df_clean, cfg=cfg),
        }

    # --- 5.3) Artifact scan preview ---
    self_manifest_name = f"governance_manifest_{rid}.json"

    preview, scan_summary = _scan_artifacts_preview(
        artifact_dir=artifacts_dir,
        cfg=cfg,
        run_id=str(rid),
        exclude_names=[self_manifest_name],
    )
    report["checks"]["artifact_scan"] = scan_summary
    report["checks"]["artifact_scan_exclusions"] = {
        "excluded_self_manifest": True,
        "self_manifest_name": self_manifest_name,
    }
    report["checks"]["artifact_counts"] = {
        "indexed": int(scan_summary.get("indexed", 0)),
        "scanned": int(scan_summary.get("scanned", 0)),
        "filtered_out": int(scan_summary.get("filtered_out", 0)),
        "excluded": int(scan_summary.get("excluded", 0)),
    }
    report["checks"]["artifacts_preview"] = preview

    clean_path = _extract_clean_parquet_path_from_dq_cleaning_artifact(
        ctx=ctx,
        run_id=rid,
        lg=lg,
        tag=TAG111,
    )

    report.setdefault("checks", {})
    report["checks"].setdefault("refs", {})

    if clean_path:
        report.setdefault("refs", {})
        report["refs"]["clean_dataset"] = {
            "path": clean_path,
            "format": "parquet",
            "producer": {
                "stage": "data_quality",
                "name": "cleaning",
                "run_id": str(rid),
                "filename": f"data_quality_cleaning_{rid}.json",
            },
        }
        report["checks"]["refs"]["clean_dataset"] = {"present": True}
    else:
        report["checks"]["refs"]["clean_dataset"] = {"present": False}

    # Optional: attach free-form metadata
    if isinstance(extra_metadata, dict) and extra_metadata:
        report["checks"]["meta"] = extra_metadata

    # --- 5.4) Event hints (audit-friendly, before Status) ---
    if raw_missing:
        add_event_hint(
            report,
            code="manifest_df_raw_missing",
            severity_signal="info",
            evidence_path="inputs.df_raw_provided",
            context={"df_raw_provided": False},
            remediation={
                "action": "pass_df_raw_if_available",
                "safe": True,
                "rationale": "Raw fingerprint improves reproducibility; manifest remains valid without it.",
                "post_check": "re-run 11.1 with df_raw for stronger traceability",
            },
        )

    if clean_missing:
        add_event_hint(
            report,
            code="manifest_df_clean_missing",
            severity_signal="risk",
            evidence_path="inputs.df_clean_provided",
            context={"df_clean_provided": False},
            remediation={
                "action": "pass_df_clean_if_available",
                "safe": True,
                "rationale": "Clean fingerprint is the DE output baseline for downstream steps (4.x+).",
                "post_check": "re-run 11.1 with df_clean to lock the baseline",
            },
        )

    if missing_required_setup:
        add_event_hint(
            report,
            code="manifest_missing_required_setup_config_snapshot",
            severity_signal="hard",
            evidence_path="checks.setup_config_snapshot.exists",
            context={"expected_path": str(setup_path) if setup_path else None},
            remediation={
                "action": "run_step1_setup_or_fix_artifact_paths",
                "safe": True,
                "rationale": "setup/config_snapshot is the authoritative SSOT snapshot for reproducibility.",
                "post_check": "re-run Step1 then re-run 11.1",
            },
        )
    elif missing_optional_setup:
        add_event_hint(
            report,
            code="manifest_setup_config_snapshot_missing",
            severity_signal="risk",
            evidence_path="checks.setup_config_snapshot.exists",
            context={"expected_path": str(setup_path) if setup_path else None},
            remediation={
                "action": "ensure_step1_snapshot_is_persisted",
                "safe": True,
                "rationale": "Missing setup snapshot weakens auditability; manifest still usable.",
                "post_check": "re-run Step1 / 11.1 for stronger traceability",
            },
        )

    # --- 6) Status + gate-friendly summary ---
    issues: List[str] = []
    warnings: List[str] = []

    if not bool(scan_summary.get("dir_exists", False)):
        warnings.append("artifacts_dir_missing")

    if int(scan_summary.get("indexed", 0)) == 0:
        warnings.append("no_artifacts_indexed")

    # If setup snapshot is required but missing -> fail
    if missing_required_setup:
        issues.append("missing_required_setup_config_snapshot")

    overall = "fail" if issues else ("warn" if warnings else "pass")

    report = finalize_module_report(
        ctx=ctx,
        report=report,
        overall_status=overall,
        issues=issues,
        warnings=warnings,
        metrics={
            "n_artifacts_indexed": int(scan_summary.get("indexed", 0)),
            "n_artifacts_scanned": int(scan_summary.get("scanned", 0)),
        },
        notes=[f"manifest_created_at_utc={utc_now_iso()}"],
        df=df_clean if isinstance(df_clean, pd.DataFrame) else None,
    )

    if overall == "pass":
        lg.info(f"[{TAG111}] ✅ Manifest pass | n_indexed={scan_summary.get('indexed', 0)}")
    elif overall == "warn":
        lg.warning(f"[{TAG111}] ⚠ Manifest warn | warnings={warnings}")
    else:
        lg.error(f"[{TAG111}] ❌ Manifest fail | issues={issues}")

    # --- 7) Persist Artifact ---
    out_path = resolve_artifact_path(
        ctx=ctx,
        stage="governance",
        name="manifest",
        run_id=str(rid),
        tag=TAG111,
    )
    write_json(out_path, report, tag=TAG111, indent=2)
    lg.info(f"[{TAG111}] 🧾 Manifest artifact saved: {out_path}")

    return report

In [ ]:
# --- 11.2 Lineage & Consistency Audit ---
#
# What:
#   Audit run-level lineage & consistency (governance control-plane):
#   - Verify required anchors exist (setup snapshot / dq orchestrator / di orchestrator / manifest).
#   - Validate consistency rules (run_id / schema_version) across run artifacts (via 11.1 manifest preview).
#   - Validate cleaned dataset evidence (from 2.9 cleaning artifact) and optional disk existence.
#
# How:
#   - Resolve YAML SSOT via ctx: pipeline.governance.lineage_audit
#   - Prefer 11.1 manifest as the artifact index (control-plane), avoid re-scanning directory.
#   - Persist ModuleReport v1: stage=governance, name=lineage_audit, step=11.2
#
# Why:
#   - Catch "broken evidence chain" and "cross-run contamination / mismatched run_id" early.
#   - Provide audit-friendly reproducibility proof for interviews.

# --- 0) Import + TAG
from typing import Any, Dict, List, Optional
from pathlib import Path

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint
from src.utils.ctx_utils import safe_get_in

from src.utils.artifact_utils import resolve_artifact_path, write_json, read_json_if_exists
from src.utils.cfg_utils import (
    get_cfg_bool,
    get_cfg_int,
    get_cfg_str,
    get_cfg_str_list,
)
from src.utils.path_utils import resolve_dir
from src.utils.time_utils import utc_now_iso

TAG112 = "LINEAGE_AUDIT"


# --- Internal helpers ---
def _safe_int(x: Any, default: int = -1) -> int:
    try:
        return int(x)
    except Exception:
        return int(default)


def _load_manifest(ctx: Dict[str, Any], *, run_id: str, lg: Any) -> Optional[Dict[str, Any]]:
    p = resolve_artifact_path(ctx=ctx, stage="governance", name="manifest", run_id=str(run_id), tag=TAG112)
    rep = read_json_if_exists(p, lg=lg, tag=TAG112)
    return rep if isinstance(rep, dict) else None


def _verify_report_identity(
    rep: Dict[str, Any],
    *,
    expected_run_id: str,
    expected_schema_version: int,
    check_run_id: bool,
    check_schema_version: bool,
) -> Dict[str, Any]:
    """Verify internal identity fields for audit consistency."""
    rid_ok = True
    schema_ok = True

    if check_run_id:
        rid_ok = (str(rep.get("run_id")) == str(expected_run_id))

    if check_schema_version:
        schema_ok = (_safe_int(rep.get("schema_version", -1)) == int(expected_schema_version))

    mod = rep.get("module") if isinstance(rep.get("module"), dict) else {}
    stage = mod.get("stage")
    name = mod.get("name")
    step = mod.get("step")

    return {
        "ok": bool(rid_ok and schema_ok),
        "run_id_ok": bool(rid_ok),
        "schema_ok": bool(schema_ok),
        "run_id": rep.get("run_id"),
        "schema_version": rep.get("schema_version"),
        "module": {"stage": stage, "name": name, "step": step},
    }


def _extract_cleaned_path_from_cleaning_artifact(
    ctx: Dict[str, Any],
    *,
    run_id: str,
    lg: Any,
    evidence_paths: List[str],
) -> Optional[str]:
    """
    Extract cleaned parquet path from 2.9 cleaning artifact using evidence_paths.
    Example evidence path: "checks.persistence.cleaned_parquet_path"
    """
    p = resolve_artifact_path(ctx=ctx, stage="data_quality", name="cleaning", run_id=str(run_id), tag=TAG112)
    rep = read_json_if_exists(p, lg=lg, tag=TAG112)
    if not isinstance(rep, dict):
        return None

    for ep in evidence_paths:
        v = safe_get_in(rep, ep, tag=TAG112, default=None)
        if isinstance(v, str) and v.strip():
            return str(Path(v))

    return None


def _resolve_lineage_cfg(ctx: Dict[str, Any]) -> Dict[str, Any]:
    """
    Resolve YAML SSOT:
      pipeline.governance.lineage_audit
    with keys:
      anchors, rules, limits
    """
    base = "pipeline.governance.lineage_audit"

    # Anchors (required flags)
    require_setup = get_cfg_bool(
        ctx,
        f"{base}.anchors.require_setup_config_snapshot",
        tag=TAG112,
        default=False,
        strict_type=False,
    )
    require_dq_orch = get_cfg_bool(
        ctx,
        f"{base}.anchors.require_data_quality_orchestrator",
        tag=TAG112,
        default=True,
        strict_type=False,
    )
    require_di_orch = get_cfg_bool(
        ctx,
        f"{base}.anchors.require_data_integrity_orchestrator",
        tag=TAG112,
        default=True,
        strict_type=False,
    )

    # anchor refs (stage/name)
    setup_stage = get_cfg_str(
        ctx,
        f"{base}.anchors.setup_config_snapshot.stage",
        tag=TAG112,
        default="setup",
        strict_type=True,
    )
    setup_name = get_cfg_str(
        ctx,
        f"{base}.anchors.setup_config_snapshot.name",
        tag=TAG112,
        default="config_snapshot",
        strict_type=True,
    )

    dq_stage = get_cfg_str(
        ctx,
        f"{base}.anchors.data_quality_orchestrator.stage",
        tag=TAG112,
        default="data_quality",
        strict_type=True,
    )
    dq_name = get_cfg_str(
        ctx,
        f"{base}.anchors.data_quality_orchestrator.name",
        tag=TAG112,
        default="orchestrator",
        strict_type=True,
    )

    di_stage = get_cfg_str(
        ctx,
        f"{base}.anchors.data_integrity_orchestrator.stage",
        tag=TAG112,
        default="data_integrity",
        strict_type=True,
    )
    di_name = get_cfg_str(
        ctx,
        f"{base}.anchors.data_integrity_orchestrator.name",
        tag=TAG112,
        default="orchestrator",
        strict_type=True,
    )

    mf_stage = get_cfg_str(
        ctx,
        f"{base}.anchors.governance_manifest.stage",
        tag=TAG112,
        default="governance",
        strict_type=True,
    )
    mf_name = get_cfg_str(
        ctx,
        f"{base}.anchors.governance_manifest.name",
        tag=TAG112,
        default="manifest",
        strict_type=True,
    )

    # Rules
    run_id_consistency = get_cfg_bool(
        ctx,
        f"{base}.rules.run_id_consistency",
        tag=TAG112,
        default=True,
        strict_type=False,
    )
    schema_version_consistency = get_cfg_bool(
        ctx,
        f"{base}.rules.schema_version_consistency",
        tag=TAG112,
        default=True,
        strict_type=False,
    )

    cleaned_enabled = get_cfg_bool(
        ctx,
        f"{base}.rules.cleaned_dataset_exists_check.enabled",
        tag=TAG112,
        default=True,
        strict_type=False,
    )

    evidence_paths: List[str] = get_cfg_str_list(
        ctx,
        f"{base}.rules.cleaned_dataset_exists_check.evidence_paths",
        tag=TAG112,
        default=[],
        allow_empty=True,     # empty means "skip evidence lookup" (still can warn)
        dedupe=True,
    )

    cleaned_required = get_cfg_bool(
        ctx,
        f"{base}.rules.cleaned_dataset_exists_check.required",
        tag=TAG112,
        default=False,
        strict_type=False,
    )

    must_exist_on_disk = get_cfg_bool(
        ctx,
        f"{base}.rules.cleaned_dataset_exists_check.must_exist_on_disk",
        tag=TAG112,
        default=True,
        strict_type=False,
    )

    # Limits
    max_examples = get_cfg_int(
        ctx,
        f"{base}.limits.max_missing_refs_examples",
        tag=TAG112,
        default=20,
        min_value=1,
    )

    return {
        "cfg_key": base,
        "anchors": {
            "require_setup_config_snapshot": require_setup,
            "require_data_quality_orchestrator": require_dq_orch,
            "require_data_integrity_orchestrator": require_di_orch,
            "setup_config_snapshot": {"stage": setup_stage, "name": setup_name},
            "data_quality_orchestrator": {"stage": dq_stage, "name": dq_name},
            "data_integrity_orchestrator": {"stage": di_stage, "name": di_name},
            "governance_manifest": {"stage": mf_stage, "name": mf_name},
        },
        "rules": {
            "run_id_consistency": run_id_consistency,
            "schema_version_consistency": schema_version_consistency,
            "cleaned_dataset_exists_check": {
                "enabled": cleaned_enabled,
                "evidence_paths": evidence_paths,
                "required": cleaned_required,
                "must_exist_on_disk": must_exist_on_disk,
            },
        },
        "limits": {
            "max_missing_refs_examples": max_examples,
        },
    }


def _extract_cleaned_path_from_manifest(
    mf: Dict[str, Any],
    *,
    evidence_paths: List[str],
) -> Optional[str]:
    """
    Prefer control-plane pointer from 11.1 manifest:
      - refs.clean_dataset.path (strong default)
      - optional: allow evidence_paths to point into manifest too
    """
    v = safe_get_in(mf, "refs.clean_dataset.path", tag=TAG112, default=None)
    if isinstance(v, str) and v.strip():
        return str(Path(v))

    for ep in evidence_paths:
        vv = safe_get_in(mf, ep, tag=TAG112, default=None)
        if isinstance(vv, str) and vv.strip():
            return str(Path(vv))

    return None


# --- 1) Public API ---
def audit_lineage_consistency(
    *,
    ctx: Dict[str, Any],
    run_id: Optional[str] = None,
    manifest_report: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """
    Step 11.2 - Lineage & Consistency Audit (ModuleReport v1).

    Tip:
      Pass manifest_report=rep_111 to avoid extra disk read.
    """
    # --- 2) Runtime validation ---
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG112, run_id=run_id)

    # --- 3) Input validation ---
    if manifest_report is not None and (not isinstance(manifest_report, dict)):
        raise TypeError(f"[{TAG112}] manifest_report must be a dict when provided")

    # --- 4) Resolve config (YAML SSOT) ---
    cfg = _resolve_lineage_cfg(ctx)
    anchors_cfg = cfg["anchors"]
    rules_cfg = cfg["rules"]
    limits_cfg = cfg["limits"]

    artifacts_dir = resolve_dir(ctx, "outputs.artifacts", required=True, tag=TAG112)

    lg.info(f"[{TAG112}] 🔎 Lineage & consistency audit | run_id={rid}")

    report = build_module_report(
        ctx=ctx,
        stage="governance",
        step="11.2",
        name="lineage_audit",
        tag=TAG112,
        cfg_key=str(cfg["cfg_key"]),
        enabled=True,
        df=None,
        run_id_override=str(rid),
    )

    report.setdefault("refs", {})
    report.setdefault("inputs", {})
    report.setdefault("checks", {})
    ensure_event_hints(report, version=1)

    report["inputs"].update(
        {
            "artifacts_dir": str(artifacts_dir),
            "manifest_report_provided": bool(isinstance(manifest_report, dict)),
        }
    )

    report["thresholds_used"] = {
        "anchors": dict(anchors_cfg),
        "rules": dict(rules_cfg),
        "limits": dict(limits_cfg),
    }

    report["used_config"] = {
        "cfg_keys": [
            "pipeline.governance.lineage_audit",
            "pipeline.governance.lineage_audit.anchors",
            "pipeline.governance.lineage_audit.rules",
            "pipeline.governance.lineage_audit.limits",
        ]
    }

    report["refs"]["self"] = make_artifact_ref(ctx=ctx, stage="governance", name="lineage_audit", run_id=str(rid))
    report["refs"]["artifacts_dir"] = str(artifacts_dir)

    # --- 5.1) Load manifest (control-plane index) ---
    mf_stage = str(anchors_cfg["governance_manifest"]["stage"])
    mf_name = str(anchors_cfg["governance_manifest"]["name"])
    mf_path = resolve_artifact_path(ctx=ctx, stage=mf_stage, name=mf_name, run_id=str(rid), tag=TAG112)

    mf = manifest_report if isinstance(manifest_report, dict) else _load_manifest(ctx, run_id=str(rid), lg=lg)

    report["checks"]["manifest_anchor"] = {
        "stage": mf_stage,
        "name": mf_name,
        "expected_path": str(mf_path),
        "exists_on_disk": bool(Path(mf_path).exists()),
        "loaded_ok": bool(isinstance(mf, dict)),
    }

    # --- 5.2) Anchor existence checks ---
    anchor_specs: List[Dict[str, Any]] = []

    # setup snapshot (required depends on config)
    anchor_specs.append(
        {
            "key": "setup_config_snapshot",
            "required": bool(anchors_cfg["require_setup_config_snapshot"]),
            "stage": str(anchors_cfg["setup_config_snapshot"]["stage"]),
            "name": str(anchors_cfg["setup_config_snapshot"]["name"]),
        }
    )

    # dq orchestrator
    anchor_specs.append(
        {
            "key": "data_quality_orchestrator",
            "required": bool(anchors_cfg["require_data_quality_orchestrator"]),
            "stage": str(anchors_cfg["data_quality_orchestrator"]["stage"]),
            "name": str(anchors_cfg["data_quality_orchestrator"]["name"]),
        }
    )

    # di orchestrator
    anchor_specs.append(
        {
            "key": "data_integrity_orchestrator",
            "required": bool(anchors_cfg["require_data_integrity_orchestrator"]),
            "stage": str(anchors_cfg["data_integrity_orchestrator"]["stage"]),
            "name": str(anchors_cfg["data_integrity_orchestrator"]["name"]),
        }
    )

    # governance manifest itself (always optional here, but we record it)
    anchor_specs.append(
        {
            "key": "governance_manifest",
            "required": False,
            "stage": mf_stage,
            "name": mf_name,
        }
    )

    anchor_results: List[Dict[str, Any]] = []
    missing_required = 0
    missing_optional = 0
    missing_examples: List[Dict[str, Any]] = []

    for a in anchor_specs:
        p = resolve_artifact_path(ctx=ctx, stage=str(a["stage"]), name=str(a["name"]), run_id=str(rid), tag=TAG112)
        exists = bool(Path(p).exists())

        anchor_results.append(
            {
                "key": str(a["key"]),
                "required": bool(a["required"]),
                "stage": str(a["stage"]),
                "name": str(a["name"]),
                "expected_path": str(p),
                "exists": bool(exists),
            }
        )

        if bool(a["required"]) and (not exists):
            missing_required += 1
            if len(missing_examples) < int(limits_cfg["max_missing_refs_examples"]):
                missing_examples.append({"key": str(a["key"]), "path": str(p)})
        elif (not bool(a["required"])) and (not exists):
            missing_optional += 1

    report["checks"]["anchors"] = {
        "items": anchor_results,
        "missing_required": int(missing_required),
        "missing_optional": int(missing_optional),
        "missing_required_examples": missing_examples,
    }

    # --- 5.3) Consistency checks across artifacts (via manifest preview) ---
    run_id_check = bool(rules_cfg["run_id_consistency"])
    schema_check = bool(rules_cfg["schema_version_consistency"])
    expected_schema_version = int(report.get("schema_version", 1))

    mismatches_run_id: List[Dict[str, Any]] = []
    mismatches_schema: List[Dict[str, Any]] = []
    checked = 0

    preview_items: List[Dict[str, Any]] = []
    if isinstance(mf, dict):
        pv = safe_get_in(mf, "checks.artifacts_preview", tag=TAG112, default=None)
        if isinstance(pv, list):
            preview_items = [x for x in pv if isinstance(x, dict) and isinstance(x.get("path"), str)]

    skipped_non_report = 0

    for item in preview_items:
        p = Path(str(item.get("path")))
        rep = read_json_if_exists(p, lg=lg, tag=TAG112)
        if not isinstance(rep, dict):
            continue

        # Skip non-ModuleReport JSONs (e.g., setup/config_snapshot)
        if not isinstance(rep.get("module"), dict) or not isinstance(rep.get("summary"), dict):
            skipped_non_report += 1
            continue

        checked += 1
        vr = _verify_report_identity(
            rep,
            expected_run_id=str(rid),
            expected_schema_version=expected_schema_version,
            check_run_id=run_id_check,
            check_schema_version=schema_check,
        )

        if run_id_check and (not bool(vr.get("run_id_ok", True))):
            if len(mismatches_run_id) < int(limits_cfg["max_missing_refs_examples"]):
                mismatches_run_id.append({"name": str(item.get("name", p.name)), "path": str(p), "verify": vr})

        if schema_check and (not bool(vr.get("schema_ok", True))):
            if len(mismatches_schema) < int(limits_cfg["max_missing_refs_examples"]):
                mismatches_schema.append({"name": str(item.get("name", p.name)), "path": str(p), "verify": vr})

    report["checks"]["consistency"] = {
        "source": "manifest.checks.artifacts_preview",
        "rules": {
            "run_id_consistency": run_id_check,
            "schema_version_consistency": schema_check,
            "expected_schema_version": int(expected_schema_version),
        },
        "n_checked": int(checked),
        "n_skipped_non_report": int(skipped_non_report),
        "run_id_mismatches": int(len(mismatches_run_id)),
        "schema_version_mismatches": int(len(mismatches_schema)),
        "examples": {
            "run_id_mismatches": mismatches_run_id,
            "schema_version_mismatches": mismatches_schema,
        },
    }

    # --- 5.4) Cleaned dataset existence check (rule-driven) ---
    cleaned_rule = rules_cfg["cleaned_dataset_exists_check"]
    cleaned_enabled = bool(cleaned_rule["enabled"])

    cleaned_path: Optional[str] = None
    cleaned_exists: Optional[bool] = None

    if cleaned_enabled:
        eps = cleaned_rule.get("evidence_paths") or []
        eps = [str(x) for x in eps if isinstance(x, str) and x.strip()]

        # (A) Prefer manifest control-plane pointer
        if isinstance(mf, dict):
            cleaned_path = _extract_cleaned_path_from_manifest(mf, evidence_paths=eps)

        # (B) Fallback: read from 2.9 cleaning artifact (evidence paths are applied there)
        if cleaned_path is None:
            cleaned_path = _extract_cleaned_path_from_cleaning_artifact(
                ctx,
                run_id=str(rid),
                lg=lg,
                evidence_paths=eps,
            )

        if isinstance(cleaned_path, str) and cleaned_path.strip():
            cleaned_exists = bool(Path(cleaned_path).exists()) if bool(cleaned_rule["must_exist_on_disk"]) else None
        else:
            cleaned_exists = False if bool(cleaned_rule["must_exist_on_disk"]) else None

    report["checks"]["cleaned_dataset_exists_check"] = {
        "enabled": cleaned_enabled,
        "required": bool(cleaned_rule["required"]),
        "must_exist_on_disk": bool(cleaned_rule["must_exist_on_disk"]),
        "cleaned_path": cleaned_path,
        "exists_on_disk": cleaned_exists,
        "evidence_paths": list(cleaned_rule.get("evidence_paths") or []),
        "source_preferred": "manifest.refs.clean_dataset.path",
        "source_fallback": "data_quality.cleaning (2.9)",
    }

    # --- 5.5 Event hints ---
    # Collect conditions first (facts already written to checks.*)
    missing_required_flag = int(missing_required) > 0
    run_id_mismatch_flag = run_id_check and (len(mismatches_run_id) > 0)
    schema_mismatch_flag = schema_check and (len(mismatches_schema) > 0)

    cleaned_required_missing_flag = (
        cleaned_enabled and bool(cleaned_rule["required"]) and (cleaned_exists is False)
    )
    cleaned_optional_missing_flag = (
        cleaned_enabled and (not bool(cleaned_rule["required"])) and (cleaned_exists is False)
    )

    manifest_missing_on_disk_flag = not bool(Path(mf_path).exists())

    # Emit hints in one place
    if missing_required_flag:
        add_event_hint(
            report,
            code="lineage_missing_required_anchors",
            severity_signal="hard",
            evidence_path="checks.anchors.missing_required",
            context={"missing_required": int(missing_required), "examples": missing_examples},
            remediation={
                "action": "re_run_missing_steps_or_fix_paths",
                "safe": True,
                "rationale": "Required anchors are the minimum reproducibility evidence chain.",
                "post_check": "re-run the missing steps and then re-run 11.2",
            },
        )
    
    if run_id_mismatch_flag:
        add_event_hint(
            report,
            code="lineage_run_id_inconsistency_detected",
            severity_signal="hard",
            evidence_path="checks.consistency.run_id_mismatches",
            context={"count": int(len(mismatches_run_id))},
            remediation={
                "action": "delete_bad_artifacts_and_rerun",
                "safe": True,
                "rationale": "Internal run_id mismatch breaks audit integrity and can cause cross-run confusion.",
                "post_check": "re-run producing steps then 11.1/11.2",
            },
        )

    if schema_mismatch_flag:
        add_event_hint(
            report,
            code="lineage_schema_version_inconsistency_detected",
            severity_signal="hard",
            evidence_path="checks.consistency.schema_version_mismatches",
            context={"count": int(len(mismatches_schema))},
            remediation={
                "action": "upgrade_or_rebuild_old_reports",
                "safe": True,
                "rationale": "Mixed schema versions reduce governance stability.",
                "post_check": "re-run producing steps with current schema_version=1",
            },
        )

    if cleaned_required_missing_flag:
        add_event_hint(
            report,
            code="lineage_missing_required_cleaned_dataset",
            severity_signal="hard",
            evidence_path="checks.cleaned_dataset_exists_check.exists_on_disk",
            context={"cleaned_path": cleaned_path},
            remediation={
                "action": "re_run_2_9_cleaning_or_fix_persistence",
                "safe": True,
                "rationale": "Cleaned dataset is a baseline reproducibility artifact for downstream stages.",
                "post_check": "ensure parquet persisted, then re-run 11.1/11.2",
            },
        )
    elif cleaned_optional_missing_flag:
        add_event_hint(
            report,
            code="lineage_cleaned_dataset_missing_optional",
            severity_signal="risk",
            evidence_path="checks.cleaned_dataset_exists_check.exists_on_disk",
            context={"cleaned_path": cleaned_path},
            remediation={
                "action": "consider_persisting_cleaned_dataset",
                "safe": True,
                "rationale": "Missing cleaned dataset weakens end-to-end reproducibility but may be acceptable.",
                "post_check": "persist cleaned dataset in 2.9 and re-run 11.1/11.2",
            },
        )
    
    if manifest_missing_on_disk_flag:
        add_event_hint(
            report,
            code="lineage_manifest_missing_on_disk",
            severity_signal="risk",
            evidence_path="checks.manifest_anchor.exists_on_disk",
            context={"expected_path": str(mf_path)},
            remediation={
                "action": "re_run_11_1_manifest",
                "safe": True,
                "rationale": "11.2 prefers manifest as the control-plane index; missing manifest reduces auditability.",
                "post_check": "re-run 11.1 then 11.2",
            },
        )

    # --- 6) Status + gate-friendly summary ---
    issues: List[str] = []
    warnings: List[str] = []

    if int(missing_required) > 0:
        issues.append("missing_required_anchors")

    if run_id_check and int(len(mismatches_run_id)) > 0:
        issues.append("run_id_inconsistency")

    if schema_check and int(len(mismatches_schema)) > 0:
        issues.append("schema_version_inconsistency")

    if cleaned_enabled and bool(cleaned_rule["required"]) and (cleaned_exists is False):
        issues.append("missing_required_cleaned_dataset")

    if cleaned_enabled and (cleaned_exists is False) and (not bool(cleaned_rule["required"])):
        warnings.append("missing_optional_cleaned_dataset")

    if (not bool(Path(mf_path).exists())):
        warnings.append("manifest_missing_on_disk")

    indexed_total = _safe_int(safe_get_in(mf, "checks.artifact_scan.indexed", tag=TAG112, default=None), default=-1) if isinstance(mf, dict) else -1
    preview_n = len(preview_items) if isinstance(preview_items, list) else 0
    if indexed_total > 0 and preview_n > 0 and indexed_total > preview_n:
        warnings.append("consistency_checked_preview_only")

    overall = "fail" if issues else ("warn" if warnings else "pass")

    report = finalize_module_report(
        ctx=ctx,
        report=report,
        overall_status=overall,
        issues=issues,
        warnings=warnings,
        metrics={
            "n_anchors_missing_required": int(missing_required),
            "n_anchors_missing_optional": int(missing_optional),
            "n_consistency_checked": int(checked),
            "n_run_id_mismatches": int(len(mismatches_run_id)),
            "n_schema_version_mismatches": int(len(mismatches_schema)),
        },
        notes=[f"lineage_audit_at_utc={utc_now_iso()}"],
        df=None,
    )

    if overall == "pass":
        lg.info(f"[{TAG112}] ✅ Lineage audit pass")
    elif overall == "warn":
        lg.warning(f"[{TAG112}] ⚠ Lineage audit warn | warnings={warnings}")
    else:
        lg.error(f"[{TAG112}] ❌ Lineage audit fail | issues={issues}")

    # --- 7) Persist artifact ---
    out_path = resolve_artifact_path(ctx=ctx, stage="governance", name="lineage_audit", run_id=str(rid), tag=TAG112)
    write_json(out_path, report, tag=TAG112, indent=2)
    lg.info(f"[{TAG112}] 🧾 Lineage audit artifact saved: {out_path}")

    return report

In [21]:
# --- 11.3 Artifact Inventory Index ---
#
# What:
#   Build a bounded artifact inventory index (control-plane):
#   - Scan outputs.artifacts for JSON artifacts (bounded).
#   - Produce a paged index for browsing/debugging and audit trails.
#   - Keep report size stable via paging + hard max_pages.
#
# How:
#   - Resolve YAML SSOT via ctx: pipeline.governance.inventory
#   - Scan artifacts directory (non-recursive by default), suffix-filtered.
#   - Sort by mtime (default) or name, bounded by max_artifacts.
#   - Persist ModuleReport v1: stage=governance, name=inventory, step=11.3
#
# Why:
#   - Provide a stable "inventory index" for audit/reproducibility and quick triage.
#   - Avoid reading/rehashing every artifact; keep it lightweight & bounded.

# --- 0) Import + TAG ---
from typing import Any, Dict, List, Optional, Tuple
from pathlib import Path
import heapq

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint

from src.utils.artifact_utils import resolve_artifact_path, write_json
from src.utils.cfg_utils import (
    get_cfg_bool,
    get_cfg_int,
    get_cfg_str,
    get_cfg_str_list,
)
from src.utils.path_utils import resolve_dir
from src.utils.time_utils import utc_now_iso, utc_from_timestamp_iso


TAG113 = "INVENTORY"

_ALLOWED_SORT_BY = {"name", "mtime"}


def _artifact_matches_run_id(p: Path, *, run_id: str) -> bool:
    """
    SSOT naming: {stage}_{name}_{run_id}.json
    We consider a match if the last '_' chunk of stem equals run_id.
    """
    stem = p.stem
    parts = stem.split("_")
    if not parts:
        return False
    return parts[-1] == str(run_id)


def _infer_step_hint_from_filename(name: str) -> str:
    """
    Infer a rough step hint from SSOT filenames:
      {stage}_{name}_{run_id}.json
    Keep consistent with 11.1 mapping.
    """
    lowered = str(name).lower()

    mapping = [
        ("setup_config_snapshot_", "1_setup_config_snapshot"),
        ("data_quality_ingestion_", "2.1_ingestion"),
        ("data_quality_contract_", "2.2_contract"),
        ("data_quality_health_", "2.3_health"),
        ("data_quality_semantic_", "2.4_semantic"),
        ("data_quality_readiness_", "2.5_readiness"),
        ("data_quality_domain_core_", "2.6.1_domain_core"),
        ("data_quality_domain_addon_", "2.6.2_domain_addon"),
        ("data_quality_distribution_", "2.7_distribution"),
        ("data_quality_pre_analysis_summary_", "2.8_pre_analysis_summary"),
        ("data_quality_cleaning_", "2.9_cleaning"),
        ("data_quality_orchestrator_", "2.x_dq_orchestrator"),
        ("data_integrity_row_identity_", "3.1_row_identity"),
        ("data_integrity_temporal_", "3.2_temporal"),
        ("data_integrity_label_alignment_", "3.3_label_alignment"),
        ("data_integrity_leakage_scan_", "3.4_leakage_scan"),
        ("data_integrity_integrity_summary_", "3.5_integrity_summary"),
        ("data_integrity_orchestrator_", "3.x_di_orchestrator"),
        ("governance_manifest_", "11.1_manifest"),
        ("governance_lineage_audit_", "11.2_lineage_audit"),
        ("governance_inventory_", "11.3_inventory"),
        ("governance_release_gate_", "11.4_release_gate"),
        ("experiment_tracking_manifest_", "11.1_manifest_legacy"),
    ]

    for key, label in mapping:
        if key in lowered:
            return label
    return "unknown"


def _resolve_inventory_cfg(ctx: Dict[str, Any]) -> Dict[str, Any]:
    """
    Resolve YAML SSOT:
      pipeline.governance.inventory
    """
    base = "pipeline.governance.inventory"

    # --- scan ---
    include_suffixes: List[str] = get_cfg_str_list(
        ctx,
        f"{base}.scan.include_suffixes",
        tag=TAG113,
        default=[".json"],
        allow_empty=False,
        dedupe=True,
    )

    max_artifacts = get_cfg_int(
        ctx,
        f"{base}.scan.max_artifacts",
        tag=TAG113,
        default=2000,
        min_value=1,
    )

    filter_by_run_id = get_cfg_bool(
        ctx,
        f"{base}.scan.filter_by_run_id",
        tag=TAG113,
        default=False,
        strict_type=False,
    )

    recursive = get_cfg_bool(
        ctx,
        f"{base}.scan.recursive",
        tag=TAG113,
        default=False,
        strict_type=False,
    )

    sort_by = get_cfg_str(
        ctx,
        f"{base}.scan.sort_by",
        tag=TAG113,
        default="mtime",
        lower=True,
        allowed=_ALLOWED_SORT_BY,
        strict_type=True,
    )

    # NOTE:
    # 11.3 is an inventory index; hashing is intentionally excluded to keep it lightweight.
    # We still accept YAML key for backward compatibility, but we do not use it.
    _include_hash_unused = get_cfg_bool(
        ctx,
        f"{base}.scan.include_hash",
        tag=TAG113,
        default=False,
        strict_type=False,
    )

    # --- paging ---
    paging_enabled = get_cfg_bool(
        ctx,
        f"{base}.paging.enabled",
        tag=TAG113,
        default=True,
        strict_type=False,
    )

    page_size = get_cfg_int(
        ctx,
        f"{base}.paging.page_size",
        tag=TAG113,
        default=200,
        min_value=1,
    )

    max_pages = get_cfg_int(
        ctx,
        f"{base}.paging.max_pages",
        tag=TAG113,
        default=5,
        min_value=1,
    )

    # --- output ---
    index_version = get_cfg_int(
        ctx,
        f"{base}.output.index_version",
        tag=TAG113,
        default=1,
        min_value=1,
    )

    include_step_hint = get_cfg_bool(
        ctx,
        f"{base}.output.include_step_hint",
        tag=TAG113,
        default=True,
        strict_type=False,
    )

    include_size_bytes = get_cfg_bool(
        ctx,
        f"{base}.output.include_size_bytes",
        tag=TAG113,
        default=True,
        strict_type=False,
    )

    include_modified_at_utc = get_cfg_bool(
        ctx,
        f"{base}.output.include_modified_at_utc",
        tag=TAG113,
        default=True,
        strict_type=False,
    )

    return {
        "cfg_key": base,
        "scan": {
            "include_suffixes": include_suffixes,
            "max_artifacts": int(max_artifacts),
            "filter_by_run_id": bool(filter_by_run_id),
            "recursive": bool(recursive),
            "sort_by": str(sort_by),
            # Keep explicit to avoid confusion: hashing is not supported in 11.3.
            "include_hash": False,
            "include_hash_supported": False,
        },
        "paging": {
            "enabled": bool(paging_enabled),
            "page_size": int(page_size),
            "max_pages": int(max_pages),
        },
        "output": {
            "index_version": int(index_version),
            "include_step_hint": bool(include_step_hint),
            "include_size_bytes": bool(include_size_bytes),
            "include_modified_at_utc": bool(include_modified_at_utc),
        },
    }


def _scan_inventory_items(
    *,
    artifact_dir: Path,
    run_id: str,
    cfg: Dict[str, Any],
) -> Tuple[List[Path], Dict[str, Any]]:
    """
    Scan artifacts dir and return bounded candidates + scan summary.
    - Supports name / mtime sorting.
    - Uses heap for mtime to keep top-K newest.
    """
    scan_cfg = cfg["scan"]

    include_suffixes = [str(s).lower() for s in (scan_cfg.get("include_suffixes") or [])]
    suffix_set = set(include_suffixes) if include_suffixes else {".json"}

    max_artifacts = int(scan_cfg.get("max_artifacts", 2000))
    sort_by = str(scan_cfg.get("sort_by", "mtime"))
    filter_by_run_id = bool(scan_cfg.get("filter_by_run_id", False))
    recursive = bool(scan_cfg.get("recursive", False))

    scanned = 0
    filtered_out = 0
    stopped_early = False
    stop_reason: Optional[str] = None

    if not artifact_dir.exists():
        return [], {
            "dir_exists": False,
            "recursive": recursive,
            "scanned": 0,
            "indexed": 0,
            "filtered_out": 0,
            "include_suffixes": list(suffix_set),
            "max_artifacts": int(max_artifacts),
            "filter_by_run_id": bool(filter_by_run_id),
            "sort_by": sort_by,
        }

    if recursive:
        it = (p for p in artifact_dir.rglob("*") if p.is_file())
    else:
        it = (p for p in artifact_dir.iterdir() if p.is_file())

    candidates: List[Path] = []
    heap: List[Tuple[float, Path]] = []

    for p in it:
        scanned += 1

        if p.suffix.lower() not in suffix_set:
            filtered_out += 1
            continue

        if filter_by_run_id and (not _artifact_matches_run_id(p, run_id=run_id)):
            filtered_out += 1
            continue

        if sort_by == "mtime":
            try:
                mtime = float(p.stat().st_mtime)
            except Exception:
                mtime = 0.0

            if len(heap) < max_artifacts:
                heapq.heappush(heap, (mtime, p))
            else:
                # keep top-K newest
                if mtime > heap[0][0]:
                    heapq.heapreplace(heap, (mtime, p))
        else:
            if len(candidates) < max_artifacts:
                candidates.append(p)
            else:
                # bounded early stop in name mode
                stopped_early = True
                stop_reason = "max_artifacts_reached"
                break

    if sort_by == "mtime":
        candidates = [p for _, p in sorted(heap, key=lambda x: x[0], reverse=True)]
    else:
        candidates = sorted(candidates, key=lambda x: x.name)

    return candidates, {
        "dir_exists": True,
        "recursive": recursive,
        "scanned": int(scanned),
        "indexed": int(len(candidates)),
        "filtered_out": int(filtered_out),
        "stopped_early": bool(stopped_early),
        "stop_reason": stop_reason,
        "include_suffixes": list(suffix_set),
        "max_artifacts": int(max_artifacts),
        "filter_by_run_id": bool(filter_by_run_id),
        "sort_by": sort_by,
        "run_id_match_rule": "stem_last_underscore_chunk_equals_run_id" if filter_by_run_id else None,
    }


def _build_index_item(
    *,
    p: Path,
    output_cfg: Dict[str, Any],
) -> Dict[str, Any]:
    """
    Convert a path into an index record, controlled by output cfg.
    Hashing is intentionally excluded in 11.3 (per YAML).
    """
    item: Dict[str, Any] = {
        "name": p.name,
        "path": str(p),
        "suffix": p.suffix,
    }

    try:
        st = p.stat()
        if bool(output_cfg.get("include_size_bytes", True)):
            item["size_bytes"] = int(st.st_size)
        if bool(output_cfg.get("include_modified_at_utc", True)):
            item["modified_at_utc"] = utc_from_timestamp_iso(st.st_mtime)
    except Exception:
        if bool(output_cfg.get("include_size_bytes", True)):
            item["size_bytes"] = None
        if bool(output_cfg.get("include_modified_at_utc", True)):
            item["modified_at_utc"] = None

    if bool(output_cfg.get("include_step_hint", True)):
        item["step_hint"] = _infer_step_hint_from_filename(p.name)

    return item


def _build_paged_index(
    *,
    items: List[Path],
    paging_cfg: Dict[str, Any],
    output_cfg: Dict[str, Any],
) -> Dict[str, Any]:
    """
    Build paged index payload:
      - pages: list[{page, n_items, items:[...]}]
      - truncated: whether we had to cut due to max_pages
      - totals
    """
    enabled = bool(paging_cfg.get("enabled", True))
    page_size = int(paging_cfg.get("page_size", 200))
    max_pages = int(paging_cfg.get("max_pages", 5))
    capacity = (page_size * max_pages) if (enabled and page_size > 0 and max_pages > 0) else None

    if (not enabled) or page_size <= 0:
        # no paging: still bound by max_artifacts from scan
        recs = [_build_index_item(p=p, output_cfg=output_cfg) for p in items]
        return {
            "paging_enabled": False,
            "page_size": None,
            "max_pages": None,
            "capacity": capacity,
            "n_total_items": int(len(recs)),
            "n_indexed_items": int(len(recs)),
            "truncated": False,
            "pages": [{"page": 1, "n_items": int(len(recs)), "items": recs}],
        }

    truncated = len(items) > capacity
    used = items[:capacity]

    pages: List[Dict[str, Any]] = []
    for i in range(max_pages):
        start = i * page_size
        end = start + page_size
        chunk = used[start:end]
        if not chunk:
            break
        recs = [_build_index_item(p=p, output_cfg=output_cfg) for p in chunk]
        pages.append({"page": i + 1, "n_items": int(len(recs)), "items": recs})

    return {
        "paging_enabled": True,
        "page_size": int(page_size),
        "max_pages": int(max_pages),
        "capacity": int(capacity) if isinstance(capacity, int) else None,
        "n_total_items": int(len(items)),
        "n_indexed_items": int(len(used)),
        "truncated": bool(truncated),
        "pages": pages,
    }

# --- 1) Public API ---
def build_artifact_inventory_index(
    *,
    ctx: Dict[str, Any],
    run_id: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Step 11.3 - Build and persist the artifact inventory index (ModuleReport v1).
    """
    # --- 2) Runtime validation ---
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG113, run_id=run_id)

    # --- 4) Resolve config (YAML SSOT) ---
    cfg = _resolve_inventory_cfg(ctx)
    scan_cfg = cfg["scan"]
    paging_cfg = cfg["paging"]
    output_cfg = cfg["output"]

    artifacts_dir = resolve_dir(ctx, "outputs.artifacts", required=True, tag=TAG113)

    # --- 5) Core workflows---
    lg.info(f"[{TAG113}] 📚 Building artifact inventory | run_id={rid}")

    # --- Build ModuleReport skeleton ---
    report = build_module_report(
        ctx=ctx,
        stage="governance",
        step="11.3",
        name="inventory",
        tag=TAG113,
        cfg_key=str(cfg["cfg_key"]),
        enabled=True,
        df=None,
        run_id_override=str(rid),
    )

    report.setdefault("refs", {})
    report.setdefault("inputs", {})
    report.setdefault("checks", {})
    ensure_event_hints(report, version=1)

    report["inputs"].update(
        {
            "artifacts_dir": str(artifacts_dir),
        }
    )

    report["thresholds_used"] = {
        "scan": dict(scan_cfg),
        "paging": dict(paging_cfg),
        "output": dict(output_cfg),
    }

    report["used_config"] = {
        "cfg_keys": [
            "pipeline.governance.inventory",
            "pipeline.governance.inventory.scan",
            "pipeline.governance.inventory.paging",
            "pipeline.governance.inventory.output",
        ]
    }

    report["refs"]["self"] = make_artifact_ref(
        ctx=ctx,
        stage="governance",
        name="inventory",
        run_id=str(rid),
    )

    # --- 5.1) Scan (bounded) ---
    paths, scan_summary = _scan_inventory_items(
        artifact_dir=artifacts_dir,
        run_id=str(rid),
        cfg=cfg,
    )
    report["checks"]["scan"] = scan_summary

    # --- 5.2) Build paged index ---
    index_payload = _build_paged_index(
        items=paths,
        paging_cfg=paging_cfg,
        output_cfg=output_cfg,
    )

    report["checks"]["inventory_index"] = {
        "index_version": int(output_cfg.get("index_version", 1)),
        **index_payload,
    }

    # --- 5.3) Event hints (audit-friendly, before Status) ---
    if not bool(scan_summary.get("dir_exists", False)):
        add_event_hint(
            report,
            code="inventory_artifacts_dir_missing",
            severity_signal="risk",
            evidence_path="checks.scan.dir_exists",
            context={"artifacts_dir": str(artifacts_dir)},
            remediation={
                "action": "ensure_outputs_artifacts_dir_is_created",
                "safe": True,
                "rationale": "Inventory needs the artifacts directory to index run evidence.",
                "post_check": "re-run Step1 setup and producing steps, then re-run 11.3",
            },
        )

    if bool(index_payload.get("truncated", False)):
        add_event_hint(
            report,
            code="inventory_truncated_by_paging_capacity",
            severity_signal="info",
            evidence_path="checks.inventory_index.truncated",
            context={
                "capacity": index_payload.get("capacity"),
                "n_total_items": int(index_payload.get("n_total_items", -1)),
            },
            remediation={
                "action": "increase_paging_capacity_or_reduce_max_artifacts",
                "safe": True,
                "rationale": "Inventory is intentionally bounded to keep artifacts stable and report size controlled.",
                "post_check": "re-run 11.3 after adjusting pipeline.governance.inventory.paging.*",
            },
        )

    # --- 6) Status + gate-friendly summary ---
    issues: List[str] = []
    warnings: List[str] = []

    if not bool(scan_summary.get("dir_exists", False)):
        warnings.append("artifacts_dir_missing")

    indexed = int(scan_summary.get("indexed", 0))
    if indexed == 0:
        warnings.append("no_artifacts_indexed")

    if bool(index_payload.get("truncated", False)):
        warnings.append("inventory_truncated")

    overall = "fail" if issues else ("warn" if warnings else "pass")

    report = finalize_module_report(
        ctx=ctx,
        report=report,
        overall_status=overall,
        issues=issues,
        warnings=warnings,
        metrics={
            "n_scanned": int(scan_summary.get("scanned", 0)),
            "n_indexed": int(scan_summary.get("indexed", 0)),
            "n_filtered_out": int(scan_summary.get("filtered_out", 0)),
            "n_total_items": int(index_payload.get("n_total_items", -1)),
            "n_indexed_items": int(index_payload.get("n_indexed_items", -1)),
            "n_pages": int(len(index_payload.get("pages", []) or [])),
        },
        notes=[f"inventory_built_at_utc={utc_now_iso()}"],
        df=None,
    )

    if overall == "pass":
        lg.info(f"[{TAG113}] ✅ Inventory pass | n_indexed={indexed}")
    elif overall == "warn":
        lg.warning(f"[{TAG113}] ⚠ Inventory warn | warnings={warnings}")
    else:
        lg.error(f"[{TAG113}] ❌ Inventory fail | issues={issues}")

    # --- 7) Persist artifact ---
    out_path = resolve_artifact_path(
        ctx=ctx,
        stage="governance",
        name="inventory",
        run_id=str(rid),
        tag=TAG113,
    )
    write_json(out_path, report, tag=TAG113, indent=2)
    lg.info(f"[{TAG113}] 🧾 Inventory artifact saved: {out_path}")

    return report

In [29]:
# --- 11.4 Release Decision & Policy Gate ---
#
# What:
#   Final policy-driven "go/no-go" gate for a run's evidence chain (control-plane):
#   - Require anchor reports with allowed statuses (policy.required_reports).
#   - Require evidence for enabled pipeline steps (policy.conditional_required_steps).
#   - Block on failures and/or blocking event_hints (policy.blocking).
#
# How:
#   - Resolve YAML SSOT via ctx: pipeline.governance.release_gate
#   - Load required artifacts via SSOT naming: {stage}_{name}_{run_id}.json
#   - Compute releasable flag + reasons + remediation
#   - Persist ModuleReport v1: stage=governance, name=release_gate, step=11.4
#
# Why:
#   - Deterministic, audit-friendly release decision for CI / demos / downstream usage.

from typing import Any, Dict, List, Optional, Tuple
from pathlib import Path

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint
from src.utils.ctx_utils import safe_get_in

from src.utils.artifact_utils import resolve_artifact_path, write_json, read_json_if_exists
from src.utils.cfg_utils import (
    get_cfg_bool,
    get_cfg_dict,
    get_cfg_str_list,
)
from src.utils.time_utils import utc_now_iso

TAG114 = "RELEASE_GATE"

_ALLOWED_STATUS = {"pass", "warn", "fail", "skipped"}


# --- Internal helpers ---
def _default_allowed_status(*, allow_warn: bool, allow_skipped: bool) -> List[str]:
    out = ["pass"]
    if bool(allow_warn):
        out.append("warn")
    if bool(allow_skipped):
        out.append("skipped")
    return out


def _status_from_report(rep: Optional[Dict[str, Any]]) -> Optional[str]:
    v = safe_get_in(rep, "summary.overall_status", tag=TAG114, default=None)
    if isinstance(v, str) and v.strip():
        return v.strip().lower()
    return None


def _event_hints_from_report(rep: Optional[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    event_hints v1 at:
      rep["checks"]["event_hints"] = {"version": 1, "hints": [...]}
    """
    v = safe_get_in(rep, "checks.event_hints.hints", tag=TAG114, default=[])
    return v if isinstance(v, list) else []


def _dedupe_preserve_order(items: List[str]) -> List[str]:
    seen = set()
    out: List[str] = []
    for x in items:
        if x in seen:
            continue
        seen.add(x)
        out.append(x)
    return out


def _load_report(
    *,
    ctx: Dict[str, Any],
    lg: Any,
    stage: str,
    name: str,
    run_id: str,
) -> Tuple[Optional[Dict[str, Any]], str, bool, bool]:
    """
    Returns: (report_obj_or_none, expected_path, exists_on_disk, loaded_ok)
    """
    p = resolve_artifact_path(ctx=ctx, stage=str(stage), name=str(name), run_id=str(run_id), tag=TAG114)
    exists = bool(Path(p).exists())
    obj = read_json_if_exists(p, lg=lg, tag=TAG114)
    loaded_ok = bool(isinstance(obj, dict))
    return (obj if loaded_ok else None), str(p), bool(exists), bool(loaded_ok)


def _resolve_enabled_step_artifacts_from_ssot(ctx: Dict[str, Any]) -> List[Dict[str, str]]:
    """
    SSOT-driven enabled steps list (no manual dict crawling beyond the final steps list).
    Uses cfg_utils getters to keep behavior consistent project-wide.

    Expects:
      pipeline.plan: list[str]
      pipeline.orchestrator.stages.<stage>.steps: list[dict{name, enabled}]
    """
    plan = get_cfg_str_list(
        ctx,
        "pipeline.plan",
        tag=TAG114,
        default=[],
        allow_empty=True,
        dedupe=True,
    )

    stages_cfg = get_cfg_dict(
        ctx,
        "pipeline.orchestrator.stages",
        tag=TAG114,
        default={},
    )

    out: List[Dict[str, str]] = []
    for stage_key in plan:
        st = str(stage_key).strip()
        if not st:
            continue

        stage_cfg = stages_cfg.get(st, {})
        if not isinstance(stage_cfg, dict):
            continue

        steps = stage_cfg.get("steps", [])
        if not isinstance(steps, list):
            continue

        for s in steps:
            if not isinstance(s, dict):
                continue
            if not bool(s.get("enabled", True)):
                continue
            nm = str(s.get("name", "")).strip()
            if not nm:
                continue
            out.append({"stage": st, "name": nm})

    return out


def _resolve_release_gate_cfg(ctx: Dict[str, Any]) -> Dict[str, Any]:
    """
    Resolve YAML SSOT (strictly aligned to your YAML):

      pipeline.governance.release_gate.allow_warn
      pipeline.governance.release_gate.allow_skipped

      pipeline.governance.release_gate.policy.required_reports: list[dict{stage,name,allowed_status}]
      pipeline.governance.release_gate.policy.conditional_required_steps: {enabled,enforce_enabled_steps_presence,allowed_status}
      pipeline.governance.release_gate.policy.blocking: {block_on_fail, block_on_event_hints{enabled,severity_signals_blocking}}

      pipeline.governance.release_gate.outputs: {include_releasable_flag, include_reasons, include_remediation}
    """
    base = "pipeline.governance.release_gate"

    allow_warn = get_cfg_bool(ctx, f"{base}.allow_warn", tag=TAG114, default=True, strict_type=False)
    allow_skipped = get_cfg_bool(ctx, f"{base}.allow_skipped", tag=TAG114, default=True, strict_type=False)

    # --- policy ---
    policy = get_cfg_dict(ctx, f"{base}.policy", tag=TAG114, default={})

    req_raw = policy.get("required_reports", [])
    required_reports: List[Dict[str, Any]] = [x for x in req_raw if isinstance(x, dict)] if isinstance(req_raw, list) else []

    # Fill defaults for required_reports.allowed_status if missing
    default_allowed = _default_allowed_status(allow_warn=bool(allow_warn), allow_skipped=bool(allow_skipped))
    for rr in required_reports:
        allowed = rr.get("allowed_status", None)
        if isinstance(allowed, list):
            rr["allowed_status"] = [str(x).strip().lower() for x in allowed if str(x).strip()]
        elif isinstance(allowed, str) and allowed.strip():
            rr["allowed_status"] = [allowed.strip().lower()]
        else:
            rr["allowed_status"] = list(default_allowed)

    # conditional_required_steps
    cond = policy.get("conditional_required_steps", {})
    cond = cond if isinstance(cond, dict) else {}
    cond_enabled = bool(cond.get("enabled", False))
    enforce_enabled_steps_presence = bool(cond.get("enforce_enabled_steps_presence", True))
    cond_allowed = cond.get("allowed_status", ["pass", "warn", "skipped"])
    if isinstance(cond_allowed, list):
        cond_allowed = [str(x).strip().lower() for x in cond_allowed if str(x).strip()]
    else:
        cond_allowed = ["pass", "warn", "skipped"]

    # blocking
    blocking = policy.get("blocking", {})
    blocking = blocking if isinstance(blocking, dict) else {}
    block_on_fail = bool(blocking.get("block_on_fail", True))

    boh = blocking.get("block_on_event_hints", {})
    boh = boh if isinstance(boh, dict) else {}
    boh_enabled = bool(boh.get("enabled", True))
    sev_block = boh.get("severity_signals_blocking", ["hard"])
    if isinstance(sev_block, list):
        sev_block = [str(x).strip().lower() for x in sev_block if str(x).strip()]
    else:
        sev_block = ["hard"]

    # outputs
    out_cfg = get_cfg_dict(ctx, f"{base}.outputs", tag=TAG114, default={})
    include_releasable_flag = bool(out_cfg.get("include_releasable_flag", True))
    include_reasons = bool(out_cfg.get("include_reasons", True))
    include_remediation = bool(out_cfg.get("include_remediation", True))

    return {
        "cfg_key": base,
        "allow_warn": bool(allow_warn),
        "allow_skipped": bool(allow_skipped),
        "policy": {
            "required_reports": required_reports,
            "conditional_required_steps": {
                "enabled": bool(cond_enabled),
                "enforce_enabled_steps_presence": bool(enforce_enabled_steps_presence),
                "allowed_status": cond_allowed,
                "limits": {"max_steps_preview": 200, "max_examples": 20},
            },
            "blocking": {
                "block_on_fail": bool(block_on_fail),
                "block_on_event_hints": {
                    "enabled": bool(boh_enabled),
                    "severity_signals_blocking": sev_block,
                    "limits": {"max_hits_preview": 20},
                },
            },
        },
        "outputs": {
            "include_releasable_flag": bool(include_releasable_flag),
            "include_reasons": bool(include_reasons),
            "include_remediation": bool(include_remediation),
            "limits": {"max_remediation": 50},
        },
    }


# --- 1) Public API ---
def run_release_gate(
    *,
    ctx: Dict[str, Any],
    run_id: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Step 11.4 - Release Decision & Policy Gate (ModuleReport v1).
    """
    # --- 2) Runtime validation ---
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG114, run_id=run_id)

    # --- 4) Resolve config (YAML SSOT) ---
    cfg = _resolve_release_gate_cfg(ctx)
    policy = cfg["policy"]
    outputs_cfg = cfg["outputs"]

    # --- 5）Core Workflow ---
    lg.info(f"[{TAG114}] 🚦 Release gate | run_id={rid}")

    # --- Build ModuleReport skeleton ---
    report = build_module_report(
        ctx=ctx,
        stage="governance",
        step="11.4",
        name="release_gate",
        tag=TAG114,
        cfg_key=str(cfg["cfg_key"]),
        enabled=True,
        df=None,
        run_id_override=str(rid),
    )

    report.setdefault("refs", {})
    report.setdefault("inputs", {})
    report.setdefault("checks", {})
    ensure_event_hints(report, version=1)

    report["refs"]["self"] = make_artifact_ref(ctx=ctx, stage="governance", name="release_gate", run_id=str(rid))

    report["thresholds_used"] = {
        "allow_warn": bool(cfg["allow_warn"]),
        "allow_skipped": bool(cfg["allow_skipped"]),
        "policy": {
            "required_reports": policy["required_reports"],
            "conditional_required_steps": dict(policy["conditional_required_steps"]),
            "blocking": dict(policy["blocking"]),
        },
        "outputs": dict(outputs_cfg),
    }

    report["used_config"] = {
        "cfg_keys": [
            "pipeline.governance.release_gate",
            "pipeline.governance.release_gate.policy.required_reports",
            "pipeline.governance.release_gate.policy.conditional_required_steps",
            "pipeline.governance.release_gate.policy.blocking",
            "pipeline.governance.release_gate.outputs",
        ]
    }

    # --- Facts layer ---
    reasons: List[str] = []
    remediation: List[Dict[str, Any]] = []
    reasons_detail: List[Dict[str, Any]] = []

    # 5.1 Required reports (anchors)
    required_reports = policy.get("required_reports", [])
    anchors_rows: List[Dict[str, Any]] = []

    missing_required: List[Dict[str, Any]] = []
    invalid_status: List[Dict[str, Any]] = []
    disallowed_status: List[Dict[str, Any]] = []

    blocking_cfg = policy["blocking"]
    block_on_fail = bool(blocking_cfg.get("block_on_fail", True))
    boh_cfg = blocking_cfg.get("block_on_event_hints", {})
    boh_enabled = bool(boh_cfg.get("enabled", True))
    sev_blocking = set([str(x).strip().lower() for x in (boh_cfg.get("severity_signals_blocking") or ["hard"]) if str(x).strip()])

    blocking_hint_hits: List[Dict[str, Any]] = []

    for rr in (required_reports if isinstance(required_reports, list) else []):
        if not isinstance(rr, dict):
            continue

        st = str(rr.get("stage", "")).strip()
        nm = str(rr.get("name", "")).strip()
        allowed = rr.get("allowed_status", ["pass", "warn"])
        allowed_list = [str(x).strip().lower() for x in (allowed if isinstance(allowed, list) else [allowed]) if str(x).strip()]
        allowed_set = set(allowed_list) if allowed_list else {"pass"}

        is_setup_snapshot = (st == "setup" and nm == "config_snapshot")

        if is_setup_snapshot:
            # For setup snapshot, do NOT parse JSON; treat as evidence file.
            p = resolve_artifact_path(ctx=ctx, stage=st, name=nm, run_id=str(rid), tag=TAG114)
            expected_path = str(p)
            exists_on_disk = bool(Path(p).exists())
            size_bytes = int(Path(p).stat().st_size) if exists_on_disk else 0

            row = {
                "stage": st,
                "name": nm,
                "exists_on_disk": bool(exists_on_disk),
                "loaded_ok": None,  # N/A for snapshot evidence
                "expected_path": expected_path,
                "status": "pass" if (exists_on_disk and size_bytes > 0) else "fail",
                "allowed_status": sorted(list(allowed_set)),
                "note": "setup/config_snapshot is treated as snapshot evidence (existence + non-empty).",
                "size_bytes": size_bytes,
            }
            anchors_rows.append(row)

            if not exists_on_disk:
                reasons.append(f"missing_required_report:{st}:{nm}")
                reasons_detail.append({"code": "missing_required_report", "stage": st, "name": nm, "path": expected_path})
                continue

            if size_bytes <= 0:
                reasons.append(f"empty_required_report:{st}:{nm}")
                reasons_detail.append({"code": "empty_required_report", "stage": st, "name": nm, "path": expected_path})
                continue

            # satisfied snapshot evidence
            continue
        
        obj, expected_path, exists_on_disk, loaded_ok = _load_report(ctx=ctx, lg=lg, stage=st, name=nm, run_id=str(rid))
        status = _status_from_report(obj)

        anchors_rows.append(
            {
                "stage": st,
                "name": nm,
                "exists_on_disk": bool(exists_on_disk),
                "loaded_ok": bool(loaded_ok),
                "expected_path": expected_path,
                "status": status,
                "allowed_status": sorted(list(allowed_set)),
            }
        )

        if not exists_on_disk:
            reasons.append(f"missing_required_report:{st}:{nm}")
            missing_required.append({"stage": st, "name": nm, "expected_path": expected_path})
            reasons_detail.append({"code": "missing_required_report", "stage": st, "name": nm, "path": expected_path})
            remediation.append(
                {
                    "action": "produce_required_report",
                    "safe": True,
                    "rationale": "Required anchor report is missing; evidence chain is incomplete.",
                    "post_check": f"run producing step for {st}:{nm}, then re-run 11.4",
                    "context": {"stage": st, "name": nm, "expected_path": expected_path},
                }
            )
            continue

        # file exists but cannot be loaded as dict -> unreadable (more severe than missing)
        if exists_on_disk and (not loaded_ok):
            reasons.append(f"unreadable_required_report:{st}:{nm}")
            reasons_detail.append({"code": "unreadable_required_report", "stage": st, "name": nm, "path": expected_path})
            remediation.append(
                {
                    "action": "rebuild_corrupted_report",
                    "safe": True,
                    "rationale": "Artifact exists but is unreadable (corrupted/partial/non-JSON).",
                    "post_check": f"re-run producing step for {st}:{nm}, then re-run 11.4",
                    "context": {"stage": st, "name": nm, "expected_path": expected_path},
                }
            )
            continue

        if (status is None) or (status not in _ALLOWED_STATUS):
            reasons.append(f"invalid_required_report_status:{st}:{nm}:{status}")
            invalid_status.append({"stage": st, "name": nm, "status": status, "expected_path": expected_path})
            reasons_detail.append({"code": "invalid_required_report_status", "stage": st, "name": nm, "status": status, "path": expected_path})
            remediation.append(
                {
                    "action": "fix_required_report_status",
                    "safe": True,
                    "rationale": "ModuleReport must contain a valid summary.overall_status.",
                    "post_check": f"fix {st}:{nm} report schema/status, then re-run 11.4",
                    "context": {"stage": st, "name": nm, "status": status},
                }
            )
            continue

        if status not in allowed_set:
            reasons.append(f"required_report_disallowed_status:{st}:{nm}:{status}")
            disallowed_status.append({"stage": st, "name": nm, "status": status, "allowed_status": sorted(list(allowed_set))})
            reasons_detail.append({"code": "required_report_disallowed_status", "stage": st, "name": nm, "status": status, "allowed_status": sorted(list(allowed_set))})
            remediation.append(
                {
                    "action": "resolve_required_report_issues",
                    "safe": True,
                    "rationale": "Required anchor report status is not allowed by policy.",
                    "post_check": f"fix {st}:{nm} until status in {sorted(list(allowed_set))}, then re-run 11.4",
                    "context": {"stage": st, "name": nm, "status": status, "allowed_status": sorted(list(allowed_set))},
                }
            )

        if block_on_fail and status == "fail":
            reasons.append(f"required_report_failed:{st}:{nm}")
            reasons_detail.append({"code": "required_report_failed", "stage": st, "name": nm})

        # Blocking event_hints scan (policy-driven)
        if boh_enabled and sev_blocking:
            for h in _event_hints_from_report(obj):
                if not isinstance(h, dict):
                    continue
                sig = str(h.get("severity_signal", "")).strip().lower()
                if sig and (sig in sev_blocking):
                    blocking_hint_hits.append(
                        {
                            "stage": st,
                            "name": nm,
                            "severity_signal": sig,
                            "code": h.get("code"),
                            "evidence_path": h.get("evidence_path"),
                        }
                    )

    report["checks"]["required_reports"] = {
        "n_required": int(len(required_reports)) if isinstance(required_reports, list) else 0,
        "items": anchors_rows,
        "missing_required_examples": missing_required[:20],
        "invalid_status_examples": invalid_status[:20],
        "disallowed_status_examples": disallowed_status[:20],
    }

    # 5.2 Conditional required steps (SSOT-driven)
    cond_cfg = policy["conditional_required_steps"]
    cond_enabled = bool(cond_cfg.get("enabled", False))
    enforce_presence = bool(cond_cfg.get("enforce_enabled_steps_presence", True))
    cond_allowed = cond_cfg.get("allowed_status", ["pass", "warn", "skipped"])
    cond_allowed_set = set([str(x).strip().lower() for x in cond_allowed if str(x).strip()])

    missing_steps: List[str] = []
    disallowed_steps: List[str] = []
    steps_preview: List[Dict[str, Any]] = []

    if cond_enabled:
        enabled_steps = _resolve_enabled_step_artifacts_from_ssot(ctx)
        max_steps_preview = int(cond_cfg["limits"]["max_steps_preview"])
        max_examples = int(cond_cfg["limits"]["max_examples"])

        for s in enabled_steps:
            st = str(s.get("stage", "")).strip()
            nm = str(s.get("name", "")).strip()

            obj, expected_path, exists_on_disk, loaded_ok = _load_report(ctx=ctx, lg=lg, stage=st, name=nm, run_id=str(rid))
            status = _status_from_report(obj)

            if len(steps_preview) < max_steps_preview:
                steps_preview.append(
                    {
                        "stage": st,
                        "name": nm,
                        "exists_on_disk": bool(exists_on_disk),
                        "loaded_ok": bool(loaded_ok),
                        "expected_path": expected_path,
                        "status": status,
                    }
                )

            if enforce_presence and (not exists_on_disk):
                reasons.append(f"missing_enabled_step_report:{st}:{nm}")
                if len(missing_steps) < max_examples:
                    missing_steps.append(f"{st}:{nm}")
                reasons_detail.append({"code": "missing_enabled_step_report", "stage": st, "name": nm, "path": expected_path})
                continue
            
            if enforce_presence and exists_on_disk and (not loaded_ok):
                reasons.append(f"unreadable_enabled_step_report:{st}:{nm}")
                reasons_detail.append({"code": "unreadable_enabled_step_report", "stage": st, "name": nm, "path": expected_path})
                continue

            if loaded_ok and (status is not None) and cond_allowed_set and (status not in cond_allowed_set):
                reasons.append(f"enabled_step_disallowed_status:{st}:{nm}:{status}")
                if len(disallowed_steps) < max_examples:
                    disallowed_steps.append(f"{st}:{nm}:{status}")
                reasons_detail.append({"code": "enabled_step_disallowed_status", "stage": st, "name": nm, "status": status, "allowed_status": sorted(list(cond_allowed_set))})

    report["checks"]["conditional_required_steps"] = {
        "enabled": bool(cond_enabled),
        "enforce_enabled_steps_presence": bool(enforce_presence),
        "allowed_status": sorted(list(cond_allowed_set)),
        "n_missing_examples": int(len(missing_steps)),
        "n_disallowed_examples": int(len(disallowed_steps)),
        "missing_examples": missing_steps,
        "disallowed_examples": disallowed_steps,
        "steps_preview": steps_preview,
        "note": "Use 11.3 inventory for full browsing; 11.4 stores only a bounded preview.",
    }

    # 5.3 Blocking event hints summary
    max_hits_preview = int(policy["blocking"]["block_on_event_hints"]["limits"]["max_hits_preview"])
    report["checks"]["blocking_event_hints"] = {
        "enabled": bool(boh_enabled),
        "severity_signals_blocking": sorted(list(sev_blocking)),
        "n_hits": int(len(blocking_hint_hits)),
        "hits_preview": blocking_hint_hits[:max_hits_preview],
    }

    if blocking_hint_hits:
        reasons.append("blocking_event_hints_present")
        reasons_detail.append({"code": "blocking_event_hints_present", "n_hits": int(len(blocking_hint_hits)), "severity_blocking": sorted(list(sev_blocking))})
        remediation.append(
            {
                "action": "resolve_blocking_event_hints",
                "safe": True,
                "rationale": "Release policy blocks when upstream reports contain blocking severity event hints.",
                "post_check": "fix upstream steps producing those hints, then re-run 11.4",
                "context": {"n_hits": int(len(blocking_hint_hits)), "severity_blocking": sorted(list(sev_blocking))},
            }
        )

    # --- 5.4 Event hints (single block, before Status) ---
    reasons_u = _dedupe_preserve_order(reasons)

    hard_prefixes = (
        "missing_required_report:",
        "unreadable_required_report:",
        "invalid_required_report_status:",
        "required_report_failed:",
        "missing_enabled_step_report:",
        "unreadable_enabled_step_report:",
        "blocking_event_hints_present",
    )
    hard_flag = any([r.startswith(hard_prefixes) for r in reasons_u])

    if reasons_u and hard_flag:
        add_event_hint(
            report,
            code="release_gate_blocked",
            severity_signal="hard",
            evidence_path="checks.release_gate.reasons",
            context={"n_reasons": int(len(reasons_u)), "examples": reasons_u[:10]},
            remediation={
                "action": "resolve_release_blockers",
                "safe": True,
                "rationale": "Release gate blocked by policy.",
                "post_check": "re-run 11.4 after fixing blockers",
            },
        )
    elif reasons_u:
        add_event_hint(
            report,
            code="release_gate_not_releasable",
            severity_signal="risk",
            evidence_path="checks.release_gate.reasons",
            context={"n_reasons": int(len(reasons_u)), "examples": reasons_u[:10]},
            remediation={
                "action": "review_policy_and_reports",
                "safe": True,
                "rationale": "Release gate not releasable under current policy.",
                "post_check": "adjust policy or fix upstream reports, then re-run 11.4",
            },
        )

    # --- 6) Decision payload + overall status ---
    releasable = len(reasons_u) == 0

    gate_payload: Dict[str, Any] = {"evaluated_at_utc": utc_now_iso()}
    if bool(outputs_cfg.get("include_releasable_flag", True)):
        gate_payload["releasable"] = bool(releasable)
    if bool(outputs_cfg.get("include_reasons", True)):
        gate_payload["reasons"] = reasons_u
        gate_payload["reasons_detail"] = reasons_detail[:200]
    if bool(outputs_cfg.get("include_remediation", True)):
        gate_payload["remediation"] = remediation[: int(outputs_cfg["limits"]["max_remediation"])]

    report["checks"]["release_gate"] = gate_payload

    issues: List[str] = []
    warnings: List[str] = []

    if not releasable and hard_flag:
        issues.append("not_releasable")
    elif not releasable:
        warnings.append("not_releasable")

    overall = "pass" if releasable else ("fail" if issues else "warn")

    report = finalize_module_report(
        ctx=ctx,
        report=report,
        overall_status=overall,
        issues=issues,
        warnings=warnings,
        metrics={
            "releasable": bool(releasable),
            "n_reasons": int(len(reasons_u)),
            "n_blocking_event_hints": int(len(blocking_hint_hits)),
        },
        notes=[f"release_gate_evaluated_at_utc={utc_now_iso()}"],
        df=None,
    )

    if overall == "pass":
        lg.info(f"[{TAG114}] ✅ Release gate pass | releasable=True")
    elif overall == "warn":
        lg.warning(f"[{TAG114}] ⚠ Release gate warn | releasable=False | reasons={reasons_u[:6]}")
    else:
        lg.error(f"[{TAG114}] ❌ Release gate fail | releasable=False | reasons={reasons_u[:6]}")

    # --- 7) Persist artifact ---
    out_path = resolve_artifact_path(
        ctx=ctx,
        stage="governance",
        name="release_gate",
        run_id=str(rid),
        tag=TAG114,
    )
    write_json(out_path, report, tag=TAG114, indent=2)
    lg.info(f"[{TAG114}] 🧾 Release gate artifact saved: {out_path}")

    return report

In [30]:
# --- 11.x Governance Orchestrator (Control-Plane) ---
#
# What:
#   Run Step-11 "governance" as one controller:
#     - Execute 11.1 manifest -> 11.2 lineage_audit -> 11.3 inventory -> 11.4 release_gate
#     - Reuse control-plane outputs to reduce I/O (pass manifest_report into 11.2)
#     - Aggregate module outcomes into one orchestrator ModuleReport v1 (single entrypoint)
#     - Surface final release decision as the run-level governance outcome
#
# How:
#   - ensure_runtime_ctx() -> ctx/logger/run_id
#   - Call:
#       build_run_manifest(...)
#       audit_lineage_consistency(..., manifest_report=rep_111)
#       build_artifact_inventory_index(...)
#       run_release_gate(...)
#   - Persist orchestrator ModuleReport v1 under stage=governance, name=orchestrator, step=11.x
#
# Why:
#   - Provide a deterministic "control-plane" entrypoint for audit/reproducibility
#   - Keep downstream usage safe (CI/demos/sharing) via final release gate
#   - Make interviews easier: one artifact shows the whole governance story

from typing import Any, Dict, List, Optional

import pandas as pd

from src.pipeline.runtime import ensure_runtime_ctx
from src.pipeline.reporting import build_module_report, finalize_module_report, make_artifact_ref
from src.pipeline.event_hints import ensure_event_hints, add_event_hint

from src.utils.artifact_utils import resolve_artifact_path, write_json

TAG11X = "GOV_ORCH"


def _safe_status(rep: Optional[Dict[str, Any]]) -> Optional[str]:
    if not isinstance(rep, dict):
        return None
    s = rep.get("summary", {}) if isinstance(rep.get("summary"), dict) else {}
    v = s.get("overall_status")
    return str(v).strip().lower() if isinstance(v, str) and v.strip() else None


def _safe_ref(ctx: Dict[str, Any], *, stage: str, name: str, run_id: str) -> Dict[str, Any]:
    return make_artifact_ref(ctx=ctx, stage=str(stage), name=str(name), run_id=str(run_id))


def _extract_release_block(rep_114: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Extract release decision payload from 11.4 report in a stable way.
    """
    out: Dict[str, Any] = {"present": False}
    if not isinstance(rep_114, dict):
        return out

    checks = rep_114.get("checks", {}) if isinstance(rep_114.get("checks"), dict) else {}
    rg = checks.get("release_gate", {}) if isinstance(checks.get("release_gate"), dict) else {}

    out["present"] = True
    out["releasable"] = rg.get("releasable")
    out["reasons"] = rg.get("reasons")
    out["reasons_detail"] = rg.get("reasons_detail")
    out["remediation"] = rg.get("remediation")
    out["evaluated_at_utc"] = rg.get("evaluated_at_utc")
    return out


# --- 1) Public API ---
def run_governance_orchestrator(
    *,
    ctx: Dict[str, Any],
    # Optional: pass dfs to strengthen manifest fingerprint. Safe to omit.
    df_raw: Optional[pd.DataFrame] = None,
    df_clean: Optional[pd.DataFrame] = None,
    run_id: Optional[str] = None,
    # Optional: attach extra metadata to 11.1
    extra_metadata: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """
    Step 11.x - Governance Orchestrator (ModuleReport v1).

    Requires the following functions to be available in scope (you already implemented them):
      - build_run_manifest (11.1)
      - audit_lineage_consistency (11.2)
      - build_artifact_inventory_index (11.3)
      - run_release_gate (11.4)
    """
    # --- 2) Runtime validation ---
    ctx, lg, rid = ensure_runtime_ctx(ctx=ctx, tag=TAG11X, run_id=run_id)

    # --- 3) Input validation ---
    if df_raw is not None and (not isinstance(df_raw, pd.DataFrame)):
        raise TypeError(f"[{TAG11X}] df_raw must be a pandas DataFrame when provided")
    if df_clean is not None and (not isinstance(df_clean, pd.DataFrame)):
        raise TypeError(f"[{TAG11X}] df_clean must be a pandas DataFrame when provided")
    if extra_metadata is not None and (not isinstance(extra_metadata, dict)):
        raise TypeError(f"[{TAG11X}] extra_metadata must be a dict when provided")

    # --- 4) Resolve config (SSOT) ---
    # Governance is a control-plane stage. We keep orchestrator itself simple.
    # If later you add pipeline.orchestrator.stages.governance, you can wire enabled/gates there.
    stage_enabled = True

    # --- 5) Core workflows ---
    lg.info(f"[{TAG11X}] 🧭 Governance orchestrator start | run_id={rid}")

    orch = build_module_report(
        ctx=ctx,
        stage="governance",
        step="11.x",
        name="orchestrator",
        tag=TAG11X,
        cfg_key="pipeline.governance",  # broad, since it orchestrates 11.1-11.4
        enabled=stage_enabled,
        df=df_clean if isinstance(df_clean, pd.DataFrame) else None,
        run_id_override=str(rid),
    )

    orch.setdefault("refs", {})
    orch.setdefault("inputs", {})
    orch.setdefault("checks", {})
    ensure_event_hints(orch, version=1)

    orch["refs"]["self"] = make_artifact_ref(ctx=ctx, stage="governance", name="orchestrator", run_id=str(rid))
    orch["inputs"] = {
        "stage_enabled": bool(stage_enabled),
        "df_raw_provided": bool(isinstance(df_raw, pd.DataFrame)),
        "df_clean_provided": bool(isinstance(df_clean, pd.DataFrame)),
        "extra_metadata_keys": sorted(list(extra_metadata.keys())) if isinstance(extra_metadata, dict) else [],
    }

    orch["checks"]["plan"] = {
        "steps": ["11.1_manifest", "11.2_lineage_audit", "11.3_inventory", "11.4_release_gate"],
        "notes": [
            "11.2 reuses 11.1 manifest_report to reduce disk I/O",
            "orchestrator overall_status inherits 11.4 outcome",
        ],
    }

    # --- Execute 11.1 -> 11.2 -> 11.3 -> 11.4 ---
    rep_111: Optional[Dict[str, Any]] = None
    rep_112: Optional[Dict[str, Any]] = None
    rep_113: Optional[Dict[str, Any]] = None
    rep_114: Optional[Dict[str, Any]] = None

    execution: List[Dict[str, Any]] = []
    halted = False
    halted_at: Optional[str] = None

    # 11.1 Manifest
    try:
        lg.info(f"[{TAG11X}] ▶ 11.1 manifest")
        rep_111 = build_run_manifest(  # type: ignore[name-defined]
            ctx=ctx,
            df_raw=df_raw,
            df_clean=df_clean,
            run_id=str(rid),
            extra_metadata=extra_metadata,
        )
        st = _safe_status(rep_111) or "unknown"
        execution.append({"step": "11.1", "name": "manifest", "status": st})
    except Exception as e:
        lg.exception(f"[{TAG11X}] ❌ 11.1 manifest crashed: {e!r}")
        execution.append({"step": "11.1", "name": "manifest", "status": "fail", "error": repr(e)})
        halted, halted_at = True, "11.1"

    # 11.2 Lineage audit (prefer manifest_report)
    if not halted:
        try:
            lg.info(f"[{TAG11X}] ▶ 11.2 lineage_audit (reuse manifest_report)")
            rep_112 = audit_lineage_consistency(  # type: ignore[name-defined]
                ctx=ctx,
                run_id=str(rid),
                manifest_report=rep_111 if isinstance(rep_111, dict) else None,
            )
            st = _safe_status(rep_112) or "unknown"
            execution.append({"step": "11.2", "name": "lineage_audit", "status": st, "used_manifest_report": True})
        except Exception as e:
            lg.exception(f"[{TAG11X}] ❌ 11.2 lineage_audit crashed: {e!r}")
            execution.append({"step": "11.2", "name": "lineage_audit", "status": "fail", "error": repr(e)})
            halted, halted_at = True, "11.2"

    # 11.3 Inventory
    if not halted:
        try:
            lg.info(f"[{TAG11X}] ▶ 11.3 inventory")
            rep_113 = build_artifact_inventory_index(  # type: ignore[name-defined]
                ctx=ctx,
                run_id=str(rid),
            )
            st = _safe_status(rep_113) or "unknown"
            execution.append({"step": "11.3", "name": "inventory", "status": st})
        except Exception as e:
            lg.exception(f"[{TAG11X}] ❌ 11.3 inventory crashed: {e!r}")
            execution.append({"step": "11.3", "name": "inventory", "status": "fail", "error": repr(e)})
            halted, halted_at = True, "11.3"

    # 11.4 Release gate (final)
    if not halted:
        try:
            lg.info(f"[{TAG11X}] ▶ 11.4 release_gate")
            rep_114 = run_release_gate(  # type: ignore[name-defined]
                ctx=ctx,
                run_id=str(rid),
            )
            st = _safe_status(rep_114) or "unknown"
            execution.append({"step": "11.4", "name": "release_gate", "status": st})
        except Exception as e:
            lg.exception(f"[{TAG11X}] ❌ 11.4 release_gate crashed: {e!r}")
            execution.append({"step": "11.4", "name": "release_gate", "status": "fail", "error": repr(e)})
            halted, halted_at = True, "11.4"

    # --- Collect module refs + statuses ---
    orch["checks"]["execution"] = execution
    orch["checks"]["halted"] = {"halted": bool(halted), "halted_at": halted_at or ""}

    # Canonical refs for downstream audit consumption
    orch["checks"]["modules"] = {
        "manifest": {
            "step": "11.1",
            "status": _safe_status(rep_111),
            "ref": _safe_ref(ctx, stage="governance", name="manifest", run_id=str(rid)),
        },
        "lineage_audit": {
            "step": "11.2",
            "status": _safe_status(rep_112),
            "ref": _safe_ref(ctx, stage="governance", name="lineage_audit", run_id=str(rid)),
        },
        "inventory": {
            "step": "11.3",
            "status": _safe_status(rep_113),
            "ref": _safe_ref(ctx, stage="governance", name="inventory", run_id=str(rid)),
        },
        "release_gate": {
            "step": "11.4",
            "status": _safe_status(rep_114),
            "ref": _safe_ref(ctx, stage="governance", name="release_gate", run_id=str(rid)),
        },
    }

    # Surface release decision at orchestrator level
    orch["checks"]["release"] = _extract_release_block(rep_114)

    if halted:
        add_event_hint(
            orch,
            code="governance_orchestrator_halted",
            severity_signal="hard",
            evidence_path="checks.halted",
            context={"halted": True, "halted_at": halted_at or ""},
            remediation={
                "action": "fix_crashing_step_then_rerun",
                "safe": True,
                "rationale": "Governance orchestrator crashed; release decision is unreliable.",
                "post_check": "Re-run 11.x after fixing the crashing step.",
            },
        )

    # --- 6) Status mapping (inherit 11.4 for clarity) ---
    issues: List[str] = []
    warnings: List[str] = []

    st_114 = _safe_status(rep_114)
    if halted:
        issues.append("governance_orchestrator_halted")

    # If we reached 11.4, inherit its status; otherwise fail (halted)
    if not halted and st_114 in {"pass", "warn", "fail"}:
        overall = st_114
    elif halted:
        overall = "fail"
    else:
        overall = "warn"

    # Optional: if release decision says not releasable, treat as warn/fail aligned with 11.4 status already
    release_block = orch["checks"].get("release", {}) if isinstance(orch.get("checks"), dict) else {}
    if isinstance(release_block, dict) and release_block.get("present") and (release_block.get("releasable") is False):
        # keep aligned with overall derived from 11.4; just annotate
        warnings.append("not_releasable")

    orch = finalize_module_report(
        ctx=ctx,
        report=orch,
        overall_status=overall,
        issues=issues,
        warnings=warnings,
        metrics={
            "halted": bool(halted),
            "n_steps_executed": int(len(execution)),
            "release_releasable": bool(release_block.get("releasable")) if isinstance(release_block, dict) else None,
        },
        notes=[f"governance_orchestrator_at_utc={utc_now_iso()}"],
        df=df_clean if isinstance(df_clean, pd.DataFrame) else None,
    )

    # --- 7) Persist orchestrator artifact ---
    out_path = resolve_artifact_path(
        ctx=ctx,
        stage="governance",
        name="orchestrator",
        run_id=str(rid),
        tag=TAG11X,
    )
    write_json(out_path, orch, tag=TAG11X, indent=2)
    lg.info(f"[{TAG11X}] 🧾 Governance orchestrator saved: {out_path}")

    lg.info(f"[{TAG11X}] ✅ Finished | overall={overall} | halted={halted} | run_id={rid}")

    return {
        "orchestrator_report": orch,
        "reports": {
            "manifest": rep_111,
            "lineage_audit": rep_112,
            "inventory": rep_113,
            "release_gate": rep_114,
        },
        "execution": execution,
        "halted": halted,
        "halted_at": halted_at,
    }


gov = run_governance_orchestrator(
    ctx=ctx,
    df_raw=df_raw,        
    df_clean=df_clean,    
    run_id=rid,           
    extra_metadata={"entrypoint": "notebook", "pipeline": "1-2x-3x-11x"},
)

rep11x = gov["orchestrator_report"]
print("[GOV] overall:", rep11x["summary"]["overall_status"])
print("[GOV] releasable:", rep11x["checks"]["release"].get("releasable"))
print("[GOV] reasons:", (rep11x["checks"]["release"].get("reasons") or [])[:10])

00:41:26 | INFO | [GOV_ORCH] 🧭 Governance orchestrator start | run_id=20260222-231948-167
00:41:26 | INFO | [GOV_ORCH] ▶ 11.1 manifest
00:41:26 | INFO | [MANIFEST] 📦 Building governance manifest | run_id=20260222-231948-167
00:41:26 | INFO | [MANIFEST] ✅ Manifest pass | n_indexed=22
00:41:26 | INFO | [MANIFEST] 🧾 Manifest artifact saved: D:\DS_project\telco-churn-project\artifacts\governance_manifest_20260222-231948-167.json
00:41:26 | INFO | [GOV_ORCH] ▶ 11.2 lineage_audit (reuse manifest_report)
00:41:26 | INFO | [LINEAGE_AUDIT] 🔎 Lineage & consistency audit | run_id=20260222-231948-167
00:41:26 | INFO | [LINEAGE_AUDIT] ✅ Lineage audit pass
00:41:26 | INFO | [LINEAGE_AUDIT] 🧾 Lineage audit artifact saved: D:\DS_project\telco-churn-project\artifacts\governance_lineage_audit_20260222-231948-167.json
00:41:26 | INFO | [GOV_ORCH] ▶ 11.3 inventory
00:41:26 | INFO | [INVENTORY] 📚 Building artifact inventory | run_id=20260222-231948-167
00:41:26 | INFO | [INVENTORY] ✅ Inventory pass | n_ind

[GOV] overall: pass
[GOV] releasable: True
[GOV] reasons: []
